# Phase 2 — Object-Oriented Programming and the Data Model

**Outcome:** build classes that behave like built-ins, design with inheritance and
composition, and understand the descriptor and metaclass machinery behind Pydantic,
SQLAlchemy and dataclasses.

| Unit | Topic |
| --- | --- |
| 2.1 | Classes and instances |
| 2.2 | Inheritance, MRO and composition |
| 2.3 | Abstractions: ABCs, `collections.abc` and Protocols |
| 2.4 | The data model: special (dunder) methods |
| 2.5 | Attribute access, descriptors and metaprogramming |
| 2.6 | Dataclasses, enums and Pythonic design patterns |
| Project | Inventory domain model |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- *Predict the output* questions hide the answer in a collapsible block: decide first,
  then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check cell.
  Until then the checks print FAIL; that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.15*, …).

## What you need from Phase 1

Phase 2 builds directly on these Phase 1 ideas. If one feels shaky, re-read the unit
before going on.

| Phase 1 idea | Unit | Why it matters for classes |
| --- | --- | --- |
| Everything is an object; names are references | 1.2 | Classes, methods and instances are all objects you can inspect and pass around |
| Mutable vs immutable, identity vs equality | 1.2.4–1.2.5 | The shared mutable class-attribute trap; `__eq__` vs `is` |
| `dict` internals and key-sharing dictionaries | 1.6.1 | Every instance stores its attributes in a `__dict__` that shares keys with its siblings |
| `__copy__` / `__deepcopy__` | 1.7.4 | Classes customize built-in behaviour through special methods |
| Class patterns and `__match_args__` | 1.8.4 | `match` destructures your own classes |
| Functions are first-class; `*args` / `**kwargs` | 1.10 | Methods are functions stored on the class; cooperative `**kwargs` in `super()` chains |
| Scope, closures and cells | 1.11 | Zero-argument `super()` reads a hidden `__class__` cell |
| Decorators, including class decorators and `__get__` | 1.12 | `@property`, `@classmethod` and `@staticmethod` are decorators built on descriptors |
| The iterator protocol | 1.13 | `collections.abc.Iterator` and `Iterable` formalize it |
| Custom exception hierarchies, `__enter__` / `__exit__` | 1.14 | You already designed classes around inheritance and special methods |

> **Note:** Phase 1 used small classes (`Price`, `LinkedListNode`, `Timer`, `AppError`)
> before explaining them fully. This phase explains everything those classes relied on.

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 2.1 — Classes and Instances

**Objectives**

- Decide when a class is the right tool, and when a function, a module or a
  dataclass is better.
- Explain what the `class` statement does at run time, down to the `type(...)` call.
- Trace object creation through `__new__` and `__init__`, and inspect instances with
  `vars()` and `__class__`.
- Predict attribute lookup (instance → class → bases) and avoid the shared mutable
  class-attribute trap.
- Choose between instance methods, `@classmethod` and `@staticmethod`, and write
  subclass-aware alternative constructors.
- Use `@property` for computed, read-only and validated attributes.
- Use Python's privacy conventions and compare them with JavaScript `#private`.

**JS bridge**

| JavaScript / TypeScript | Python | The difference that bites |
| --- | --- | --- |
| `class Cart { constructor(items) { … } }` | `class Cart:` + `def __init__(self, items):` | Construction is two steps (`__new__` allocates, `__init__` fills in) |
| `this` (set by the call site) | `self` (an explicit first parameter) | `self` is bound when you look the method up, so it is never "lost" |
| `obj.method.bind(obj)` | `obj.method` | A bound method already remembers its instance |
| Class field `items = []` (one array **per instance**) | Class attribute `items = []` (one list **shared** by all instances) | The most common Python class bug, see 2.1.4 |
| `static create() {}` | `@classmethod` / `@staticmethod` | `@classmethod` receives the class, so it builds the right subclass |
| `get total() {}` / `set total(v) {}` | `@property` / `@total.setter` | Same idea; Python has a deleter too |
| `#balance` (engine-enforced) / TS `private` (compile-time) | `_balance` (convention) / `__balance` (name mangling) | Nothing in Python is truly private |
| `Counter.prototype.reset = …` | `Counter.reset = reset` | Python classes are mutable objects too; instances see the change |
| `class` expression, `Object.create` | `type(name, bases, namespace)` | A class is built at run time by calling its metaclass |

The next cell is this unit's setup. `raised_message(thunk)` is new: like
`raised_error_name` from Phase 1, but it also returns the error message.

In [2]:
import dis
import functools
import inspect
import sys
import types
from abc import ABC, abstractmethod
from collections.abc import Callable, Mapping
from dataclasses import dataclass
from datetime import date
from decimal import ROUND_HALF_UP, Decimal
from typing import Any, ClassVar, NamedTuple, Self

PAISE = Decimal("0.01")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.1.1 Why classes

A class **bundles state with the behaviour that keeps that state valid**. Without a
class you can still keep data in a `dict` and write functions that work on it, but
nothing ties the rules to the data: any code can put the dict into an invalid state.

The next cell keeps stock levels in a plain dict. `ship_stock` refuses to go below
zero, but the last assignment ignores the rule completely.

In [3]:
def make_stock_item(sku: str, quantity: int) -> dict[str, Any]:
    """Create a stock record as a plain dict."""
    return {"sku": sku, "quantity": quantity}


def ship_stock(stock_item: dict[str, Any], units: int) -> None:
    """Ship units, refusing to go below zero."""
    if units > stock_item["quantity"]:
        raise ValueError(f"only {stock_item['quantity']} left")
    stock_item["quantity"] -= units


soap_stock = make_stock_item("SOAP", 10)
ship_stock(soap_stock, 3)
soap_stock["quantity"] = -50  # nothing stops this: the rule lives in ship_stock only
print(f"{soap_stock=}")

soap_stock={'sku': 'SOAP', 'quantity': -50}


The same idea as a class. The quantity can only change through `receive()` and
`ship()`, and the read-only `quantity` property (2.1.6) refuses direct assignment.
Notice the `AttributeError` at the end: the invalid state is now impossible from
outside.

In [4]:
class StockItem:
    """Stock for one SKU; quantity changes only through receive() and ship()."""

    def __init__(self, sku: str, quantity: int) -> None:
        self.sku = sku
        self._quantity = quantity

    @property
    def quantity(self) -> int:
        """Units on hand (read-only from outside)."""
        return self._quantity

    def receive(self, units: int) -> None:
        """Add units from a delivery."""
        self._quantity += units

    def ship(self, units: int) -> None:
        """Remove units, refusing to go below zero."""
        if units > self._quantity:
            raise ValueError(f"only {self._quantity} left")
        self._quantity -= units


soap = StockItem("SOAP", 10)
soap.ship(3)
soap.receive(5)
print(f"{soap.quantity=}")
print(raised_message(lambda: setattr(soap, "quantity", -50)))

soap.quantity=12
AttributeError: property 'quantity' of 'StockItem' object has no setter


A class is not always the answer. Python has three lighter tools, and idiomatic code
uses them first:

| You need | Best tool | Why |
| --- | --- | --- |
| A stateless calculation | a **function** | Nothing to remember between calls (`add_gst(amount)`) |
| A group of related functions and constants, one copy | a **module** | A module is already a singleton namespace (`import settings`) |
| A record of data with little behaviour | a **dataclass** or `NamedTuple` (2.6, 1.5.6) | `__init__`, `__repr__`, `__eq__` are generated for you |
| State plus rules that must always hold (invariants) | a **class** | Methods guard the state |
| Several interchangeable implementations | a **class** + ABC / Protocol (2.3) | Callers depend on the shared interface |

A common smell is a class with only `__init__` and one method: it is a function in
disguise. Both versions below compute the same thing; the function is shorter and
easier to test.

In [5]:
class GstCalculator:
    """A class that only wraps one function (a smell)."""

    def __init__(self, rate_percent: Decimal) -> None:
        self.rate_percent = rate_percent

    def calculate(self, amount: Decimal) -> Decimal:
        """Return amount plus GST, rounded to the paisa."""
        return (amount * (1 + self.rate_percent / 100)).quantize(PAISE, ROUND_HALF_UP)


def add_gst(amount: Decimal, rate_percent: Decimal = Decimal("18")) -> Decimal:
    """The same thing as one function."""
    return (amount * (1 + rate_percent / 100)).quantize(PAISE, ROUND_HALF_UP)


print(f"{GstCalculator(Decimal('18')).calculate(Decimal('499'))=}")
print(f"{add_gst(Decimal('499'))=}")

GstCalculator(Decimal('18')).calculate(Decimal('499'))=Decimal('588.82')
add_gst(Decimal('499'))=Decimal('588.82')


A **module** is an object too: `import math` gives you a `module` instance whose
attributes are the functions and constants defined in the file. That is why Python
needs no "static utility class" (`class MathUtils { static … }` in Java). The cell
builds a module object by hand to show it is just a namespace.

In [6]:
import math

print(f"{type(math).__name__=}, {math.tau=:.4f}")
settings = types.ModuleType("settings")  # what `import settings` would give you
settings.DEBUG = False
settings.PAGE_SIZE = 20
print(f"{type(settings).__name__=}, {settings.PAGE_SIZE=}")

type(math).__name__='module', math.tau=6.2832
type(settings).__name__='module', settings.PAGE_SIZE=20


For plain records, a **dataclass** (Unit 2.6) writes the boring methods for you.
Here `@dataclass` generated `__init__`, a readable `__repr__` and a value-based
`__eq__` from three annotated lines.

In [7]:
@dataclass
class ShippingAddress:
    """A plain record: the decorator writes __init__, __repr__ and __eq__."""

    city: str
    pin_code: str


home = ShippingAddress("Pune", "411001")
print(home)
print(f"{home == ShippingAddress('Pune', '411001')=}")

ShippingAddress(city='Pune', pin_code='411001')
home == ShippingAddress('Pune', '411001')=True


### 2.1.2 The `class` statement

`class` is an **executable statement**, like `def` (1.10.1). When Python reaches it:

1. It creates a fresh namespace (a dict).
2. It **runs the class body** top to bottom inside that namespace. Every assignment
   and every `def` in the body becomes an entry.
3. It calls the **metaclass**, normally `type`, as `type(name, bases, namespace)`.
   The result is the class object.
4. It binds the class name to that object in the enclosing scope.

The body is ordinary code, so `print` runs while the class is being defined, before
any instance exists. Watch the namespace grow as the body runs.

In [8]:
print("before the class statement")


class Checkout:
    """Prints while its body runs."""

    print("  class body starts")
    currency = "INR"
    print(f"  {sorted(name for name in locals() if not name.startswith('__'))=}")

    def total(self) -> int:
        """Never called here."""
        return 0

    print(f"  {sorted(name for name in locals() if not name.startswith('__'))=}")


print("after the class statement")

before the class statement
  class body starts
  sorted(name for name in locals() if not name.startswith('__'))=['currency']
  sorted(name for name in locals() if not name.startswith('__'))=['currency', 'total']
after the class statement


Step 3 can be done by hand. Calling `type` with three arguments creates a class:
a name, a tuple of base classes and a namespace dict. The result behaves exactly
like a class written with the `class` statement; JavaScript's closest equivalent is
a class expression or `Object.create(proto)`.

In [9]:
def describe_coupon(self: Any) -> str:
    """A plain function that becomes a method once it is in the namespace."""
    return f"{self.code}: {self.percent_off}% off"


Coupon = type(
    "Coupon",
    (object,),
    {"percent_off": 10, "describe": describe_coupon, "__doc__": "Built by type()."},
)
welcome = Coupon()
welcome.code = "WELCOME10"
print(f"{welcome.describe()=}")
print(f"{type(Coupon)=}, {Coupon.__name__=}, {Coupon.__bases__=}")
print(f"{type(Checkout) is type=}  (every ordinary class is an instance of type)")

welcome.describe()='WELCOME10: 10% off'
type(Coupon)=<class 'type'>, Coupon.__name__='Coupon', Coupon.__bases__=(<class 'object'>,)
type(Checkout) is type=True  (every ordinary class is an instance of type)


After creation, the namespace is stored as `ClassName.__dict__`. It is a
**`mappingproxy`**: a read-only view of the real dict. You cannot write through it,
but `setattr` (or `Checkout.currency = …`) works, because it goes through
`type.__setattr__`, which also invalidates CPython's method-lookup caches.

The keys include entries Python added for you: `__module__`, `__qualname__`,
`__doc__`, descriptors for `__dict__` and `__weakref__`, and two newer ones:
`__firstlineno__` and `__static_attributes__` (*New in 3.13*, explained in 2.1.3).

In [10]:
print(f"{type(Checkout.__dict__).__name__=}")
print(sorted(Checkout.__dict__))
try:
    Checkout.__dict__["currency"] = "USD"
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
Checkout.currency = "USD"  # goes through type.__setattr__, which updates the real dict
print(f"{Checkout.__dict__['currency']=}")

type(Checkout.__dict__).__name__='mappingproxy'
['__dict__', '__doc__', '__firstlineno__', '__module__', '__static_attributes__', '__weakref__', 'currency', 'total']
TypeError: 'mappingproxy' object does not support item assignment
Checkout.__dict__['currency']='USD'


#### Under the hood: the bytecode of a `class` statement

Compiling a tiny class shows the steps above. `LOAD_BUILD_CLASS` pushes the built-in
`__build_class__`, the class body is a separate **code object** turned into a
function by `MAKE_FUNCTION`, and the `CALL` runs it: `__build_class__` executes the
body function in a new namespace and then calls the metaclass. Code objects are
shown by name only (their reprs contain memory addresses).

> **Note:** opcode names change between Python versions; the shape stays the same.

In [11]:
class_source = "class Demo:\n    rate = 18\n\n    def total(self):\n        return 0\n"
module_code = compile(class_source, "<demo>", "exec")
for instruction in dis.get_instructions(module_code):
    if isinstance(instruction.argval, types.CodeType):
        shown_argument = f"<code object {instruction.argval.co_name}>"
    else:
        shown_argument = instruction.argrepr
    print(f"{instruction.opname:<20} {shown_argument}")

RESUME               
LOAD_BUILD_CLASS     
PUSH_NULL            
LOAD_CONST           <code object Demo>
MAKE_FUNCTION        
LOAD_CONST           'Demo'
CALL                 
STORE_NAME           Demo
LOAD_COMMON_CONSTANT None
RETURN_VALUE         


### 2.1.3 Object creation

Calling a class, `StockItem("SOAP", 10)`, runs `type.__call__`, which does two
separate steps:

1. **`__new__(cls, *args)`** allocates and returns a new, empty instance. It is a
   static method that receives the class. The default, `object.__new__`, is almost
   always enough.
2. **`__init__(self, *args)`** initializes that instance: it sets attributes. It
   must return `None`.

JavaScript's `constructor` does both at once. The traced class below prints each
step; notice the instance has an empty `__dict__` between the two.

In [12]:
class TracedOrder:
    """Prints each creation step."""

    def __new__(cls, order_id: str) -> Self:
        print(f"__new__(cls={cls.__name__}, order_id={order_id!r}) allocates")
        instance = super().__new__(cls)  # object.__new__ takes only the class
        print(f"  allocated, vars={vars(instance)}")
        return instance

    def __init__(self, order_id: str) -> None:
        print(f"__init__(self, order_id={order_id!r}) fills in attributes")
        self.order_id = order_id


first_order = TracedOrder("ORD-1")
print(f"{vars(first_order)=}")

__new__(cls=TracedOrder, order_id='ORD-1') allocates
  allocated, vars={}
__init__(self, order_id='ORD-1') fills in attributes
vars(first_order)={'order_id': 'ORD-1'}


The same two steps, done by hand. This is what `TracedOrder("ORD-2")` does for you.

In [13]:
manual_order = TracedOrder.__new__(TracedOrder, "ORD-2")
TracedOrder.__init__(manual_order, "ORD-2")
print(f"{vars(manual_order)=}")

__new__(cls=TracedOrder, order_id='ORD-2') allocates
  allocated, vars={}
__init__(self, order_id='ORD-2') fills in attributes
vars(manual_order)={'order_id': 'ORD-2'}


When do you write `__new__`? Mainly when subclassing an **immutable** type such as
`int`, `str` or `tuple`: by the time `__init__` runs the value is already fixed, so
validation or conversion has to happen in `__new__`.

In [14]:
class Paise(int):
    """An int that refuses negative amounts; int is immutable, so this needs __new__."""

    def __new__(cls, amount: int) -> Self:
        if amount < 0:
            raise ValueError("amount cannot be negative")
        return super().__new__(cls, amount)


print(f"{Paise(2500) + 100=}, {isinstance(Paise(1), int)=}")
print(raised_message(lambda: Paise(-1)))

Paise(2500) + 100=2600, isinstance(Paise(1), int)=True
ValueError: amount cannot be negative


If `__new__` returns something that is **not** an instance of the class,
`type.__call__` skips `__init__` entirely. That is legal but surprising, so keep it
for special cases (caching, singletons, factories).

In [15]:
class Surprising:
    """__new__ returns an int, so __init__ never runs."""

    def __new__(cls) -> Any:
        return 42

    def __init__(self) -> None:
        print("never printed")


print(f"{Surprising()=}")

Surprising()=42


**Explicit `self`.** A method is an ordinary function whose first parameter receives
the instance. `soap.ship(2)` is shorthand for `StockItem.ship(soap, 2)`. The name
`self` is only a convention (any name works), but always use it.

In [16]:
soap = StockItem("SOAP", 10)
soap.ship(2)  # what you write
StockItem.ship(soap, 2)  # what it means
print(f"{soap.quantity=}")

soap.quantity=6


**Inspecting an instance.** `vars(instance)` returns the instance's own attribute
dict (the same object as `instance.__dict__`). `instance.__class__` is its class,
the same as `type(instance)`.

`__static_attributes__` (*New in 3.13*) is a tuple of the names assigned through
`self.<name>` anywhere in the class body, collected by the compiler. Tools use it
to know an instance's likely attributes without creating one.

In [17]:
print(f"{vars(soap)=}")
print(f"{soap.__dict__ is vars(soap)=}")
print(f"{soap.__class__=}, {type(soap) is soap.__class__=}")
if sys.version_info >= (3, 13):
    print(f"{StockItem.__static_attributes__=}")
else:
    print("__static_attributes__ needs Python 3.13+")

vars(soap)={'sku': 'SOAP', '_quantity': 6}
soap.__dict__ is vars(soap)=True
soap.__class__=<class '__main__.StockItem'>, type(soap) is soap.__class__=True
StockItem.__static_attributes__=('_quantity', 'sku')


#### Under the hood: what an instance costs

An ordinary instance is a small header plus a pointer to its attribute storage.
Since 3.11 CPython stores the attributes of most instances inline and only builds a
real `__dict__` when you ask for one (`vars()` does). Instances of the same class
share one key table (key-sharing dicts, 1.6.1). Unit 2.5 shows how `__slots__`
removes the dict altogether.

> **Note:** the byte counts depend on the Python version and platform.

In [18]:
print(f"{sys.getsizeof(soap)=} bytes (the instance itself)")
print(f"{sys.getsizeof(vars(soap))=} bytes (its materialized __dict__)")
print(f"{sys.getsizeof(StockItem)=} bytes (the class object)")

sys.getsizeof(soap)=48 bytes (the instance itself)
sys.getsizeof(vars(soap))=288 bytes (its materialized __dict__)
sys.getsizeof(StockItem)=1728 bytes (the class object)


### 2.1.4 Instance vs class attributes

Assignments in the **class body** create **class attributes**, shared by every
instance. Assignments to `self.<name>` create **instance attributes**, stored in each
instance's own `__dict__`.

Reading `instance.name` searches in this order (descriptors such as properties
change this slightly; see 2.1.6 and 2.5):

1. the instance's `__dict__`
2. the class's `__dict__`
3. each base class's `__dict__`, in MRO order (2.2.3)

```mermaid
graph LR
    instance["soap.__dict__"] -->|not found| klass["Product.__dict__"]
    klass -->|not found| base["object.__dict__"]
    base -->|not found| error["AttributeError"]
```

<details><summary>ASCII version</summary>

```text
soap.__dict__ --miss--> Product.__dict__ --miss--> object.__dict__ --miss--> AttributeError
```

</details>

In [19]:
class Product:
    """currency is a class attribute; name and price are instance attributes."""

    currency = "INR"

    def __init__(self, name: str, price: int) -> None:
        self.name = name
        self.price = price


soap = Product("Soap", 49)
tea = Product("Tea", 120)
print(f"{vars(soap)=}")
print(f"{soap.currency=} (found on the class)")
Product.currency = "USD"
print(f"{tea.currency=} (every instance sees the change)")
Product.currency = "INR"

vars(soap)={'name': 'Soap', 'price': 49}
soap.currency='INR' (found on the class)
tea.currency='USD' (every instance sees the change)


**Shadowing.** Assigning through an instance never changes the class: it creates an
instance attribute with the same name, which hides the class attribute for that one
instance. Deleting it makes the class attribute visible again.

In [20]:
tea.currency = "EUR"  # creates an instance attribute that shadows the class one
print(f"{vars(tea)=}")
print(f"{tea.currency=}, {soap.currency=}, {Product.currency=}")
del tea.currency
print(f"{tea.currency=} (the class attribute is visible again)")

vars(tea)={'name': 'Tea', 'price': 120, 'currency': 'EUR'}
tea.currency='EUR', soap.currency='INR', Product.currency='INR'
tea.currency='INR' (the class attribute is visible again)


The lookup continues into base classes. `find_attribute_owner` repeats the search
by hand (without descriptors) to show where each name is found. `__repr__` comes
all the way from `object`.

In [21]:
def find_attribute_owner(instance: object, attribute_name: str) -> str:
    """Report where attribute lookup finds attribute_name (ignoring descriptors)."""
    if attribute_name in vars(instance):
        return "instance __dict__"
    for klass in type(instance).__mro__:
        if attribute_name in vars(klass):
            return f"class {klass.__name__}"
    return "not found -> AttributeError"


class DigitalProduct(Product):
    """Adds a class attribute of its own."""

    shipping_required = False


ebook = DigitalProduct("Python e-book", 499)
for attribute_name in ["name", "shipping_required", "currency", "__repr__", "weight"]:
    print(f"{attribute_name:<18} -> {find_attribute_owner(ebook, attribute_name)}")

name               -> instance __dict__
shipping_required  -> class DigitalProduct
currency           -> class Product
__repr__           -> class object
weight             -> not found -> AttributeError


**The shared mutable class-attribute trap.** A mutable value (list, dict, set) in
the class body is created **once**, when the class body runs, and every instance
shares it. `self.line_items.append(...)` does not assign, it mutates the shared list.

> **Warning:** this is the opposite of JavaScript. In JS, a class field
> `lineItems = [];` runs once **per instance**, inside the constructor.

In [22]:
class Invoice:
    """BUG: line_items is one list shared by every invoice."""

    line_items: list[str] = []

    def add_line(self, description: str) -> None:
        """Append a line (mutates the shared class attribute)."""
        self.line_items.append(description)


march_invoice = Invoice()
april_invoice = Invoice()
march_invoice.add_line("Hosting")
print(f"{april_invoice.line_items=}")
print(f"{march_invoice.line_items is Invoice.line_items=}")

april_invoice.line_items=['Hosting']
march_invoice.line_items is Invoice.line_items=True


The fix: create mutable state in `__init__`, so each instance gets its own object.

In [23]:
class FixedInvoice:
    """Each invoice gets its own list."""

    def __init__(self) -> None:
        self.line_items: list[str] = []

    def add_line(self, description: str) -> None:
        """Append a line to this invoice only."""
        self.line_items.append(description)


march_invoice = FixedInvoice()
april_invoice = FixedInvoice()
march_invoice.add_line("Hosting")
print(f"{march_invoice.line_items=}, {april_invoice.line_items=}")

march_invoice.line_items=['Hosting'], april_invoice.line_items=[]


**Predict the output.**

```python
class Team:
    size = 0
    members = []

alpha, beta = Team(), Team()
alpha.size += 1
alpha.members += ["Asha"]
print(Team.size, beta.size, Team.members)
print("size" in vars(alpha), "members" in vars(alpha))
```

<details><summary>Answer</summary>

```text
0 0 ['Asha']
True True
```

`alpha.size += 1` reads `Team.size` (0), adds 1 and **assigns** to `alpha.size`,
creating an instance attribute; the class keeps 0. `alpha.members += ["Asha"]`
calls `list.__iadd__`, which **mutates the shared list in place**, and then also
assigns the same list to `alpha.members`. So both names end up in `vars(alpha)`,
but only the list change is visible to `beta` and the class.

</details>

Run the next cell to check your prediction.

In [24]:
class Team:
    size: int = 0
    members: list[str] = []


alpha, beta = Team(), Team()
alpha.size += 1
alpha.members += ["Asha"]
print(Team.size, beta.size, Team.members)
print("size" in vars(alpha), "members" in vars(alpha))

0 0 ['Asha']
True True


**Good uses of class attributes:** constants and counters. Constants use
`UPPER_CASE` (PEP 8) and can be annotated `ClassVar` so type checkers know they
belong to the class. A counter must be updated **through the class**
(`Ticket.issued_count += 1`); updating it through `self` would create an instance
attribute, as the predict question showed.

In [25]:
class Ticket:
    """Class constants in UPPER_CASE; a class-level counter for ticket numbers."""

    MAX_SEATS: ClassVar[int] = 4
    issued_count: ClassVar[int] = 0

    def __init__(self, seats: int) -> None:
        if not 1 <= seats <= Ticket.MAX_SEATS:
            raise ValueError(f"seats must be 1-{Ticket.MAX_SEATS}")
        Ticket.issued_count += 1  # write through the class, not self
        self.ticket_number = Ticket.issued_count
        self.seats = seats


tickets = [Ticket(2), Ticket(1), Ticket(4)]
print(f"{[ticket.ticket_number for ticket in tickets]=}, {Ticket.issued_count=}")
print(raised_message(lambda: Ticket(9)))

[ticket.ticket_number for ticket in tickets]=[1, 2, 3], Ticket.issued_count=3
ValueError: seats must be 1-4


The broken version: `self.issued_count += 1` reads the class value (0) and writes
an instance attribute every time, so the class counter never moves.

In [26]:
class BrokenTicket:
    """BUG: the counter is updated through self."""

    issued_count = 0

    def __init__(self) -> None:
        self.issued_count += 1  # reads the class value, writes an instance attribute


broken_tickets = [BrokenTicket(), BrokenTicket()]
print(f"{BrokenTicket.issued_count=}")
print(f"{[ticket.issued_count for ticket in broken_tickets]=}")

BrokenTicket.issued_count=0
[ticket.issued_count for ticket in broken_tickets]=[1, 1]


### 2.1.5 Methods

A function stored on a class is a plain function. When you read it **through an
instance**, Python returns a **bound method**: a small object that remembers the
function (`__func__`) and the instance (`__self__`). Calling the bound method calls
the function with the instance as the first argument.

`Money` below is the start of a value type that Unit 2.4 will turn into a full
numeric class.

In [27]:
class Money:
    """An amount of money in one currency (Unit 2.4 adds arithmetic)."""

    def __init__(self, amount: Decimal, currency: str = "INR") -> None:
        self.amount = amount
        self.currency = currency

    def formatted(self) -> str:
        """Return e.g. 'INR 250.00'."""
        return f"{self.currency} {self.amount:.2f}"


price = Money(Decimal("250"))
print(f"{type(Money.formatted).__name__=}  (a plain function on the class)")
bound_formatted = price.formatted
print(f"{type(bound_formatted).__name__=}")
print(f"{bound_formatted.__self__ is price=}")
print(f"{bound_formatted.__func__ is Money.formatted=}")
print(f"{bound_formatted()=}")

type(Money.formatted).__name__='function'  (a plain function on the class)
type(bound_formatted).__name__='method'
bound_formatted.__self__ is price=True
bound_formatted.__func__ is Money.formatted=True
bound_formatted()='INR 250.00'


Because the bound method carries its instance, you can store it, pass it as a
callback or put it in a dict, and it still works. No `.bind(this)` is ever needed.

Each attribute access builds a **new** bound-method object, so `is` is `False`,
but two bound methods of the same function and instance compare equal.

In [28]:
callbacks: list[Callable[[], str]] = [
    Money(Decimal("99")).formatted,
    Money(Decimal("5"), "USD").formatted,
]
print([callback() for callback in callbacks])
print(f"{price.formatted is price.formatted=}  (a new bound method on each access)")
print(f"{price.formatted == price.formatted=}")

['INR 99.00', 'USD 5.00']
price.formatted is price.formatted=False  (a new bound method on each access)
price.formatted == price.formatted=True


#### Under the hood: binding is the descriptor protocol

Functions have a `__get__` method. When attribute lookup finds a function on the
class, it calls `function.__get__(instance, owner)`, which returns the bound method.
`types.MethodType(function, instance)` builds the same object directly. Unit 2.5
covers descriptors in full; `@property`, `@classmethod` and `@staticmethod` all use
this hook.

In [29]:
function_object = Money.__dict__["formatted"]
manually_bound = function_object.__get__(price, Money)
print(f"{manually_bound()=}, {type(manually_bound) is types.MethodType=}")
print(f"{types.MethodType(function_object, price)()=}")

manually_bound()='INR 250.00', type(manually_bound) is types.MethodType=True
types.MethodType(function_object, price)()='INR 250.00'


**`@classmethod`** receives the **class** as its first argument (`cls`) instead of
an instance. Its classic use is an **alternative constructor**: another way to build
an instance, named after its input (`from_string`, `from_paise`). Because it calls
`cls(...)` rather than `Money(...)`, it builds whichever class it was called on.

**`@staticmethod`** receives neither. It is a plain function that lives in the class
namespace because it belongs to the class conceptually (here, currency-code
validation used by the constructors).

In [30]:
class Money:
    """An amount of money in one currency."""

    def __init__(self, amount: Decimal, currency: str = "INR") -> None:
        if not Money.is_valid_currency(currency):
            raise ValueError(f"invalid currency code: {currency!r}")
        self.amount = amount
        self.currency = currency

    @classmethod
    def from_string(cls, text: str) -> Self:
        """Parse 'INR 250.00' (alternative constructor)."""
        currency, amount_text = text.split()
        return cls(Decimal(amount_text), currency)

    @classmethod
    def from_paise(cls, paise: int) -> Self:
        """Build an INR amount from whole paise."""
        return cls(Decimal(paise) / 100, "INR")

    @staticmethod
    def is_valid_currency(code: str) -> bool:
        """Three upper-case letters, like 'INR' or 'USD'."""
        return len(code) == 3 and code.isalpha() and code.isupper()

    def formatted(self) -> str:
        """Return e.g. 'INR 250.00'."""
        return f"{self.currency} {self.amount:.2f}"


print(f"{Money.from_string('USD 19.99').formatted()=}")
print(f"{Money.from_paise(25_050).formatted()=}")

Money.from_string('USD 19.99').formatted()='USD 19.99'
Money.from_paise(25_050).formatted()='INR 250.50'


**Subclass-aware.** `Refund` inherits `from_paise`, and because the method uses
`cls`, `Refund.from_paise(...)` returns a `Refund`. A version that hard-codes
`Money(...)` silently returns the wrong type for every subclass.

In [31]:
class Refund(Money):
    """Money flowing back to a customer."""

    def formatted(self) -> str:
        """Prefix the amount with a minus sign."""
        return f"-{super().formatted()}"


class HardCodedMoney(Money):
    """BUG: the alternative constructor ignores cls."""

    @classmethod
    def from_paise(cls, paise: int) -> Money:
        return Money(Decimal(paise) / 100, "INR")


class HardCodedRefund(HardCodedMoney):
    """Inherits the broken constructor."""


refund = Refund.from_paise(4_950)
print(f"{type(refund).__name__=}, {refund.formatted()=}")
print(f"{type(HardCodedRefund.from_paise(4_950)).__name__=}  (wrong class)")

type(refund).__name__='Refund', refund.formatted()='-INR 49.50'
type(HardCodedRefund.from_paise(4_950)).__name__='Money'  (wrong class)


In application code the most common alternative constructor is **`from_dict`**:
building an object from a JSON payload or a database row. The same `cls(...)` rule
makes it subclass-aware. (Lab 2.1 exercise 3 asks for a stricter one, with
validation and cleaning.)

In [32]:
class ApiMoney(Money):
    """Money as it arrives in JSON payloads."""

    @classmethod
    def from_dict(cls, payload: Mapping[str, str]) -> Self:
        """Build from {"amount": "250.00", "currency": "INR"}."""
        return cls(Decimal(payload["amount"]), payload.get("currency", "INR"))


class ApiRefund(ApiMoney):
    """Inherits from_dict and gets ApiRefund instances from it."""


payload = {"amount": "99.90", "currency": "USD"}
print(f"{ApiMoney.from_dict(payload).formatted()=}")
print(f"{type(ApiRefund.from_dict({'amount': '10'})).__name__=}")

ApiMoney.from_dict(payload).formatted()='USD 99.90'
type(ApiRefund.from_dict({'amount': '10'})).__name__='ApiRefund'


The standard library uses the same pattern everywhere: `dict.fromkeys`,
`int.from_bytes`, `date.fromisoformat`, `Decimal.from_float`. Each one is a
`classmethod`, so subclasses of these types get instances of themselves.

In [33]:
print(f"{dict.fromkeys(['soap', 'tea'], 0)=}")
print(f"{int.from_bytes(bytes([1, 0]), 'big')=}")
print(f"{date.fromisoformat('2026-11-07')=}")
print(f"{Decimal.from_float(0.5)=}")
print(f"{type(dict.__dict__['fromkeys']).__name__=}")

dict.fromkeys(['soap', 'tea'], 0)={'soap': 0, 'tea': 0}
int.from_bytes(bytes([1, 0]), 'big')=256
date.fromisoformat('2026-11-07')=datetime.date(2026, 11, 7)
Decimal.from_float(0.5)=Decimal('0.5')
type(dict.__dict__['fromkeys']).__name__='classmethod_descriptor'


A static method can be called through the class or an instance; it receives no
automatic argument either way.

In [34]:
print(f"{Money.is_valid_currency('INR')=}")
print(f"{Money.from_paise(100).is_valid_currency('rupees')=}")
print(raised_message(lambda: Money(Decimal("1"), "inr")))

Money.is_valid_currency('INR')=True
Money.from_paise(100).is_valid_currency('rupees')=False
ValueError: invalid currency code: 'inr'


**Decision table**

| Kind | First argument | Use it when | Example |
| --- | --- | --- | --- |
| Instance method | `self` (the instance) | It reads or changes one object's state | `cart.add(item)` |
| `@classmethod` | `cls` (the class it was called on) | It builds instances or works with class-level data | `Money.from_paise(500)` |
| `@staticmethod` | nothing | It is a helper that belongs with the class but needs no state | `Money.is_valid_currency("INR")` |
| Module-level function | nothing | It does not belong to any one class | `add_gst(amount)` |

The cell calls each kind through an instance, a subclass instance and the subclass
itself. Notice that `cls` follows the subclass, and that an instance method called on
the class has no `self` to receive.

In [35]:
class MethodKinds:
    """One method of each kind."""

    def instance_method(self) -> str:
        return f"self is a {type(self).__name__} instance"

    @classmethod
    def class_method(cls) -> str:
        return f"cls is {cls.__name__}"

    @staticmethod
    def static_method() -> str:
        return "nothing passed"


class SubKinds(MethodKinds):
    """Inherits all three."""


for caller in [MethodKinds(), SubKinds(), SubKinds]:
    caller_label = getattr(caller, "__name__", f"{type(caller).__name__}()")
    for method_name in ["instance_method", "class_method", "static_method"]:
        result = raised_message(getattr(caller, method_name))
        if result == "no error":
            result = getattr(caller, method_name)()
        print(f"{caller_label:<14} {method_name:<16} {result}")

MethodKinds()  instance_method  self is a MethodKinds instance
MethodKinds()  class_method     cls is MethodKinds
MethodKinds()  static_method    nothing passed
SubKinds()     instance_method  self is a SubKinds instance
SubKinds()     class_method     cls is SubKinds
SubKinds()     static_method    nothing passed
SubKinds       instance_method  TypeError: MethodKinds.instance_method() missing 1 required positional argument: 'self'
SubKinds       class_method     cls is SubKinds
SubKinds       static_method    nothing passed


### 2.1.6 Properties

`@property` turns a method into an attribute that runs code when it is read.
Callers write `plan.total_fee`, not `plan.total_fee()`. Add a `.setter` to run code
on assignment and a `.deleter` to run code on `del`.

| Pattern | How | Example below |
| --- | --- | --- |
| Computed attribute | getter only, derived from other state | `total_fee` |
| Read-only attribute | getter only, backed by `_name` | `subscription_id` |
| Validation on assignment | getter + setter that checks, then stores `_name` | `months` |
| Clean-up on delete | getter + setter + deleter | `auth_token` |

First, a computed and a read-only property. Assigning to either raises
`AttributeError`, and the computed value is recalculated after `months` changes.

In [36]:
class Subscription:
    """A paid plan billed monthly."""

    def __init__(self, subscription_id: str, monthly_fee: Decimal, months: int) -> None:
        self._subscription_id = subscription_id
        self.monthly_fee = monthly_fee
        self.months = months

    @property
    def subscription_id(self) -> str:
        """Read-only: there is no setter."""
        return self._subscription_id

    @property
    def total_fee(self) -> Decimal:
        """Computed on every access."""
        return self.monthly_fee * self.months


plan = Subscription("SUB-7", Decimal("299"), 12)
print(f"{plan.total_fee=}")
plan.months = 6
print(f"{plan.total_fee=} (recomputed)")
for attribute_name in ["subscription_id", "total_fee"]:
    print(raised_message(functools.partial(setattr, plan, attribute_name, "x")))

plan.total_fee=Decimal('3588')
plan.total_fee=Decimal('1794') (recomputed)
AttributeError: property 'subscription_id' of 'Subscription' object has no setter
AttributeError: property 'total_fee' of 'Subscription' object has no setter


**Validation on assignment.** The subclass adds a `months` property whose setter
rejects bad values. The inherited `__init__` already does `self.months = months`, and
that assignment now goes through the setter too, so validation also happens at
construction. A failed assignment leaves the old value in place.

In [37]:
class ValidatedSubscription(Subscription):
    """months must be a whole number from 1 to 36, on every assignment."""

    @property
    def months(self) -> int:
        """Billing period in months."""
        return self._months

    @months.setter
    def months(self, value: int) -> None:
        if not isinstance(value, int) or not 1 <= value <= 36:
            raise ValueError(f"months must be an int from 1 to 36, got {value!r}")
        self._months = value


plan = ValidatedSubscription("SUB-8", Decimal("299"), 12)
print(raised_message(lambda: setattr(plan, "months", 48)))
print(f"{plan.months=} (unchanged after the failed assignment)")
print(raised_message(lambda: ValidatedSubscription("SUB-9", Decimal("99"), 0)))

ValueError: months must be an int from 1 to 36, got 48
plan.months=12 (unchanged after the failed assignment)
ValueError: months must be an int from 1 to 36, got 0


**Deleter.** `del session.auth_token` runs the deleter, which can clean up instead
of just removing a value. Here it records a logout and leaves the token as `None`.

In [38]:
class UserSession:
    """A login session whose token can be revoked with del."""

    def __init__(self, username: str) -> None:
        self.username = username
        self._auth_token: str | None = None
        self.events: list[str] = []

    @property
    def auth_token(self) -> str | None:
        """The current token, or None when logged out."""
        return self._auth_token

    @auth_token.setter
    def auth_token(self, token: str) -> None:
        self._auth_token = token
        self.events.append("login")

    @auth_token.deleter
    def auth_token(self) -> None:
        self._auth_token = None
        self.events.append("logout")


session = UserSession("asha")
session.auth_token = "tok_123"
del session.auth_token
print(f"{session.auth_token=}, {session.events=}")

session.auth_token=None, session.events=['login', 'logout']


`@property` is shorthand for `property(fget, fset, fdel, doc)`. The decorator form
is preferred, but the call form shows what a property object holds: up to three
functions and a docstring.

In [39]:
class SpeakerVolume:
    """Volume clamped to 0-10, written with the property() call form."""

    def __init__(self) -> None:
        self._level = 5

    def _get_level(self) -> int:
        return self._level

    def _set_level(self, value: int) -> None:
        self._level = max(0, min(10, value))

    level = property(_get_level, _set_level, doc="Volume from 0 to 10, clamped.")


speaker = SpeakerVolume()
speaker.level = 14
print(f"{speaker.level=}")
months_property = ValidatedSubscription.__dict__["months"]
print(f"{type(months_property).__name__=}, {months_property.fget.__name__=}")
print(f"{months_property.fset is not None=}, {months_property.fdel=}")
print(f"{SpeakerVolume.level.__doc__=}")

speaker.level=10
type(months_property).__name__='property', months_property.fget.__name__='months'
months_property.fset is not None=True, months_property.fdel=None
SpeakerVolume.level.__doc__='Volume from 0 to 10, clamped.'


**Uniform access principle.** Callers should not care whether a value is stored or
computed. Start with a plain attribute; if you later need validation or
normalization, turn it into a property and **no caller changes**. That is why Python
code does not write `get_email()` / `set_email()` "just in case", as Java code
often does. JavaScript getters and setters give the same freedom.

In [40]:
def send_welcome(account: Any) -> str:
    """Client code: only reads account.email."""
    return f"Welcome mail to {account.email}"


class AccountV1:
    """Version 1: a plain attribute."""

    def __init__(self, email: str) -> None:
        self.email = email


class AccountV2:
    """Version 2: the same name, now a normalizing property."""

    def __init__(self, email: str) -> None:
        self.email = email  # goes through the setter

    @property
    def email(self) -> str:
        return self._email

    @email.setter
    def email(self, value: str) -> None:
        self._email = value.strip().lower()


for account_class in [AccountV1, AccountV2]:
    print(f"{account_class.__name__}: {send_welcome(account_class('  Asha@Example.com '))!r}")

AccountV1: 'Welcome mail to   Asha@Example.com '
AccountV2: 'Welcome mail to asha@example.com'


A computed property runs on **every** read. For an expensive value that does not
change, `functools.cached_property` computes it once and stores the result in the
instance `__dict__`; later reads never call the function again.

In [41]:
class SalesSummary:
    """grand_total is computed once, on first access."""

    def __init__(self, daily_sales: list[int]) -> None:
        self.daily_sales = daily_sales

    @functools.cached_property
    def grand_total(self) -> int:
        """Sum of every day's sales."""
        print("  computing grand_total ...")
        return sum(self.daily_sales)


summary = SalesSummary([1200, 950, 1430])
print(f"{summary.grand_total=}")
print(f"{summary.grand_total=}  (no 'computing' line: cached)")
print(f"{vars(summary)=}")

  computing grand_total ...
summary.grand_total=3580
summary.grand_total=3580  (no 'computing' line: cached)
vars(summary)={'daily_sales': [1200, 950, 1430], 'grand_total': 3580}


#### Under the hood: a property beats the instance `__dict__`

A property is a **data descriptor** stored on the class (it defines `__set__`), and
data descriptors take priority over the instance `__dict__` during lookup. Even if
a value with the same name is forced into `vars(plan)`, the property still answers.
Unit 2.5 explains the full lookup order.

In [42]:
plan = Subscription("SUB-10", Decimal("100"), 3)
vars(plan)["total_fee"] = Decimal("0")  # sneak a value into the instance dict
print(f"{vars(plan)=}")
print(f"{plan.total_fee=} (the property on the class still wins)")
print(f"{inspect.isdatadescriptor(Subscription.__dict__['total_fee'])=}")

vars(plan)={'_subscription_id': 'SUB-10', 'monthly_fee': Decimal('100'), 'months': 3, 'total_fee': Decimal('0')}
plan.total_fee=Decimal('300') (the property on the class still wins)
inspect.isdatadescriptor(Subscription.__dict__['total_fee'])=True


### 2.1.7 Encapsulation by convention

Python has no `private` keyword. Instead:

- **`_name`** (one underscore) means "internal, not part of the public API". Nothing
  enforces it; it is a message to other developers. Tools respect it:
  `from module import *` skips underscore names and IDEs hide them from completion.
- **`__name`** (two leading underscores, at most one trailing) triggers **name
  mangling**: inside a class body the compiler rewrites it to `_ClassName__name`.

The motto is "we are all consenting adults": the underscore warns, it does not lock.

In [43]:
class PaymentClient:
    """Public method send(); everything with an underscore is internal."""

    def __init__(self, api_key: str) -> None:
        self._api_key = api_key  # internal: please do not touch

    def _sign(self, payload: str) -> str:
        return f"{payload}|signed-with-{self._api_key[-4:]}"

    def send(self, payload: str) -> str:
        """Sign and 'send' a payload."""
        return self._sign(payload)


client = PaymentClient("sk_test_12345678")
print(client.send("amount=500"))
print(f"{client._api_key=}  (reachable: the underscore is only a warning)")

amount=500|signed-with-5678
client._api_key='sk_test_12345678'  (reachable: the underscore is only a warning)


**Name mangling.** `self.__pin` is stored as `_Vault__pin`. Code outside the class
that writes `vault.__pin` is not mangled, so it fails; the mangled name still works
for anyone who really wants it.

In [44]:
class Vault:
    """Stores a PIN under a mangled name."""

    def __init__(self, pin: str) -> None:
        self.__pin = pin  # stored as _Vault__pin

    def check(self, attempt: str) -> bool:
        """Compare an attempt with the stored PIN."""
        return attempt == self.__pin


vault = Vault("4321")
print(f"{vars(vault)=}")
print(f"{vault.check('4321')=}")
print(raised_message(lambda: vault.__pin))
print(f"{vault._Vault__pin=}  (still reachable by the mangled name)")

vars(vault)={'_Vault__pin': '4321'}
vault.check('4321')=True
AttributeError: 'Vault' object has no attribute '__pin'
vault._Vault__pin='4321'  (still reachable by the mangled name)


Mangling exists to prevent **accidental name clashes in subclasses**, not to hide
data. Here `Cache` and its subclass both use `__entries`, but they end up as two
different attributes, so the subclass cannot break the base class by accident.

In [45]:
class Cache:
    """Base class with its own private storage."""

    def __init__(self) -> None:
        self.__entries: dict[str, str] = {}  # becomes _Cache__entries

    def cache_size(self) -> int:
        """Entries in the base-class storage."""
        return len(self.__entries)


class UserCache(Cache):
    """Reuses the name __entries without clobbering the base class."""

    def __init__(self) -> None:
        super().__init__()
        self.__entries = ["recent", "users"]  # becomes _UserCache__entries


user_cache = UserCache()
print(f"{vars(user_cache)=}")
print(f"{user_cache.cache_size()=}  (the base class still sees its own dict)")

vars(user_cache)={'_Cache__entries': {}, '_UserCache__entries': ['recent', 'users']}
user_cache.cache_size()=0  (the base class still sees its own dict)


#### Under the hood: mangling happens at compile time

The rewrite is done by the compiler, so the mangled name is already in the
method's code object (`co_names`). Strings passed to `getattr` / `hasattr` are never
mangled, and dunder names such as `__init__` (two trailing underscores) are left
alone.

In [46]:
print(f"{Vault.check.__code__.co_names=}")
mangled_name = "_Vault__pin"
print(f"{getattr(vault, mangled_name)=}")
print(f"{hasattr(vault, '__pin')=}  (strings are not mangled)")

Vault.check.__code__.co_names=('_Vault__pin',)
getattr(vault, mangled_name)='4321'
hasattr(vault, '__pin')=False  (strings are not mangled)


**Comparison with JavaScript and TypeScript**

| | Python `_x` | Python `__x` | TypeScript `private x` | JavaScript `#x` |
| --- | --- | --- | --- | --- |
| Enforced by | nobody (convention) | the compiler renames it | the TS compiler only; erased at run time | the JS engine |
| Reachable from outside | yes | yes, as `obj._Class__x` | yes, at run time | no (`SyntaxError` outside the class) |
| Visible to reflection | `vars(obj)` | `vars(obj)` (mangled) | `Object.keys(obj)` | no |
| Main purpose | mark internal API | avoid subclass clashes | design-time checking | real privacy |

> **Note:** prefer a single underscore. Use `__x` only when you expect subclasses
> from other authors that might reuse the name. Python's closest match to `#x` is a
> closure variable (1.11), which really is unreachable from outside.

### 2.1.8 JS bridge

**Classes vs prototypes.** JavaScript classes are syntax over prototypes: every
object has a hidden `[[Prototype]]` link, and methods live on `Class.prototype`.
Python has no per-object prototype link. Every instance has a `__class__`, methods
live in the class's `__dict__`, and lookup walks the class's `__mro__`.

| JavaScript | Python |
| --- | --- |
| `Object.getPrototypeOf(obj)` | `type(obj)` / `obj.__class__` |
| prototype chain | `type(obj).__mro__` |
| `Counter.prototype.reset = function () {}` | `Counter.reset = reset` |
| `Object.setPrototypeOf(obj, Other.prototype)` | `obj.__class__ = Other` (only for compatible layouts; rarely wise) |
| `this` set by the call site (`obj.m()`, `m()`, `new`, `call`, `apply`, `bind`) | `self` bound when the method is read from the instance |
| arrow functions capture the outer `this` | lambdas and closures capture `self` like any other variable |
| `static m()`: `this` is the constructor | `@classmethod`: `cls` is the class it was called on |
| `new Counter()` | `Counter()`: a class is called like a function |

**`self` vs `this`.** In JavaScript, pulling a method off its object loses `this`:

```javascript
class ClickCounter {
  count = 0;
  increment() { this.count += 1; }
}
const counter = new ClickCounter();
const increment = counter.increment;
increment();                          // TypeError: cannot read 'count' of undefined
button.addEventListener("click", counter.increment);  // same bug
const fixed = counter.increment.bind(counter);         // the usual fix
```

In Python the bound method already carries the instance, so the same code just works.

In [47]:
class ClickCounter:
    """Counts clicks."""

    def __init__(self) -> None:
        self.count = 0

    def increment(self) -> None:
        """Add one click."""
        self.count += 1


click_counter = ClickCounter()
increment = click_counter.increment  # self is already bound
for _ in range(3):
    increment()
handlers: dict[str, Callable[[], None]] = {"click": click_counter.increment}
handlers["click"]()
print(f"{click_counter.count=}")

click_counter.count=4


Like adding to `ClickCounter.prototype`, adding a function to a Python class later
makes it available to **existing** instances, because lookup reaches the class at
call time. The second line shows the Python view of the "prototype chain".

In [48]:
def reset(self: ClickCounter) -> None:
    """Set the count back to zero."""
    self.count = 0


ClickCounter.reset = reset  # like ClickCounter.prototype.reset = function () {...}
click_counter.reset()
print(f"{click_counter.count=}")
print(f"{[klass.__name__ for klass in type(click_counter).__mro__]=}")

click_counter.count=0
[klass.__name__ for klass in type(click_counter).__mro__]=['ClickCounter', 'object']


**Predict the output.**

```python
class Greeter:
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"Hi, I am {self.name}"

asha = Greeter("Asha")
say_hello = asha.greet
asha.name = "Asha K"
print(say_hello())
print(Greeter.greet(asha))
print(Greeter.greet())
```

<details><summary>Answer</summary>

```text
Hi, I am Asha K
Hi, I am Asha K
TypeError: Greeter.greet() missing 1 required positional argument: 'self'
```

`say_hello` is bound to the **object**, not to a snapshot of its attributes, so it
sees the new name. `Greeter.greet(asha)` passes the instance explicitly. Reading the
method from the **class** gives the plain function, and with no instance there is
nothing to fill `self`: Python's version of JavaScript's lost `this`, except it
fails loudly with a clear message.

</details>

Run the next cell to check your prediction.

In [49]:
class Greeter:
    """Says hello."""

    def __init__(self, name: str) -> None:
        self.name = name

    def greet(self) -> str:
        """Introduce yourself."""
        return f"Hi, I am {self.name}"


asha = Greeter("Asha")
say_hello = asha.greet
asha.name = "Asha K"
print(say_hello())
print(Greeter.greet(asha))
print(raised_message(lambda: Greeter.greet()))  # type: ignore[call-arg]

Hi, I am Asha K
Hi, I am Asha K
TypeError: Greeter.greet() missing 1 required positional argument: 'self'


### 2.1.9 The four pillars of OOP in Python

The four classic pillars all exist in Python, but they look lighter than in Java
or TypeScript, and this phase covers each one:

| Pillar | Where in this phase | Python mechanism | Java | TypeScript |
| --- | --- | --- | --- | --- |
| Encapsulation | 2.1.6–2.1.7 | properties, `_name` convention, name mangling | `private` fields + getters/setters | `private` (compile time), `#x` (run time) |
| Inheritance | 2.2 | `class B(A)`, **multiple** inheritance, C3 MRO, mixins | single `extends` + `implements` | single `extends`, mixin functions |
| Polymorphism | 2.3.4, 2.4 | duck typing, special (dunder) methods, `singledispatch` | overriding + overloading by signature | structural typing, overload signatures (types only) |
| Abstraction | 2.3 | ABCs (checked at instantiation), Protocols (checked by type checkers) | `abstract class`, `interface` | `abstract class`, `interface` (erased at run time) |

The four cells below show one pillar each on a tiny media library.

**Encapsulation**: a read-only property protects the size.

In [50]:
class MediaFile:
    """A file in a media library."""

    def __init__(self, name: str, size_bytes: int) -> None:
        self.name = name
        self._size_bytes = size_bytes

    @property
    def size_bytes(self) -> int:
        """Read-only size."""
        return self._size_bytes

    def summary(self) -> str:
        """One-line description."""
        return f"{self.name} ({self.size_bytes:,} bytes)"


print(MediaFile("notes.txt", 2_048).summary())
print(raised_error_name(lambda: setattr(MediaFile("x", 1), "size_bytes", 0)))

notes.txt (2,048 bytes)
AttributeError


**Inheritance**: `AudioFile` reuses `MediaFile` and extends `summary()` with
`super()` (Unit 2.2).

In [51]:
class AudioFile(MediaFile):
    """A media file with a duration."""

    def __init__(self, name: str, size_bytes: int, duration_seconds: int) -> None:
        super().__init__(name, size_bytes)
        self.duration_seconds = duration_seconds

    def summary(self) -> str:
        """Add the duration to the base summary."""
        minutes, seconds = divmod(self.duration_seconds, 60)
        return f"{super().summary()}, {minutes}:{seconds:02d}"


print(AudioFile("raga.mp3", 4_500_000, 245).summary())

raga.mp3 (4,500,000 bytes), 4:05


**Polymorphism**: the loop calls `summary()` without knowing the type. `VideoLink`
shares no base class with the others; having the method is enough (duck typing,
2.3.4).

In [52]:
class VideoLink:
    """Not a MediaFile at all, but it has summary()."""

    def __init__(self, url: str) -> None:
        self.url = url

    def summary(self) -> str:
        """One-line description."""
        return f"link to {self.url}"


library = [MediaFile("notes.txt", 2_048), AudioFile("raga.mp3", 4_500_000, 245)]
library.append(VideoLink("https://example.com/talk"))
for item in library:
    print(item.summary())

notes.txt (2,048 bytes)
raga.mp3 (4,500,000 bytes), 4:05
link to https://example.com/talk


**Abstraction**: an abstract base class (Unit 2.3) states what every playable item
must provide, and refuses to create incomplete objects.

In [53]:
class Playable(ABC):
    """Anything that can be played."""

    @abstractmethod
    def play(self) -> str:
        """Start playback and describe it."""


class Podcast(Playable):
    """A concrete playable item."""

    def play(self) -> str:
        return "playing podcast"


print(Podcast().play())
print(raised_message(Playable))

playing podcast
TypeError: Can't instantiate abstract class Playable without an implementation for abstract method 'play'


**How Python differs from Java/TypeScript in practice**

- **No access modifiers.** Encapsulation is a convention plus properties, not a
  compiler rule (2.1.7).
- **Multiple inheritance is allowed**, made safe by the C3 MRO and cooperative
  `super()` (2.2.2–2.2.4). Java and TypeScript allow one base class.
- **No method overloading.** A class namespace is a dict, so a second `def` with
  the same name **replaces** the first. Python uses default arguments,
  `*args`/`**kwargs` or `functools.singledispatchmethod` instead.
- **Interfaces are optional.** Duck typing works with no declaration at all;
  Protocols (2.3.3) add TypeScript-style structural checks for type checkers.

The cell shows the "no overloading" rule: only the second `add` exists.

In [54]:
class Calculator:
    """Two methods with the same name: the second replaces the first."""

    def add(self, first: int, second: int) -> int:
        return first + second

    def add(self, first: int, second: int, third: int) -> int:  # replaces the first
        return first + second + third


print(f"{Calculator().add(1, 2, 3)=}")
print(raised_message(lambda: Calculator().add(1, 2)))  # type: ignore[call-arg]

Calculator().add(1, 2, 3)=6
TypeError: Calculator.add() missing 1 required positional argument: 'third'


### Lab 2.1

Four exercises. For each, replace every `raise NotImplementedError` in the stub,
then run the check cell. Until then the checks print FAIL; that is expected. Two
structural checks ("is a classmethod" in exercise 3, "line_total is a property" in
exercise 4) already pass on the stub, because the stub has the right shape.

#### Exercise 1 — `BankAccount` with a validated balance and history

`BankAccount(owner, opening_balance=Decimal("0"))`:

- `balance` is a **read-only** property (assigning to it raises `AttributeError`).
- `deposit(amount)` and `withdraw(amount)` return the new balance. The amount must be
  a `Decimal` (otherwise `TypeError(f"amount must be a Decimal, got {type_name}")`)
  and greater than zero (otherwise `ValueError("amount must be positive")`).
- A withdrawal larger than the balance raises
  `ValueError(f"insufficient funds: balance {balance:.2f}, requested {amount:.2f}")`.
- `history` is a property returning a **tuple** of `Transaction` records, oldest
  first. The first one is `Transaction("open", opening_balance, opening_balance)`;
  then `"deposit"` / `"withdrawal"` with the balance after each.
- A failed operation changes nothing. A negative opening balance raises
  `ValueError("opening balance cannot be negative")`; a non-`Decimal` one raises
  `TypeError(f"opening_balance must be a Decimal, got {type_name}")`.

In [55]:
class Transaction(NamedTuple):
    """One entry in a BankAccount history."""

    kind: str  # "open", "deposit" or "withdrawal"
    amount: Decimal
    balance_after: Decimal


class BankAccount:
    """A bank account with a read-only, validated balance and an append-only history."""

    def __init__(self, owner: str, opening_balance: Decimal = Decimal("0")) -> None:
        raise NotImplementedError("Lab 2.1: implement BankAccount.__init__")

    @property
    def balance(self) -> Decimal:
        """Current balance (read-only)."""
        raise NotImplementedError("Lab 2.1: implement BankAccount.balance")

    @property
    def history(self) -> tuple[Transaction, ...]:
        """Every transaction so far, oldest first."""
        raise NotImplementedError("Lab 2.1: implement BankAccount.history")

    def deposit(self, amount: Decimal) -> Decimal:
        """Add amount and return the new balance."""
        raise NotImplementedError("Lab 2.1: implement BankAccount.deposit")

    def withdraw(self, amount: Decimal) -> Decimal:
        """Take amount and return the new balance."""
        raise NotImplementedError("Lab 2.1: implement BankAccount.withdraw")

Run the checks.

In [56]:
def sample_account() -> BankAccount:
    """An account opened with 100.00, then +50.00 and -30.00."""
    account = BankAccount("Asha", Decimal("100.00"))
    account.deposit(Decimal("50.00"))
    account.withdraw(Decimal("30.00"))
    return account


def balance_after_failed_withdrawal() -> tuple[Decimal, int]:
    """Balance and history length after an overdraft attempt."""
    account = sample_account()
    raised_error_name(lambda: account.withdraw(Decimal("500.00")))
    return account.balance, len(account.history)


def assign_balance() -> None:
    """Try to overwrite the read-only balance."""
    account = sample_account()
    account.balance = Decimal("1000000")  # type: ignore[misc]


def history_is_a_copy() -> bool:
    """True if the returned history is a tuple and the account still has 3 entries."""
    account = sample_account()
    history = account.history
    return type(history) is tuple and len(account.history) == 3


run_checks(
    [
        ("balance", lambda: sample_account().balance, Decimal("120.00")),
        (
            "history",
            lambda: sample_account().history,
            (
                Transaction("open", Decimal("100.00"), Decimal("100.00")),
                Transaction("deposit", Decimal("50.00"), Decimal("150.00")),
                Transaction("withdrawal", Decimal("30.00"), Decimal("120.00")),
            ),
        ),
        ("history is a tuple", history_is_a_copy, True),
        ("balance is read-only", lambda: raised_error_name(assign_balance), "AttributeError"),
        ("default opening", lambda: BankAccount("Ravi").balance, Decimal("0")),
        (
            "returns new balance",
            lambda: BankAccount("Ravi").deposit(Decimal("10.50")),
            Decimal("10.50"),
        ),
        (
            "non-positive amount",
            lambda: raised_message(lambda: BankAccount("Ravi").deposit(Decimal("0"))),
            "ValueError: amount must be positive",
        ),
        (
            "float amount",
            lambda: raised_message(lambda: BankAccount("Ravi").deposit(10.5)),  # type: ignore[arg-type]
            "TypeError: amount must be a Decimal, got float",
        ),
        (
            "overdraft",
            lambda: raised_message(lambda: sample_account().withdraw(Decimal("500"))),
            "ValueError: insufficient funds: balance 120.00, requested 500.00",
        ),
        (
            "failed withdrawal changes nothing",
            balance_after_failed_withdrawal,
            (Decimal("120.00"), 3),
        ),
        (
            "negative opening",
            lambda: raised_message(lambda: BankAccount("Ravi", Decimal("-1"))),
            "ValueError: opening balance cannot be negative",
        ),
    ]
)

FAIL  balance: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  history: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  history is a tuple: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  balance is read-only: expected 'AttributeError', got 'NotImplementedError'
FAIL  default opening: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  returns new balance: raised NotImplementedError: Lab 2.1: implement BankAccount.__init__
FAIL  non-positive amount: expected 'ValueError: amount must be positive', got 'NotImplementedError: Lab 2.1: implement BankAccount.__init__'
FAIL  float amount: expected 'TypeError: amount must be a Decimal, got float', got 'NotImplementedError: Lab 2.1: implement BankAccount.__init__'
FAIL  overdraft: expected 'ValueError: insufficient funds: balance 120.00, requested 500.00', got 'NotImplementedError: Lab 2.1: implement BankAccount.__init__'
FAIL  failed withdrawal 

#### Exercise 2 — `Temperature` with a `fahrenheit` property

`Temperature(celsius)` stores degrees Celsius:

- `celsius` is a property whose setter rejects values below absolute zero
  (−273.15) with `ValueError(f"below absolute zero: {value} °C")`. `__init__` must
  go through the setter.
- `fahrenheit` is a property computed from Celsius (`c * 9 / 5 + 32`); its
  **setter** converts back and stores Celsius, so it is validated too.
- `kelvin` is a **read-only** property (`c + 273.15`).
- A rejected assignment leaves the old value.

In [57]:
ABSOLUTE_ZERO_CELSIUS = -273.15


class Temperature:
    """A temperature stored in Celsius, readable and writable in Fahrenheit."""

    def __init__(self, celsius: float) -> None:
        raise NotImplementedError("Lab 2.1: implement Temperature")

Run the checks. Floats are rounded to two places before comparing.

In [58]:
def set_fahrenheit(value: float) -> float:
    """Set Fahrenheit on a 20 °C temperature and return the Celsius value."""
    temperature = Temperature(20)
    temperature.fahrenheit = value
    return round(temperature.celsius, 2)


def rejected_fahrenheit_keeps_value() -> float:
    """Celsius after an invalid Fahrenheit assignment."""
    temperature = Temperature(20)
    raised_error_name(lambda: setattr(temperature, "fahrenheit", -500))
    return temperature.celsius


run_checks(
    [
        ("100 °C in °F", lambda: round(Temperature(100).fahrenheit, 2), 212.0),
        ("-40 is -40", lambda: round(Temperature(-40).fahrenheit, 2), -40.0),
        ("set °F", lambda: set_fahrenheit(212), 100.0),
        ("set °F to 32", lambda: set_fahrenheit(32), 0.0),
        ("kelvin", lambda: round(Temperature(25).kelvin, 2), 298.15),
        (
            "kelvin is read-only",
            lambda: raised_error_name(lambda: setattr(Temperature(25), "kelvin", 0)),
            "AttributeError",
        ),
        (
            "below absolute zero",
            lambda: raised_message(lambda: Temperature(-300)),
            "ValueError: below absolute zero: -300 °C",
        ),
        (
            "°F setter validates",
            lambda: raised_error_name(lambda: setattr(Temperature(0), "fahrenheit", -500)),
            "ValueError",
        ),
        ("rejected value leaves state", rejected_fahrenheit_keeps_value, 20),
        (
            "fahrenheit is a property",
            lambda: type(Temperature.__dict__["fahrenheit"]).__name__,
            "property",
        ),
    ]
)

FAIL  100 °C in °F: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  -40 is -40: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  set °F: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  set °F to 32: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  kelvin: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  kelvin is read-only: expected 'AttributeError', got 'NotImplementedError'
FAIL  below absolute zero: expected 'ValueError: below absolute zero: -300 °C', got 'NotImplementedError: Lab 2.1: implement Temperature'
FAIL  °F setter validates: expected 'ValueError', got 'NotImplementedError'
FAIL  rejected value leaves state: raised NotImplementedError: Lab 2.1: implement Temperature
FAIL  fahrenheit is a property: raised KeyError: 'fahrenheit'
0/10 checks passed


#### Exercise 3 — subclass-aware `User.from_dict`

Write the `classmethod` `User.from_dict(data)`:

- `username` and `email` are required; check them in that order and raise
  `ValueError(f"missing field: {name}")` for the first one missing.
- Strip the username; strip and lower-case the email. `is_active` is optional
  (default `True`). Ignore unknown keys and do not modify `data`.
- It must build an instance of **the class it is called on**, so
  `AdminUser.from_dict(...)` returns an `AdminUser`, and so does any subclass
  written later (the check defines a new `GuestUser`).

`__init__`, `to_dict` and `AdminUser` are given.

In [59]:
class User:
    """A site user."""

    role = "member"

    def __init__(self, username: str, email: str, is_active: bool = True) -> None:
        self.username = username
        self.email = email
        self.is_active = is_active

    @classmethod
    def from_dict(cls, data: Mapping[str, Any]) -> Self:
        """Build a user from a mapping (username and email required)."""
        raise NotImplementedError("Lab 2.1: implement User.from_dict")

    def to_dict(self) -> dict[str, Any]:
        """Serialize to a plain dict, including the class-level role."""
        return {
            "username": self.username,
            "email": self.email,
            "is_active": self.is_active,
            "role": self.role,
        }


class AdminUser(User):
    """A user with the admin role."""

    role = "admin"

Run the checks.

In [60]:
class GuestUser(User):
    """A third subclass the stub has never seen."""

    role = "guest"


raw_user = {"username": "  meena ", "email": " Meena@Example.COM ", "plan": "pro"}
run_checks(
    [
        ("returns User", lambda: type(User.from_dict(raw_user)).__name__, "User"),
        ("subclass-aware", lambda: type(AdminUser.from_dict(raw_user)).__name__, "AdminUser"),
        ("unseen subclass", lambda: type(GuestUser.from_dict(raw_user)).__name__, "GuestUser"),
        (
            "fields cleaned",
            lambda: User.from_dict(raw_user).to_dict(),
            {
                "username": "meena",
                "email": "meena@example.com",
                "is_active": True,
                "role": "member",
            },
        ),
        ("admin role", lambda: AdminUser.from_dict(raw_user).to_dict()["role"], "admin"),
        (
            "is_active read",
            lambda: User.from_dict({**raw_user, "is_active": False}).is_active,
            False,
        ),
        (
            "round trip",
            lambda: AdminUser.from_dict(AdminUser.from_dict(raw_user).to_dict()).to_dict(),
            {
                "username": "meena",
                "email": "meena@example.com",
                "is_active": True,
                "role": "admin",
            },
        ),
        (
            "missing email",
            lambda: raised_message(lambda: User.from_dict({"username": "x"})),
            "ValueError: missing field: email",
        ),
        (
            "missing both",
            lambda: raised_message(lambda: User.from_dict({})),
            "ValueError: missing field: username",
        ),
        (
            "input untouched",
            lambda: (User.from_dict(raw_user), raw_user["username"])[1],
            "  meena ",
        ),
        ("is a classmethod", lambda: type(User.__dict__["from_dict"]).__name__, "classmethod"),
    ]
)

FAIL  returns User: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  subclass-aware: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  unseen subclass: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  fields cleaned: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  admin role: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  is_active read: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  round trip: raised NotImplementedError: Lab 2.1: implement User.from_dict
FAIL  missing email: expected 'ValueError: missing field: email', got 'NotImplementedError: Lab 2.1: implement User.from_dict'
FAIL  missing both: expected 'ValueError: missing field: username', got 'NotImplementedError: Lab 2.1: implement User.from_dict'
FAIL  input untouched: raised NotImplementedError: Lab 2.1: implement User.from_dict
PASS  is a classmethod
1/11 checks passed


#### Exercise 4 — `ShoppingCart` and `CartItem` with `Decimal` totals

`CartItem(sku, unit_price, quantity=1)`:

- `unit_price` must be a `Decimal`
  (`TypeError(f"unit_price must be a Decimal, got {type_name}")`) and not negative
  (`ValueError("unit price cannot be negative")`); `quantity` must be at least 1
  (`ValueError("quantity must be at least 1")`).
- `line_total` is a property: `unit_price * quantity`.

`ShoppingCart()`:

- `add(sku, unit_price, quantity=1)` returns the `CartItem`. Adding an existing SKU
  increases its quantity; a different unit price raises
  `ValueError(f"price mismatch for {sku}")`.
- `remove(sku, quantity=None)` removes that many units (all of them when `None`, or
  when the quantity reaches zero); an unknown SKU raises `KeyError(sku)`.
- Properties: `items` (a tuple of `CartItem`s in first-added order), `item_count`
  (total units) and `subtotal` (a `Decimal`, `Decimal("0")` when empty).
- `total(tax_rate_percent=Decimal("18"))` returns subtotal plus tax, rounded
  `ROUND_HALF_UP` to the paisa (`PAISE`, from the setup cell).

In [61]:
class CartItem:
    """One line of a shopping cart."""

    def __init__(self, sku: str, unit_price: Decimal, quantity: int = 1) -> None:
        raise NotImplementedError("Lab 2.1: implement CartItem.__init__")

    @property
    def line_total(self) -> Decimal:
        """unit_price × quantity."""
        raise NotImplementedError("Lab 2.1: implement CartItem.line_total")


class ShoppingCart:
    """A cart of CartItems keyed by SKU, with exact Decimal totals."""

    def __init__(self) -> None:
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.__init__")

    def add(self, sku: str, unit_price: Decimal, quantity: int = 1) -> CartItem:
        """Add quantity of sku."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.add")

    def remove(self, sku: str, quantity: int | None = None) -> None:
        """Remove quantity of sku (all of it when quantity is None)."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.remove")

    @property
    def items(self) -> tuple[CartItem, ...]:
        """The cart lines in first-added order."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.items")

    @property
    def item_count(self) -> int:
        """Total quantity across every line."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.item_count")

    @property
    def subtotal(self) -> Decimal:
        """Sum of line totals before tax."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.subtotal")

    def total(self, tax_rate_percent: Decimal = Decimal("18")) -> Decimal:
        """Subtotal plus tax, rounded half-up to the paisa."""
        raise NotImplementedError("Lab 2.1: implement ShoppingCart.total")

Run the checks.

In [62]:
def sample_cart() -> ShoppingCart:
    """Two lines: 2 × 499.00 soap (added in two calls) and 1 × 1250.50 rice."""
    cart = ShoppingCart()
    cart.add("SOAP", Decimal("499.00"))
    cart.add("RICE-5KG", Decimal("1250.50"))
    cart.add("SOAP", Decimal("499.00"))
    return cart


def lines(cart: ShoppingCart) -> list[tuple[str, int, Decimal]]:
    """(sku, quantity, line_total) for each cart line."""
    return [(item.sku, item.quantity, item.line_total) for item in cart.items]


def after_remove(sku: str, quantity: int | None) -> list[tuple[str, int, Decimal]]:
    """Lines of the sample cart after one remove call."""
    cart = sample_cart()
    cart.remove(sku, quantity)
    return lines(cart)


run_checks(
    [
        (
            "lines merge",
            lambda: lines(sample_cart()),
            [("SOAP", 2, Decimal("998.00")), ("RICE-5KG", 1, Decimal("1250.50"))],
        ),
        ("item_count", lambda: sample_cart().item_count, 3),
        ("subtotal", lambda: sample_cart().subtotal, Decimal("2248.50")),
        ("total with 18% GST", lambda: sample_cart().total(), Decimal("2653.23")),
        ("total with 5% GST", lambda: sample_cart().total(Decimal("5")), Decimal("2360.93")),
        (
            "empty cart",
            lambda: (ShoppingCart().subtotal, ShoppingCart().total()),
            (Decimal("0"), Decimal("0.00")),
        ),
        ("items is a tuple", lambda: type(sample_cart().items).__name__, "tuple"),
        (
            "price mismatch",
            lambda: raised_message(lambda: sample_cart().add("SOAP", Decimal("450.00"))),
            "ValueError: price mismatch for SOAP",
        ),
        (
            "remove some",
            lambda: after_remove("SOAP", 1),
            [("SOAP", 1, Decimal("499.00")), ("RICE-5KG", 1, Decimal("1250.50"))],
        ),
        ("remove all", lambda: after_remove("SOAP", None), [("RICE-5KG", 1, Decimal("1250.50"))]),
        (
            "remove unknown",
            lambda: raised_error_name(lambda: sample_cart().remove("TEA")),
            "KeyError",
        ),
        (
            "negative price",
            lambda: raised_message(lambda: CartItem("X", Decimal("-1"))),
            "ValueError: unit price cannot be negative",
        ),
        (
            "zero quantity",
            lambda: raised_message(lambda: CartItem("X", Decimal("1"), 0)),
            "ValueError: quantity must be at least 1",
        ),
        (
            "float price",
            lambda: raised_message(lambda: CartItem("X", 9.99)),  # type: ignore[arg-type]
            "TypeError: unit_price must be a Decimal, got float",
        ),
        (
            "line_total is a property",
            lambda: type(CartItem.__dict__["line_total"]).__name__,
            "property",
        ),
    ]
)

FAIL  lines merge: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  item_count: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  subtotal: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  total with 18% GST: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  total with 5% GST: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  empty cart: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  items is a tuple: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  price mismatch: expected 'ValueError: price mismatch for SOAP', got 'NotImplementedError: Lab 2.1: implement ShoppingCart.__init__'
FAIL  remove some: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  remove all: raised NotImplementedError: Lab 2.1: implement ShoppingCart.__init__
FAIL  remove unknown: expected 'KeyError', got 'NotImpleme

### Recap — Unit 2.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Why classes | Bundle state with the rules that keep it valid; prefer functions, modules or dataclasses when there are no rules | `class` vs plain objects and functions |
| `class` statement | Runs the body in a namespace, then calls `type(name, bases, namespace)` | class expression / `Object.create` |
| `__dict__` | A class's namespace is a read-only `mappingproxy`; change it with `setattr` | `Class.prototype` |
| Creation | `__new__` allocates, `__init__` initializes; `__new__` matters for immutable bases | `constructor` (both at once) |
| `self` | Explicit first parameter; `obj.m(x)` is `Class.m(obj, x)` | implicit `this` |
| Attribute lookup | instance `__dict__` → class → bases; assignment through an instance shadows | prototype chain |
| Class-attribute trap | A mutable class attribute is shared by every instance; create it in `__init__` | class fields are per instance |
| Bound methods | `__self__` + `__func__`; usable as callbacks without binding | `.bind(this)` |
| `@classmethod` | Receives `cls`; use for subclass-aware alternative constructors | `static` + `this` |
| `@staticmethod` | Receives nothing; a helper that lives with the class | `static` |
| `@property` | Computed, read-only and validated attributes behind attribute syntax | `get` / `set` accessors |
| Privacy | `_x` by convention, `__x` mangled to `_Class__x`; nothing is truly private | `#x` (enforced), TS `private` (compile time) |
| Four pillars | All four exist, with lighter mechanisms: conventions, multiple inheritance, duck typing, ABCs/Protocols | Java/TS keywords |

**Further reading**

- [Tutorial: classes](https://docs.python.org/3/tutorial/classes.html),
  [data model: classes and instances](https://docs.python.org/3/reference/datamodel.html#classes),
  [class definitions](https://docs.python.org/3/reference/compound_stmts.html#class-definitions)
- [`type`](https://docs.python.org/3/library/functions.html#type),
  [`vars`](https://docs.python.org/3/library/functions.html#vars),
  [`property`](https://docs.python.org/3/library/functions.html#property),
  [`classmethod`](https://docs.python.org/3/library/functions.html#classmethod),
  [`staticmethod`](https://docs.python.org/3/library/functions.html#staticmethod)
- [Descriptor HowTo Guide](https://docs.python.org/3/howto/descriptor.html),
  [private name mangling](https://docs.python.org/3/reference/expressions.html#private-name-mangling),
  [`functools.cached_property`](https://docs.python.org/3/library/functools.html#functools.cached_property)
- [PEP 8 — naming conventions](https://peps.python.org/pep-0008/#naming-conventions),
  [PEP 3115 — Metaclasses in Python 3](https://peps.python.org/pep-3115/),
  [PEP 526 — `ClassVar`](https://peps.python.org/pep-0526/),
  [PEP 673 — `Self`](https://peps.python.org/pep-0673/),
  [PEP 557 — Data classes](https://peps.python.org/pep-0557/)

## Unit 2.2 — Inheritance, MRO and Composition

**Objectives**

- Override and extend methods, and call the base class with `super().__init__`.
- Explain zero-argument `super()` as "the next class in the MRO", not "the parent".
- Compute a C3 linearization (MRO) by hand and recognize an inconsistent one.
- Write mixins and order base classes correctly.
- Prefer composition when inheritance would explode into many classes or break the
  Liskov substitution principle.
- Recognize each SOLID principle in a short Python example.

**JS bridge**

| JavaScript / TypeScript | Python | The difference that bites |
| --- | --- | --- |
| `class B extends A` | `class B(A):` | Python allows **several** base classes |
| `super(...)` must run before `this` is used | `super().__init__(...)` is optional and can run anywhere | Forget it and base attributes are silently missing |
| `super.method()` (always the static parent, via `[[HomeObject]]`) | `super().method()` (the next class in `type(self).__mro__`) | Under multiple inheritance `super()` can call a **sibling** |
| Prototype chain | `__mro__` (C3 linearization) | Computed once per class; base order matters |
| Mixins via `Object.assign` or `Base => class extends Base {}` | Mixin classes listed before the base class | Order of bases decides which method wins |
| `obj instanceof A` | `isinstance(obj, A)` / `issubclass(B, A)` | Accept tuples and `A | B` unions |
| `Object.prototype` | `object` | Every class inherits from `object` |
| TS `override` keyword | `@typing.override` (3.12+) | Checked by type checkers, not at run time |

The next cell is this unit's setup. `mro_names(cls)` lists class names along
`cls.__mro__`.

In [63]:
import importlib.util
import inspect
import itertools
import json
import socketserver
import sys
from collections.abc import Callable, Iterable, Iterator
from dataclasses import dataclass
from datetime import date, datetime
from decimal import Decimal
from typing import Any, ClassVar, Protocol, override


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def mro_names(cls: type) -> list[str]:
    """Class names along cls.__mro__."""
    return [klass.__name__ for klass in cls.__mro__]


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.2.1 Single inheritance

A subclass inherits every attribute of its base class and can **override** (replace)
or **extend** (replace and call the original) any of them. The base class for this
section is a parcel shipment.

In [64]:
class Shipment:
    """A parcel to deliver."""

    RATE_PER_KG = Decimal("40")

    def __init__(self, tracking_id: str, weight_kg: Decimal) -> None:
        self.tracking_id = tracking_id
        self.weight_kg = weight_kg

    def shipping_cost(self) -> Decimal:
        """Price of delivery in rupees."""
        return self.weight_kg * self.RATE_PER_KG

    def label(self) -> str:
        """Text printed on the parcel."""
        return f"{self.tracking_id}: {self.weight_kg} kg, INR {self.shipping_cost():.2f}"


print(Shipment("TRK-1", Decimal("2.5")).label())

TRK-1: 2.5 kg, INR 100.00


**Overriding.** `FragileShipment` replaces `shipping_cost` and reuses the original
through `super()`. It does not touch `label`, yet the inherited `label` prints the
new cost: `self.shipping_cost()` is looked up on the **instance's** class at call
time (dynamic dispatch).

`@override` (from `typing`, 3.12+) marks the intent. At run time it only sets
`__override__ = True`; type checkers report an error if the base class has no
method with that name, which catches typos like `shiping_cost`.

In [65]:
class FragileShipment(Shipment):
    """Costs more: a fixed handling fee on top of the weight-based price."""

    HANDLING_FEE = Decimal("75")

    @override
    def shipping_cost(self) -> Decimal:
        return super().shipping_cost() + self.HANDLING_FEE


vase = FragileShipment("TRK-2", Decimal("2.5"))
print(vase.label())
print(f"{FragileShipment.shipping_cost.__override__=}")
print(f"{'label' in vars(FragileShipment)=}  (label is inherited, not copied)")

TRK-2: 2.5 kg, INR 175.00
FragileShipment.shipping_cost.__override__=True
'label' in vars(FragileShipment)=False  (label is inherited, not copied)


**Extending `__init__`.** A subclass that needs extra state defines its own
`__init__`, calls `super().__init__(...)` for the base part, then sets its own
attributes. It can extend other methods the same way.

In [66]:
class ExpressShipment(Shipment):
    """A shipment with a promised delivery date."""

    def __init__(self, tracking_id: str, weight_kg: Decimal, promised_by: date) -> None:
        super().__init__(tracking_id, weight_kg)
        self.promised_by = promised_by

    @override
    def label(self) -> str:
        return f"{super().label()} | EXPRESS by {self.promised_by:%d %b}"


express = ExpressShipment("TRK-3", Decimal("1"), date(2026, 11, 9))
print(express.label())
print(f"{vars(express)=}")

TRK-3: 1 kg, INR 40.00 | EXPRESS by 09 Nov
vars(express)={'tracking_id': 'TRK-3', 'weight_kg': Decimal('1'), 'promised_by': datetime.date(2026, 11, 9)}


Forgetting `super().__init__` is not an error in Python (unlike JavaScript, where
using `this` before `super()` throws). The object is created without the base
attributes, and the failure shows up later, far from the cause.

In [67]:
class BrokenExpressShipment(Shipment):
    """BUG: never calls super().__init__."""

    def __init__(self, tracking_id: str, weight_kg: Decimal, promised_by: date) -> None:
        self.promised_by = promised_by


broken = BrokenExpressShipment("TRK-4", Decimal("1"), date(2026, 11, 9))
print(f"{vars(broken)=}")
print(raised_message(broken.label))

vars(broken)={'promised_by': datetime.date(2026, 11, 9)}
AttributeError: 'BrokenExpressShipment' object has no attribute 'tracking_id'


**`isinstance` and `issubclass`** follow the inheritance chain. Both accept a tuple
of classes or a `X | Y` union (3.10+). Note that `bool` is a subclass of `int`, a
classic surprise in validation code.

In [68]:
print(f"{isinstance(vase, FragileShipment)=}, {isinstance(vase, Shipment)=}")
print(f"{isinstance(vase, ExpressShipment)=}")
print(f"{issubclass(ExpressShipment, Shipment)=}, {issubclass(Shipment, ExpressShipment)=}")
print(f"{isinstance(vase, (ExpressShipment, FragileShipment))=}")
print(f"{isinstance(42, int | str)=}, {issubclass(bool, int)=}, {isinstance(True, int)=}")

isinstance(vase, FragileShipment)=True, isinstance(vase, Shipment)=True
isinstance(vase, ExpressShipment)=False
issubclass(ExpressShipment, Shipment)=True, issubclass(Shipment, ExpressShipment)=False
isinstance(vase, (ExpressShipment, FragileShipment))=True
isinstance(42, int | str)=True, issubclass(bool, int)=True, isinstance(True, int)=True


**Everything inherits from `object`.** A class with no bases gets `object` as its
only base, which is where default `__repr__`, `__eq__`, `__hash__`, `__init__` and
friends come from. `__subclasses__()` lists direct subclasses (it holds weak
references, so it reflects classes that still exist).

In [69]:
print(f"{Shipment.__bases__=}")
print(f"{mro_names(FragileShipment)=}")
print(f"{sorted(klass.__name__ for klass in Shipment.__subclasses__())=}")
print(f"{len(dir(object))=} attributes on object, e.g. {sorted(dir(object))[:5]}")
print(f"{issubclass(type, object)=}, {isinstance(object, type)=}")

Shipment.__bases__=(<class 'object'>,)
mro_names(FragileShipment)=['FragileShipment', 'Shipment', 'object']
sorted(klass.__name__ for klass in Shipment.__subclasses__())=['BrokenExpressShipment', 'ExpressShipment', 'FragileShipment']
len(dir(object))=24 attributes on object, e.g. ['__class__', '__delattr__', '__dir__', '__doc__', '__eq__']
issubclass(type, object)=True, isinstance(object, type)=True


### 2.2.2 `super()` in depth

**Zero-argument `super()`** is shorthand for `super(CurrentClass, self)`. The
compiler sees `super` (or `__class__`) inside a method and gives the method a hidden
closure cell named `__class__`, holding the class being defined (closures and cells:
1.11). At call time `super()` reads that cell and the method's first argument.

In [70]:
init_function = ExpressShipment.__init__
print(f"{init_function.__code__.co_freevars=}")
print(f"{init_function.__closure__[0].cell_contents=}")
print(f"{super(ExpressShipment, express).label()=}  (the explicit two-argument form)")

init_function.__code__.co_freevars=('__class__',)
init_function.__closure__[0].cell_contents=<class '__main__.ExpressShipment'>
super(ExpressShipment, express).label()='TRK-3: 1 kg, INR 40.00'  (the explicit two-argument form)


Because the cell is created by the class statement, `super()` in a function defined
**outside** a class fails, even if you attach the function to a class later.

In [71]:
def orphan_label(self: Shipment) -> str:
    """Uses super() but was not defined inside a class body."""
    return super().label()


Shipment.orphan_label = orphan_label
print(raised_message(lambda: Shipment("TRK-5", Decimal("1")).orphan_label()))
del Shipment.orphan_label

RuntimeError: super(): __class__ cell not found


**"Next in MRO", not "parent".** `super()` means: in the MRO of `type(self)`, find
the class *after* the one where this method is defined, and look the name up from
there. With single inheritance that is the parent. With multiple inheritance it can
be a **sibling**.

Below, `Left.setup` calls `super().setup(...)`. On a plain `Left` instance that
reaches `Base`. On a `Child` instance, whose MRO is `Child, Left, Right, Base`, the
very same line reaches **`Right`**, a class `Left` knows nothing about.

In [72]:
class Base:
    def setup(self, trace: list[str]) -> None:
        trace.append("Base")


class Left(Base):
    def setup(self, trace: list[str]) -> None:
        trace.append("Left")
        super().setup(trace)


class Right(Base):
    def setup(self, trace: list[str]) -> None:
        trace.append("Right")
        super().setup(trace)


class Child(Left, Right):
    def setup(self, trace: list[str]) -> None:
        trace.append("Child")
        super().setup(trace)


for instance in [Left(), Child()]:
    setup_trace: list[str] = []
    instance.setup(setup_trace)
    print(f"{type(instance).__name__:<5} MRO {mro_names(type(instance))}  calls {setup_trace}")
print(f"{super(Left, Child()).setup.__qualname__=}")

Left  MRO ['Left', 'Base', 'object']  calls ['Left', 'Base']
Child MRO ['Child', 'Left', 'Right', 'Base', 'object']  calls ['Child', 'Left', 'Right', 'Base']
super(Left, Child()).setup.__qualname__='Right.setup'


**Cooperative `**kwargs`.** When several classes in one hierarchy each need
constructor arguments, the pattern that works with any MRO is:

1. Every `__init__` takes its own arguments as **keyword-only** and the rest as
   `**kwargs`.
2. It removes what it needs and passes the rest on with `super().__init__(**kwargs)`.
3. Every class calls `super()`, so the chain reaches `object.__init__`, which must
   receive **nothing**: a leftover keyword is a bug and raises `TypeError`.

Notice that `CatalogItem` defines no `__init__` at all, yet both base classes run.

In [73]:
class Model:
    """Root of the cooperative chain."""

    def __init__(self, **kwargs: Any) -> None:
        super().__init__(**kwargs)  # object.__init__ must receive nothing


class Named(Model):
    def __init__(self, *, name: str, **kwargs: Any) -> None:
        self.name = name
        super().__init__(**kwargs)


class Priced(Model):
    def __init__(self, *, price: Decimal, **kwargs: Any) -> None:
        self.price = price
        super().__init__(**kwargs)


class CatalogItem(Named, Priced):
    """Gets both constructors through super(), in MRO order."""


tea = CatalogItem(name="Tea", price=Decimal("120"))
print(f"{vars(tea)=}, {mro_names(CatalogItem)=}")
print(raised_message(lambda: CatalogItem(name="Tea", price=Decimal("1"), colour="red")))

vars(tea)={'name': 'Tea', 'price': Decimal('120')}, mro_names(CatalogItem)=['CatalogItem', 'Named', 'Priced', 'Model', 'object']
TypeError: object.__init__() takes exactly one argument (the instance to initialize)


### 2.2.3 Multiple inheritance and the MRO

The **method resolution order** (MRO) is the list of classes Python searches for an
attribute. Since Python 2.3 it is computed with the **C3 linearization**, which
guarantees three things: a class comes before its bases, the order of bases in the
`class` line is kept, and every class's MRO is consistent with its bases' MROs.

**The rule.** For `class C(B1, B2, …, Bn)`:

`L[C] = C + merge(L[B1], L[B2], …, L[Bn], [B1, B2, …, Bn])`

`merge` repeatedly takes the **first head** (first element of a list) that does
**not appear in the tail** (any later position) of any list, appends it to the
result and removes it from the front of every list. If no head qualifies, the
hierarchy is inconsistent and Python raises `TypeError`.

**Worked example: a diamond.**

```text
class Vehicle: ...
class LandVehicle(Vehicle): ...
class WaterVehicle(Vehicle): ...
class Amphibious(LandVehicle, WaterVehicle): ...
```

| Step | Lists still to merge | Candidate heads | Taken | Why |
| --- | --- | --- | --- | --- |
| 0 | `[LandVehicle, Vehicle, object]`, `[WaterVehicle, Vehicle, object]`, `[LandVehicle, WaterVehicle]` | — | `Amphibious` | the class itself goes first |
| 1 | (same) | `LandVehicle` | `LandVehicle` | not in any tail |
| 2 | `[Vehicle, object]`, `[WaterVehicle, Vehicle, object]`, `[WaterVehicle]` | `Vehicle` | — | `Vehicle` is in the tail of the second list: skip |
| 3 | (same) | `WaterVehicle` | `WaterVehicle` | not in any tail |
| 4 | `[Vehicle, object]`, `[Vehicle, object]` | `Vehicle` | `Vehicle` | now only ever a head |
| 5 | `[object]`, `[object]` | `object` | `object` | done |

Result: `Amphibious, LandVehicle, WaterVehicle, Vehicle, object`. The shared base
`Vehicle` appears **once**, after both of its subclasses.

In [74]:
class Vehicle:
    def describe(self) -> str:
        return "vehicle"


class LandVehicle(Vehicle):
    def describe(self) -> str:
        return "drives on land"


class WaterVehicle(Vehicle):
    def describe(self) -> str:
        return "floats on water"


class Amphibious(LandVehicle, WaterVehicle):
    """Inherits describe() from the first class in its MRO that defines it."""


print(f"{mro_names(Amphibious)=}")
print(f"{Amphibious.mro() == list(Amphibious.__mro__) == list(inspect.getmro(Amphibious))=}")
print(f"{Amphibious().describe()=}")

mro_names(Amphibious)=['Amphibious', 'LandVehicle', 'WaterVehicle', 'Vehicle', 'object']
Amphibious.mro() == list(Amphibious.__mro__) == list(inspect.getmro(Amphibious))=True
Amphibious().describe()='drives on land'


#### Under the hood: C3 step by step

`c3_linearize` implements the rule above on class **names** and prints every step
for the top-level class. The last line checks it against Python's own `__mro__`.

In [75]:
def c3_linearize(
    class_name: str, bases_of: dict[str, list[str]], verbose: bool = True
) -> list[str]:
    """C3 linearization of class_name, given each class's direct bases by name."""
    if not bases_of[class_name]:
        return [class_name]
    lists = [c3_linearize(base, bases_of, False) for base in bases_of[class_name]]
    lists.append(list(bases_of[class_name]))
    result = [class_name]
    while any(lists):
        for candidate_list in filter(None, lists):
            head = candidate_list[0]
            if not any(head in other[1:] for other in lists):
                break
        else:
            raise TypeError(f"inconsistent hierarchy, stuck at {lists}")
        if verbose:
            print(f"merge {lists} -> take {head}")
        result.append(head)
        lists = [other[1:] if other and other[0] == head else other for other in lists]
    return result


vehicle_bases = {"object": [], "Vehicle": ["object"], "LandVehicle": ["Vehicle"]}
vehicle_bases |= {"WaterVehicle": ["Vehicle"], "Amphibious": ["LandVehicle", "WaterVehicle"]}
print(f"{c3_linearize('Amphibious', vehicle_bases) == mro_names(Amphibious)=}")

merge [['LandVehicle', 'Vehicle', 'object'], ['WaterVehicle', 'Vehicle', 'object'], ['LandVehicle', 'WaterVehicle']] -> take LandVehicle
merge [['Vehicle', 'object'], ['WaterVehicle', 'Vehicle', 'object'], ['WaterVehicle']] -> take WaterVehicle
merge [['Vehicle', 'object'], ['Vehicle', 'object'], []] -> take Vehicle
merge [['object'], ['object'], []] -> take object
c3_linearize('Amphibious', vehicle_bases) == mro_names(Amphibious)=True


**The diamond problem.** In C++ a diamond can give an object two copies of the
shared base. In Python the MRO lists `Vehicle` once, so with cooperative `super()`
its `__init__` runs **once**. Calling base classes by name instead
(`Vehicle.__init__(self)` from both branches) runs it **twice**, which is how
diamonds go wrong in practice.

In [76]:
class Engine:
    def __init__(self, init_log: list[str]) -> None:
        init_log.append("Engine")


class ByNamePetrol(Engine):
    def __init__(self, init_log: list[str]) -> None:
        Engine.__init__(self, init_log)  # explicit call by name


class ByNameElectric(Engine):
    def __init__(self, init_log: list[str]) -> None:
        Engine.__init__(self, init_log)


class ByNameHybrid(ByNamePetrol, ByNameElectric):
    def __init__(self, init_log: list[str]) -> None:
        ByNamePetrol.__init__(self, init_log)
        ByNameElectric.__init__(self, init_log)


by_name_log: list[str] = []
ByNameHybrid(by_name_log)
print(f"{by_name_log=}  (Engine initialized twice)")

by_name_log=['Engine', 'Engine']  (Engine initialized twice)


The same diamond with `super()` everywhere: `Engine` runs exactly once.

In [77]:
class Petrol(Engine):
    def __init__(self, init_log: list[str]) -> None:
        init_log.append("Petrol")
        super().__init__(init_log)


class Electric(Engine):
    def __init__(self, init_log: list[str]) -> None:
        init_log.append("Electric")
        super().__init__(init_log)


class Hybrid(Petrol, Electric):
    def __init__(self, init_log: list[str]) -> None:
        init_log.append("Hybrid")
        super().__init__(init_log)


super_log: list[str] = []
Hybrid(super_log)
print(f"{super_log=}")

super_log=['Hybrid', 'Petrol', 'Electric', 'Engine']


**Inconsistent MROs.** Some base lists cannot be linearized, and Python refuses to
create the class. Two common causes:

1. Listing a base **before its own subclass** (`class Bad(Vehicle, LandVehicle)`):
   `Vehicle` must come first (base order) and after `LandVehicle` (subclass first).
2. Two parents that order the same bases **differently**.

In [78]:
def define_class(name: str, bases: tuple[type, ...]) -> str:
    """Try to create a class; report its MRO or the error."""
    try:
        return f"{mro_names(type(name, bases, {}))}"
    except TypeError as error:
        return f"{type(error).__name__}: {error}"


class FirstOrder(LandVehicle, WaterVehicle): ...


class SecondOrder(WaterVehicle, LandVehicle): ...


print(define_class("Bad", (Vehicle, LandVehicle)))
print(define_class("Conflicted", (FirstOrder, SecondOrder)))
print(define_class("Fine", (LandVehicle, Vehicle)))

TypeError: Cannot create a consistent method resolution order (MRO) for bases Vehicle, LandVehicle
TypeError: Cannot create a consistent method resolution order (MRO) for bases LandVehicle, WaterVehicle
['Fine', 'LandVehicle', 'Vehicle', 'object']


**Predict the output.**

```python
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B+" + super().who()
class C(A):
    def who(self): return "C+" + super().who()
class D(C, B):
    pass

print(D().who())
print(B().who())
```

<details><summary>Answer</summary>

```text
C+B+A
B+A
```

`D`'s MRO is `D, C, B, A, object` (bases in the order written: `C` before `B`).
`D` has no `who`, so `C.who` runs; its `super()` is the next class after `C` in
**`D`'s** MRO, which is `B`, not `A`. On a plain `B` instance the MRO is
`B, A, object`, so `B`'s `super()` reaches `A`.

</details>

Run the next cell to check your prediction.

In [79]:
class A:
    def who(self) -> str:
        return "A"


class B(A):
    def who(self) -> str:
        return "B+" + super().who()


class C(A):
    def who(self) -> str:
        return "C+" + super().who()


class D(C, B):
    pass


print(D().who())
print(B().who())

C+B+A
B+A


### 2.2.4 Mixins

A **mixin** is a small class that adds one capability and is meant to be combined
with other classes, never used alone. Good mixins:

- do one thing (a `__repr__`, JSON export, timestamps);
- keep little or no state of their own, and if they need `__init__`, take
  `**kwargs` and call `super().__init__(**kwargs)` (2.2.2);
- are named `…Mixin` so readers know they are not standalone.

Three mixins first. `ReprMixin` builds a `__repr__` from `vars(self)`.

In [80]:
class ReprMixin:
    """Adds a __repr__ built from the instance attributes."""

    def __repr__(self) -> str:
        fields = ", ".join(f"{name}={value!r}" for name, value in vars(self).items())
        return f"{type(self).__name__}({fields})"

`JsonSerializableMixin` dumps the instance attributes. This simple version only
handles values `json` already understands; Lab 2.2 exercise 2 extends it to nested
objects and datetimes.

In [81]:
class JsonSerializableMixin:
    """Adds to_json() for objects whose attributes are plain JSON values."""

    def to_json(self) -> str:
        """Serialize vars(self) with json.dumps."""
        return json.dumps(vars(self), sort_keys=True)

`TimestampMixin` needs state, so it has a **cooperative** `__init__`. Its clock is a
class attribute, so tests (and this notebook) can swap in a fake clock and get
deterministic output.

In [82]:
class TimestampMixin:
    """Adds created_at / updated_at, set from a replaceable clock."""

    clock: ClassVar[Callable[[], datetime]] = datetime.now

    def __init__(self, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.created_at = self.clock()
        self.updated_at = self.created_at

    def touch(self) -> None:
        """Record a modification."""
        self.updated_at = self.clock()


fake_ticks = iter([datetime(2026, 11, 7, 9, 0), datetime(2026, 11, 7, 9, 45)])


def fake_now() -> datetime:
    """Return the next scripted time."""
    return next(fake_ticks)

**Combining them.** The rule for base order: **mixins on the left, the main base
class on the right**. The MRO then tries each mixin before the base, so a mixin can
add or override behaviour, and the base sits at the end of the cooperative
`__init__` chain.

In [83]:
class Entity:
    """The main base class: every stored object has an id."""

    def __init__(self, *, entity_id: int, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.entity_id = entity_id


class Product(ReprMixin, JsonSerializableMixin, TimestampMixin, Entity):
    """A product assembled from three mixins and one base class."""

    clock = staticmethod(fake_now)

    def __init__(self, *, sku: str, price_paise: int, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.sku = sku
        self.price_paise = price_paise


soap = Product(entity_id=7, sku="SOAP", price_paise=4_950)
soap.touch()
print(f"{mro_names(Product)=}")
print(soap)

mro_names(Product)=['Product', 'ReprMixin', 'JsonSerializableMixin', 'TimestampMixin', 'Entity', 'object']
Product(entity_id=7, created_at=datetime.datetime(2026, 11, 7, 9, 0), updated_at=datetime.datetime(2026, 11, 7, 9, 45), sku='SOAP', price_paise=4950)


The flat JSON mixin fails on the `datetime` attributes the timestamp mixin added:
`json` does not know how to encode them. A plain class with only JSON-friendly
values works fine. (Lab 2.2 exercise 2 fixes this.)

In [84]:
class Tag(JsonSerializableMixin):
    """Only JSON-friendly attributes."""

    def __init__(self, label: str, colour: str) -> None:
        self.label = label
        self.colour = colour


print(Tag("organic", "green").to_json())
print(raised_message(soap.to_json))

{"colour": "green", "label": "organic"}
TypeError: Object of type datetime is not JSON serializable


**Wrong order.** If the base class comes first and defines the same method, the
mixin never gets a chance: `AuditedEntity.__repr__` is found before
`ReprMixin.__repr__`.

In [85]:
class AuditedEntity(Entity):
    """A base class with its own __repr__."""

    def __repr__(self) -> str:
        return f"<AuditedEntity {self.entity_id}>"


class WrongOrder(AuditedEntity, ReprMixin):
    """Base first, mixin last: the mixin's __repr__ is shadowed."""


class RightOrder(ReprMixin, AuditedEntity):
    """Mixin first: the mixin's __repr__ wins."""


print(f"{mro_names(WrongOrder)=}")
print(f"{WrongOrder(entity_id=1)!r}  vs  {RightOrder(entity_id=1)!r}")

mro_names(WrongOrder)=['WrongOrder', 'AuditedEntity', 'Entity', 'ReprMixin', 'object']
<AuditedEntity 1>  vs  RightOrder(entity_id=1)


**Mixins in Django and DRF.** Web frameworks lean heavily on mixins. Django's
class-based views combine permission and behaviour mixins with a generic view, and
Django REST Framework builds its generic API views from one mixin per action:

```python
# Django: permission mixin first, generic view last
class ArticleListView(LoginRequiredMixin, ListView):
    model = Article

# Django REST Framework: one mixin per HTTP action, base view last
class ArticleList(mixins.ListModelMixin, mixins.CreateModelMixin, generics.GenericAPIView):
    queryset = Article.objects.all()
    serializer_class = ArticleSerializer

    def get(self, request, *args, **kwargs):
        return self.list(request, *args, **kwargs)   # from ListModelMixin

    def post(self, request, *args, **kwargs):
        return self.create(request, *args, **kwargs)  # from CreateModelMixin
```

Neither framework is a course dependency, so the cell below only checks whether
they are installed. The standard library uses the same pattern:
`socketserver.ThreadingTCPServer` is literally `ThreadingMixIn` + `TCPServer`.

In [86]:
for package_name in ["django", "rest_framework"]:
    if importlib.util.find_spec(package_name) is None:
        print(f"{package_name} is not installed: skipping (not needed for this course)")
    else:
        print(f"{package_name} is installed")
print(f"{mro_names(socketserver.ThreadingTCPServer)=}")

django is not installed: skipping (not needed for this course)
rest_framework is not installed: skipping (not needed for this course)
mro_names(socketserver.ThreadingTCPServer)=['ThreadingTCPServer', 'ThreadingMixIn', 'TCPServer', 'BaseServer', 'object']


### 2.2.5 Composition over inheritance

**Inheritance** models *is-a*: an `ExpressShipment` is a `Shipment` and can be used
anywhere a `Shipment` is expected. **Composition** models *has-a*: an object keeps
other objects as attributes and **delegates** work to them.

| Question | If yes |
| --- | --- |
| Is every subclass instance usable wherever the base is expected, with no surprises? | Inheritance may fit |
| Do you only want to *reuse some code*? | Composition |
| Would you need one subclass per *combination* of features? | Composition |
| Should the behaviour be swappable at run time or in tests? | Composition |

**Inheriting from a built-in exposes too much.** A "top 3" list that subclasses
`list` guards `append`, but every other mutating method (`extend`, `insert`, `+=`)
walks straight past the rule.

In [87]:
class TopThreeList(list[str]):
    """BUG: is-a list, so every list method can break the size rule."""

    def append(self, item: str) -> None:
        if len(self) >= 3:
            raise ValueError("top three is full")
        super().append(item)


podium = TopThreeList()
for runner in ["Asha", "Ravi", "Meena"]:
    podium.append(runner)
print(raised_message(lambda: podium.append("John")))
podium.extend(["John", "Fatima"])  # not guarded
podium.insert(0, "Arjun")  # not guarded
print(f"{podium=}, {len(podium)=}")

ValueError: top three is full
podium=['Arjun', 'Asha', 'Ravi', 'Meena', 'John', 'Fatima'], len(podium)=6


**Composition with explicit delegation.** `TopThree` *has* a list and exposes only
what it wants. The rule cannot be bypassed because there is no `extend` or
`insert` to call.

In [88]:
class TopThree:
    """Has a list; delegates only the safe operations to it."""

    def __init__(self) -> None:
        self._runners: list[str] = []

    def add(self, runner: str) -> None:
        """Add a finisher, refusing a fourth one."""
        if len(self._runners) >= 3:
            raise ValueError("top three is full")
        self._runners.append(runner)

    def __len__(self) -> int:
        return len(self._runners)

    def __iter__(self) -> Iterator[str]:
        return iter(self._runners)


podium = TopThree()
for runner in ["Asha", "Ravi", "Meena"]:
    podium.add(runner)
print(raised_message(lambda: podium.add("John")))
print(f"{list(podium)=}, {len(podium)=}, {hasattr(podium, 'extend')=}")

ValueError: top three is full
list(podium)=['Asha', 'Ravi', 'Meena'], len(podium)=3, hasattr(podium, 'extend')=False


**Automatic delegation with `__getattr__`.** Python calls `__getattr__` only when
normal lookup fails, so a wrapper can forward every unknown attribute to the object
it wraps (a proxy). Unit 2.5 covers attribute hooks in detail.

In [89]:
class RecordingProxy:
    """Forwards every unknown attribute to the wrapped object and records the name."""

    def __init__(self, target: object) -> None:
        self._target = target
        self.accessed: list[str] = []

    def __getattr__(self, name: str) -> Any:
        self.accessed.append(name)
        return getattr(self._target, name)


proxied_text = RecordingProxy("  Hello, Pune  ")
print(f"{proxied_text.strip().upper()=}")
print(f"{proxied_text.count('l')=}")
print(f"{proxied_text.accessed=}")

proxied_text.strip().upper()='HELLO, PUNE'
proxied_text.count('l')=2
proxied_text.accessed=['strip', 'count']


**Refactoring a `Notifier` hierarchy.** A notifier sends messages over a channel
(e-mail, SMS) and may add features (logging, retries). With inheritance every
combination needs its own class, and the feature code is copied into each one.

In [90]:
class Notifier:
    def send(self, recipient: str, message: str) -> str:
        raise NotImplementedError


class EmailNotifier(Notifier):
    def send(self, recipient: str, message: str) -> str:
        return f"email to {recipient}: {message}"


class SmsNotifier(Notifier):
    def send(self, recipient: str, message: str) -> str:
        return f"sms to {recipient}: {message}"


class LoggingEmailNotifier(EmailNotifier):
    def __init__(self, log: list[str]) -> None:
        self.log = log

    def send(self, recipient: str, message: str) -> str:
        self.log.append(f"sending to {recipient}")  # duplicated in every Logging* class
        return super().send(recipient, message)


notifier_log: list[str] = []
print(LoggingEmailNotifier(notifier_log).send("asha@example.com", "Order shipped"))
print(f"{notifier_log=}")

email to asha@example.com: Order shipped
notifier_log=['sending to asha@example.com']


The class count grows **multiplicatively**: every channel times every subset of
features. Adding one channel or one feature means writing many new classes.

In [91]:
def inheritance_class_count(channel_count: int, feature_count: int) -> int:
    """Classes needed for every channel with every subset of features."""
    return channel_count * 2**feature_count


channels = ["Email", "Sms"]
features = ["Logging", "Retrying"]
combinations = [
    "".join(chosen) + channel + "Notifier"
    for channel in channels
    for size in range(len(features) + 1)
    for chosen in itertools.combinations(features, size)
]
print(combinations)
for channel_count, feature_count in [(2, 2), (3, 2), (3, 3), (4, 4)]:
    print(
        f"{channel_count} channels, {feature_count} features -> "
        f"{inheritance_class_count(channel_count, feature_count)} classes"
    )

['EmailNotifier', 'LoggingEmailNotifier', 'RetryingEmailNotifier', 'LoggingRetryingEmailNotifier', 'SmsNotifier', 'LoggingSmsNotifier', 'RetryingSmsNotifier', 'LoggingRetryingSmsNotifier']
2 channels, 2 features -> 8 classes
3 channels, 2 features -> 12 classes
3 channels, 3 features -> 24 classes
4 channels, 4 features -> 64 classes


**The composition version.** Each channel is a small class with `send()`. Each
feature is a **wrapper** that has an inner channel and adds one behaviour (the
decorator pattern, as objects). The `Notifier` has a channel. Classes now grow
**additively** (channels + features), and combinations are built at run time.

A `FlakyChannel` fails a fixed number of times so the retry feature can be
demonstrated deterministically.

In [92]:
class EmailChannel:
    def send(self, recipient: str, message: str) -> str:
        return f"email to {recipient}: {message}"


class SmsChannel:
    def send(self, recipient: str, message: str) -> str:
        return f"sms to {recipient}: {message}"


class FlakyChannel:
    """Wraps a channel and raises ConnectionError for the first `failures` calls."""

    def __init__(self, inner: Any, failures: int) -> None:
        self.inner = inner
        self.failures_left = failures

    def send(self, recipient: str, message: str) -> str:
        if self.failures_left > 0:
            self.failures_left -= 1
            raise ConnectionError("gateway timeout")
        return self.inner.send(recipient, message)

The two feature wrappers. Each has an `inner` channel and adds one behaviour:

In [93]:
class LoggingChannel:
    """Feature: log each send attempt, then delegate."""

    def __init__(self, inner: Any, log: list[str]) -> None:
        self.inner = inner
        self.log = log

    def send(self, recipient: str, message: str) -> str:
        self.log.append(f"sending to {recipient}")
        return self.inner.send(recipient, message)


class RetryingChannel:
    """Feature: retry on ConnectionError, up to `attempts` tries in total."""

    def __init__(self, inner: Any, attempts: int) -> None:
        self.inner = inner
        self.attempts = attempts

    def send(self, recipient: str, message: str) -> str:
        for attempt in range(1, self.attempts + 1):
            try:
                return self.inner.send(recipient, message)
            except ConnectionError:
                if attempt == self.attempts:
                    raise
        raise AssertionError("unreachable")

The `Notifier` only has a channel. It knows nothing about logging or retries, so it
never changes when a feature is added.

In [94]:
class Notifier:
    """Has a channel; knows nothing about logging or retries."""

    def __init__(self, channel: Any) -> None:
        self.channel = channel

    def notify(self, recipient: str, message: str) -> str:
        return self.channel.send(recipient, message)

Any combination is now one expression. The SMS gateway fails twice; the retrying
wrapper tries three times; the logging wrapper (inside the retry) records every
attempt.

In [95]:
delivery_log: list[str] = []
reliable_sms = Notifier(
    RetryingChannel(
        LoggingChannel(FlakyChannel(SmsChannel(), failures=2), delivery_log), attempts=3
    )
)
print(reliable_sms.notify("+91-98xxxxxx01", "OTP 4821"))
print(f"{delivery_log=}")
print(Notifier(EmailChannel()).notify("asha@example.com", "Order shipped"))
print(
    raised_message(
        lambda: Notifier(RetryingChannel(FlakyChannel(EmailChannel(), 5), 2)).notify("x", "y")
    )
)

sms to +91-98xxxxxx01: OTP 4821
delivery_log=['sending to +91-98xxxxxx01', 'sending to +91-98xxxxxx01', 'sending to +91-98xxxxxx01']
email to asha@example.com: Order shipped
ConnectionError: gateway timeout


**The Square/Rectangle Liskov violation.** The **Liskov substitution principle**
(LSP) says code written for a base class must keep working, with no surprises, when
given a subclass. Mathematically a square *is a* rectangle, so `Square(Rectangle)`
looks natural. But a mutable `Rectangle` promises that setting `width` leaves
`height` alone, and a square cannot keep that promise.

In [96]:
class Rectangle:
    """A mutable rectangle: width and height change independently."""

    def __init__(self, width: float, height: float) -> None:
        self._width = width
        self._height = height

    @property
    def width(self) -> float:
        return self._width

    @width.setter
    def width(self, value: float) -> None:
        self._width = value

    @property
    def height(self) -> float:
        return self._height

    @height.setter
    def height(self, value: float) -> None:
        self._height = value

    @property
    def area(self) -> float:
        return self._width * self._height

`Square` keeps its sides equal by overriding both setters. `stretch_width` is
written against `Rectangle` and relies on its promise; it works for a rectangle and
silently breaks for a square, even though `isinstance(square, Rectangle)` is `True`.

In [97]:
class Square(Rectangle):
    """Keeps width == height, which breaks Rectangle's promise."""

    def __init__(self, side: float) -> None:
        super().__init__(side, side)

    @Rectangle.width.setter
    def width(self, value: float) -> None:
        self._width = self._height = value

    @Rectangle.height.setter
    def height(self, value: float) -> None:
        self._width = self._height = value


def stretch_width(shape: Rectangle, factor: float) -> str:
    """Client code: doubling the width should double the area."""
    expected_area = shape.area * factor
    shape.width *= factor
    return f"{type(shape).__name__}: expected {expected_area}, got {shape.area}"


print(stretch_width(Rectangle(2, 3), 2))
print(stretch_width(Square(3), 2))
print(f"{isinstance(Square(3), Rectangle)=}")

Rectangle: expected 12, got 12
Square: expected 18, got 36
isinstance(Square(3), Rectangle)=True


**One fix:** make the shapes immutable values. "Changing" the width returns a **new**
shape, so a square stretched in one direction honestly becomes a rectangle, and
`Square` no longer needs to inherit from `Rectangle` at all. Lab 2.3 builds shapes
this way behind a shared abstract base class.

In [98]:
@dataclass(frozen=True)
class FixedRectangle:
    width: float
    height: float

    def with_width(self, width: float) -> FixedRectangle:
        return FixedRectangle(width, self.height)


@dataclass(frozen=True)
class FixedSquare:
    side: float

    def with_width(self, width: float) -> FixedRectangle:
        return FixedRectangle(width, self.side)  # a stretched square is a rectangle


print(f"{FixedSquare(3).with_width(6)=}")
print(raised_error_name(lambda: setattr(FixedSquare(3), "side", 6)))

FixedSquare(3).with_width(6)=FixedRectangle(width=6, height=3)
FrozenInstanceError


### 2.2.6 SOLID in Python

SOLID is five design principles from the object-oriented world. In Python they
often take a lighter form (a function, a Protocol), but the ideas carry over.

| Letter | Principle | One-line meaning |
| --- | --- | --- |
| S | Single responsibility | A class should have one reason to change |
| O | Open/closed | Add behaviour by adding code, not by editing working code |
| L | Liskov substitution | Subclasses must honour the base class's promises |
| I | Interface segregation | Depend on small interfaces, not fat ones |
| D | Dependency inversion | Depend on abstractions; inject concrete implementations |

**S — Single responsibility.** An `Invoice` that computes totals, renders text and
saves itself changes for three unrelated reasons (tax rules, layout, storage).
Split it so each class has one.

In [99]:
class InvoiceCalculator:
    """Changes when tax rules change."""

    def total(self, line_amounts: list[Decimal], tax_rate_percent: Decimal) -> Decimal:
        return sum(line_amounts, Decimal("0")) * (1 + tax_rate_percent / 100)


class InvoiceRenderer:
    """Changes when the layout changes."""

    def render(self, invoice_number: str, total: Decimal) -> str:
        return f"Invoice {invoice_number}: INR {total:.2f}"


class InvoiceArchive:
    """Changes when storage changes."""

    def __init__(self) -> None:
        self.saved: dict[str, str] = {}

    def save(self, invoice_number: str, text: str) -> None:
        self.saved[invoice_number] = text


invoice_total = InvoiceCalculator().total([Decimal("100"), Decimal("250")], Decimal("18"))
invoice_archive = InvoiceArchive()
invoice_archive.save("INV-9", InvoiceRenderer().render("INV-9", invoice_total))
print(invoice_archive.saved)

{'INV-9': 'Invoice INV-9: INR 413.00'}

**O — Open/closed.** `checkout_total` applies whatever discount rules it is given.
A new rule is a new class; `checkout_total` never changes. (An `if rule == "…"`
chain would have to be edited for every new rule.)

In [100]:
class PercentOff:
    def __init__(self, percent: Decimal) -> None:
        self.percent = percent

    def apply(self, amount: Decimal) -> Decimal:
        return amount * (1 - self.percent / 100)


class FlatOff:
    def __init__(self, rupees: Decimal) -> None:
        self.rupees = rupees

    def apply(self, amount: Decimal) -> Decimal:
        return max(amount - self.rupees, Decimal("0"))


def checkout_total(amount: Decimal, rules: Iterable[Any]) -> Decimal:
    """Closed for modification: new rules plug in without editing this function."""
    for rule in rules:
        amount = rule.apply(amount)
    return amount


weekend_rules = [PercentOff(Decimal("10")), FlatOff(Decimal("50"))]
print(f"{checkout_total(Decimal('1000'), weekend_rules)=}")

checkout_total(Decimal('1000'), weekend_rules)=Decimal('850.0')

**L — Liskov substitution.** Besides Square/Rectangle (2.2.5), a classic violation
is a subclass that refuses an operation the base class promises. `warm_up` works for
any `Cache`, and crashes for the `ReadOnlyCache` "subtype". The fix is to not make
it a subclass (or to split the interface, as the next principle shows).

In [101]:
class Cache:
    def __init__(self) -> None:
        self._entries: dict[str, str] = {}

    def get(self, key: str) -> str | None:
        return self._entries.get(key)

    def set(self, key: str, value: str) -> None:
        self._entries[key] = value


class ReadOnlyCache(Cache):
    """BUG: strengthens the precondition of set() to 'never'."""

    def set(self, key: str, value: str) -> None:
        raise PermissionError("read-only cache")


def warm_up(cache: Cache) -> str | None:
    """Written against Cache: set, then get."""
    cache.set("home", "<html>…</html>")
    return cache.get("home")


print(f"{warm_up(Cache())=}")
print(raised_message(lambda: warm_up(ReadOnlyCache())))

warm_up(Cache())='<html>…</html>'
PermissionError: read-only cache


**I — Interface segregation.** A report viewer only reads, so it should depend on a
small "readable" interface, not on a fat storage interface with writes and deletes.
`typing.Protocol` (2.3.3) describes such interfaces; `PublicMirror` only implements
reading and still fits the viewer.

In [102]:
class Readable(Protocol):
    def read(self, key: str) -> str: ...


class Writable(Protocol):
    def write(self, key: str, value: str) -> None: ...


class ArchiveStore:
    """Implements both small interfaces."""

    def __init__(self) -> None:
        self.data = {"q3": "revenue up 12%"}

    def read(self, key: str) -> str:
        return self.data[key]

    def write(self, key: str, value: str) -> None:
        self.data[key] = value


class PublicMirror:
    """Read-only copy: implements only Readable."""

    def read(self, key: str) -> str:
        return "revenue up 12% (mirror)"


def show_report(store: Readable, key: str) -> str:
    """Needs reading only, so it asks for Readable."""
    return f"[{key}] {store.read(key)}"


print(show_report(ArchiveStore(), "q3"))
print(show_report(PublicMirror(), "q3"))

[q3] revenue up 12%
[q3] revenue up 12% (mirror)


**D — Dependency inversion.** `OrderService` does not create its own database
connection; it receives any object with `save` and `find`. Production passes a real
repository, tests pass an in-memory one. FastAPI's `Depends` (Phase 6) is built on
this idea.

In [103]:
class InMemoryOrderRepository:
    """A test double: stores orders in a dict."""

    def __init__(self) -> None:
        self.orders: dict[str, Decimal] = {}

    def save(self, order_id: str, amount: Decimal) -> None:
        self.orders[order_id] = amount

    def find(self, order_id: str) -> Decimal | None:
        return self.orders.get(order_id)


class OrderService:
    """High-level policy; the repository is injected, not constructed here."""

    def __init__(self, repository: Any) -> None:
        self.repository = repository

    def place(self, order_id: str, amount: Decimal) -> str:
        if self.repository.find(order_id) is not None:
            return f"{order_id} already placed"
        self.repository.save(order_id, amount)
        return f"{order_id} placed"


service = OrderService(InMemoryOrderRepository())
print(service.place("ORD-1", Decimal("499")), "|", service.place("ORD-1", Decimal("499")))

ORD-1 placed | ORD-1 already placed


### Lab 2.2

Three exercises. For each, replace every `raise NotImplementedError` in the stub,
then run the check cell. Until then the checks print FAIL; that is expected. Three
structural checks already pass on the stub ("mixin has no __init__" and "in the MRO"
in exercise 2, "Report has no base class" in exercise 3): keep them passing.

#### Exercise 1 — compute three MROs by hand

The cell below defines three hierarchies. For each, work out the MRO **on paper**
with `L[C] = C + merge(...)` (2.2.3), writing down every merge step, then make
`predict_mro(case_name)` return your answer as a list of class names (ending with
`"object"`). Do not call `__mro__` in your answer; the check compares it with
`__mro__` for you.

> **Note:** a failing check prints the real MRO. Finish all three on paper before
> running it.

In [104]:
# Case "diamond"
class Animal: ...


class Swimmer(Animal): ...


class Flyer(Animal): ...


class Duck(Swimmer, Flyer): ...


# Case "services"
class Base: ...


class LoggingMixin: ...


class CacheMixin: ...


class Service(LoggingMixin, CacheMixin, Base): ...


class AuditMixin(LoggingMixin): ...


class AuditedService(AuditMixin, Service): ...


# Case "classic"
class Persistent: ...


class Versioned: ...


class Searchable: ...


class Document(Persistent, Versioned): ...


class Index(Persistent, Searchable): ...


class SearchableDocument(Document, Index): ...


MRO_CASES: dict[str, type] = {
    "diamond": Duck,
    "services": AuditedService,
    "classic": SearchableDocument,
}


def predict_mro(case_name: str) -> list[str]:
    """Return the class names of the case's MRO, computed by hand."""
    raise NotImplementedError("Lab 2.2: write your three MROs in predict_mro")

Run the checks.

In [105]:
run_checks(
    [
        (case_name, lambda case_name=case_name: predict_mro(case_name), mro_names(cls))
        for case_name, cls in MRO_CASES.items()
    ]
)

FAIL  diamond: raised NotImplementedError: Lab 2.2: write your three MROs in predict_mro
FAIL  services: raised NotImplementedError: Lab 2.2: write your three MROs in predict_mro
FAIL  classic: raised NotImplementedError: Lab 2.2: write your three MROs in predict_mro
0/3 checks passed


#### Exercise 2 — a JSON mixin for nested objects and datetimes

Write `JsonMixin` with two methods and **no `__init__`**:

- `to_dict()` returns the instance's attributes (from `vars(self)`) **except** names
  starting with `_`, with each value converted recursively:

  | Value | Becomes |
  | --- | --- |
  | another `JsonMixin` object | its `to_dict()` |
  | `datetime` or `date` | `isoformat()` string |
  | `Decimal` | `str(value)` |
  | `list` or `tuple` | a list of converted items |
  | `dict` | a dict with `str` keys and converted values |
  | anything else | unchanged |

- `to_json(**dumps_options)` returns `json.dumps(self.to_dict(), **dumps_options)`.

The sample classes in the cell after the stub use the mixin.

In [106]:
class JsonMixin:
    """Adds to_dict() / to_json() to any class whose state lives in instance attributes."""

    def to_dict(self) -> dict[str, Any]:
        """Public attributes as JSON-ready values."""
        raise NotImplementedError("Lab 2.2: implement JsonMixin.to_dict")

    def to_json(self, **dumps_options: Any) -> str:
        """Serialize to_dict() with json.dumps."""
        raise NotImplementedError("Lab 2.2: implement JsonMixin.to_json")

Sample classes (given): a `Customer` with nested `Address` and `LineItem` objects,
a `datetime`, a `date`, a tuple, a dict holding another object, and a private
`_password_hash` that must not be exported.

In [107]:
class Address(JsonMixin):
    """A postal address."""

    def __init__(self, city: str, pin_code: str) -> None:
        self.city = city
        self.pin_code = pin_code


class LineItem(JsonMixin):
    """One purchased product."""

    def __init__(self, sku: str, unit_price: Decimal, quantity: int) -> None:
        self.sku = sku
        self.unit_price = unit_price
        self.quantity = quantity


class Customer(JsonMixin):
    """A customer with nested objects and a private field."""

    def __init__(
        self,
        name: str,
        joined_at: datetime,
        birthday: date,
        addresses: list[Address],
        last_order: list[LineItem],
        tags: tuple[str, ...],
        preferences: dict[str, object],
    ) -> None:
        self.name, self.joined_at, self.birthday = name, joined_at, birthday
        self.addresses, self.last_order = addresses, last_order
        self.tags, self.preferences = tags, preferences
        self._password_hash = "pbkdf2$not-for-json"


def make_sample_customer() -> Customer:
    """A fresh customer for the checks."""
    return Customer(
        name="Fatima",
        joined_at=datetime(2026, 1, 15, 9, 30),
        birthday=date(1994, 8, 2),
        addresses=[Address("Pune", "411001"), Address("Goa", "403001")],
        last_order=[LineItem("SOAP", Decimal("49.50"), 2)],
        tags=("vip", "early-adopter"),
        preferences={"newsletter": True, "home": Address("Pune", "411001")},
    )

Run the checks.

In [108]:
EXPECTED_CUSTOMER_DICT = {
    "name": "Fatima",
    "joined_at": "2026-01-15T09:30:00",
    "birthday": "1994-08-02",
    "addresses": [{"city": "Pune", "pin_code": "411001"}, {"city": "Goa", "pin_code": "403001"}],
    "last_order": [{"sku": "SOAP", "unit_price": "49.50", "quantity": 2}],
    "tags": ["vip", "early-adopter"],
    "preferences": {"newsletter": True, "home": {"city": "Pune", "pin_code": "411001"}},
}


def address_with_landmark() -> dict[str, Any]:
    """to_dict() sees attributes added after __init__."""
    address = Address("Goa", "403001")
    address.landmark = "Fort"  # type: ignore[attr-defined]
    return address.to_dict()


run_checks(
    [
        ("to_dict", lambda: make_sample_customer().to_dict(), EXPECTED_CUSTOMER_DICT),
        (
            "private field skipped",
            lambda: "_password_hash" in make_sample_customer().to_dict(),
            False,
        ),
        (
            "to_json round trip",
            lambda: json.loads(make_sample_customer().to_json()),
            EXPECTED_CUSTOMER_DICT,
        ),
        (
            "dumps options forwarded",
            lambda: Address("Pune", "411001").to_json(sort_keys=True, indent=None),
            '{"city": "Pune", "pin_code": "411001"}',
        ),
        (
            "indent forwarded",
            lambda: Address("Goa", "403001").to_json(indent=2).splitlines()[1],
            '  "city": "Goa",',
        ),
        (
            "attributes added later",
            address_with_landmark,
            {"city": "Goa", "pin_code": "403001", "landmark": "Fort"},
        ),
        ("mixin has no __init__", lambda: "__init__" in JsonMixin.__dict__, False),
        ("in the MRO", lambda: JsonMixin in Customer.__mro__, True),
    ]
)

FAIL  to_dict: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_dict
FAIL  private field skipped: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_dict
FAIL  to_json round trip: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_json
FAIL  dumps options forwarded: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_json
FAIL  indent forwarded: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_json
FAIL  attributes added later: raised NotImplementedError: Lab 2.2: implement JsonMixin.to_dict
PASS  mixin has no __init__
PASS  in the MRO
2/8 checks passed


#### Exercise 3 — refactor an inheritance tree into composition

The legacy code below picks a report **format** by subclassing and adds
**deliveries** (e-mail, archive) by subclassing again. It already needs five classes
for three combinations, `EmailedTextReport` duplicates `EmailedCsvReport`, and an
"archived text report" would need yet another class.

Rewrite it with composition:

- `CsvFormatter` and `TextFormatter`, each with `format(title, rows) -> str`
  producing exactly what `CsvReport.render` / `TextReport.render` produce.
- `EmailDelivery(outbox)` and `ArchiveDelivery(archive)`, each with
  `deliver(title, content) -> None` doing what the legacy `publish` overrides do.
- `Report(title, rows, formatter, deliveries=())` with no base class (other than
  `object`); `publish()` formats, calls every delivery **in order**, and returns
  the content. Store the deliveries as a list in `self.deliveries`.

First, the legacy code (given, working).

In [109]:
type SalesRows = list[tuple[str, int]]

SAMPLE_SALES: SalesRows = [("Soap", 120), ("Rice 5 kg", 45), ("Tea", 300)]


class LegacyReport:
    """Legacy base: subclasses pick a format by overriding render()."""

    def __init__(self, title: str, rows: SalesRows) -> None:
        self.title = title
        self.rows = rows

    def render(self) -> str:
        raise NotImplementedError  # subclasses override this

    def publish(self) -> str:
        return self.render()


class CsvReport(LegacyReport):
    def render(self) -> str:
        body = "\n".join(f"{product},{units}" for product, units in self.rows)
        return f"# {self.title}\nproduct,units\n{body}"


class TextReport(LegacyReport):
    def render(self) -> str:
        lines = [self.title, "=" * len(self.title)]
        lines += [f"{product:<12}{units:>5}" for product, units in self.rows]
        return "\n".join(lines)

The legacy delivery subclasses (given):

In [110]:
class EmailedCsvReport(CsvReport):
    def __init__(self, title: str, rows: SalesRows, outbox: list[tuple[str, str]]) -> None:
        super().__init__(title, rows)
        self.outbox = outbox

    def publish(self) -> str:
        content = super().publish()
        self.outbox.append((self.title, content))
        return content


class EmailedTextReport(TextReport):  # the same code as EmailedCsvReport, duplicated
    def __init__(self, title: str, rows: SalesRows, outbox: list[tuple[str, str]]) -> None:
        super().__init__(title, rows)
        self.outbox = outbox

    def publish(self) -> str:
        content = super().publish()
        self.outbox.append((self.title, content))
        return content


class ArchivedEmailedCsvReport(EmailedCsvReport):
    def __init__(
        self, title: str, rows: SalesRows, outbox: list[tuple[str, str]], archive: dict[str, str]
    ) -> None:
        super().__init__(title, rows, outbox)
        self.archive = archive

    def publish(self) -> str:
        content = super().publish()
        self.archive[self.title] = content
        return content


print(TextReport("Weekly sales", SAMPLE_SALES).publish())

Weekly sales
Soap          120
Rice 5 kg      45
Tea           300

Your composition-based version:

In [111]:
class CsvFormatter:
    def format(self, title: str, rows: SalesRows) -> str:
        """Same output as CsvReport.render."""
        raise NotImplementedError("Lab 2.2: implement CsvFormatter.format")


class TextFormatter:
    def format(self, title: str, rows: SalesRows) -> str:
        """Same output as TextReport.render."""
        raise NotImplementedError("Lab 2.2: implement TextFormatter.format")


class EmailDelivery:
    def __init__(self, outbox: list[tuple[str, str]]) -> None:
        raise NotImplementedError("Lab 2.2: implement EmailDelivery")

    def deliver(self, title: str, content: str) -> None:
        raise NotImplementedError("Lab 2.2: implement EmailDelivery.deliver")


class ArchiveDelivery:
    def __init__(self, archive: dict[str, str]) -> None:
        raise NotImplementedError("Lab 2.2: implement ArchiveDelivery")

    def deliver(self, title: str, content: str) -> None:
        raise NotImplementedError("Lab 2.2: implement ArchiveDelivery.deliver")


class Report:
    def __init__(
        self, title: str, rows: SalesRows, formatter: Any, deliveries: Iterable[Any] = ()
    ) -> None:
        raise NotImplementedError("Lab 2.2: implement Report")

    def publish(self) -> str:
        raise NotImplementedError("Lab 2.2: implement Report.publish")

Run the checks. They compare every output and side effect with the legacy classes.

In [112]:
def legacy_outputs() -> tuple[str, str, list[tuple[str, str]], dict[str, str]]:
    """Publish through the legacy tree and capture every side effect."""
    outbox: list[tuple[str, str]] = []
    archive: dict[str, str] = {}
    text_content = EmailedTextReport("Weekly sales", SAMPLE_SALES, outbox).publish()
    csv_content = ArchivedEmailedCsvReport("Weekly sales", SAMPLE_SALES, outbox, archive).publish()
    return text_content, csv_content, outbox, archive


def composed_outputs() -> tuple[str, str, list[tuple[str, str]], dict[str, str]]:
    """Publish the same two reports through Report + formatters + deliveries."""
    outbox: list[tuple[str, str]] = []
    archive: dict[str, str] = {}
    text_content = Report(
        "Weekly sales", SAMPLE_SALES, TextFormatter(), [EmailDelivery(outbox)]
    ).publish()
    csv_deliveries = [EmailDelivery(outbox), ArchiveDelivery(archive)]
    csv_content = Report("Weekly sales", SAMPLE_SALES, CsvFormatter(), csv_deliveries).publish()
    return text_content, csv_content, outbox, archive


def new_combination() -> tuple[str, dict[str, str]]:
    """Archived text report: a combination the legacy tree has no class for."""
    archive: dict[str, str] = {}
    content = Report("Stock", [("Tea", 7)], TextFormatter(), [ArchiveDelivery(archive)]).publish()
    return content, archive


run_checks(
    [
        (
            "plain CSV",
            lambda: Report("Q3", SAMPLE_SALES, CsvFormatter()).publish(),
            CsvReport("Q3", SAMPLE_SALES).publish(),
        ),
        (
            "plain text",
            lambda: Report("Q3", SAMPLE_SALES, TextFormatter()).publish(),
            TextReport("Q3", SAMPLE_SALES).publish(),
        ),
        ("same side effects as legacy", composed_outputs, legacy_outputs()),
        (
            "new combination, no new class",
            new_combination,
            ("Stock\n=====\nTea             7", {"Stock": "Stock\n=====\nTea             7"}),
        ),
        ("Report has no base class", lambda: Report.__bases__, (object,)),
        ("deliveries default to none", lambda: Report("Q3", [], CsvFormatter()).deliveries, []),
    ]
)

FAIL  plain CSV: raised NotImplementedError: Lab 2.2: implement Report


FAIL  plain text: raised NotImplementedError: Lab 2.2: implement Report
FAIL  same side effects as legacy: raised NotImplementedError: Lab 2.2: implement EmailDelivery
FAIL  new combination, no new class: raised NotImplementedError: Lab 2.2: implement ArchiveDelivery
PASS  Report has no base class
FAIL  deliveries default to none: raised NotImplementedError: Lab 2.2: implement Report
1/6 checks passed


### Recap — Unit 2.2

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Overriding | A subclass method replaces the base one; `self.method()` dispatches on the instance's class | method overriding |
| Extending | Call the original with `super().method(...)`; forgetting `super().__init__` fails silently | `super.method()` |
| `@override` | Marks intent; type checkers catch typos (3.12+) | TS `override` |
| `isinstance` / `issubclass` | Follow the MRO; accept tuples and `A | B` | `instanceof` |
| `object` | The root of every class; source of default dunders | `Object.prototype` |
| Zero-arg `super()` | Reads the hidden `__class__` cell and the first argument | `[[HomeObject]]` |
| Next in MRO | `super()` can reach a sibling under multiple inheritance | — (always the static parent) |
| Cooperative `**kwargs` | Keyword-only args, take yours, pass the rest, `object` gets nothing | — |
| C3 MRO | `L[C] = C + merge(...)`; take the first head not in any tail | prototype chain |
| Inconsistent MRO | Base listed before its subclass, or conflicting orders → `TypeError` | — |
| Mixins | One capability each, mixins left, main base right | `Object.assign`, class-factory mixins |
| Composition | Has-a + delegation; wrappers combine features additively | object composition |
| Liskov | Subclasses must keep the base class's promises (Square/Rectangle) | same |
| SOLID | S, O, L, I, D: small focused classes, plug-in extension, injected dependencies | same |

**Further reading**

- [Tutorial: inheritance](https://docs.python.org/3/tutorial/classes.html#inheritance),
  [multiple inheritance](https://docs.python.org/3/tutorial/classes.html#multiple-inheritance)
- [`super`](https://docs.python.org/3/library/functions.html#super),
  [`isinstance`](https://docs.python.org/3/library/functions.html#isinstance),
  [`issubclass`](https://docs.python.org/3/library/functions.html#issubclass),
  [`typing.override`](https://docs.python.org/3/library/typing.html#typing.override)
- [The Python 2.3 Method Resolution Order](https://docs.python.org/3/howto/mro.html)
  (the C3 algorithm, with worked examples)
- [`socketserver` mix-in classes](https://docs.python.org/3/library/socketserver.html#server-objects),
  [`__getattr__`](https://docs.python.org/3/reference/datamodel.html#object.__getattr__)
- [PEP 3135 — New `super`](https://peps.python.org/pep-3135/),
  [PEP 698 — `@override`](https://peps.python.org/pep-0698/),
  [PEP 253 — Subtyping built-in types](https://peps.python.org/pep-0253/)

## Unit 2.3 — Abstractions: ABCs, `collections.abc` and Protocols

**Objectives**

- Define abstract base classes with abstract methods and properties, and use the
  template-method pattern.
- Implement a few methods of a `collections.abc` class and get the rest for free;
  use `register` and `__subclasshook__`.
- Describe interfaces structurally with `typing.Protocol`, as in TypeScript, and
  know what `@runtime_checkable` can and cannot check.
- Choose between an ABC and a Protocol.
- Write polymorphic code with duck typing and EAFP instead of type checks.

**JS / TS bridge**

| JavaScript / TypeScript | Python | The difference that bites |
| --- | --- | --- |
| TS `abstract class` / `abstract method()` | `class X(ABC)` + `@abstractmethod` | Checked at **run time**, when you instantiate, not at compile time |
| TS `interface` (structural) | `typing.Protocol` | Checked by a type checker (mypy, pyright); ignored at run time by default |
| TS `implements Shape` | optional: `class Circle(Shape)` with a Protocol or ABC base | Not needed for Protocols: matching the shape is enough |
| `x instanceof SomeInterface` (impossible: interfaces are erased) | `isinstance(x, Proto)` with `@runtime_checkable` | Only checks that the **names** exist |
| `Symbol.hasInstance` | `__subclasshook__` / `ABC.register` | Customizes `isinstance` / `issubclass` |
| `Symbol.iterator` protocol | `collections.abc.Iterable` (`__iter__`) | Built-in ABCs recognize the protocol structurally |
| `ReadonlyMap<K, V>` / `Map<K, V>` | `Mapping[K, V]` / `MutableMapping[K, V]` | Implement 3 (or 5) methods, inherit the rest |
| `typeof x.close === "function"` | `try: x.close()` / `except AttributeError:` | EAFP over checks (2.3.4) |

The next cell is this unit's setup.

In [113]:
import functools
import importlib.util
import inspect
import io
import json
import os
import shutil
import sys
import timeit
import typing
from abc import ABC, ABCMeta, abstractmethod
from collections.abc import (
    Callable,
    Hashable,
    Iterable,
    Iterator,
    Mapping,
    MutableMapping,
    MutableSequence,
    Sequence,
    Set,
    Sized,
)
from decimal import Decimal
from math import pi  # noqa: F401 (Lab 2.3)
from typing import Any, NamedTuple, Protocol, runtime_checkable


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.3.1 Abstract base classes

An **abstract base class** (ABC) declares methods that subclasses **must** provide.
Inherit from `abc.ABC` and decorate those methods with `@abstractmethod`. Python then
refuses to instantiate the ABC itself, or any subclass that leaves an abstract method
unimplemented. An ABC can also have ordinary (concrete) methods built on top of the
abstract ones; here `exists()` works for every storage backend.

In [114]:
class Storage(ABC):
    """Where uploaded files go. Subclasses decide how."""

    @abstractmethod
    def save(self, name: str, data: bytes) -> str:
        """Store data and return its location."""

    @abstractmethod
    def load(self, location: str) -> bytes:
        """Return the stored bytes, or raise KeyError."""

    def exists(self, location: str) -> bool:
        """Concrete method built on the abstract load()."""
        try:
            self.load(location)
        except KeyError:
            return False
        return True

The error comes at **instantiation time**: defining an incomplete subclass is fine,
creating an instance of it is not. The message names every missing method. A
complete subclass works and inherits `exists()`.

In [115]:
class HalfDoneStorage(Storage):
    """Implements save() but forgets load()."""

    def save(self, name: str, data: bytes) -> str:
        return name


class MemoryStorage(Storage):
    """Keeps files in a dict."""

    def __init__(self) -> None:
        self._files: dict[str, bytes] = {}

    def save(self, name: str, data: bytes) -> str:
        location = f"mem://{name}"
        self._files[location] = data
        return location

    def load(self, location: str) -> bytes:
        return self._files[location]


print(raised_message(Storage))
print(raised_message(HalfDoneStorage))
memory_storage = MemoryStorage()
invoice_location = memory_storage.save("invoice.pdf", b"%PDF-1.7")
print(
    f"{invoice_location=}, {memory_storage.exists(invoice_location)=}, {memory_storage.exists('mem://x')=}"
)

TypeError: Can't instantiate abstract class Storage without an implementation for abstract methods 'load', 'save'
TypeError: Can't instantiate abstract class HalfDoneStorage without an implementation for abstract method 'load'
invoice_location='mem://invoice.pdf', memory_storage.exists(invoice_location)=True, memory_storage.exists('mem://x')=False


#### Under the hood: `ABCMeta` and `__abstractmethods__`

`ABC` is just a convenience base whose metaclass is **`ABCMeta`**. When the class is
created, `ABCMeta` collects every name still marked abstract into the frozenset
`__abstractmethods__`; `object.__new__` refuses to instantiate a class whose set is
not empty. `inspect.isabstract` reads the same information.

In [116]:
print(f"{type(Storage)=}")
print(f"{sorted(Storage.__abstractmethods__)=}")
print(f"{sorted(HalfDoneStorage.__abstractmethods__)=}")
print(f"{MemoryStorage.__abstractmethods__=}")
print(f"{inspect.isabstract(Storage)=}, {inspect.isabstract(MemoryStorage)=}")


class ExplicitMetaclass(metaclass=ABCMeta):  # the same as inheriting from ABC
    @abstractmethod
    def run(self) -> None: ...


print(f"{inspect.isabstract(ExplicitMetaclass)=}")

type(Storage)=<class 'abc.ABCMeta'>
sorted(Storage.__abstractmethods__)=['load', 'save']
sorted(HalfDoneStorage.__abstractmethods__)=['load']
MemoryStorage.__abstractmethods__=frozenset()
inspect.isabstract(Storage)=True, inspect.isabstract(MemoryStorage)=False
inspect.isabstract(ExplicitMetaclass)=True


Unlike Java interfaces, an abstract method **can have a body**. Subclasses must still
override it, but they can call the shared part with `super()`.

In [117]:
class Uploader(ABC):
    @abstractmethod
    def validate(self, file_name: str) -> list[str]:
        """Checks every uploader shares; subclasses extend them."""
        return [] if file_name else ["empty file name"]


class ImageUploader(Uploader):
    def validate(self, file_name: str) -> list[str]:
        problems = super().validate(file_name)
        if not file_name.endswith((".png", ".jpg")):
            problems.append("not an image")
        return problems


print(f"{ImageUploader().validate('logo.png')=}")
print(f"{ImageUploader().validate('')=}")

ImageUploader().validate('logo.png')=[]
ImageUploader().validate('')=['empty file name', 'not an image']


**Abstract properties and class methods.** Stack `@abstractmethod` **under**
`@property` (or `@classmethod`), so `abstractmethod` is applied first. A subclass can
satisfy an abstract property with a real property or simply with a class attribute
of the same name.

In [118]:
class Backend(ABC):
    @property
    @abstractmethod
    def scheme(self) -> str:
        """URL scheme this backend handles, e.g. "s3"."""

    @classmethod
    @abstractmethod
    def from_url(cls, url: str) -> Backend:
        """Build a backend from a URL."""


class S3Backend(Backend):
    def __init__(self, bucket: str) -> None:
        self.bucket = bucket

    @property
    def scheme(self) -> str:
        return "s3"

    @classmethod
    def from_url(cls, url: str) -> S3Backend:
        return cls(url.removeprefix("s3://").split("/")[0])


class LocalBackend(Backend):
    scheme = "file"  # a plain class attribute also satisfies the abstract property

    @classmethod
    def from_url(cls, url: str) -> LocalBackend:
        return cls()


print(f"{S3Backend.from_url('s3://invoices/2026/11').bucket=}, {S3Backend('x').scheme=}")
print(f"{LocalBackend.from_url('file:///tmp').scheme=}, {sorted(Backend.__abstractmethods__)=}")

S3Backend.from_url('s3://invoices/2026/11').bucket='invoices', S3Backend('x').scheme='s3'
LocalBackend.from_url('file:///tmp').scheme='file', sorted(Backend.__abstractmethods__)=['from_url', 'scheme']


**The template-method pattern.** The base class owns the **algorithm** (a concrete
method that calls steps in a fixed order); subclasses fill in the **steps**. Here
`OrderImporter.run` always parses, cleans, validates and collects errors; the CSV
and JSON importers only say how to parse and what makes a record valid. `clean` is
an optional **hook** with a default.

In [119]:
class OrderImporter(ABC):
    """Template method: run() fixes the steps, subclasses implement them."""

    def run(self, raw_text: str) -> tuple[list[dict[str, Any]], list[str]]:
        accepted: list[dict[str, Any]] = []
        errors: list[str] = []
        for record in self.parse(raw_text):
            record = self.clean(record)
            problem = self.validate(record)
            if problem is None:
                accepted.append(record)
            else:
                errors.append(problem)
        return accepted, errors

    @abstractmethod
    def parse(self, raw_text: str) -> list[dict[str, Any]]:
        """Turn raw text into records."""

    def clean(self, record: dict[str, Any]) -> dict[str, Any]:
        """Optional hook: normalize a record (default: unchanged)."""
        return record

    @abstractmethod
    def validate(self, record: dict[str, Any]) -> str | None:
        """Return an error message, or None when the record is valid."""

Two importers that only supply the steps:

In [120]:
class CsvOrderImporter(OrderImporter):
    def parse(self, raw_text: str) -> list[dict[str, Any]]:
        rows = [line.split(",") for line in raw_text.strip().splitlines()[1:]]
        return [{"order_id": order_id, "quantity": quantity} for order_id, quantity in rows]

    def clean(self, record: dict[str, Any]) -> dict[str, Any]:
        return {**record, "quantity": int(record["quantity"])}

    def validate(self, record: dict[str, Any]) -> str | None:
        if record["quantity"] > 0:
            return None
        return f"{record['order_id']}: quantity must be positive"


class JsonOrderImporter(OrderImporter):
    def parse(self, raw_text: str) -> list[dict[str, Any]]:
        return json.loads(raw_text)

    def validate(self, record: dict[str, Any]) -> str | None:
        return None if "order_id" in record else "record without order_id"


print(CsvOrderImporter().run("order_id,quantity\nORD-1,2\nORD-2,0"))
print(JsonOrderImporter().run('[{"order_id": "ORD-3", "quantity": 1}, {"quantity": 4}]'))

([{'order_id': 'ORD-1', 'quantity': 2}], ['ORD-2: quantity must be positive'])
([{'order_id': 'ORD-3', 'quantity': 1}], ['record without order_id'])


### 2.3.2 `collections.abc`

The `collections.abc` module defines ABCs for Python's container protocols. Each
one has a few **abstract methods** you write, and **mixin methods** it implements for
you on top of them. Inheriting also makes `isinstance(obj, Mapping)` (and so on)
true, so code that accepts "any mapping" accepts your class.

| ABC | You write | You get for free |
| --- | --- | --- |
| `Iterable` | `__iter__` | (recognized structurally, see `__subclasshook__` below) |
| `Iterator` | `__next__` | `__iter__` (returns `self`) |
| `Sequence` | `__getitem__`, `__len__` | `__contains__`, `__iter__`, `__reversed__`, `index`, `count` |
| `Mapping` | `__getitem__`, `__iter__`, `__len__` | `__contains__`, `keys`, `items`, `values`, `get`, `__eq__`, `__ne__` |
| `MutableMapping` | the `Mapping` three + `__setitem__`, `__delitem__` | the `Mapping` mixins + `pop`, `popitem`, `clear`, `update`, `setdefault` |
| `Set` | `__contains__`, `__iter__`, `__len__` | `<=`, `<`, `==`, `!=`, `>`, `>=`, `&`, `|`, `-`, `^`, `isdisjoint` |

> **Note:** `Set` here is `collections.abc.Set`, the abstract read-only set, not the
> built-in `set` type.

#### `Iterable`

Any class with `__iter__` counts as `Iterable`, without inheriting or registering.
A class that only has the old `__getitem__` protocol can still be iterated by
`iter()`, but `isinstance(…, Iterable)` says `False`. To ask "can I iterate this?",
call `iter()` and catch `TypeError` (EAFP, 2.3.4).

In [121]:
class Playlist:
    """Iterable by defining __iter__; no base class."""

    def __init__(self, *songs: str) -> None:
        self._songs = list(songs)

    def __iter__(self) -> Iterator[str]:
        return iter(self._songs)


class LegacyPlaylist:
    """Only the old __getitem__ protocol."""

    def __getitem__(self, index: int) -> str:
        return ["Raga Yaman", "Raga Bhairav"][index]


print(f"{isinstance(Playlist('Raga Yaman'), Iterable)=}")
print(f"{list(LegacyPlaylist())=}, {isinstance(LegacyPlaylist(), Iterable)=}")
print(raised_message(lambda: iter(42)))

isinstance(Playlist('Raga Yaman'), Iterable)=True
list(LegacyPlaylist())=['Raga Yaman', 'Raga Bhairav'], isinstance(LegacyPlaylist(), Iterable)=False
TypeError: 'int' object is not iterable


#### `Iterator`

Inherit from `Iterator` and write only `__next__`; the ABC supplies the
`__iter__` that returns `self` (1.13).

In [122]:
class TicketDispenser(Iterator[int]):
    """Hands out ticket numbers first..last."""

    def __init__(self, first: int, last: int) -> None:
        self._next_number = first
        self._last = last

    def __next__(self) -> int:
        if self._next_number > self._last:
            raise StopIteration
        number = self._next_number
        self._next_number += 1
        return number


dispenser = TicketDispenser(101, 104)
print(f"{iter(dispenser) is dispenser=}, {'__iter__' in vars(TicketDispenser)=}")
print(f"{list(dispenser)=}, {list(dispenser)=}  (exhausted)")

iter(dispenser) is dispenser=True, '__iter__' in vars(TicketDispenser)=False
list(dispenser)=[101, 102, 103, 104], list(dispenser)=[]  (exhausted)


#### `Sequence`

`PageRange` describes the page boundaries for paginating `total_items` results.
It computes each page on demand instead of storing a list, writes only `__len__`
and `__getitem__` (with negative indexes and slices), and gets `in`, iteration,
`reversed`, `index` and `count` from `Sequence`.

In [123]:
class PageRange(Sequence[tuple[int, int]]):
    """(start, stop) item offsets for each page of total_items results."""

    def __init__(self, total_items: int, page_size: int) -> None:
        self.total_items = total_items
        self.page_size = page_size

    def __len__(self) -> int:
        return -(-self.total_items // self.page_size)  # ceiling division

    def __getitem__(self, index: int | slice) -> Any:
        if isinstance(index, slice):
            return [self[position] for position in range(*index.indices(len(self)))]
        if index < 0:
            index += len(self)
        if not 0 <= index < len(self):
            raise IndexError("page out of range")
        start = index * self.page_size
        return start, min(start + self.page_size, self.total_items)

Every line below except the first two uses a method `PageRange` never wrote.

In [124]:
pages = PageRange(total_items=45, page_size=10)
print(f"{len(pages)=}, {pages[-1]=}, {pages[1:3]=}")
print(f"{list(pages)=}")
print(f"{(40, 45) in pages=}, {pages.index((20, 30))=}, {pages.count((0, 10))=}")
print(f"{list(reversed(pages))[:2]=}")
print(f"{isinstance(pages, Sequence)=}")

len(pages)=5, pages[-1]=(40, 45), pages[1:3]=[(10, 20), (20, 30)]
list(pages)=[(0, 10), (10, 20), (20, 30), (30, 40), (40, 45)]
(40, 45) in pages=True, pages.index((20, 30))=2, pages.count((0, 10))=1
list(reversed(pages))[:2]=[(40, 45), (30, 40)]
isinstance(pages, Sequence)=True


#### `Mapping`

`EnvironmentConfig` exposes the environment variables that share a prefix, with the
prefix removed and the keys lower-cased. Three methods are enough for `get`, `in`,
`keys`, `items`, `values`, `==` and `dict(...)` to work. The real `os.environ` is
itself a `MutableMapping`; the demo uses a fake environment so the output is the
same on every machine.

In [125]:
class EnvironmentConfig(Mapping[str, str]):
    """Settings from prefixed environment variables, e.g. APP_PORT -> "port"."""

    def __init__(self, environ: Mapping[str, str], prefix: str) -> None:
        self._values = {
            name.removeprefix(prefix).lower(): value
            for name, value in environ.items()
            if name.startswith(prefix)
        }

    def __getitem__(self, key: str) -> str:
        return self._values[key]

    def __iter__(self) -> Iterator[str]:
        return iter(self._values)

    def __len__(self) -> int:
        return len(self._values)


fake_environ = {"APP_DEBUG": "1", "APP_PORT": "8000", "HOME": "/home/asha"}
app_config = EnvironmentConfig(fake_environ, prefix="APP_")
print(f"{dict(app_config)=}, {app_config.get('port')=}, {app_config.get('host', 'localhost')=}")
print(f"{'debug' in app_config=}, {app_config == {'debug': '1', 'port': '8000'}=}")
print(f"{isinstance(os.environ, MutableMapping)=}")

dict(app_config)={'debug': '1', 'port': '8000'}, app_config.get('port')='8000', app_config.get('host', 'localhost')='localhost'
'debug' in app_config=True, app_config == {'debug': '1', 'port': '8000'}=True
isinstance(os.environ, MutableMapping)=True


#### `MutableMapping`

HTTP header names are case-insensitive. `CaseInsensitiveDict` stores each entry
under the lower-cased key but remembers the original spelling. Five methods give it
`update`, `setdefault`, `pop`, `popitem` and `clear` too; the constructor itself
uses the free `update`.

In [126]:
class CaseInsensitiveDict(MutableMapping[str, str]):
    """A str -> str mapping whose keys ignore case (like HTTP headers)."""

    def __init__(self, initial: Mapping[str, str] | None = None) -> None:
        self._store: dict[str, tuple[str, str]] = {}
        if initial:
            self.update(initial)  # update() is a free mixin method

    def __setitem__(self, key: str, value: str) -> None:
        self._store[key.lower()] = (key, value)

    def __getitem__(self, key: str) -> str:
        return self._store[key.lower()][1]

    def __delitem__(self, key: str) -> None:
        del self._store[key.lower()]

    def __iter__(self) -> Iterator[str]:
        return (original_key for original_key, _ in self._store.values())

    def __len__(self) -> int:
        return len(self._store)


headers = CaseInsensitiveDict({"Content-Type": "application/json"})
headers.setdefault("X-Request-ID", "req-42")
print(f"{headers['CONTENT-TYPE']=}, {'x-request-id' in headers=}, {list(headers)=}")
print(f"{headers.pop('content-type')=}, {dict(headers)=}")

headers['CONTENT-TYPE']='application/json', 'x-request-id' in headers=True, list(headers)=['Content-Type', 'X-Request-ID']
headers.pop('content-type')='application/json', dict(headers)={'X-Request-ID': 'req-42'}


#### `Set`

A read-only set of permissions. Three methods give it every comparison and set
operator. Operators build their result with the class method `_from_iterable`,
which by default calls `cls(iterable)`; that works here because the constructor
accepts an iterable (otherwise you would override `_from_iterable`).

In [127]:
class PermissionSet(Set[str]):
    """An immutable set of permission names."""

    def __init__(self, permissions: Iterable[str] = ()) -> None:
        self._permissions = frozenset(permissions)

    def __contains__(self, permission: object) -> bool:
        return permission in self._permissions

    def __iter__(self) -> Iterator[str]:
        return iter(sorted(self._permissions))

    def __len__(self) -> int:
        return len(self._permissions)


editor = PermissionSet({"read", "write"})
admin = PermissionSet({"read", "write", "delete"})
print(f"{editor <= admin=}, {editor < admin=}, {editor == PermissionSet(['write', 'read'])=}")
print(f"{list(admin - editor)=}, {list(editor | {'comment'})=}, {type(admin & editor).__name__=}")
print(f"{editor.isdisjoint({'delete'})=}")

editor <= admin=True, editor < admin=True, editor == PermissionSet(['write', 'read'])=True
list(admin - editor)=['delete'], list(editor | {'comment'})=['comment', 'read', 'write'], type(admin & editor).__name__='PermissionSet'
editor.isdisjoint({'delete'})=True


#### Mixin methods for free, counted

`free_methods` lists what each ABC implements on top of its abstract methods, by
walking the ABC's MRO (an `inspect`-style look at the class dictionaries).

In [128]:
def free_methods(abc_class: type) -> list[str]:
    """Callable members an ABC defines on top of its abstract methods."""
    found: set[str] = set()
    for klass in abc_class.__mro__[:-1]:  # skip object
        for name, value in vars(klass).items():
            is_public_or_dunder = not name.startswith("_") or name.endswith("__")
            if (
                callable(value)
                and is_public_or_dunder
                and name not in abc_class.__abstractmethods__
            ):
                found.add(name)
    return sorted(found - {"__subclasshook__", "__class_getitem__", "__init_subclass__"})


for abc_class in [Iterator, Sequence, Mapping, MutableMapping, Set]:
    print(f"{abc_class.__name__:<15} write {sorted(abc_class.__abstractmethods__)}")
    print(f"{'':<15} free  {free_methods(abc_class)}")

Iterator        write ['__next__']
                free  ['__iter__']
Sequence        write ['__getitem__', '__len__']
                free  ['__contains__', '__iter__', '__reversed__', 'count', 'index']
Mapping         write ['__getitem__', '__iter__', '__len__']
                free  ['__contains__', '__eq__', 'get', 'items', 'keys', 'values']
MutableMapping  write ['__delitem__', '__getitem__', '__iter__', '__len__', '__setitem__']
                free  ['__contains__', '__eq__', 'clear', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']
Set             write ['__contains__', '__iter__', '__len__']
                free  ['__and__', '__eq__', '__ge__', '__gt__', '__le__', '__lt__', '__or__', '__rand__', '__ror__', '__rsub__', '__rxor__', '__sub__', '__xor__', 'isdisjoint']


#### `register`: virtual subclasses

`SomeABC.register(cls)` declares that `cls` implements the ABC **without
inheriting**. `isinstance` and `issubclass` then return `True`, but the class gets
**no mixin methods** and the ABC is **not** in its MRO. Python also does **no
checking**: registering is a promise you make. The built-ins are wired up this way:
`list` is a registered `MutableSequence`.

In [129]:
class LegacyRecordSet:
    """An old class we cannot change: it has __len__ and __getitem__."""

    def __init__(self, records: list[str]) -> None:
        self._records = records

    def __len__(self) -> int:
        return len(self._records)

    def __getitem__(self, index: int) -> str:
        return self._records[index]


Sequence.register(LegacyRecordSet)
legacy_records = LegacyRecordSet(["r1", "r2"])
print(f"{isinstance(legacy_records, Sequence)=}, {Sequence in LegacyRecordSet.__mro__=}")
print(f"{hasattr(legacy_records, 'index')=}  (no mixin methods for virtual subclasses)")
print(f"{issubclass(list, MutableSequence)=}, {MutableSequence in list.__mro__=}")

isinstance(legacy_records, Sequence)=True, Sequence in LegacyRecordSet.__mro__=False
hasattr(legacy_records, 'index')=False  (no mixin methods for virtual subclasses)
issubclass(list, MutableSequence)=True, MutableSequence in list.__mro__=False


#### `__subclasshook__`: structural checks

An ABC can decide membership by **inspecting the candidate class**. `Iterable`,
`Sized`, `Hashable` and friends do exactly this: their `__subclasshook__` looks for
the method in the class's MRO. Returning `NotImplemented` means "no opinion", so
normal inheritance and registration still apply. A list is not `Hashable`, because
`list` sets `__hash__ = None`, and the hook treats a `None` method as missing.

In [130]:
print(inspect.getsource(Sized.__subclasshook__))
print(f"{isinstance(PageRange(10, 5), Sized)=}, {isinstance([], Hashable)=}")

    @classmethod
    def __subclasshook__(cls, C):
        if cls is Sized:
            return _check_methods(C, "__len__")
        return NotImplemented

isinstance(PageRange(10, 5), Sized)=True, isinstance([], Hashable)=False


Your own ABC can do the same. `SupportsToJson` accepts any class that defines a
callable `to_json`, anywhere in its MRO, without inheritance or registration
(`JsonSerializableMixin` from 2.2.4 would qualify).

In [131]:
class SupportsToJson(ABC):  # noqa: B024 (membership comes from __subclasshook__)
    """Any class with a callable to_json() is a virtual subclass."""

    @classmethod
    def __subclasshook__(cls, candidate: type) -> Any:
        if cls is SupportsToJson:
            for klass in candidate.__mro__:
                if "to_json" in vars(klass):
                    return callable(vars(klass)["to_json"])
        return NotImplemented


class InvoiceRecord:
    def to_json(self) -> str:
        return '{"invoice": "INV-9"}'


class PlainRecord:
    pass


print(f"{isinstance(InvoiceRecord(), SupportsToJson)=}")
print(f"{isinstance(PlainRecord(), SupportsToJson)=}, {issubclass(dict, SupportsToJson)=}")

isinstance(InvoiceRecord(), SupportsToJson)=True
isinstance(PlainRecord(), SupportsToJson)=False, issubclass(dict, SupportsToJson)=False


### 2.3.3 Protocols (PEP 544)

An ABC is **nominal**: a class belongs because it inherits (or is registered).
A **Protocol** is **structural**, exactly like a TypeScript interface: any class
with matching methods satisfies it, with no inheritance and no registration.

```typescript
interface MessageChannel {
  send(recipient: string, message: string): boolean;
}
function broadcast(channel: MessageChannel, recipients: string[], message: string) { … }
broadcast({ send: (recipient, message) => true }, ["asha"], "hi");  // any matching shape
```

The Python version: `MessageChannel` describes the shape, and two unrelated classes
match it.

In [132]:
class MessageChannel(Protocol):
    """Anything that can send a message and report success."""

    def send(self, recipient: str, message: str) -> bool: ...


def broadcast(channel: MessageChannel, recipients: list[str], message: str) -> int:
    """Send to every recipient; return how many succeeded."""
    return sum(channel.send(recipient, message) for recipient in recipients)

Two channels that never mention `MessageChannel`. They match because they have a
`send(recipient, message) -> bool` method, and `broadcast` accepts both.

In [133]:
class ConsoleChannel:
    """No base class, no registration: it just has send()."""

    def __init__(self) -> None:
        self.sent: list[str] = []

    def send(self, recipient: str, message: str) -> bool:
        self.sent.append(f"{recipient}: {message}")
        return True


class PushChannel:
    """Only reaches users with a registered device."""

    def __init__(self, device_owners: set[str]) -> None:
        self.device_owners = device_owners

    def send(self, recipient: str, message: str) -> bool:
        return recipient in self.device_owners


console = ConsoleChannel()
print(f"{broadcast(console, ['asha', 'ravi'], 'Sale starts now')=}, {console.sent=}")
print(f"{broadcast(PushChannel({'ravi'}), ['asha', 'ravi'], 'Sale starts now')=}")

broadcast(console, ['asha', 'ravi'], 'Sale starts now')=2, console.sent=['asha: Sale starts now', 'ravi: Sale starts now']
broadcast(PushChannel({'ravi'}), ['asha', 'ravi'], 'Sale starts now')=1


**At run time a Protocol does nothing.** Annotations are not enforced: passing a
string fails only when `send` is called. And a plain Protocol cannot be used with
`isinstance` at all; that needs `@runtime_checkable` (below).

The checking happens in a **type checker** (mypy or pyright), before the code runs.
mypy would report the bad call as:

```text
error: Argument 1 to "broadcast" has incompatible type "str"; expected "MessageChannel"  [arg-type]
```

In [134]:
print(raised_message(lambda: broadcast("not a channel", ["asha"], "hi")))  # type: ignore[arg-type]
print(raised_message(lambda: isinstance(console, MessageChannel)))

AttributeError: 'str' object has no attribute 'send'
TypeError: Instance and class checks can only be used with @runtime_checkable protocols


The type checkers are Phase 4 tools and are not installed yet, so this cell only
reports whether one is available.

In [135]:
for checker_module, checker_command in [("mypy", "mypy"), ("pyright", "pyright")]:
    installed = importlib.util.find_spec(checker_module) or shutil.which(checker_command)
    status = "available" if installed else "not installed yet (Phase 4): skipping"
    print(f"{checker_module}: {status}")

mypy: not installed yet (Phase 4): skipping


pyright: not installed yet (Phase 4): skipping


**Inspecting a Protocol.** `typing.get_protocol_members` and `typing.is_protocol`
(*New in 3.13*) list the members a class must have. You *may* inherit from a
Protocol explicitly, like TypeScript's `implements`; the class then also inherits
any default method bodies. A Protocol itself cannot be instantiated.

In [136]:
class LoggedChannel(MessageChannel):
    """Explicitly declares that it implements MessageChannel."""

    def send(self, recipient: str, message: str) -> bool:
        return bool(recipient)


if sys.version_info >= (3, 13):
    print(f"{typing.get_protocol_members(MessageChannel)=}")
    print(f"{typing.is_protocol(MessageChannel)=}, {typing.is_protocol(LoggedChannel)=}")
else:
    print("get_protocol_members / is_protocol need Python 3.13+")
print(f"{LoggedChannel().send('asha', 'hi')=}")
print(raised_message(MessageChannel))

typing.get_protocol_members(MessageChannel)=frozenset({'send'})
typing.is_protocol(MessageChannel)=True, typing.is_protocol(LoggedChannel)=False
LoggedChannel().send('asha', 'hi')=True
TypeError: Protocols cannot be instantiated


Protocols can describe **attributes** as well as methods, and they combine with
generics. `cheapest` accepts any items that support `<`, so it works for `Decimal`,
`str` and anything else comparable, while a type checker still rejects items that
cannot be compared. (PEP 695 syntax `[T: Bound]`, 3.12+.)

In [137]:
class SupportsLessThan(Protocol):
    def __lt__(self, other: Any, /) -> bool: ...


class Named(Protocol):
    name: str  # an attribute member


def cheapest[T: SupportsLessThan](items: Iterable[T]) -> T:
    """The smallest item; T can be any comparable type."""
    return min(items)


def greeting(thing: Named) -> str:
    return f"Hello, {thing.name}"


class Shop:
    def __init__(self, name: str) -> None:
        self.name = name


print(f"{cheapest([Decimal('49.50'), Decimal('12.00'), Decimal('120')])=}")
print(f"{cheapest(['tea', 'coffee', 'soap'])=}")
print(greeting(Shop("Kirana Corner")))

cheapest([Decimal('49.50'), Decimal('12.00'), Decimal('120')])=Decimal('12.00')
cheapest(['tea', 'coffee', 'soap'])='coffee'
Hello, Kirana Corner


#### `@runtime_checkable` and its limits

Decorating a Protocol with `@runtime_checkable` makes `isinstance` work. It checks
only that every member **name exists** on the object:

In [138]:
@runtime_checkable
class Flushable(Protocol):
    def flush(self) -> None: ...


print(f"{isinstance(io.StringIO(), Flushable)=}, {isinstance(sys.stdout, Flushable)=}")
print(f"{isinstance(42, Flushable)=}, {issubclass(io.BytesIO, Flushable)=}")

isinstance(io.StringIO(), Flushable)=True, isinstance(sys.stdout, Flushable)=True
isinstance(42, Flushable)=False, issubclass(io.BytesIO, Flushable)=True


**Limit 1: names, not signatures.** The check ignores parameters and return types,
and does not even require the member to be callable. Only a member explicitly set
to `None` counts as missing.

In [139]:
class WrongSignature:
    def flush(self, force: bool, level: int) -> str:
        return "nothing like the protocol"


class NotEvenAMethod:
    flush = "yes"


class ExplicitlyNone:
    flush = None


for candidate in [WrongSignature, NotEvenAMethod, ExplicitlyNone]:
    print(f"{candidate.__name__:<15} isinstance -> {isinstance(candidate(), Flushable)}")

WrongSignature  isinstance -> True
NotEvenAMethod  isinstance -> True
ExplicitlyNone  isinstance -> False


**Limit 2: no `issubclass` with attribute members.** Without an instance Python
cannot know whether an attribute will exist, so Protocols with data members only
support `isinstance`.

**Limit 3: properties are not run.** Since 3.12 the check uses
`inspect.getattr_static`, so it never executes a property or `__getattr__`; a
property that would raise still counts as present.

In [140]:
@runtime_checkable
class HasName(Protocol):
    name: str


class LazyRecord:
    @property
    def name(self) -> str:
        raise RuntimeError("database unavailable")


print(f"{isinstance(LazyRecord(), HasName)=}  (the property was never called)")
print(raised_message(lambda: issubclass(LazyRecord, HasName)))

isinstance(LazyRecord(), HasName)=True  (the property was never called)
TypeError: Protocols with non-method members don't support issubclass(). Non-method members: 'name'.


**Limit 4: speed.** A runtime Protocol check looks up every member on every call. It
is slower than an ABC check (whose result is cached per class) and several times
slower than a plain `hasattr`. Avoid it in hot loops.

> **Note:** timings vary by machine and Python version; compare the ratios.

In [141]:
buffer = io.StringIO()
for label, check in [
    ("isinstance(Protocol)", lambda: isinstance(buffer, Flushable)),
    ("isinstance(ABC)", lambda: isinstance(buffer, Iterable)),
    ("hasattr", lambda: hasattr(buffer, "flush")),
]:
    seconds = min(timeit.repeat(check, number=20_000, repeat=3))
    print(f"{label:<22} {seconds / 20_000 * 1e9:8.0f} ns per check")

isinstance(Protocol)        152 ns per check
isinstance(ABC)              75 ns per check


hasattr                      34 ns per check


**Protocol or ABC?**

| Question | Protocol | ABC |
| --- | --- | --- |
| How does a class qualify? | By having the right members (structural) | By inheriting or `register` (nominal) |
| Can third-party classes qualify without changes? | Yes | Only via `register` |
| Checked when? | By the type checker; `isinstance` only with `@runtime_checkable` (names only) | At instantiation (missing abstract methods) and by `isinstance` |
| Provides shared code? | Only to classes that inherit explicitly | Yes: concrete and mixin methods |
| Typical use | Describing what a function **needs** from its arguments | A base class **family** with shared behaviour (template method) |
| TypeScript analogue | `interface` | `abstract class` |

Rule of thumb: describe **parameters** with Protocols, build **class hierarchies**
with ABCs. The standard library has both: `typing.SupportsIndex` and friends are
runtime-checkable Protocols, while `collections.abc.Sized` is an ABC with a
structural `__subclasshook__`.

In [142]:
print(f"{isinstance(3, typing.SupportsIndex)=}, {isinstance(3.5, typing.SupportsIndex)=}")
print(f"{typing.is_protocol(typing.SupportsIndex)=}, {typing.is_protocol(Sized)=}")
print(f"{isinstance([], Sized)=}, {type(Sized).__name__=}")

isinstance(3, typing.SupportsIndex)=True, isinstance(3.5, typing.SupportsIndex)=False
typing.is_protocol(typing.SupportsIndex)=True, typing.is_protocol(Sized)=False
isinstance([], Sized)=True, type(Sized).__name__='ABCMeta'


### 2.3.4 Duck typing and polymorphism

"If it walks like a duck and quacks like a duck, it is a duck." Python code usually
does not ask what **type** an object is, only whether it **can** do what is needed.
That gives polymorphism without any shared base class: `len()` works on anything
with `__len__`, including `PageRange` from 2.3.2.

In [143]:
def total_length(collections_of_things: Iterable[Sized]) -> int:
    """Works with any objects that have __len__."""
    return sum(len(things) for things in collections_of_things)


mixed_collections: list[Sized] = ["chai", [1, 2, 3], {"a": 1}, PageRange(45, 10), range(5)]
print(f"{[len(things) for things in mixed_collections]=}")
print(f"{total_length(mixed_collections)=}")

[len(things) for things in mixed_collections]=[4, 3, 1, 5, 5]
total_length(mixed_collections)=18


Operators are polymorphic in the same way: `+` calls `__add__` on whatever the
left operand is (Unit 2.4 shows how to write these methods).

In [144]:
for left, right in [(2, 3), ("chai", " latte"), ([1], [2]), (Decimal("1.10"), Decimal("2.20"))]:
    print(f"{type(left).__name__:<8} {left!r} + {right!r} = {left + right!r}")

int      2 + 3 = 5
str      'chai' + ' latte' = 'chai latte'
list     [1] + [2] = [1, 2]
Decimal  Decimal('1.10') + Decimal('2.20') = Decimal('3.30')


**EAFP over type checks.** "Easier to ask forgiveness than permission": try the
operation and handle the failure, instead of checking types first (LBYL, 1.9.6 and
1.14.7). The type-checking version below rejects `FakeSocket` even though it would
work perfectly; the EAFP version accepts anything with a `read()` method.

In [145]:
class FakeSocket:
    """Not a file at all, but it has read()."""

    def read(self) -> str:
        return "data from the network"


def read_all_lbyl(source: object) -> str:
    """Type checks: only the types the author thought of."""
    if isinstance(source, io.StringIO):
        return source.read()
    if isinstance(source, str):
        return source
    raise TypeError(f"unsupported source: {type(source).__name__}")


def read_all(source: Any) -> str:
    """EAFP: anything with .read() works; otherwise treat it as text."""
    try:
        reader = source.read
    except AttributeError:
        return str(source)
    return reader()

Each source goes through both versions. `result_or_error` shows either the result or
the `TypeError` the type-checking version raises.

In [146]:
def result_or_error(thunk: Callable[[], str]) -> str:
    """The call's result, or "TypeError: message" if it raised TypeError."""
    try:
        return thunk()
    except TypeError as error:
        return f"{type(error).__name__}: {error}"


source_makers = [("StringIO", lambda: io.StringIO("from a buffer")), ("FakeSocket", FakeSocket)]
source_makers.append(("str", lambda: "plain text"))
for label, make_source in source_makers:
    lbyl_result = result_or_error(functools.partial(read_all_lbyl, make_source()))
    print(f"{label:<10} LBYL: {lbyl_result:<43} EAFP: {read_all(make_source())}")

StringIO   LBYL: from a buffer                               EAFP: from a buffer
FakeSocket LBYL: TypeError: unsupported source: FakeSocket   EAFP: data from the network
str        LBYL: plain text                                  EAFP: plain text


> **Warning:** keep the `try` block **narrow**. If the `try` also covers the call
> to `read()`, an `AttributeError` raised by a bug *inside* `read()` is swallowed and
> silently treated as "not readable". The version above only looks up the attribute
> inside the `try`, so the bug surfaces.

In [147]:
class BrokenReader:
    """read() has a bug that raises AttributeError internally."""

    def read(self) -> str:
        return self.buffer.decode()  # bug: there is no self.buffer

    def __str__(self) -> str:
        return "BrokenReader (fallback used: the bug is hidden)"


def read_all_too_wide(source: Any) -> str:
    """BUG: the try block also covers the call."""
    try:
        return source.read()
    except AttributeError:
        return str(source)


print(read_all_too_wide(BrokenReader()))
print(raised_message(lambda: read_all(BrokenReader())))

BrokenReader (fallback used: the bug is hidden)
AttributeError: 'BrokenReader' object has no attribute 'buffer'


**When type checks are right.** Duck typing is the default, but `isinstance` has
its place:

| Situation | Tool |
| --- | --- |
| Validating untrusted input at a boundary (JSON, CLI args) | `isinstance` against built-in types |
| "Is this any kind of mapping / sequence?" | `isinstance(x, Mapping)` with `collections.abc`, never `type(x) is dict` |
| Different behaviour per type, kept open for extension | `functools.singledispatch` (1.12.7) |
| Describing what a function needs, for the type checker | a Protocol (2.3.3) |

`singledispatch` dispatches on the type of the first argument and accepts ABCs, so
any `Mapping` (including `EnvironmentConfig`) takes the mapping branch.

In [148]:
@functools.singledispatch
def to_display(value: object) -> str:
    """Default: str()."""
    return str(value)


@to_display.register
def _(value: Decimal) -> str:
    return f"INR {value:,.2f}"


@to_display.register
def _(value: Mapping) -> str:
    return ", ".join(f"{key}={item}" for key, item in value.items())


for value in [42, Decimal("1234.5"), {"port": 8000}, app_config]:
    print(f"{type(value).__name__:<18} -> {to_display(value)}")

int                -> 42
Decimal            -> INR 1,234.50
dict               -> port=8000
EnvironmentConfig  -> debug=1, port=8000


**Predict the output.**

```python
from collections.abc import Iterable

class Countdown:
    def __getitem__(self, index):
        if index >= 3:
            raise IndexError
        return 3 - index

print(list(Countdown()))
print(isinstance(Countdown(), Iterable))
```

<details><summary>Answer</summary>

```text
[3, 2, 1]
False
```

`iter()` falls back to the old sequence protocol: it calls `__getitem__(0)`,
`__getitem__(1)`, … until `IndexError`. But `Iterable.__subclasshook__` only looks
for `__iter__`, so the ABC check says `False`. Asking "can I iterate it?" with
`isinstance(x, Iterable)` is therefore not fully reliable; trying `iter(x)` is.

</details>

Run the next cell to check your prediction.

In [149]:
class Countdown:
    """Iterable only through the old __getitem__ protocol."""

    def __getitem__(self, index: int) -> int:
        if index >= 3:
            raise IndexError
        return 3 - index


print(list(Countdown()))
print(isinstance(Countdown(), Iterable))

[3, 2, 1]
False


### Lab 2.3

Four exercises. For each, complete the stub, then run the check cell. Until then
the checks print FAIL; that is expected. Some structural checks already pass on the
stub because the class shape is given: "Square is not a Rectangle" (exercise 1),
"no assignment", "is a Mapping", "not a MutableMapping" and "only three methods
written" (exercise 2), and "issubclass refused" (exercise 4). Keep them passing.

#### Exercise 1 — a `Shape` ABC with sortable subclasses

- `Shape(ABC)` declares an **abstract property** `area` and an **abstract method**
  `perimeter()`.
- `Shape.__lt__(other)` compares areas, so `sorted()` orders shapes from smallest to
  largest; return `NotImplemented` when `other` is not a `Shape` (Python then raises
  `TypeError`). `sorted` needs only `<`; Unit 2.4 covers comparison methods fully.
- `Shape.describe()` is a template method returning
  `f"{class name}: area={area:.2f}, perimeter={perimeter:.2f}"`.
- `Circle(radius)`, `Rectangle(width, height)` and `Square(side)` are concrete
  shapes. Every dimension must be greater than zero, otherwise
  `ValueError("dimensions must be positive")`. `Square` must **not** subclass
  `Rectangle` (2.2.5).

In [150]:
class Shape(ABC):  # noqa: B024 (add the abstract members)
    """Abstract shape: subclasses provide area and perimeter; shapes sort by area."""

    # Declare the abstract property `area` and the abstract method `perimeter()` here.

    def __lt__(self, other: object) -> bool:
        raise NotImplementedError("Lab 2.3: implement Shape.__lt__")

    def describe(self) -> str:
        raise NotImplementedError("Lab 2.3: implement Shape.describe")


class Circle(Shape):
    def __init__(self, radius: float) -> None:
        raise NotImplementedError("Lab 2.3: implement Circle")


class Rectangle(Shape):
    def __init__(self, width: float, height: float) -> None:
        raise NotImplementedError("Lab 2.3: implement Rectangle")


class Square(Shape):
    def __init__(self, side: float) -> None:
        raise NotImplementedError("Lab 2.3: implement Square")

Run the checks. `Triangle` deliberately forgets `perimeter()`, so creating one must
fail once `Shape` is abstract.

In [151]:
class Triangle(Shape):
    """Forgets perimeter()."""

    @property
    def area(self) -> float:
        return 1.0


run_checks(
    [
        ("Shape is abstract", lambda: raised_error_name(Shape), "TypeError"),
        ("incomplete subclass", lambda: raised_error_name(Triangle), "TypeError"),
        ("abstract members", lambda: sorted(Shape.__abstractmethods__), ["area", "perimeter"]),
        (
            "sorted by area",
            lambda: [
                type(shape).__name__
                for shape in sorted([Rectangle(3, 4), Circle(1), Square(3), Circle(2)])
            ],
            ["Circle", "Square", "Rectangle", "Circle"],
        ),
        ("less than", lambda: (Circle(1) < Square(2), Square(2) < Circle(1)), (True, False)),
        ("describe circle", lambda: Circle(1).describe(), "Circle: area=3.14, perimeter=6.28"),
        (
            "describe rectangle",
            lambda: Rectangle(3, 4).describe(),
            "Rectangle: area=12.00, perimeter=14.00",
        ),
        ("describe square", lambda: Square(2.5).describe(), "Square: area=6.25, perimeter=10.00"),
        ("area is a property", lambda: Square(3).area, 9),
        ("Square is not a Rectangle", lambda: issubclass(Square, Rectangle), False),
        (
            "positive dimensions",
            lambda: raised_message(lambda: Rectangle(0, 2)),
            "ValueError: dimensions must be positive",
        ),
        ("compare with non-shape", lambda: raised_error_name(lambda: Circle(1) < 5), "TypeError"),
    ]
)

FAIL  Shape is abstract: expected 'TypeError', got 'no error'
FAIL  incomplete subclass: expected 'TypeError', got 'no error'
FAIL  abstract members: expected ['area', 'perimeter'], got []
FAIL  sorted by area: raised NotImplementedError: Lab 2.3: implement Rectangle
FAIL  less than: raised NotImplementedError: Lab 2.3: implement Circle
FAIL  describe circle: raised NotImplementedError: Lab 2.3: implement Circle
FAIL  describe rectangle: raised NotImplementedError: Lab 2.3: implement Rectangle
FAIL  describe square: raised NotImplementedError: Lab 2.3: implement Square
FAIL  area is a property: raised NotImplementedError: Lab 2.3: implement Square
PASS  Square is not a Rectangle
FAIL  positive dimensions: expected 'ValueError: dimensions must be positive', got 'NotImplementedError: Lab 2.3: implement Rectangle'
FAIL  compare with non-shape: expected 'TypeError', got 'NotImplementedError'
1/12 checks passed


#### Exercise 2 — `ReadOnlyView(Mapping)` with three methods

`ReadOnlyView(data)` wraps another mapping **without copying it**: changes to the
wrapped dict show through the view, but the view itself cannot be changed (item
assignment raises `TypeError`, because `Mapping` has no `__setitem__`).

Implement only `__getitem__`, `__iter__` and `__len__`; `get`, `keys`, `items`,
`values`, `in` and `==` must come from `Mapping`.

In [152]:
class ReadOnlyView[K, V](Mapping[K, V]):
    """A live, read-only view of another mapping."""

    def __init__(self, data: Mapping[K, V]) -> None:
        self._data = data

    def __getitem__(self, key: K) -> V:
        raise NotImplementedError("Lab 2.3: implement ReadOnlyView.__getitem__")

    def __iter__(self) -> Iterator[K]:
        raise NotImplementedError("Lab 2.3: implement ReadOnlyView.__iter__")

    def __len__(self) -> int:
        raise NotImplementedError("Lab 2.3: implement ReadOnlyView.__len__")

Run the checks.

In [153]:
prices = {"soap": 49, "tea": 120}
MAPPING_METHOD_NAMES = {
    "__getitem__",
    "__iter__",
    "__len__",
    "get",
    "keys",
    "items",
    "values",
    "__contains__",
}


def live_update() -> int:
    """Change the wrapped dict and read through the view."""
    backing = {"soap": 49}
    view = ReadOnlyView(backing)
    backing["soap"] = 55
    return view["soap"]


def try_assign() -> None:
    """Item assignment on the view."""
    view = ReadOnlyView({"soap": 49})
    view["soap"] = 1  # type: ignore[index]


run_checks(
    [
        ("getitem", lambda: ReadOnlyView(prices)["tea"], 120),
        ("len", lambda: len(ReadOnlyView(prices)), 2),
        ("iteration order", lambda: list(ReadOnlyView(prices)), ["soap", "tea"]),
        (
            "get (free)",
            lambda: (ReadOnlyView(prices).get("rice"), ReadOnlyView(prices).get("rice", 0)),
            (None, 0),
        ),
        ("items (free)", lambda: list(ReadOnlyView(prices).items()), [("soap", 49), ("tea", 120)]),
        (
            "in (free)",
            lambda: ("soap" in ReadOnlyView(prices), "rice" in ReadOnlyView(prices)),
            (True, False),
        ),
        ("== dict (free)", lambda: ReadOnlyView(prices) == {"tea": 120, "soap": 49}, True),
        (
            "missing key",
            lambda: raised_error_name(lambda: ReadOnlyView(prices)["rice"]),
            "KeyError",
        ),
        ("live view", live_update, 55),
        ("no assignment", lambda: raised_error_name(try_assign), "TypeError"),
        ("is a Mapping", lambda: isinstance(ReadOnlyView(prices), Mapping), True),
        ("not a MutableMapping", lambda: isinstance(ReadOnlyView(prices), MutableMapping), False),
        (
            "only three methods written",
            lambda: sorted(name for name in vars(ReadOnlyView) if name in MAPPING_METHOD_NAMES),
            ["__getitem__", "__iter__", "__len__"],
        ),
    ]
)

FAIL  getitem: raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__getitem__
FAIL  len: raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__len__
FAIL  iteration order: raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__iter__
FAIL  get (free): raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__getitem__
FAIL  items (free): raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__len__
FAIL  in (free): raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__getitem__
FAIL  == dict (free): raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__iter__
FAIL  missing key: expected 'KeyError', got 'NotImplementedError'
FAIL  live view: raised NotImplementedError: Lab 2.3: implement ReadOnlyView.__getitem__
PASS  no assignment
PASS  is a Mapping
PASS  not a MutableMapping
PASS  only three methods written
4/13 checks passed


#### Exercise 3 — a `SupportsClose` Protocol

- Make `SupportsClose` a **runtime-checkable** Protocol with one member,
  `close(self) -> None`.
- Write `close_all(resources)`: for every item that satisfies `SupportsClose` (in
  order), call `close()`; skip items that do not. Keep going when a `close()` raises
  an `Exception`. Return `(closed_count, failures)` where `closed_count` counts the
  successful closes and `failures` is a list of
  `f"{type(resource).__name__}: {error}"` strings.
- `resources` can be any iterable, including a generator.

In [154]:
class SupportsClose(Protocol):
    """Anything with a close() method. Add the member and make it runtime-checkable."""


def close_all(resources: Iterable[object]) -> tuple[int, list[str]]:
    """Close every resource that supports close(), even if some fail."""
    raise NotImplementedError("Lab 2.3: implement close_all")

Run the checks. `Recorder` and `FlakyResource` inherit from nothing.

In [155]:
class Recorder:
    """Closes cleanly and remembers it."""

    def __init__(self) -> None:
        self.closed = False

    def close(self) -> None:
        self.closed = True


class FlakyResource:
    """close() fails."""

    def close(self) -> None:
        raise OSError("disk unplugged")


def close_mixed() -> tuple[tuple[int, list[str]], bool, bool]:
    """close_all over a mix; also report whether the good resources really closed."""
    buffer = io.StringIO("data")
    recorder = Recorder()
    report = close_all([buffer, "not closable", FlakyResource(), 42, recorder])
    return report, buffer.closed, recorder.closed


run_checks(
    [
        ("StringIO", lambda: isinstance(io.StringIO(), SupportsClose), True),
        ("plain class, no inheritance", lambda: isinstance(Recorder(), SupportsClose), True),
        ("int", lambda: isinstance(42, SupportsClose), False),
        ("issubclass", lambda: issubclass(FlakyResource, SupportsClose), True),
        ("members", lambda: typing.get_protocol_members(SupportsClose), frozenset({"close"})),
        ("close_all", close_mixed, ((2, ["FlakyResource: disk unplugged"]), True, True)),
        ("generator input", lambda: close_all(Recorder() for _ in range(3)), (3, [])),
        ("empty", lambda: close_all([]), (0, [])),
    ]
)

FAIL  StringIO: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  plain class, no inheritance: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  int: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  issubclass: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  members: expected frozenset({'close'}), got frozenset()
FAIL  close_all: raised NotImplementedError: Lab 2.3: implement close_all
FAIL  generator input: raised NotImplementedError: Lab 2.3: implement close_all
FAIL  empty: raised NotImplementedError: Lab 2.3: implement close_all
0/8 checks passed


#### Exercise 4 — a payment gateway behind a `PaymentProvider` Protocol

Write a runtime-checkable Protocol `PaymentProvider` with three members: the
attribute `name: str`, `charge(amount_paise, token) -> str` (returns a transaction
id, or raises `PaymentDeclinedError`) and `refund(transaction_id) -> None`.

Two fake providers implement it **without inheriting from it**. Both keep
`charges: dict[str, int]` (transaction id → amount) and `refunded: list[str]`, number
their transactions `f"{name}_{n:04d}"` from 1, and raise
`ValueError("amount must be positive")` (a programming error, not a decline) when
`amount_paise <= 0`. `refund` moves a charge from `charges` to `refunded` and raises
`KeyError(transaction_id)` for an unknown id.

| Provider | `name` | Declines with `PaymentDeclinedError(…)` |
| --- | --- | --- |
| `CardGateway()` | `"card"` | `"invalid card token"` unless the token starts with `"tok_"` |
| `UpiGateway()` | `"upi"` | `"invalid UPI ID"` unless the token contains `"@"`; `"amount exceeds UPI limit"` above `UPI_LIMIT_PAISE` |

`CheckoutService(provider)` depends only on the Protocol:

- `pay(order_id, amount_paise, token)` returns a `PaymentResult`: status `"paid"`
  with the transaction id and message `""`, or `"declined"` with
  `transaction_id=None` and the decline message. Every result is appended to
  `self.history`. A `ValueError` from the provider is **not** caught.
- `refund(result)` refunds a paid result through the provider; anything else raises
  `ValueError("only paid orders can be refunded")`.

The given types:

In [156]:
class PaymentDeclinedError(Exception):
    """The provider refused the charge; the message says why."""


class PaymentResult(NamedTuple):
    """Outcome of CheckoutService.pay."""

    order_id: str
    provider: str
    status: str  # "paid" or "declined"
    transaction_id: str | None
    message: str


UPI_LIMIT_PAISE = 1_00_000_00  # ₹1,00,000

Your Protocol, providers and service:

In [157]:
class PaymentProvider(Protocol):
    """What CheckoutService needs from a payment provider. Add the three members."""


class CardGateway:
    def __init__(self) -> None:
        raise NotImplementedError("Lab 2.3: implement CardGateway")


class UpiGateway:
    def __init__(self) -> None:
        raise NotImplementedError("Lab 2.3: implement UpiGateway")


class CheckoutService:
    def __init__(self, provider: PaymentProvider) -> None:
        raise NotImplementedError("Lab 2.3: implement CheckoutService")

    def pay(self, order_id: str, amount_paise: int, token: str) -> PaymentResult:
        raise NotImplementedError("Lab 2.3: implement CheckoutService.pay")

    def refund(self, result: PaymentResult) -> None:
        raise NotImplementedError("Lab 2.3: implement CheckoutService.refund")

Run the checks. `WalletGateway` is a third provider written later, inheriting from
nothing; the service must accept it unchanged.

In [158]:
class WalletGateway:
    name = "wallet"

    def charge(self, amount_paise: int, token: str) -> str:
        return f"wallet-{amount_paise}"

    def refund(self, transaction_id: str) -> None:
        pass


def card_flow() -> list[PaymentResult]:
    """Two card payments: one paid, one declined."""
    checkout = CheckoutService(CardGateway())
    checkout.pay("ORD-1", 49_900, "tok_visa")
    checkout.pay("ORD-2", 10_000, "4111-1111")
    return checkout.history


def upi_flow() -> tuple[PaymentResult, PaymentResult, PaymentResult]:
    """UPI: paid, bad ID, over the limit."""
    checkout = CheckoutService(UpiGateway())
    return (
        checkout.pay("ORD-3", 25_000, "asha@okbank"),
        checkout.pay("ORD-4", 25_000, "asha"),
        checkout.pay("ORD-5", UPI_LIMIT_PAISE + 1, "asha@okbank"),
    )


def refund_flow() -> tuple[dict[str, int], list[str]]:
    """Pay twice, refund the first."""
    gateway = UpiGateway()
    checkout = CheckoutService(gateway)
    first = checkout.pay("ORD-6", 1_000, "ravi@okbank")
    checkout.pay("ORD-7", 2_000, "ravi@okbank")
    checkout.refund(first)
    return gateway.charges, gateway.refunded


def refund_declined() -> None:
    """Refunding a declined payment."""
    checkout = CheckoutService(CardGateway())
    checkout.refund(checkout.pay("ORD-8", 100, "bad-token"))

The checks themselves:

In [159]:
run_checks(
    [
        ("card satisfies", lambda: isinstance(CardGateway(), PaymentProvider), True),
        ("upi satisfies", lambda: isinstance(UpiGateway(), PaymentProvider), True),
        ("wallet satisfies", lambda: isinstance(WalletGateway(), PaymentProvider), True),
        ("str does not", lambda: isinstance("card", PaymentProvider), False),
        (
            "members",
            lambda: sorted(typing.get_protocol_members(PaymentProvider)),
            ["charge", "name", "refund"],
        ),
        (
            "issubclass refused (data member)",
            lambda: raised_error_name(lambda: issubclass(CardGateway, PaymentProvider)),
            "TypeError",
        ),
        (
            "card flow",
            card_flow,
            [
                PaymentResult("ORD-1", "card", "paid", "card_0001", ""),
                PaymentResult("ORD-2", "card", "declined", None, "invalid card token"),
            ],
        ),
        (
            "upi flow",
            upi_flow,
            (
                PaymentResult("ORD-3", "upi", "paid", "upi_0001", ""),
                PaymentResult("ORD-4", "upi", "declined", None, "invalid UPI ID"),
                PaymentResult("ORD-5", "upi", "declined", None, "amount exceeds UPI limit"),
            ),
        ),
        ("refund", refund_flow, ({"upi_0002": 2_000}, ["upi_0001"])),
        (
            "refund declined",
            lambda: raised_message(refund_declined),
            "ValueError: only paid orders can be refunded",
        ),
        (
            "unknown refund id",
            lambda: raised_error_name(lambda: CardGateway().refund("card_9999")),
            "KeyError",
        ),
        (
            "bad amount is a bug, not a decline",
            lambda: raised_error_name(
                lambda: CheckoutService(CardGateway()).pay("ORD-9", 0, "tok_x")
            ),
            "ValueError",
        ),
        (
            "third provider works",
            lambda: CheckoutService(WalletGateway()).pay("ORD-10", 500, "w"),
            PaymentResult("ORD-10", "wallet", "paid", "wallet-500", ""),
        ),
    ]
)

FAIL  card satisfies: raised NotImplementedError: Lab 2.3: implement CardGateway
FAIL  upi satisfies: raised NotImplementedError: Lab 2.3: implement UpiGateway
FAIL  wallet satisfies: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  str does not: raised TypeError: Instance and class checks can only be used with @runtime_checkable protocols
FAIL  members: expected ['charge', 'name', 'refund'], got []
PASS  issubclass refused (data member)
FAIL  card flow: raised NotImplementedError: Lab 2.3: implement CardGateway
FAIL  upi flow: raised NotImplementedError: Lab 2.3: implement UpiGateway
FAIL  refund: raised NotImplementedError: Lab 2.3: implement UpiGateway
FAIL  refund declined: expected 'ValueError: only paid orders can be refunded', got 'NotImplementedError: Lab 2.3: implement CardGateway'
FAIL  unknown refund id: expected 'KeyError', got 'NotImplementedError'
FAIL  bad amount is a bug, not a decline: expected 'ValueError', got 'NotI

### Recap — Unit 2.3

| Concept | One-line takeaway | JS / TS equivalent |
| --- | --- | --- |
| `ABC` + `@abstractmethod` | Instantiating a class with unimplemented abstract methods raises `TypeError` | TS `abstract class` |
| `ABCMeta` / `__abstractmethods__` | The metaclass records what is still abstract | — |
| Abstract property | `@property` over `@abstractmethod`; a class attribute can satisfy it | TS `abstract get x()` |
| Template method | The base owns the algorithm, subclasses fill in the steps (and optional hooks) | same pattern |
| `collections.abc` | Write the abstract methods, inherit the mixin methods | — |
| `register` | Virtual subclass: `isinstance` is true, no mixins, no checks | — |
| `__subclasshook__` | Membership decided by inspecting the class (how `Iterable` works) | `Symbol.hasInstance` |
| `Protocol` | Structural typing, checked by type checkers | TS `interface` |
| `@runtime_checkable` | `isinstance` checks member **names** only; no `issubclass` with attributes; slow | — (interfaces are erased) |
| Protocol vs ABC | Protocols for what a function needs; ABCs for families sharing code | `interface` vs `abstract class` |
| Duck typing | Ask "can it?", not "is it?"; polymorphism without a base class | same |
| EAFP | `try` the narrow operation, catch the specific error | `try/catch` |
| `singledispatch` | Type-based dispatch that understands ABCs | — |

**Further reading**

- [`abc` — Abstract base classes](https://docs.python.org/3/library/abc.html),
  [`collections.abc`](https://docs.python.org/3/library/collections.abc.html)
  (the full table of ABCs and mixin methods)
- [`typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol),
  [`runtime_checkable`](https://docs.python.org/3/library/typing.html#typing.runtime_checkable),
  [`get_protocol_members`](https://docs.python.org/3/library/typing.html#typing.get_protocol_members),
  [typing spec: protocols](https://typing.python.org/en/latest/spec/protocol.html)
- [Glossary: duck typing](https://docs.python.org/3/glossary.html#term-duck-typing),
  [EAFP](https://docs.python.org/3/glossary.html#term-EAFP),
  [`functools.singledispatch`](https://docs.python.org/3/library/functools.html#functools.singledispatch)
- [PEP 3119 — Introducing abstract base classes](https://peps.python.org/pep-3119/),
  [PEP 544 — Protocols: structural subtyping](https://peps.python.org/pep-0544/),
  [PEP 695 — Type parameter syntax](https://peps.python.org/pep-0695/)

## Unit 2.4 — The Data Model: Special (Dunder) Methods

**Objectives**

- Explain the data-model idea: syntax such as `len(x)`, `x[i]` and `a + b` calls
  special ("dunder") methods on the type.
- Write `__repr__`, `__str__` and `__format__`, and know which one each tool uses.
- Implement value equality and hashing correctly, and ordering with
  `functools.total_ordering`.
- Emulate containers (indexing, slicing, membership, iteration, `__missing__`) and
  know why `UserDict` / `UserList` are safer bases than `dict` / `list`.
- Make objects callable, and make them work with `+`, `sum()`, `+=`, `*`, `/`, `@`.
- Describe how Python looks special methods up, and read the `BINARY_OP` bytecode.

**JS bridge**

| JavaScript | Python | The difference that bites |
| --- | --- | --- |
| `toString()` | `__str__` | Python also has `__repr__`, the developer view used by the REPL, containers and `!r` |
| `Symbol.toPrimitive` / `valueOf()` | `__format__`, `__int__`, `__float__`, `__index__` | One method per conversion, no hint argument |
| No operator overloading (`a + b` on objects gives `"[object Object]..."`) | `__add__`, `__lt__`, `__matmul__`, … | Python operators are method calls; returning `NotImplemented` lets the other operand try |
| `===` on objects compares identity | `==` calls `__eq__` (default: identity) | Defining `__eq__` switches hashing **off** until you define `__hash__` |
| `Map` keys compare by identity | `dict` keys compare by `__hash__` + `__eq__` | Two equal value objects are the **same** key |
| `Symbol.iterator` | `__iter__` | Also `__contains__`, `__reversed__`, `__len__` |
| `Proxy` with `get` trap for `obj[key]` | `__getitem__` | `__getitem__` receives a `slice` object for `x[1:3]` |
| Callable objects need a function wrapper | `__call__` | Any instance can be called like a function |

The next cell is this unit's setup.

In [160]:
import dis
import functools
import inspect
import math  # noqa: F401 (Lab 2.4)
import sys
import time
from collections import UserDict, UserList
from collections.abc import Callable, Iterable, Iterator, Mapping, MutableMapping
from decimal import ROUND_HALF_UP, Decimal
from typing import Self

PAISE = Decimal("0.01")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def set_attribute(target: object, attribute_name: str, value: object) -> None:
    """`target.attribute_name = value` as a call, for use inside raised_error_name(lambda: ...)."""
    setattr(target, attribute_name, value)


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.4.1 The data-model idea

Python's syntax and built-in functions are a thin layer over **special methods**
(also called *dunder* methods, for the double underscores). `len(x)` calls
`type(x).__len__(x)`; `a + b` calls `type(a).__add__(a, b)`. If your class defines
the method, the syntax works on your objects exactly as on built-ins. This is what
"Pythonic" usually means for a class: it plugs into the language instead of
inventing `getLength()` or `addTo()` methods.

| Syntax | Special method |
| --- | --- |
| `len(x)` | `__len__` |
| `x[i]` | `__getitem__` |
| `x + y` | `__add__` |
| `x()` | `__call__` |
| `for item in x` | `__iter__` |
| `item in x` | `__contains__` |
| `with x` | `__enter__` / `__exit__` |
| `hash(x)` | `__hash__` |
| `bool(x)` | `__bool__` |
| `format(x, spec)` | `__format__` |

You rarely call a dunder yourself; you implement it and let the syntax call it. The
class below implements **every row** of the table, so it is longer than a usual
cell. Each row then gets its own demo cell, which also shows the explicit call
Python makes, `type(x).__method__(x, ...)`.

In [161]:
class OrderBatch:
    """A named batch of order ids that supports every row of the table above."""

    def __init__(self, name: str, order_ids: Iterable[str]) -> None:
        self.name = name
        self.order_ids = tuple(order_ids)

    def __repr__(self) -> str:
        return f"OrderBatch({self.name!r}, {list(self.order_ids)!r})"

    def __len__(self) -> int:
        return len(self.order_ids)

    def __getitem__(self, index: int) -> str:
        return self.order_ids[index]

    def __add__(self, other: object) -> OrderBatch:
        if not isinstance(other, OrderBatch):
            return NotImplemented
        return OrderBatch(f"{self.name}+{other.name}", self.order_ids + other.order_ids)

    def __call__(self, courier: str) -> str:
        return f"{len(self)} orders of {self.name} handed to {courier}"

    def __iter__(self) -> Iterator[str]:
        return iter(self.order_ids)

    def __contains__(self, order_id: object) -> bool:
        return order_id in self.order_ids

    def __enter__(self) -> Self:
        print(f"  lock {self.name}")
        return self

    def __exit__(self, *exc_info: object) -> None:
        print(f"  unlock {self.name}")

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, OrderBatch):
            return NotImplemented
        return (self.name, self.order_ids) == (other.name, other.order_ids)

    def __hash__(self) -> int:
        return hash((self.name, self.order_ids))

    def __bool__(self) -> bool:
        return bool(self.order_ids)

    def __format__(self, spec: str) -> str:
        if spec == "short":
            return f"{self.name} ({len(self)} orders)"
        return format(repr(self), spec)


morning_batch = OrderBatch("morning", ["ORD-1", "ORD-2", "ORD-3"])
evening_batch = OrderBatch("evening", ["ORD-4", "ORD-5"])
print(morning_batch)

OrderBatch('morning', ['ORD-1', 'ORD-2', 'ORD-3'])


#### `len(x)` → `__len__`

`len()` calls `__len__` and checks the result is a non-negative `int` (2.4.5).

In [162]:
print(f"{len(morning_batch)=}")
print(f"{type(morning_batch).__len__(morning_batch)=}  (what len() calls)")

len(morning_batch)=3
type(morning_batch).__len__(morning_batch)=3  (what len() calls)


#### `x[i]` → `__getitem__`

Square brackets pass the index (or a `slice`, see 2.4.6) to `__getitem__`. Here the
tuple underneath handles negative indexes and raises `IndexError`.

In [163]:
print(f"{morning_batch[0]=}, {morning_batch[-1]=}")
print(f"{type(morning_batch).__getitem__(morning_batch, 1)=}")
print(raised_message(lambda: morning_batch[10]))

morning_batch[0]='ORD-1', morning_batch[-1]='ORD-3'
type(morning_batch).__getitem__(morning_batch, 1)='ORD-2'
IndexError: tuple index out of range


#### `x + y` → `__add__`

`+` calls `__add__` on the **left** operand. Returning `NotImplemented` for unknown
types (instead of raising) gives the right operand a chance (2.4.9); when nobody
handles it, Python raises a `TypeError` that names both types.

In [164]:
combined_batch = morning_batch + evening_batch
print(f"{combined_batch=}")
print(f"{type(morning_batch).__add__(morning_batch, 'ORD-9')=}")
print(raised_message(lambda: morning_batch + "ORD-9"))

combined_batch=OrderBatch('morning+evening', ['ORD-1', 'ORD-2', 'ORD-3', 'ORD-4', 'ORD-5'])
type(morning_batch).__add__(morning_batch, 'ORD-9')=NotImplemented
TypeError: unsupported operand type(s) for +: 'OrderBatch' and 'str'


#### `x()` → `__call__`

Parentheses after any object call `type(obj).__call__(obj, ...)`. Functions are just
objects whose type defines `__call__` (2.4.7).

In [165]:
print(morning_batch("BlueDart"))
print(f"{callable(morning_batch)=}, {callable(42)=}")

3 orders of morning handed to BlueDart


callable(morning_batch)=True, callable(42)=False


#### `for item in x` → `__iter__`

A `for` loop calls `iter(x)`, which calls `__iter__` and must return an iterator
(Unit 1.13). Unpacking, `list()`, `sorted()` and comprehensions use the same path.

In [166]:
for order_id in morning_batch:
    print(f"  shipping {order_id}")
first_order, *other_orders = morning_batch
print(f"{first_order=}, {other_orders=}, {sorted(morning_batch, reverse=True)=}")

  shipping ORD-1
  shipping ORD-2
  shipping ORD-3
first_order='ORD-1', other_orders=['ORD-2', 'ORD-3'], sorted(morning_batch, reverse=True)=['ORD-3', 'ORD-2', 'ORD-1']


#### `item in x` → `__contains__`

`in` calls `__contains__` and converts the result to `bool`. Without it, Python
falls back to scanning `__iter__` (or `__getitem__`), which is slower (2.4.6).

In [167]:
print(f"{'ORD-2' in morning_batch=}, {'ORD-9' not in morning_batch=}")
print(f"{type(morning_batch).__contains__(morning_batch, 'ORD-3')=}")

'ORD-2' in morning_batch=True, 'ORD-9' not in morning_batch=True
type(morning_batch).__contains__(morning_batch, 'ORD-3')=True


#### `with x` → `__enter__` / `__exit__`

Covered in Unit 1.14.9: `with` calls `__enter__` (its return value is bound by
`as`) and always calls `__exit__`, even when the block raises.

In [168]:
with morning_batch as locked_batch:
    print(f"  inside the block: {locked_batch is morning_batch=}")

  lock morning
  inside the block: locked_batch is morning_batch=True
  unlock morning


#### `hash(x)` → `__hash__`

`hash()` calls `__hash__`; sets and dict keys use it together with `__eq__`
(2.4.3). Two equal batches hash the same, so a set keeps one of them.

In [169]:
same_morning_batch = OrderBatch("morning", ["ORD-1", "ORD-2", "ORD-3"])
print(f"{morning_batch == same_morning_batch=}, {morning_batch is same_morning_batch=}")
print(f"{hash(morning_batch) == hash(same_morning_batch)=}")
print(f"{len({morning_batch, same_morning_batch, evening_batch})=}")

morning_batch == same_morning_batch=True, morning_batch is same_morning_batch=False
hash(morning_batch) == hash(same_morning_batch)=True
len({morning_batch, same_morning_batch, evening_batch})=2


#### `bool(x)` → `__bool__`

`if x:`, `while x:`, `not x`, `and` / `or` all ask `bool(x)`, which calls `__bool__`
(falling back to `__len__`, 2.4.5).

In [170]:
empty_batch = OrderBatch("night", [])
for batch in (morning_batch, empty_batch):
    print(f"{batch.name}: {bool(batch)=}, {'has orders' if batch else 'empty'}")

morning: bool(batch)=True, has orders
night: bool(batch)=False, empty


#### `format(x, spec)` → `__format__`

`format(x, spec)`, `f"{x:spec}"` and `str.format` pass the text after the colon to
`__format__`. The class decides what a spec means: here `"short"` is a custom spec,
and anything else is applied to the repr text.

In [171]:
print(format(morning_batch, "short"))
print(f"[{morning_batch:short}] [{evening_batch:short}]")
print(f"[{evening_batch:>48}]")

morning (3 orders)
[morning (3 orders)] [evening (2 orders)]
[       OrderBatch('evening', ['ORD-4', 'ORD-5'])]


### 2.4.2 Representation

Python has **two** text forms for an object:

| Method | Audience | Used by | Goal |
| --- | --- | --- | --- |
| `__repr__` | developers | the REPL, `repr()`, `!r`, debuggers, logs, **containers** | unambiguous; ideally valid Python that rebuilds the object |
| `__str__` | end users | `print()`, `str()`, f-strings without a conversion | readable |

If a class defines only `__repr__`, `str()` falls back to it, so **always write
`__repr__` first**. The default from `object` shows the class and the memory
address, which helps nobody (and changes every run, so the cell below cuts it off).

In [172]:
class PlainMoney:
    """No __repr__ or __str__: object's defaults apply."""

    def __init__(self, amount: Decimal, currency: str = "INR") -> None:
        self.amount = amount
        self.currency = currency


plain_price = PlainMoney(Decimal("250"))
default_text = repr(plain_price)
print(default_text.split(" at ")[0] + " at 0x…>")
print(f"{str(plain_price) == default_text=}  (str falls back to repr)")

<__main__.PlainMoney object at 0x…>
str(plain_price) == default_text=True  (str falls back to repr)


With only `__repr__`, both `print()` and containers show it. Containers **always**
use `repr` for their elements, even inside `print()`, so that `["1"]` and `[1]` look
different.

In [173]:
class ReprOnlyMoney(PlainMoney):
    def __repr__(self) -> str:
        return f"ReprOnlyMoney({str(self.amount)!r}, {self.currency!r})"


print(ReprOnlyMoney(Decimal("250")))
print([ReprOnlyMoney(Decimal("1")), ReprOnlyMoney(Decimal("2"), "USD")])
print(["1", 1])

ReprOnlyMoney('250', 'INR')
[ReprOnlyMoney('1', 'INR'), ReprOnlyMoney('2', 'USD')]
['1', 1]


Now `Money` from Unit 2.1.5 grows. The amount is converted to a `Decimal` rounded
to the paisa, `__repr__` produces text you could paste back into Python, and
`__str__` is what a customer sees. `__format__` adds **custom format specs**:

| Spec | Result for `Money("1234.5")` |
| --- | --- |
| `""` (plain `f"{m}"`) | same as `str(m)`: `INR 1,234.50` |
| `"symbol"` | `₹1,234.50` (`$`, `€` for USD, EUR) |
| anything else, e.g. `",.0f"` | applied to the amount: `INR 1,234` |

In [174]:
CURRENCY_SYMBOLS = {"INR": "₹", "USD": "$", "EUR": "€"}


class Money:
    """An amount in one currency (Unit 2.1.5's Money, growing through Unit 2.4)."""

    def __init__(self, amount: Decimal | int | str, currency: str = "INR") -> None:
        if not (len(currency) == 3 and currency.isalpha() and currency.isupper()):
            raise ValueError(f"invalid currency code: {currency!r}")
        self.amount = Decimal(amount).quantize(PAISE, rounding=ROUND_HALF_UP)
        self.currency = currency

    def __repr__(self) -> str:
        return f"Money({str(self.amount)!r}, {self.currency!r})"

    def __str__(self) -> str:
        return f"{self.currency} {self.amount:,.2f}"

    def __format__(self, spec: str) -> str:
        if spec == "":
            return str(self)
        if spec == "symbol":
            symbol = CURRENCY_SYMBOLS.get(self.currency, self.currency + " ")
            return f"{symbol}{self.amount:,.2f}"
        return f"{self.currency} {format(self.amount, spec)}"

`repr` vs `str`, and the `!r` / `!s` conversions in f-strings. The `=` specifier
(`f"{x=}"`) uses `repr` by default.

In [175]:
laptop_price = Money("54999.5")
print(repr(laptop_price))
print(str(laptop_price))
print(f"{laptop_price!r} / {laptop_price!s} / {laptop_price=}")
print([laptop_price, Money(19, "USD")])

Money('54999.50', 'INR')
INR 54,999.50
Money('54999.50', 'INR') / INR 54,999.50 / laptop_price=Money('54999.50', 'INR')
[Money('54999.50', 'INR'), Money('19.00', 'USD')]


The custom specs. `f"{price:symbol}"` passes the string `"symbol"` to `__format__`;
any other spec is handed to `Decimal.__format__`, so the standard format
mini-language (width, `,`, precision) works on the amount.

In [176]:
print(f"{laptop_price}")
print(f"{laptop_price:symbol} / {Money('19.99', 'USD'):symbol} / {Money(5, 'JPY'):symbol}")
print(f"{laptop_price:,.0f}")
print(f"[{laptop_price:>12,.2f}]")
print(raised_message(lambda: format(laptop_price, "bogus")))

INR 54,999.50
₹54,999.50 / $19.99 / JPY 5.00
INR 55,000
[INR    54,999.50]
ValueError: invalid format string


`object.__format__` (what you inherit) accepts only the empty spec; anything else
raises `TypeError`. That is why `PlainMoney` cannot be padded in an f-string until
it defines `__format__`. A safe default for your own `__format__` is to apply the
spec to `str(self)`, as `OrderBatch` did with its repr.

In [177]:
print(f"{format(plain_price, '') == str(plain_price)=}")
print(raised_message(lambda: f"{plain_price:>20}"))

format(plain_price, '') == str(plain_price)=True
TypeError: unsupported format string passed to PlainMoney.__format__


### 2.4.3 Equality and hashing

Without `__eq__`, `==` falls back to identity (`is`), like `===` on JS objects. Two
`Money("10")` objects are different objects, so they are not equal.

In [178]:
print(f"{Money(10) == Money(10)=}  (no __eq__ yet: identity)")

Money(10) == Money(10)=False  (no __eq__ yet: identity)


**`__eq__` returning `NotImplemented`.** When `__eq__` does not know the other type
it returns the special value `NotImplemented` (not `False`, and never raises). For
`a == b` Python then tries `b.__eq__(a)`, and if that is also `NotImplemented` it
falls back to `a is b`. The trace below shows the sequence. `!=` calls `__ne__`,
which by default inverts `__eq__`, so you only write `__eq__`.

In [179]:
class LoudEquality:
    """Prints every __eq__ call and declines to answer."""

    def __init__(self, label: str) -> None:
        self.label = label

    def __eq__(self, other: object) -> bool:
        print(f"  {self.label}.__eq__({getattr(other, 'label', other)!r})")
        return NotImplemented


left_side, right_side = LoudEquality("left"), LoudEquality("right")
print(f"result: {left_side == right_side}  (both declined: identity fallback)")
print(f"result: {left_side != right_side}  (__ne__ inverts __eq__)")

  left.__eq__('right')
  right.__eq__('left')
result: False  (both declined: identity fallback)
  left.__eq__('right')
  right.__eq__('left')
result: True  (__ne__ inverts __eq__)


Value equality for `Money`: compare amount and currency, and return
`NotImplemented` for anything that is not `Money`. Comparing with an `int` then gives
`False` instead of a crash.

**Defining `__eq__` sets `__hash__ = None`.** Python assumes that if you changed
what "equal" means, the inherited identity hash is now wrong (equal objects must
have equal hashes). So the class becomes **unhashable**: no sets, no dict keys.

In [180]:
class EqualityOnlyMoney(Money):
    """Adds value equality but no __hash__."""

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Money):
            return NotImplemented
        return (self.amount, self.currency) == (other.amount, other.currency)


print(f"{EqualityOnlyMoney(10) == EqualityOnlyMoney('10.00')=}")
print(f"{EqualityOnlyMoney(10) == 10=}  (NotImplemented on both sides -> False)")
print(f"{EqualityOnlyMoney.__hash__ is None=}")
print(raised_message(lambda: {EqualityOnlyMoney(10)}))

EqualityOnlyMoney(10) == EqualityOnlyMoney('10.00')=True
EqualityOnlyMoney(10) == 10=False  (NotImplemented on both sides -> False)
EqualityOnlyMoney.__hash__ is None=True
TypeError: cannot use 'EqualityOnlyMoney' as a set element (unhashable type: 'EqualityOnlyMoney')


**Hash only immutable fields.** A hash must never change while the object is in a
set or used as a dict key. The trap below hashes a mutable field, then changes it:
the object is still in the set, but Python looks in the wrong bucket and reports it
missing.

In [181]:
class MutableTag:
    """BUG: hash depends on a field that can change."""

    def __init__(self, label: str) -> None:
        self.label = label

    def __eq__(self, other: object) -> bool:
        return isinstance(other, MutableTag) and self.label == other.label

    def __hash__(self) -> int:
        return hash(self.label)


sale_tag = MutableTag("sale")
tags = {sale_tag}
sale_tag.label = "clearance"
print(f"{sale_tag in tags=}  (lost: hash changed after insertion)")
print(f"{any(tag is sale_tag for tag in tags)=}  (but it is still there)")

sale_tag in tags=False  (lost: hash changed after insertion)
any(tag is sale_tag for tag in tags)=True  (but it is still there)


The fix is to make the fields that take part in `__eq__` and `__hash__` read-only.
This `Money` stores `_amount` and `_currency` behind properties with no setters (Unit
2.1.6), and hashes the same tuple it compares. Because the amount is quantized to
paise, `Money("10")` and `Money("10.00")` hold the same `Decimal` and hash equally.
Lab 2.4 finishes the job (ordering, arithmetic, `sum()`).

In [182]:
class Money:
    """Read-only amount and currency; equality and hashing by value."""

    def __init__(self, amount: Decimal | int | str, currency: str = "INR") -> None:
        if not (len(currency) == 3 and currency.isalpha() and currency.isupper()):
            raise ValueError(f"invalid currency code: {currency!r}")
        self._amount = Decimal(amount).quantize(PAISE, rounding=ROUND_HALF_UP)
        self._currency = currency

    @property
    def amount(self) -> Decimal:
        return self._amount

    @property
    def currency(self) -> str:
        return self._currency

    def __repr__(self) -> str:
        return f"Money({str(self._amount)!r}, {self._currency!r})"

    def __str__(self) -> str:
        return f"{self._currency} {self._amount:,.2f}"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Money):
            return NotImplemented
        return (self._amount, self._currency) == (other._amount, other._currency)

    def __hash__(self) -> int:
        return hash((self._amount, self._currency))

Equal values collapse in a set and find the same dict entry; a different currency
is a different value; assigning to `amount` fails.

In [183]:
price_list = {Money(10): "pen", Money("49.99"): "notebook"}
print(f"{price_list[Money('10.00')]=}")
print(f"{len({Money(10), Money('10.0'), Money(10, 'USD')})=}")
print(f"{Money(10) == Money(10, 'USD')=}")
print(raised_message(lambda: set_attribute(Money(10), "amount", Decimal(0))))

price_list[Money('10.00')]='pen'
len({Money(10), Money('10.0'), Money(10, 'USD')})=2
Money(10) == Money(10, 'USD')=False
AttributeError: property 'amount' of 'Money' object has no setter


#### Under the hood: `__hash__` in the class `__dict__`

The `__hash__ = None` entry is literally written into the class namespace when the
class body defines `__eq__` without `__hash__`. A subclass that only adds `__eq__`
gets its own `None`, even if the parent was hashable.

In [184]:
print(f"{'__hash__' in vars(EqualityOnlyMoney)=}, {vars(EqualityOnlyMoney)['__hash__']=}")
print(f"{vars(Money)['__hash__'].__qualname__=}")
print(f"{'__hash__' in vars(OrderBatch)=}  (defined explicitly)")

'__hash__' in vars(EqualityOnlyMoney)=True, vars(EqualityOnlyMoney)['__hash__']=None
vars(Money)['__hash__'].__qualname__='Money.__hash__'
'__hash__' in vars(OrderBatch)=True  (defined explicitly)


### 2.4.4 Ordering

There are six **rich comparison** methods. Each operator calls one of them on the
left operand; if that returns `NotImplemented`, Python tries the **reflected**
method on the right operand (the mirror image, not the negation):

| Expression | First try | Reflected try |
| --- | --- | --- |
| `a < b` | `a.__lt__(b)` | `b.__gt__(a)` |
| `a <= b` | `a.__le__(b)` | `b.__ge__(a)` |
| `a > b` | `a.__gt__(b)` | `b.__lt__(a)` |
| `a >= b` | `a.__ge__(b)` | `b.__le__(a)` |
| `a == b` | `a.__eq__(b)` | `b.__eq__(a)` (then `a is b`) |
| `a != b` | `a.__ne__(b)` | `b.__ne__(a)` (then `a is not b`) |

Python does **not** derive `<=` from `<` and `==`. The `Version` class below writes
only `__eq__` and `__lt__`: `<` works, `>` works through reflection (`b < a`), and
`<=` fails.

In [185]:
class Version:
    """A semantic version like 3.15.1, compared field by field."""

    def __init__(self, text: str) -> None:
        major, minor, patch = (int(part) for part in text.split("."))
        self.parts = (major, minor, patch)

    def __repr__(self) -> str:
        return f"Version('{'.'.join(map(str, self.parts))}')"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts == other.parts

    def __lt__(self, other: object) -> bool:
        if not isinstance(other, Version):
            return NotImplemented
        return self.parts < other.parts


old_release, new_release = Version("3.9.18"), Version("3.15.1")
print(f"{old_release < new_release=}  (string comparison would say False)")
print(f"{new_release > old_release=}  (reflected: old_release.__lt__(new_release))")
print(raised_message(lambda: old_release <= new_release))

old_release < new_release=True  (string comparison would say False)
new_release > old_release=True  (reflected: old_release.__lt__(new_release))
TypeError: '<=' not supported between instances of 'Version' and 'Version'


**Reflection, traced.** Both sides print and decline. Python tries `__lt__` on the
left, then `__gt__` on the right, and only then raises `TypeError`.

In [186]:
class LoudOrdering:
    """Prints each comparison call and declines it."""

    def __init__(self, label: str) -> None:
        self.label = label

    def __lt__(self, other: object) -> bool:
        print(f"  {self.label}.__lt__({getattr(other, 'label', other)})")
        return NotImplemented

    def __gt__(self, other: object) -> bool:
        print(f"  {self.label}.__gt__({getattr(other, 'label', other)})")
        return NotImplemented


print(raised_message(lambda: LoudOrdering("a") < LoudOrdering("b")))

  a.__lt__(b)
  b.__gt__(a)
TypeError: '<' not supported between instances of 'LoudOrdering' and 'LoudOrdering'


**Subclass first.** If the right operand's type is a **subclass** of the left
operand's type and overrides the reflected method, Python tries the right side
first. This lets a more specific type take control of mixed comparisons (the same
rule applies to arithmetic, 2.4.9).

In [187]:
class PreReleaseVersion(Version):
    """A subclass that reports when its reflected __gt__ runs first."""

    def __gt__(self, other: object) -> bool:
        print("  PreReleaseVersion.__gt__ ran first")
        return NotImplemented


print(f"{Version('1.0.0') < PreReleaseVersion('2.0.0')=}")

  PreReleaseVersion.__gt__ ran first
Version('1.0.0') < PreReleaseVersion('2.0.0')=True


**`functools.total_ordering`.** Write `__eq__` and **one** of `__lt__`, `__le__`,
`__gt__`, `__ge__`; the class decorator fills in the other three. The generated
methods still return `NotImplemented` for foreign types. They add a little call
overhead, so for a hot sorting path you can write all four by hand, or sort with a
`key=` function instead.

In [188]:
@functools.total_ordering
class OrderedVersion(Version):
    """Version plus the four ordering methods generated from __lt__ and __eq__."""


releases = [OrderedVersion(text) for text in ["3.12.4", "3.9.18", "3.15.1", "3.13.0"]]
print(f"{sorted(releases)=}")
print(f"{max(releases)=}")
print(f"{OrderedVersion('3.13.0') <= OrderedVersion('3.13.0')=}")
print(f"{OrderedVersion('3.13.0') >= OrderedVersion('3.15.1')=}")
print(f"{sorted(set(vars(OrderedVersion)) & {'__le__', '__gt__', '__ge__'})=}  (generated)")
print(raised_message(lambda: OrderedVersion("1.0.0") <= "1.0.0"))

sorted(releases)=[Version('3.9.18'), Version('3.12.4'), Version('3.13.0'), Version('3.15.1')]
max(releases)=Version('3.15.1')
OrderedVersion('3.13.0') <= OrderedVersion('3.13.0')=True
OrderedVersion('3.13.0') >= OrderedVersion('3.15.1')=False
sorted(set(vars(OrderedVersion)) & {'__le__', '__gt__', '__ge__'})=['__ge__', '__gt__', '__le__']  (generated)
TypeError: '<=' not supported between instances of 'OrderedVersion' and 'str'


### 2.4.5 Truthiness and size

`bool(x)` first looks for `__bool__`. If the class has none, it uses `__len__`
(zero means false). If it has neither, every instance is truthy. Unit 1.3.8 showed
the rule with `Wallet` and `ShoppingBag`; here are the details that bite.

A class with only `__len__` is falsy when empty:

In [189]:
class PrintQueue:
    """Only __len__: truthiness comes from the length."""

    def __init__(self, jobs: Iterable[str] = ()) -> None:
        self.jobs = list(jobs)

    def __len__(self) -> int:
        return len(self.jobs)


for queue in (PrintQueue(), PrintQueue(["invoice.pdf"])):
    print(f"{len(queue)=}, {bool(queue)=}")

len(queue)=0, bool(queue)=False
len(queue)=1, bool(queue)=True


`__bool__` wins over `__len__` when both exist. Use it when "has items" and "is
usable" differ. Here a cache with entries is still falsy once it has expired.

In [190]:
class ExpiringCache:
    """Has a length, but is only 'true' while it is fresh."""

    def __init__(self, entries: dict[str, str], expired: bool) -> None:
        self.entries = entries
        self.expired = expired

    def __len__(self) -> int:
        return len(self.entries)

    def __bool__(self) -> bool:
        return not self.expired


stale_cache = ExpiringCache({"home": "<html>"}, expired=True)
print(f"{len(stale_cache)=}, {bool(stale_cache)=}")
print("serve from cache" if stale_cache else "refetch")

len(stale_cache)=1, bool(stale_cache)=False
refetch


Both methods have contracts that `len()` and `bool()` enforce: `__len__` must return
a non-negative `int` (an `int` subclass such as `bool` is accepted), and `__bool__`
must return exactly `bool`. Breaking them raises at the call site, not in your
method.

In [191]:
class BrokenSizes:
    def __init__(self, length: object, truth: object) -> None:
        self.length, self.truth = length, truth

    def __len__(self) -> int:
        return self.length  # type: ignore[return-value]

    def __bool__(self) -> bool:
        return self.truth  # type: ignore[return-value]


print(raised_message(lambda: len(BrokenSizes(-1, True))))
print(raised_message(lambda: len(BrokenSizes(2.5, True))))
print(raised_message(lambda: bool(BrokenSizes(1, 1))))
print(f"{len(BrokenSizes(True, False))=}  (bool is an int subclass)")

ValueError: __len__() should return >= 0


TypeError: 'float' object cannot be interpreted as an integer
TypeError: BrokenSizes.__bool__() must return a bool, not int
len(BrokenSizes(True, False))=1  (bool is an int subclass)


### 2.4.6 Container emulation

A class becomes a sequence or mapping by implementing these methods. The example is
`SensorLog`, a list of temperature readings.

| Method | Called by | Notes |
| --- | --- | --- |
| `__getitem__(key)` | `x[key]` | `key` is an `int`, or a `slice` for `x[a:b:c]` |
| `__setitem__(key, value)` | `x[key] = value` | |
| `__delitem__(key)` | `del x[key]` | |
| `__contains__(item)` | `item in x` | falls back to `__iter__`, then `__getitem__` |
| `__iter__()` | `for`, `list()`, unpacking | falls back to `__getitem__(0)`, `(1)`, … until `IndexError` |
| `__reversed__()` | `reversed(x)` | falls back to `__len__` + `__getitem__` |
| `__missing__(key)` | `dict.__getitem__` in a `dict` subclass | only for `dict` subclasses |

**`__getitem__` with an `int`.** Accept negative indexes and raise `IndexError` when
out of range; the old iteration protocol and `reversed()` rely on that error.

In [192]:
class SensorLog:
    """Temperature readings in °C, indexable like a list."""

    def __init__(self, sensor: str, readings: Iterable[float]) -> None:
        self.sensor = sensor
        self.readings = list(readings)

    def __repr__(self) -> str:
        return f"{type(self).__name__}({self.sensor!r}, {self.readings!r})"

    def __len__(self) -> int:
        return len(self.readings)

    def __getitem__(self, index: int | slice) -> float | SensorLog:
        if isinstance(index, slice):
            return type(self)(self.sensor, self.readings[index])
        if not -len(self) <= index < len(self):
            raise IndexError(f"reading {index} out of range for {len(self)} readings")
        return self.readings[index]


freezer_log = SensorLog("freezer", [-18.0, -17.5, -19.2, -12.4, -18.1])
print(f"{freezer_log[0]=}, {freezer_log[-1]=}")
print(raised_message(lambda: freezer_log[5]))

freezer_log[0]=-18.0, freezer_log[-1]=-18.1
IndexError: reading 5 out of range for 5 readings


**`__getitem__` with a `slice`.** `x[1:4]` calls `x.__getitem__(slice(1, 4, None))`.
Returning the **same type** (`type(self)(...)`, not a bare list) keeps slices useful,
also for subclasses.
`slice.indices(length)` turns `None`s and negatives into concrete
`(start, stop, step)` values when you need to compute positions yourself.

In [193]:
print(f"{freezer_log[1:4]=}")
print(f"{freezer_log[::-2]=}")
print(f"{freezer_log[slice(1, 4)].readings == freezer_log[1:4].readings=}")
print(f"{slice(None, None, -2).indices(len(freezer_log))=}")

freezer_log[1:4]=SensorLog('freezer', [-17.5, -19.2, -12.4])
freezer_log[::-2]=SensorLog('freezer', [-18.1, -19.2, -18.0])
freezer_log[slice(1, 4)].readings == freezer_log[1:4].readings=True
slice(None, None, -2).indices(len(freezer_log))=(4, -1, -2)


**`__setitem__` and `__delitem__`.** The subclass adds writes with validation. Both
receive the same `int` or `slice` keys as `__getitem__`; here they hand them to the
list underneath.

In [194]:
class EditableSensorLog(SensorLog):
    """Adds validated writes and deletes."""

    def __setitem__(self, index: int | slice, value: float | list[float]) -> None:
        values = value if isinstance(index, slice) else [value]
        if any(not -90 <= reading <= 60 for reading in values):  # type: ignore[union-attr]
            raise ValueError("readings must be between -90 and 60 °C")
        self.readings[index] = value  # type: ignore[index]

    def __delitem__(self, index: int | slice) -> None:
        del self.readings[index]


editable_log = EditableSensorLog("fridge", [4.0, 4.2, 99.0, 3.9])
del editable_log[2]
editable_log[0] = 4.1
editable_log[1:3] = [4.3, 4.4]
print(f"{editable_log=}")
print(raised_message(lambda: editable_log.__setitem__(0, 150.0)))

editable_log=EditableSensorLog('fridge', [4.1, 4.3, 4.4])
ValueError: readings must be between -90 and 60 °C


**`__contains__`.** Without it, `in` scans with `__iter__` or `__getitem__`. Writing
it lets you change the meaning: this log answers `in` for a reading within ±0.5 °C,
because comparing floats exactly is fragile (Unit 1.3.2).

In [195]:
class TolerantSensorLog(SensorLog):
    def __contains__(self, target: object) -> bool:
        if not isinstance(target, (int, float)):
            return False
        return any(abs(reading - target) <= 0.5 for reading in self.readings)


tolerant_log = TolerantSensorLog("freezer", freezer_log.readings)
print(f"{-17.5 in freezer_log=}, {-17.2 in freezer_log=}  (fallback: exact scan)")
print(f"{-17.2 in tolerant_log=}, {-15.0 in tolerant_log=}, {'cold' in tolerant_log=}")

-17.5 in freezer_log=True, -17.2 in freezer_log=False  (fallback: exact scan)
-17.2 in tolerant_log=True, -15.0 in tolerant_log=False, 'cold' in tolerant_log=False


**`__iter__`.** `SensorLog` has no `__iter__`, yet `for` works through the
`__getitem__` fallback (calling indexes 0, 1, 2, … until `IndexError`). Writing
`__iter__` as a generator is clearer and faster, and it is what
`collections.abc.Iterable` checks for.

In [196]:
class IterableSensorLog(SensorLog):
    def __iter__(self) -> Iterator[float]:
        yield from self.readings


print(f"{list(freezer_log)=}  (via __getitem__ fallback)")
print(f"{[reading for reading in IterableSensorLog('freezer', [-18.0, -17.5])]=}")
print(f"{'__iter__' in dir(freezer_log)=}, {'__iter__' in dir(IterableSensorLog)=}")

list(freezer_log)=[-18.0, -17.5, -19.2, -12.4, -18.1]  (via __getitem__ fallback)
[reading for reading in IterableSensorLog('freezer', [-18.0, -17.5])]=[-18.0, -17.5]
'__iter__' in dir(freezer_log)=False, '__iter__' in dir(IterableSensorLog)=True


**`__reversed__`.** `reversed(x)` uses `__reversed__` if present; otherwise it needs
both `__len__` and `__getitem__` and walks the indexes backwards. Write it when you
can reverse more cheaply, or when the object has no indexes at all.

In [197]:
class ReversibleSensorLog(IterableSensorLog):
    def __reversed__(self) -> Iterator[float]:
        print("  (custom __reversed__)")
        return reversed(self.readings)


print(f"{list(reversed(freezer_log))=}  (len + getitem fallback)")
print(f"{list(reversed(ReversibleSensorLog('freezer', [-18.0, -17.5, -19.2])))=}")

list(reversed(freezer_log))=[-18.1, -12.4, -19.2, -17.5, -18.0]  (len + getitem fallback)
  (custom __reversed__)
list(reversed(ReversibleSensorLog('freezer', [-18.0, -17.5, -19.2])))=[-19.2, -17.5, -18.0]


**`__missing__`.** Only for `dict` subclasses: when `dict.__getitem__` cannot find a
key, it calls `self.__missing__(key)` and returns its result. `defaultdict` is built
on it (Unit 1.6.11). Note that `.get()` and `in` do **not** call it. A handy use is
`str.format_map`, which leaves unknown placeholders in place:

In [198]:
class KeepPlaceholders(dict[str, str]):
    """Unknown keys format as themselves, e.g. '{coupon}'."""

    def __missing__(self, key: str) -> str:
        return "{" + key + "}"


template = "Hi {name}, use {coupon} before {date}."
values = KeepPlaceholders(name="Asha", date="31 Dec")
print(template.format_map(values))
print(f"{values['coupon']=}, {values.get('coupon')=}, {'coupon' in values=}")

Hi Asha, use {coupon} before 31 Dec.
values['coupon']='{coupon}', values.get('coupon')=None, 'coupon' in values=False


#### Why subclass `UserDict` / `UserList` instead of `dict` / `list`

The built-in `dict` and `list` are written in C, and their methods call each other
**directly in C**, not through your overrides. Override `__setitem__` on a `dict`
subclass and `dict(...)`, `update()` and `setdefault()` silently skip it. Here a
form-data dict is meant to strip whitespace from every value:

In [199]:
class StrippedFormDict(dict[str, str]):
    """Meant to strip every value; only d[key] = value does."""

    def __setitem__(self, key: str, value: str) -> None:
        super().__setitem__(key, value.strip())


built_in_form = StrippedFormDict(name="  Asha ")
built_in_form.update(city="  Pune ")
built_in_form.setdefault("pin", " 411001 ")
built_in_form["email"] = "  asha@example.com "
print(built_in_form)

{'name': '  Asha ', 'city': '  Pune ', 'pin': ' 411001 ', 'email': 'asha@example.com'}


`collections.UserDict` is a pure-Python class that keeps the real dict in
`self.data` and implements `update`, `setdefault`, the constructor and the rest **in
terms of** `__setitem__` / `__getitem__` (it is a `MutableMapping`, Unit 2.3.2). One
override now covers every path.

In [200]:
class StrippedForm(UserDict[str, str]):
    """Strips every value, whichever method stored it."""

    def __setitem__(self, key: str, value: str) -> None:
        super().__setitem__(key, value.strip())


user_dict_form = StrippedForm(name="  Asha ")
user_dict_form.update(city="  Pune ")
user_dict_form.setdefault("pin", " 411001 ")
user_dict_form["email"] = "  asha@example.com "
print(user_dict_form)
print(f"{type(user_dict_form.data)=}, {isinstance(user_dict_form, MutableMapping)=}")

{'name': 'Asha', 'city': 'Pune', 'pin': '411001', 'email': 'asha@example.com'}
type(user_dict_form.data)=<class 'dict'>, isinstance(user_dict_form, MutableMapping)=True


`UserList` has a different benefit: its operations **return your subclass**. A
`list` subclass loses its type on slicing, `+` and `*`, so methods you added
disappear. `UserList` builds results with `self.__class__(...)`.

In [201]:
class TagList(list[str]):
    def as_hashtags(self) -> str:
        return " ".join(f"#{tag}" for tag in self)


class UserTagList(UserList[str]):
    def as_hashtags(self) -> str:
        return " ".join(f"#{tag}" for tag in self)


for tags in (TagList(["python", "oop", "dunder"]), UserTagList(["python", "oop", "dunder"])):
    slice_type, sum_type = type(tags[1:]).__name__, type(tags + tags).__name__
    print(f"{type(tags).__name__}: slice -> {slice_type}, + -> {sum_type}")
print(f"{UserTagList(['python', 'oop', 'dunder'])[1:].as_hashtags()=}")

TagList: slice -> list, + -> list
UserTagList: slice -> UserTagList, + -> UserTagList
UserTagList(['python', 'oop', 'dunder'])[1:].as_hashtags()='#oop #dunder'


> **Note:** subclassing `dict` or `list` is fine when you only **add** methods or
> use `__missing__`. Reach for `UserDict` / `UserList` (or `MutableMapping` /
> `MutableSequence`) when you need to **change** how items are stored or returned.

### 2.4.7 Callables

Any object whose type defines `__call__` can be called with `()`. A class with
`__call__` is a **stateful function object**: like a closure (Unit 1.11) it keeps
state between calls, but the state is visible as attributes and it can have more
methods. `IdGenerator` hands out readable order numbers.

In [202]:
class IdGenerator:
    """Calling it returns the next id: ORD-0001, ORD-0002, …"""

    def __init__(self, prefix: str, start: int = 1) -> None:
        self.prefix = prefix
        self.next_number = start

    def __call__(self) -> str:
        new_id = f"{self.prefix}-{self.next_number:04d}"
        self.next_number += 1
        return new_id


next_order_id = IdGenerator("ORD")
print(next_order_id(), next_order_id(), next_order_id())
print(f"{next_order_id.next_number=}  (state is a plain attribute)")
print(f"{callable(next_order_id)=}")

ORD-0001 ORD-0002 ORD-0003
next_order_id.next_number=4  (state is a plain attribute)
callable(next_order_id)=True


Because it is called like a function, it fits anywhere a function is expected:
`map`, `sorted(key=...)`, callbacks. `inspect.signature` reads the signature of
`__call__`. This `TaxCalculator` keeps its rate and counts how often it ran.

In [203]:
class TaxCalculator:
    """Adds tax to a price; remembers how many prices it handled."""

    def __init__(self, rate_percent: Decimal) -> None:
        self.rate_percent = rate_percent
        self.call_count = 0

    def __call__(self, price: Decimal) -> Decimal:
        self.call_count += 1
        with_tax = price * (1 + self.rate_percent / 100)
        return with_tax.quantize(PAISE, rounding=ROUND_HALF_UP)


add_gst = TaxCalculator(Decimal("18"))
print(list(map(add_gst, [Decimal("100"), Decimal("49.99")])))
print(f"{add_gst.call_count=}, {inspect.signature(add_gst)=}")

[Decimal('118.00'), Decimal('58.99')]
add_gst.call_count=2, inspect.signature(add_gst)=<Signature (price: decimal.Decimal) -> decimal.Decimal>


Only the **type's** `__call__` counts. Putting a `__call__` attribute on an instance
does not make it callable (2.4.9 explains why), which is a common surprise when
people try to "patch" an object into a function.

In [204]:
class NotCallable:
    pass


plain_object = NotCallable()
plain_object.__call__ = lambda: "hello"  # type: ignore[attr-defined]
print(f"{plain_object.__call__()=}  (explicit attribute call works)")
print(raised_message(lambda: plain_object()))

plain_object.__call__()='hello'  (explicit attribute call works)
TypeError: 'NotCallable' object is not callable


### 2.4.8 Numeric emulation

Arithmetic operators map to special methods; each binary one has a **reflected**
(`__r*__`) form for when your object is on the right, and an **in-place**
(`__i*__`) form for augmented assignment.

| Operator | Method | Reflected | In-place |
| --- | --- | --- | --- |
| `a + b` | `__add__` | `__radd__` | `__iadd__` (`+=`) |
| `a - b` | `__sub__` | `__rsub__` | `__isub__` |
| `a * b` | `__mul__` | `__rmul__` | `__imul__` |
| `a / b` | `__truediv__` | `__rtruediv__` | `__itruediv__` |
| `a // b`, `a % b`, `a ** b` | `__floordiv__`, `__mod__`, `__pow__` | `__r…__` | `__i…__` |
| `a @ b` | `__matmul__` | `__rmatmul__` | `__imatmul__` |
| `-a`, `+a`, `abs(a)`, `~a` | `__neg__`, `__pos__`, `__abs__`, `__invert__` | — | — |

The demos use an immutable `Duration` (whole seconds). Each method below gets its
own cell; the class is extended one step at a time with subclasses so every cell
stays small.

In [205]:
class Duration:
    """An immutable span of whole seconds."""

    def __init__(self, seconds: int) -> None:
        self._seconds = seconds

    @property
    def seconds(self) -> int:
        return self._seconds

    def __repr__(self) -> str:
        return f"Duration({self._seconds})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Duration):
            return NotImplemented
        return self._seconds == other._seconds

    def __hash__(self) -> int:
        return hash(self._seconds)

#### `__add__`

Return a **new** object (the class is immutable) and `NotImplemented` for types you
do not handle. Note `type(self)(...)`, so subclasses get their own type back.

In [206]:
class Duration(Duration):  # each step extends the previous Duration
    def __add__(self, other: object) -> Duration:
        if not isinstance(other, Duration):
            return NotImplemented
        return type(self)(self.seconds + other.seconds)


print(f"{Duration(90) + Duration(30)=}")
print(raised_message(lambda: Duration(90) + 30))

Duration(90) + Duration(30)=Duration(120)
TypeError: unsupported operand type(s) for +: 'Duration' and 'int'


#### `__radd__` (for `sum`)

`sum(items)` starts from `0` and computes `0 + first`. `int.__add__` returns
`NotImplemented` for a `Duration`, so Python calls `first.__radd__(0)`. Without
`__radd__`, `sum` fails. Treating `0` as "no time" fixes it; passing
`start=Duration(0)` also works, but callers forget it.

In [207]:
print(raised_message(lambda: sum([Duration(60), Duration(30)])))
print(f"{sum([Duration(60), Duration(30)], start=Duration(0))=}")


class Duration(Duration):
    def __radd__(self, other: object) -> Duration:
        if other == 0:  # sum() starts from int 0
            return self
        return self.__add__(other)


print(f"{sum([Duration(60), Duration(30), Duration(5)])=}")
print(f"{0 + Duration(45)=}")

TypeError: unsupported operand type(s) for +: 'int' and 'Duration'
sum([Duration(60), Duration(30)], start=Duration(0))=Duration(90)
sum([Duration(60), Duration(30), Duration(5)])=Duration(95)
0 + Duration(45)=Duration(45)


#### `__iadd__`

`total += x` calls `__iadd__` if it exists; otherwise it falls back to
`total = total + x`. An immutable type should **not** define `__iadd__`: the name is
simply rebound to a new object. A mutable type defines it to change itself in place
and return `self`, so every alias sees the change (like `list +=`, Unit 1.2).

In [208]:
running_total = Duration(0)
alias_of_total = running_total
running_total += Duration(30)
print(f"{running_total=}, {alias_of_total=}, {running_total is alias_of_total=}")


class Timesheet:
    """Mutable: += adds a duration to this very timesheet."""

    def __init__(self) -> None:
        self.entries: list[Duration] = []

    def __iadd__(self, entry: Duration) -> Self:
        self.entries.append(entry)
        return self


my_timesheet = Timesheet()
shared_view = my_timesheet
my_timesheet += Duration(3600)
print(f"{shared_view.entries=}, {my_timesheet is shared_view=}")

running_total=Duration(30), alias_of_total=Duration(0), running_total is alias_of_total=False
shared_view.entries=[Duration(3600)], my_timesheet is shared_view=True


#### `__sub__`

Subtraction is not commutative, so `__rsub__` (if you write one) must compute
`other - self`, not `self - other`.

In [209]:
class Duration(Duration):
    def __sub__(self, other: object) -> Duration:
        if not isinstance(other, Duration):
            return NotImplemented
        return type(self)(self.seconds - other.seconds)


print(f"{Duration(300) - Duration(45)=}, {Duration(45) - Duration(300)=}")

Duration(300) - Duration(45)=Duration(255), Duration(45) - Duration(300)=Duration(-255)


#### `__mul__`

Multiplying a duration by a whole number makes sense; multiplying two durations
does not (seconds²). `__rmul__` handles `3 * duration`, where `int.__mul__` gives up
first.

In [210]:
class Duration(Duration):
    def __mul__(self, factor: object) -> Duration:
        if not isinstance(factor, int):
            return NotImplemented
        return type(self)(self.seconds * factor)

    __rmul__ = __mul__  # multiplication by a number is commutative


print(f"{Duration(25) * 4=}, {4 * Duration(25)=}")
print(raised_message(lambda: Duration(25) * Duration(4)))

Duration(25) * 4=Duration(100), 4 * Duration(25)=Duration(100)
TypeError: unsupported operand type(s) for *: 'Duration' and 'Duration'


#### `__truediv__`

One method, two meanings chosen by the operand type: `Duration / int` splits a
duration into equal parts (rounded to whole seconds), `Duration / Duration` is a
plain ratio (`float`). `int / Duration` stays unsupported because there is no
`__rtruediv__`.

In [211]:
class Duration(Duration):
    def __truediv__(self, divisor: object) -> Duration | float:
        if isinstance(divisor, Duration):
            return self.seconds / divisor.seconds
        if isinstance(divisor, int):
            return type(self)(round(self.seconds / divisor))
        return NotImplemented


print(f"{Duration(3600) / 4=}")
print(f"{Duration(5400) / Duration(3600)=}")
print(raised_message(lambda: 10 / Duration(5)))

Duration(3600) / 4=Duration(900)
Duration(5400) / Duration(3600)=1.5
TypeError: unsupported operand type(s) for /: 'int' and 'Duration'


#### `__neg__`

Unary minus. Negative durations are allowed here (they come out of subtraction
anyway), so `-d` simply flips the sign.

In [212]:
class Duration(Duration):
    def __neg__(self) -> Duration:
        return type(self)(-self.seconds)


negative_span = -Duration(90)
print(f"{negative_span=}, {-negative_span=}")

negative_span=Duration(-90), -negative_span=Duration(90)


#### `__abs__`

`abs(x)` calls `__abs__`. Combined with subtraction it answers "how far apart are
two times", whichever comes first.

In [213]:
class Duration(Duration):
    def __abs__(self) -> Duration:
        return type(self)(abs(self.seconds))


planned, actual = Duration(1800), Duration(2100)
print(f"{abs(planned - actual)=}, {abs(actual - planned)=}")

abs(planned - actual)=Duration(300), abs(actual - planned)=Duration(300)


#### `__matmul__`

`@` (PEP 465) exists for matrix multiplication and no built-in type uses it; NumPy
does. A tiny 2×2 matrix shows it: rotating by 90° twice equals rotating by 180°, and
`matrix @ (x, y)` transforms a point (reflected `__rmatmul__` is not needed because
the matrix is always on the left).

In [214]:
class Matrix2x2:
    """A 2×2 integer matrix ((a, b), (c, d))."""

    def __init__(self, a: int, b: int, c: int, d: int) -> None:
        self.rows = ((a, b), (c, d))

    def __repr__(self) -> str:
        return f"Matrix2x2{self.rows}"

    def __matmul__(self, other: object) -> Matrix2x2 | tuple[int, int]:
        (a, b), (c, d) = self.rows
        if isinstance(other, Matrix2x2):
            (e, f), (g, h) = other.rows
            return Matrix2x2(a * e + b * g, a * f + b * h, c * e + d * g, c * f + d * h)
        if isinstance(other, tuple) and len(other) == 2:
            x, y = other
            return (a * x + b * y, c * x + d * y)
        return NotImplemented


rotate_90 = Matrix2x2(0, -1, 1, 0)
print(f"{rotate_90 @ rotate_90=}  (rotate 180°)")
print(f"{rotate_90 @ (3, 1)=}")

rotate_90 @ rotate_90=Matrix2x2((-1, 0), (0, -1))  (rotate 180°)
rotate_90 @ (3, 1)=(-1, 3)


The finished `Duration` now works with every operator above, plus `sum()`:

In [215]:
work_sessions = [Duration(1500), Duration(1500), Duration(900)]
total_work = sum(work_sessions)
print(f"{total_work=}, {total_work / len(work_sessions)=}")
print(f"{total_work / Duration(3600)=}, {abs(-total_work)=}")
print(f"{len(Duration.__mro__)=}  (eight Duration steps, then object)")

total_work=Duration(3900), total_work / len(work_sessions)=Duration(1300)
total_work / Duration(3600)=1.0833333333333333, abs(-total_work)=Duration(3900)
len(Duration.__mro__)=9  (eight Duration steps, then object)


### 2.4.9 Special-method lookup

**Implicit calls look up the type, not the instance.** `len(x)` runs
`type(x).__len__(x)`; it never looks in `x.__dict__`. An explicit `x.__len__()` is a
normal attribute lookup and **does** see the instance attribute.

In [216]:
class Shelf:
    def __len__(self) -> int:
        return 3


oak_shelf = Shelf()
oak_shelf.__len__ = lambda: 99  # type: ignore[method-assign]
print(f"{len(oak_shelf)=}  (type lookup: Shelf.__len__)")
print(f"{oak_shelf.__len__()=}  (normal attribute lookup finds the instance attribute)")

len(oak_shelf)=3  (type lookup: Shelf.__len__)
oak_shelf.__len__()=99  (normal attribute lookup finds the instance attribute)


Two reasons for the rule. **Speed**: the interpreter reads the method from a slot in
the type struct (Unit 3.2) instead of doing a dictionary lookup. **Correctness**:
classes are objects too. `repr(Money)` must call `type.__repr__` (the metaclass,
Unit 2.5.7), not the `Money.__repr__` that is meant for instances, and `hash(int)`
must hash the class, not call `int.__hash__` with no argument.

In [217]:
print(f"{repr(Money)=}  (type(Money).__repr__)")
print(raised_message(lambda: Money.__repr__()))
print(f"{hash(int) == type(int).__hash__(int)=}")
print(raised_message(lambda: int.__hash__()))

repr(Money)="<class '__main__.Money'>"  (type(Money).__repr__)
TypeError: Money.__repr__() missing 1 required positional argument: 'self'
hash(int) == type(int).__hash__(int)=True
TypeError: descriptor '__hash__' of 'int' object needs an argument


#### Under the hood: `dis` and `BINARY_OP`

Every binary operator compiles to one `BINARY_OP` instruction whose argument says
which operator (`+`, `+=`, `@`, …); `x += y` uses the in-place variant. The
instruction does the whole protocol: `__add__`, `NotImplemented`, `__radd__`,
`TypeError`. (Instruction names change between versions; this is 3.15.)

In [218]:
def apply_charges(subtotal: Money, delivery_fee: Money) -> Money:
    """Two operators: + and +=."""
    total = subtotal + delivery_fee
    total += delivery_fee
    return total


for instruction in dis.get_instructions(apply_charges):
    if instruction.opname == "BINARY_OP":
        print(f"{instruction.opname} {instruction.arg} ({instruction.argrepr})")

BINARY_OP 0 (+)
BINARY_OP 13 (+=)


**The `__add__` → `__radd__` sequence, traced.** For `left + right`:

1. If `type(right)` is a **subclass** of `type(left)` and overrides `__radd__`, try
   `right.__radd__(left)` first.
2. Otherwise try `left.__add__(right)`.
3. If that is missing or returns `NotImplemented`, try `right.__radd__(left)`
   (skipped when both operands have the same type).
4. If everything returns `NotImplemented`, raise `TypeError: unsupported operand
   type(s)`.

In [219]:
class LoudLeft:
    def __add__(self, other: object) -> object:
        print("  LoudLeft.__add__ -> NotImplemented")
        return NotImplemented


class LoudRight:
    def __radd__(self, other: object) -> str:
        print("  LoudRight.__radd__ -> handled")
        return "sum from the right"


class Decliner:
    def __radd__(self, other: object) -> object:
        print("  Decliner.__radd__ -> NotImplemented")
        return NotImplemented


print(f"result: {LoudLeft() + LoudRight()!r}")
print(raised_message(lambda: LoudLeft() + Decliner()))

  LoudLeft.__add__ -> NotImplemented
  LoudRight.__radd__ -> handled
result: 'sum from the right'
  LoudLeft.__add__ -> NotImplemented
  Decliner.__radd__ -> NotImplemented
TypeError: unsupported operand type(s) for +: 'LoudLeft' and 'Decliner'


Step 1, the subclass rule, in action: the right operand is tried first even though
the left one could have answered.

In [220]:
class Points:
    def __init__(self, value: int) -> None:
        self.value = value

    def __add__(self, other: object) -> object:
        print("  Points.__add__")
        if not isinstance(other, Points):
            return NotImplemented
        return Points(self.value + other.value)


class BonusPoints(Points):
    def __radd__(self, other: object) -> object:
        print("  BonusPoints.__radd__ (subclass goes first)")
        if not isinstance(other, Points):
            return NotImplemented
        return BonusPoints(other.value + self.value * 2)


print(f"{type(Points(10) + BonusPoints(5)).__name__=}")

  BonusPoints.__radd__ (subclass goes first)
type(Points(10) + BonusPoints(5)).__name__='BonusPoints'


A pure-Python model of the algorithm makes the rules concrete. It agrees with the
real `+` on every pair below (the real one is in C, inside `BINARY_OP`).

In [221]:
def model_add(left: object, right: object) -> object:
    """Mimic how `left + right` picks __add__ / __radd__ (ignores C-level slots)."""
    left_type, right_type = type(left), type(right)
    reflected = getattr(right_type, "__radd__", None)
    try_right_first = (
        right_type is not left_type
        and issubclass(right_type, left_type)
        and reflected is not getattr(left_type, "__radd__", None)
    )
    attempts = []
    if try_right_first:
        attempts.append(lambda: reflected(right, left))
    if (forward := getattr(left_type, "__add__", None)) is not None:
        attempts.append(lambda: forward(left, right))
    if reflected is not None and right_type is not left_type and not try_right_first:
        attempts.append(lambda: reflected(right, left))
    for attempt in attempts:
        if (result := attempt()) is not NotImplemented:
            return result
    operand_names = f"{left_type.__name__!r} and {right_type.__name__!r}"
    raise TypeError(f"unsupported operand type(s) for +: {operand_names}")


def outcome(thunk: Callable[[], object]) -> str:
    """repr of the result, or the TypeError message."""
    try:
        return repr(thunk())
    except TypeError as error:
        return f"TypeError: {error}"

Compare the model with the real operator on a few pairs, including the
`0 + Duration` step that `sum()` performs and one unsupported pair.

In [222]:
pairs = [(2, 3), ("ab", "cd"), (0, Duration(5)), (Duration(5), Duration(7)), (Duration(5), "x")]
for left, right in pairs:
    model_result = outcome(lambda: model_add(left, right))  # noqa: B023 (called at once)
    real_result = outcome(lambda: left + right)  # noqa: B023
    print(f"{left!r} + {right!r}: {model_result}  same={model_result == real_result}")

2 + 3: 5  same=True
'ab' + 'cd': 'abcd'  same=True
0 + Duration(5): Duration(5)  same=True
Duration(5) + Duration(7): Duration(12)  same=True
Duration(5) + 'x': TypeError: unsupported operand type(s) for +: 'Duration' and 'str'  same=True


**Predict the output.**

```python
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

print(Point(1, 2) == Point(1, 2))
print(len({Point(1, 2), Point(1, 2)}))
```

<details><summary>Answer</summary>

```text
True
TypeError: cannot use 'Point' as a set element (unhashable type: 'Point')
```

The first line works: `__eq__` compares values. The second line never prints a
number: defining `__eq__` without `__hash__` sets `Point.__hash__ = None`, so the
set literal raises `TypeError` (the message names the set element on 3.15; older
versions say only `unhashable type: 'Point'`). Fix it with
`def __hash__(self): return hash((self.x, self.y))` and keep `x` and `y` immutable
(or use `@dataclass(frozen=True)`, Unit 2.6).

</details>

Run the next cell to check your prediction.

In [223]:
class Point:
    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y

    def __eq__(self, other: object) -> bool:
        return (self.x, self.y) == (other.x, other.y)  # type: ignore[attr-defined]


print(Point(1, 2) == Point(1, 2))
print(raised_message(lambda: len({Point(1, 2), Point(1, 2)})))

True
TypeError: cannot use 'Point' as a set element (unhashable type: 'Point')


### Lab 2.4

Six exercises. For each, complete the stub, then run the check cell. Until then the
checks print FAIL; that is expected. In exercise 4 four checks already pass on the
stub, because they test behaviour `UserDict` gives for free: "missing key", "copy
keeps type", "is a UserDict" and "is a MutableMapping". Keep them passing.

#### Exercise 1 — an immutable `Money`

Finish the `Money` started in 2.4.2–2.4.3. This is the `Money` the Phase 2 Project
uses.

- `Money(amount, currency="INR")`: `amount` is a `Decimal`, `int` or `str`,
  converted to `Decimal` and rounded to the paisa with `ROUND_HALF_UP`. A `float`
  raises `TypeError("amount must be Decimal, int or str, not float")`. A currency
  that is not three upper-case letters raises
  `ValueError(f"invalid currency code: {currency!r}")`.
- **Immutable**: `amount` and `currency` are read-only properties, and
  `__slots__ = ("_amount", "_currency")` stops new attributes (Unit 2.5.5 explains
  slots).
- `repr` → `Money('1234.50', 'INR')`; `str` → `INR 1,234.50`; `format` as in
  2.4.2 (`""`, `"symbol"`, otherwise the spec applies to the amount).
- `==` and `hash` by value (amount and currency); comparing with a non-`Money`
  is simply unequal.
- Ordering (`<`, `<=`, `>`, `>=`) within one currency. Different currencies raise
  `CurrencyMismatchError(f"currency mismatch: {a} vs {b}")` with the two codes;
  a non-`Money` operand raises `TypeError` (return `NotImplemented`).
- `+` and `-` between `Money` of the same currency (mismatch: the same error);
  `0 + money` returns the money, so `sum()` works; any other `int` is a `TypeError`.
- `money * n` and `n * money` for `int` or `Decimal` (result rounded to the paisa);
  `float` or `Money` factors raise `TypeError`.
- `-money`, `abs(money)`; `bool(money)` is `False` only for zero.

In [224]:
class CurrencyMismatchError(ValueError):
    """Two Money values in different currencies were combined or compared."""


CURRENCY_SYMBOLS = {"INR": "₹", "USD": "$", "EUR": "€"}


class Money:
    """An immutable amount of money in one currency."""

    __slots__ = ("_amount", "_currency")

    def __init__(self, amount: Decimal | int | str, currency: str = "INR") -> None:
        raise NotImplementedError("Lab 2.4: implement Money")

Run the checks.

In [225]:
def money_errors() -> list[str]:
    """Messages for the constructor and mismatch errors."""
    return [
        raised_message(lambda: Money(1.5)),
        raised_message(lambda: Money(1, "inr")),
        raised_message(lambda: Money(1) + Money(1, "USD")),
        raised_message(lambda: Money(1) < Money(1, "USD")),
    ]


run_checks(
    [
        ("rounds to paise", lambda: Money("10.005").amount, Decimal("10.01")),
        (
            "int and str amounts",
            lambda: (Money(10).amount, Money("2.5").currency),
            (Decimal("10.00"), "INR"),
        ),
        (
            "error messages",
            money_errors,
            [
                "TypeError: amount must be Decimal, int or str, not float",
                "ValueError: invalid currency code: 'inr'",
                "CurrencyMismatchError: currency mismatch: INR vs USD",
                "CurrencyMismatchError: currency mismatch: INR vs USD",
            ],
        ),
        (
            "read-only amount",
            lambda: raised_error_name(lambda: set_attribute(Money(1), "amount", 5)),
            "AttributeError",
        ),
        (
            "no new attributes",
            lambda: raised_error_name(lambda: set_attribute(Money(1), "note", "x")),
            "AttributeError",
        ),
        ("repr", lambda: repr(Money("1234.5")), "Money('1234.50', 'INR')"),
        ("str", lambda: str(Money("1234.5", "USD")), "USD 1,234.50"),
        (
            "format",
            lambda: f"{Money('1234.5'):symbol}|{Money(7, 'USD'):symbol}|{Money(1234):,.0f}",
            "₹1,234.50|$7.00|INR 1,234",
        ),
        (
            "equality",
            lambda: (Money(10) == Money("10.00"), Money(10) == Money(10, "USD"), Money(10) == 10),
            (True, False, False),
        ),
        ("hash", lambda: len({Money(10), Money("10.0"), Money(10, "USD")}), 2),
        (
            "ordering",
            lambda: (
                Money(5) < Money(7),
                Money(7) <= Money(7),
                Money(9) > Money(7),
                Money(1) >= Money(2),
            ),
            (True, True, True, False),
        ),
        (
            "sorted and max",
            lambda: [
                str(money)
                for money in [*sorted([Money(3), Money(1), Money(2)]), max(Money(3), Money(9))]
            ],
            ["INR 1.00", "INR 2.00", "INR 3.00", "INR 9.00"],
        ),
        ("compare with int", lambda: raised_error_name(lambda: Money(5) < 5), "TypeError"),
        (
            "add and subtract",
            lambda: (str(Money("10.50") + Money("4.75")), str(Money(10) - Money("12.5"))),
            ("INR 15.25", "INR -2.50"),
        ),
        ("sum()", lambda: str(sum([Money(100), Money("49.99"), Money("0.01")])), "INR 150.00"),
        ("0 + money", lambda: repr(0 + Money(5)), "Money('5.00', 'INR')"),
        ("other int", lambda: raised_error_name(lambda: Money(5) + 1), "TypeError"),
        (
            "multiply",
            lambda: (str(Money("19.99") * 3), str(2 * Money(5)), str(Money(10) * Decimal("0.125"))),
            ("INR 59.97", "INR 10.00", "INR 1.25"),
        ),
        ("float factor", lambda: raised_error_name(lambda: Money(10) * 1.5), "TypeError"),
        ("money factor", lambda: raised_error_name(lambda: Money(10) * Money(2)), "TypeError"),
        (
            "neg and abs",
            lambda: (str(-Money(5)), str(abs(Money(-5, "EUR")))),
            ("INR -5.00", "EUR 5.00"),
        ),
        ("bool", lambda: (bool(Money(0)), bool(Money("0.01"))), (False, True)),
    ]
)

FAIL  rounds to paise: raised NotImplementedError: Lab 2.4: implement Money
FAIL  int and str amounts: raised NotImplementedError: Lab 2.4: implement Money
FAIL  error messages: expected ['TypeError: amount must be Decimal, int or str, not float', "ValueError: invalid currency code: 'inr'", 'CurrencyMismatchError: currency mismatch: INR vs USD', 'CurrencyMismatchError: currency mismatch: INR vs USD'], got ['NotImplementedError: Lab 2.4: implement Money', 'NotImplementedError: Lab 2.4: implement Money', 'NotImplementedError: Lab 2.4: implement Money', 'NotImplementedError: Lab 2.4: implement Money']
FAIL  read-only amount: expected 'AttributeError', got 'NotImplementedError'
FAIL  no new attributes: expected 'AttributeError', got 'NotImplementedError'
FAIL  repr: raised NotImplementedError: Lab 2.4: implement Money
FAIL  str: raised NotImplementedError: Lab 2.4: implement Money
FAIL  format: raised NotImplementedError: Lab 2.4: implement Money
FAIL  equality: raised NotImplementedError:

#### Exercise 2 — `Vector2D`

`Vector2D(x, y)` with numeric components:

- read-only `x` and `y`; `repr` → `Vector2D(3, 4)` (each component with `!r`).
- `==` and `hash` by components; iteration yields `x` then `y`, so
  `x, y = vector` works.
- `v + w`, `v - w` → vectors; `v * k` and `k * v` for an `int` or `float` scalar;
  `-v`; `abs(v)` → length (`math.hypot`); `bool(v)` → `False` only for the zero
  vector; `v @ w` → dot product (a number).
- Unsupported operands raise `TypeError` (return `NotImplemented`).

In [226]:
class Vector2D:
    """An immutable 2-D vector."""

    def __init__(self, x: float, y: float) -> None:
        raise NotImplementedError("Lab 2.4: implement Vector2D")

Run the checks.

In [227]:
run_checks(
    [
        ("repr", lambda: repr(Vector2D(3, 4)), "Vector2D(3, 4)"),
        ("repr floats", lambda: repr(Vector2D(1.5, -2.0)), "Vector2D(1.5, -2.0)"),
        (
            "read-only",
            lambda: raised_error_name(lambda: set_attribute(Vector2D(1, 2), "x", 5)),
            "AttributeError",
        ),
        (
            "equality and hash",
            lambda: (Vector2D(1, 2) == Vector2D(1, 2), len({Vector2D(1, 2), Vector2D(1, 2)})),
            (True, 1),
        ),
        ("not equal to tuple", lambda: Vector2D(1, 2) == (1, 2), False),
        ("unpacking", lambda: tuple(Vector2D(7, 8)), (7, 8)),
        (
            "add and subtract",
            lambda: (repr(Vector2D(1, 2) + Vector2D(3, 4)), repr(Vector2D(1, 2) - Vector2D(3, 4))),
            ("Vector2D(4, 6)", "Vector2D(-2, -2)"),
        ),
        (
            "scalar both sides",
            lambda: (repr(Vector2D(1, 2) * 3), repr(0.5 * Vector2D(4, 6))),
            ("Vector2D(3, 6)", "Vector2D(2.0, 3.0)"),
        ),
        ("negation", lambda: repr(-Vector2D(1, -2)), "Vector2D(-1, 2)"),
        ("abs", lambda: abs(Vector2D(3, 4)), 5.0),
        ("bool", lambda: (bool(Vector2D(0, 0)), bool(Vector2D(0, 1))), (False, True)),
        ("dot product", lambda: Vector2D(1, 2) @ Vector2D(3, 4), 11),
        ("add a number", lambda: raised_error_name(lambda: Vector2D(1, 2) + 5), "TypeError"),
        (
            "vector * vector",
            lambda: raised_error_name(lambda: Vector2D(1, 2) * Vector2D(1, 2)),
            "TypeError",
        ),
    ]
)

FAIL  repr: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  repr floats: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  read-only: expected 'AttributeError', got 'NotImplementedError'
FAIL  equality and hash: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  not equal to tuple: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  unpacking: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  add and subtract: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  scalar both sides: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  negation: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  abs: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  bool: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  dot product: raised NotImplementedError: Lab 2.4: implement Vector2D
FAIL  add a number: expected 'TypeError', got 'NotImplementedError'
FAIL  vector * vector: expected 'TypeErr

#### Exercise 3 — `Playlist` with slicing

`Playlist(name, songs=())` keeps its **own copy** of the songs as a list.

- `len`, `in`, iteration in order, and `reversed()` (write `__reversed__`).
- `playlist[i]` → a song (negative indexes work, out of range is `IndexError`);
  `playlist[a:b:c]` → a **new `Playlist`** with the same name.
- `playlist[i] = song` and `del playlist[i]` (slices allowed for both).
- `==` compares name and songs; `repr` → `Playlist('Road trip', ['A', 'B'])`.

In [228]:
class Playlist:
    """A named, ordered list of songs."""

    def __init__(self, name: str, songs: Iterable[str] = ()) -> None:
        raise NotImplementedError("Lab 2.4: implement Playlist")

Run the checks.

In [229]:
SONGS = ["Kun Faya Kun", "Ilahi", "Kabira", "Safarnama", "Raabta"]


def playlist_edits() -> Playlist:
    """Replace, delete and slice-delete."""
    road_trip = Playlist("Road trip", SONGS)
    road_trip[0] = "Patakha Guddi"
    del road_trip[-1]
    del road_trip[1:3]
    return road_trip


def keeps_own_copy() -> list[str]:
    """Changing the source list must not change the playlist."""
    source = ["A", "B"]
    playlist = Playlist("Copy", source)
    source.append("C")
    return list(playlist)


run_checks(
    [
        ("len", lambda: len(Playlist("Road trip", SONGS)), 5),
        (
            "index",
            lambda: (Playlist("Road trip", SONGS)[0], Playlist("Road trip", SONGS)[-1]),
            ("Kun Faya Kun", "Raabta"),
        ),
        (
            "index error",
            lambda: raised_error_name(lambda: Playlist("Road trip", SONGS)[5]),
            "IndexError",
        ),
        ("slice type", lambda: type(Playlist("Road trip", SONGS)[1:3]).__name__, "Playlist"),
        (
            "slice content",
            lambda: repr(Playlist("Road trip", SONGS)[1:3]),
            "Playlist('Road trip', ['Ilahi', 'Kabira'])",
        ),
        (
            "step slice",
            lambda: list(Playlist("Road trip", SONGS)[::2]),
            ["Kun Faya Kun", "Kabira", "Raabta"],
        ),
        (
            "contains",
            lambda: (
                "Kabira" in Playlist("Road trip", SONGS),
                "Tum Hi Ho" in Playlist("Road trip", SONGS),
            ),
            (True, False),
        ),
        ("reversed", lambda: list(reversed(Playlist("x", ["A", "B", "C"]))), ["C", "B", "A"]),
        ("has __reversed__", lambda: "__reversed__" in vars(Playlist), True),
        (
            "edits",
            lambda: repr(playlist_edits()),
            "Playlist('Road trip', ['Patakha Guddi', 'Safarnama'])",
        ),
        ("own copy", keeps_own_copy, ["A", "B"]),
        ("equality uses name", lambda: Playlist("A", ["x"]) == Playlist("B", ["x"]), False),
        (
            "repr",
            lambda: repr(Playlist("Road trip", ["A", "B"])),
            "Playlist('Road trip', ['A', 'B'])",
        ),
    ]
)

FAIL  len: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  index: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  index error: expected 'IndexError', got 'NotImplementedError'
FAIL  slice type: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  slice content: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  step slice: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  contains: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  reversed: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  has __reversed__: expected True, got False
FAIL  edits: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  own copy: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  equality uses name: raised NotImplementedError: Lab 2.4: implement Playlist
FAIL  repr: raised NotImplementedError: Lab 2.4: implement Playlist
0/13 checks passed


#### Exercise 4 — `CaseInsensitiveDict(UserDict)`

Unit 2.3.2 built a case-insensitive mapping on `MutableMapping`; this time use
`UserDict` and override as little as possible.

- Keys must be `str` (otherwise `TypeError("keys must be str")`) and are stored
  **case-folded** (`key.casefold()`), so iteration yields lower-case keys.
- `d[key]`, `d[key] = value`, `del d[key]` and `key in d` ignore case.
- Because `UserDict` routes through those methods, the constructor, `update`,
  `get`, `setdefault`, `pop` and `copy` must then work case-insensitively with no
  extra code.

In [230]:
class CaseInsensitiveDict(UserDict[str, str]):
    """HTTP-header style dict: keys compare without case."""

Run the checks.

In [231]:
def header_session() -> tuple[object, ...]:
    """Exercise every inherited method."""
    headers = CaseInsensitiveDict({"Content-Type": "text/html"}, Accept="*/*")
    headers.update({"ACCEPT": "application/json"})
    headers.setdefault("X-Request-Id", "abc123")
    headers.setdefault("x-request-id", "ignored")
    popped = headers.pop("CONTENT-TYPE")
    return popped, headers.get("accept"), headers.get("missing", "-"), sorted(headers)


run_checks(
    [
        (
            "getitem ignores case",
            lambda: CaseInsensitiveDict({"Content-Type": "text/html"})["content-TYPE"],
            "text/html",
        ),
        ("contains ignores case", lambda: "ACCEPT" in CaseInsensitiveDict(accept="*/*"), True),
        (
            "one key per spelling",
            lambda: len(CaseInsensitiveDict({"Accept": "a", "ACCEPT": "b"})),
            1,
        ),
        ("keys case-folded", lambda: list(CaseInsensitiveDict({"Accept": "a"})), ["accept"]),
        (
            "inherited methods",
            header_session,
            ("text/html", "application/json", "-", ["accept", "x-request-id"]),
        ),
        (
            "delete",
            lambda: raised_error_name(
                lambda: CaseInsensitiveDict(Accept="a").__delitem__("ACCEPT")
            ),
            "no error",
        ),
        (
            "missing key",
            lambda: raised_error_name(lambda: CaseInsensitiveDict()["accept"]),
            "KeyError",
        ),
        (
            "str keys only",
            lambda: raised_message(lambda: CaseInsensitiveDict({1: "a"})),
            "TypeError: keys must be str",
        ),
        (
            "copy keeps type",
            lambda: type(CaseInsensitiveDict(a="1").copy()).__name__,
            "CaseInsensitiveDict",
        ),
        ("is a UserDict", lambda: isinstance(CaseInsensitiveDict(), UserDict), True),
        ("is a MutableMapping", lambda: isinstance(CaseInsensitiveDict(), MutableMapping), True),
    ]
)

FAIL  getitem ignores case: raised KeyError: 'content-TYPE'
FAIL  contains ignores case: expected True, got False
FAIL  one key per spelling: expected 1, got 2
FAIL  keys case-folded: expected ['accept'], got ['Accept']
FAIL  inherited methods: raised KeyError: 'CONTENT-TYPE'
FAIL  delete: expected 'no error', got 'KeyError'
PASS  missing key
FAIL  str keys only: expected 'TypeError: keys must be str', got 'no error'
PASS  copy keeps type
PASS  is a UserDict
PASS  is a MutableMapping
4/11 checks passed


#### Exercise 5 — a callable `RateLimiter`

`RateLimiter(max_calls, period_seconds, clock=time.monotonic)` allows at most
`max_calls` calls in any window of `period_seconds` (a sliding window: a call at
time `t` still counts while `now - t < period_seconds`).

- `limiter()` returns `True` and records the call when allowed, `False` (and records
  nothing) when the limit is reached.
- `limiter.remaining()` → how many calls are still allowed right now.
- `repr` → `RateLimiter(max_calls=3, period_seconds=10)`.
- `max_calls < 1` → `ValueError("max_calls must be at least 1")`;
  `period_seconds <= 0` → `ValueError("period_seconds must be positive")`.
- `clock` is injected so the checks can control time.

In [232]:
class RateLimiter:
    """Calling the limiter asks: may I make one more call now?"""

    def __init__(
        self,
        max_calls: int,
        period_seconds: float,
        clock: Callable[[], float] = time.monotonic,
    ) -> None:
        raise NotImplementedError("Lab 2.4: implement RateLimiter")

Run the checks. `ManualClock` is a fake clock you move by hand.

In [233]:
class ManualClock:
    """A clock that only moves when told to."""

    def __init__(self) -> None:
        self.now = 0.0

    def __call__(self) -> float:
        return self.now


def limiter_timeline() -> list[object]:
    """Three calls allowed per 10 s; record answers over time."""
    clock = ManualClock()
    limiter = RateLimiter(3, 10, clock=clock)
    answers: list[object] = [limiter(), limiter(), limiter(), limiter(), limiter.remaining()]
    clock.now = 9.9
    answers.append(limiter())
    clock.now = 10.0
    answers.extend([limiter.remaining(), limiter(), limiter()])
    return answers


run_checks(
    [
        ("callable", lambda: callable(RateLimiter(1, 1)), True),
        ("timeline", limiter_timeline, [True, True, True, False, 0, False, 3, True, True]),
        ("remaining at start", lambda: RateLimiter(5, 60, clock=ManualClock()).remaining(), 5),
        ("repr", lambda: repr(RateLimiter(3, 10)), "RateLimiter(max_calls=3, period_seconds=10)"),
        (
            "max_calls",
            lambda: raised_message(lambda: RateLimiter(0, 10)),
            "ValueError: max_calls must be at least 1",
        ),
        (
            "period",
            lambda: raised_message(lambda: RateLimiter(1, 0)),
            "ValueError: period_seconds must be positive",
        ),
    ]
)

FAIL  callable: raised NotImplementedError: Lab 2.4: implement RateLimiter
FAIL  timeline: raised NotImplementedError: Lab 2.4: implement RateLimiter
FAIL  remaining at start: raised NotImplementedError: Lab 2.4: implement RateLimiter
FAIL  repr: raised NotImplementedError: Lab 2.4: implement RateLimiter
FAIL  max_calls: expected 'ValueError: max_calls must be at least 1', got 'NotImplementedError: Lab 2.4: implement RateLimiter'
FAIL  period: expected 'ValueError: period_seconds must be positive', got 'NotImplementedError: Lab 2.4: implement RateLimiter'
0/6 checks passed


#### Exercise 6 — an `Inventory` keyed by SKU

`Inventory(stock=None)` maps SKU → quantity on hand (`stock` is an optional
mapping to start from).

- `inventory[sku]` → quantity; an unknown SKU raises `KeyError`.
- `inventory[sku] = quantity` requires an `int >= 0`, otherwise
  `ValueError("quantity must be a non-negative int")` (the constructor checks
  `stock` the same way); `del inventory[sku]`.
- `len` (number of SKUs), `in`, `bool`; iteration yields SKUs in **sorted** order.
- `add(sku, quantity)` increases stock (a new SKU starts at 0);
  `remove(sku, quantity)` decreases it and raises
  `ValueError(f"not enough {sku}: have {have}, need {quantity}")` when short.
- `total_units` property; `inventory + other` → a **new** `Inventory` with summed
  quantities; `==` compares stock; `repr` → `Inventory({'PEN-01': 5, 'TEA-500': 2})`
  with SKUs sorted.

In [234]:
class Inventory:
    """Quantities on hand, keyed by SKU."""

    def __init__(self, stock: Mapping[str, int] | None = None) -> None:
        raise NotImplementedError("Lab 2.4: implement Inventory")

Run the checks.

In [235]:
def stock_session() -> Inventory:
    """Add, remove, set and delete."""
    inventory = Inventory({"TEA-500": 2})
    inventory.add("PEN-01", 10)
    inventory.add("TEA-500", 3)
    inventory.remove("PEN-01", 4)
    inventory["MUG-02"] = 1
    del inventory["MUG-02"]
    return inventory


def merge_leaves_operands() -> tuple[str, str]:
    """first + second must build a new Inventory; return both reprs."""
    first = Inventory({"A": 1})
    merged = first + Inventory({"A": 1})
    return repr(merged), repr(first)


run_checks(
    [
        ("session", lambda: repr(stock_session()), "Inventory({'PEN-01': 6, 'TEA-500': 5})"),
        ("getitem", lambda: Inventory({"PEN-01": 6})["PEN-01"], 6),
        ("unknown sku", lambda: raised_error_name(lambda: Inventory()["NOPE"]), "KeyError"),
        (
            "sorted iteration",
            lambda: list(Inventory({"TEA-500": 1, "MUG-02": 0, "PEN-01": 3})),
            ["MUG-02", "PEN-01", "TEA-500"],
        ),
        (
            "len, in, bool",
            lambda: (len(Inventory({"A": 1})), "A" in Inventory({"A": 1}), bool(Inventory())),
            (1, True, False),
        ),
        (
            "bad quantity",
            lambda: raised_message(lambda: Inventory().__setitem__("A", -1)),
            "ValueError: quantity must be a non-negative int",
        ),
        (
            "not an int",
            lambda: raised_message(lambda: Inventory({"A": 1.5})),
            "ValueError: quantity must be a non-negative int",
        ),
        (
            "not enough",
            lambda: raised_message(lambda: Inventory({"PEN-01": 2}).remove("PEN-01", 5)),
            "ValueError: not enough PEN-01: have 2, need 5",
        ),
        ("total_units", lambda: Inventory({"A": 2, "B": 3}).total_units, 5),
        (
            "merge with +",
            lambda: repr(Inventory({"A": 2, "B": 1}) + Inventory({"B": 4, "C": 1})),
            "Inventory({'A': 2, 'B': 5, 'C': 1})",
        ),
        (
            "+ leaves operands alone",
            merge_leaves_operands,
            ("Inventory({'A': 2})", "Inventory({'A': 1})"),
        ),
        ("+ with dict", lambda: raised_error_name(lambda: Inventory() + {"A": 1}), "TypeError"),
        (
            "repr",
            lambda: repr(Inventory({"TEA-500": 2, "PEN-01": 5})),
            "Inventory({'PEN-01': 5, 'TEA-500': 2})",
        ),
    ]
)

FAIL  session: raised NotImplementedError: Lab 2.4: implement Inventory
FAIL  getitem: raised NotImplementedError: Lab 2.4: implement Inventory
FAIL  unknown sku: expected 'KeyError', got 'NotImplementedError'
FAIL  sorted iteration: raised NotImplementedError: Lab 2.4: implement Inventory
FAIL  len, in, bool: raised NotImplementedError: Lab 2.4: implement Inventory
FAIL  bad quantity: expected 'ValueError: quantity must be a non-negative int', got 'NotImplementedError: Lab 2.4: implement Inventory'
FAIL  not an int: expected 'ValueError: quantity must be a non-negative int', got 'NotImplementedError: Lab 2.4: implement Inventory'
FAIL  not enough: expected 'ValueError: not enough PEN-01: have 2, need 5', got 'NotImplementedError: Lab 2.4: implement Inventory'
FAIL  total_units: raised NotImplementedError: Lab 2.4: implement Inventory
FAIL  merge with +: raised NotImplementedError: Lab 2.4: implement Inventory
FAIL  + leaves operands alone: raised NotImplementedError: Lab 2.4: implemen

### Recap — Unit 2.4

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Data model | Syntax calls `type(x).__dunder__(x, …)`; implement the method, get the syntax | `Symbol.iterator`, `Symbol.toPrimitive` |
| `__repr__` / `__str__` | repr for developers (and containers), str for users; write repr first | `toString()` |
| `__format__` | Receives the text after `:`; custom specs are yours to define | — |
| `__eq__` + `NotImplemented` | Decline unknown types so the other side (then identity) can answer | — |
| `__hash__` | Defining `__eq__` sets it to `None`; hash only immutable fields | — (`Map` uses identity) |
| Rich comparisons | Reflection is `a < b` → `b > a`; `total_ordering` fills in the rest | — |
| `__bool__` / `__len__` | `__bool__` wins; `len` must be a non-negative int | — |
| Container methods | `__getitem__` gets ints and `slice`s; `__missing__` for dict subclasses | `Proxy` traps |
| `UserDict` / `UserList` | Overrides are honoured by every method; results keep your type | — |
| `__call__` | Stateful function objects | functions with properties |
| Numeric methods | `__add__`, `__radd__` (for `sum`), `__iadd__` only for mutable types | no operator overloading |
| Special-method lookup | Implicit calls use the type, never the instance; `BINARY_OP` runs the protocol | — |

**Further reading**

- [Data model — special method names](https://docs.python.org/3/reference/datamodel.html#special-method-names)
  (the complete list), and
  [special method lookup](https://docs.python.org/3/reference/datamodel.html#special-method-lookup)
- [`functools.total_ordering`](https://docs.python.org/3/library/functools.html#functools.total_ordering),
  [`collections.UserDict` / `UserList`](https://docs.python.org/3/library/collections.html#userdict-objects),
  [`NotImplemented`](https://docs.python.org/3/library/constants.html#NotImplemented)
- [Format specification mini-language](https://docs.python.org/3/library/string.html#formatspec),
  [`dis` — `BINARY_OP`](https://docs.python.org/3/library/dis.html#opcode-BINARY_OP)
- [PEP 207 — Rich comparisons](https://peps.python.org/pep-0207/),
  [PEP 465 — The `@` operator](https://peps.python.org/pep-0465/),
  [PEP 3101 — Advanced string formatting](https://peps.python.org/pep-3101/)

## Unit 2.5 — Attribute Access, Descriptors and Metaprogramming

**Objectives**

- Trace exactly what `obj.name` does: data descriptors, the instance `__dict__`,
  non-data descriptors and class attributes, then `__getattr__`.
- Hook attribute access with `__getattr__`, `__getattribute__`, `__setattr__` and
  `__delattr__` without falling into recursion traps.
- Write descriptors (`__get__`, `__set__`, `__delete__`, `__set_name__`), and see
  that methods, `property`, `classmethod` and `staticmethod` are descriptors.
- Use `__slots__` and measure what it saves.
- Create classes at run time with `type(...)`, write metaclasses, and prefer the
  lighter `__init_subclass__` and class decorators when they are enough.
- Recognize these mechanisms inside Pydantic, SQLAlchemy, Django and dataclasses.

**JS bridge**

| JavaScript | Python | The difference that bites |
| --- | --- | --- |
| Prototype chain lookup | MRO lookup + the descriptor protocol | A class attribute can **intercept** instance access (`__get__`) |
| `Object.defineProperty(obj, "x", {get, set})` | a descriptor object on the class | Defined once on the class, reused by every attribute that needs it |
| `Proxy` with `get` / `set` traps | `__getattr__`, `__getattribute__`, `__setattr__` | `__getattr__` runs only for **missing** names |
| `Object.freeze` / sealed objects | `__slots__` (no new attributes), read-only descriptors | Slots are mainly a memory feature |
| `class` expressions, `Function` constructor | `type(name, bases, namespace)` | Classes are ordinary objects created by calling their metaclass |
| No metaclasses (decorators proposal is the nearest) | `metaclass=` / `__init_subclass__` | Code that runs when a **class** is defined |
| `this` binding with `.bind` | functions are descriptors: `function.__get__` makes bound methods | Binding happens on attribute access, automatically |

The next cell is this unit's setup.

In [236]:
import annotationlib
import copy
import dataclasses
import enum
import functools
import importlib.util
import inspect
import sqlite3
import sys
import timeit
import tracemalloc
import types
import weakref
from abc import ABC, ABCMeta, abstractmethod
from collections.abc import Callable
from decimal import Decimal
from typing import Any


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def set_attribute(target: object, attribute_name: str, value: object) -> None:
    """`target.attribute_name = value` as a call, for use inside raised_error_name(lambda: ...)."""
    setattr(target, attribute_name, value)


def mro_names(cls: type) -> list[str]:
    """Class names along cls.__mro__."""
    return [klass.__name__ for klass in cls.__mro__]


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.5.1 The attribute-lookup algorithm

Reading `obj.name` calls `type(obj).__getattribute__(obj, "name")`. For ordinary
classes that is `object.__getattribute__`, which follows this order:

1. Search `type(obj).__mro__` for `name`. If found and it is a **data descriptor**
   (its type defines `__set__` or `__delete__`, like `property`), call its
   `__get__(obj, type(obj))` and stop.
2. Otherwise, if `name` is in `obj.__dict__`, return that value.
3. Otherwise, if the class attribute from step 1 is a **non-data descriptor**
   (only `__get__`, like a function), call its `__get__`; if it is a plain class
   attribute, return it as is.
4. Otherwise raise `AttributeError`, and if the class defines `__getattr__`, call it
   instead of raising.

```mermaid
flowchart TD
    A["obj.name"] --> B{"name found on type(obj).__mro__?"}
    B -- "yes, data descriptor" --> C["descriptor.__get__(obj, type)"]
    B -- "no, or not a data descriptor" --> D{"name in obj.__dict__?"}
    D -- yes --> E["return obj.__dict__[name]"]
    D -- no --> F{"class attribute found?"}
    F -- "non-data descriptor" --> G["descriptor.__get__(obj, type)"]
    F -- "plain attribute" --> H["return class attribute"]
    F -- no --> I{"__getattr__ defined?"}
    I -- yes --> J["type(obj).__getattr__(obj, name)"]
    I -- no --> K["raise AttributeError"]
```

<details><summary>ASCII fallback</summary>

```text
obj.name
  |
  +-- found on type(obj).__mro__ AND is a data descriptor? --> descriptor.__get__(obj, type)
  |
  +-- name in obj.__dict__? -----------------------------> obj.__dict__[name]
  |
  +-- found on the class?
  |      non-data descriptor (has only __get__) ---------> descriptor.__get__(obj, type)
  |      plain attribute ---------------------------------> the attribute itself
  |
  +-- __getattr__ defined? -------------------------------> type(obj).__getattr__(obj, name)
  |
  +-- AttributeError
```

</details>

The class below has one attribute of each kind, plus `__getattr__`.

In [237]:
class Product:
    """One attribute of every kind."""

    category = "stationery"  # plain class attribute

    def __init__(self, name: str) -> None:
        self.name = name  # instance attribute

    @property
    def label(self) -> str:  # data descriptor
        return f"[{self.name}]"

    def describe(self) -> str:  # function: non-data descriptor
        return f"{self.name} in {self.category}"

    def __getattr__(self, attribute_name: str) -> str:  # last resort
        return f"<no {attribute_name}>"


pen = Product("pen")
for attribute_name in ["label", "name", "describe", "category", "colour"]:
    value = getattr(pen, attribute_name)
    shown = f"bound method {value.__qualname__}" if inspect.ismethod(value) else repr(value)
    print(f"{attribute_name:<9} -> {shown}")

label     -> '[pen]'
name      -> 'pen'
describe  -> bound method Product.describe
category  -> 'stationery'
colour    -> '<no colour>'


**Precedence in action.** Write the same names straight into `pen.__dict__` (to
bypass any setter). The data descriptor `label` still wins over the instance
dictionary; the method and the class attribute are **shadowed** by it.

In [238]:
vars(pen).update(label="shadow label", describe="shadow describe", category="shadow category")
print(f"{pen.label=}  (data descriptor beats the instance dict)")
print(f"{pen.describe=}  (instance dict beats a non-data descriptor)")
print(f"{pen.category=}  (instance dict beats a plain class attribute)")
print(f"{Product.category=}  (the class itself is unchanged)")

pen.label='[pen]'  (data descriptor beats the instance dict)
pen.describe='shadow describe'  (instance dict beats a non-data descriptor)
pen.category='shadow category'  (instance dict beats a plain class attribute)
Product.category='stationery'  (the class itself is unchanged)


#### Under the hood: the algorithm in Python

A simplified `find_attribute` that follows the four steps (it ignores metaclasses
and `__getattribute__` overrides). `inspect.getattr_static` does a similar
descriptor-free lookup and is useful for debugging, because it never runs your
code.

In [239]:
def find_attribute(obj: object, attribute_name: str) -> object:
    """Simplified object.__getattribute__ followed by the __getattr__ fallback."""
    class_attribute, found_on_class = None, False
    for klass in type(obj).__mro__:
        if attribute_name in vars(klass):
            class_attribute, found_on_class = vars(klass)[attribute_name], True
            break
    descriptor_type = type(class_attribute)
    if found_on_class and (
        hasattr(descriptor_type, "__set__") or hasattr(descriptor_type, "__delete__")
    ):
        return descriptor_type.__get__(class_attribute, obj, type(obj))
    if attribute_name in getattr(obj, "__dict__", {}):
        return obj.__dict__[attribute_name]
    if found_on_class and hasattr(descriptor_type, "__get__"):
        return descriptor_type.__get__(class_attribute, obj, type(obj))
    if found_on_class:
        return class_attribute
    if hasattr(type(obj), "__getattr__"):
        return type(obj).__getattr__(obj, attribute_name)
    raise AttributeError(attribute_name)


fresh_pen = Product("pen")
for attribute_name in ["label", "name", "category", "colour"]:
    same_answer = find_attribute(fresh_pen, attribute_name) == getattr(fresh_pen, attribute_name)
    print(f"{attribute_name:<9} same as getattr: {same_answer}")
print(f"{find_attribute(fresh_pen, 'describe')()=}")
print(
    f"{type(inspect.getattr_static(fresh_pen, 'label')).__name__=}  (the descriptor, not its value)"
)

label     same as getattr: True
name      same as getattr: True
category  same as getattr: True
colour    same as getattr: True
find_attribute(fresh_pen, 'describe')()='pen in stationery'
type(inspect.getattr_static(fresh_pen, 'label')).__name__='property'  (the descriptor, not its value)


### 2.5.2 Attribute hooks

| Hook | Runs | Typical use |
| --- | --- | --- |
| `__getattr__(self, name)` | only when normal lookup **fails** | defaults, delegation, lazy loading |
| `__getattribute__(self, name)` | on **every** attribute read | tracing, access control (rarely needed) |
| `__setattr__(self, name, value)` | on every assignment `obj.name = value` | validation, freezing, change tracking |
| `__delattr__(self, name)` | on every `del obj.name` | protecting required attributes |

**`__getattr__`** is the safe one: real attributes never reach it. This settings
object answers from a defaults table for anything not set explicitly.

In [240]:
class Settings:
    """Explicit values win; anything else comes from DEFAULTS."""

    DEFAULTS = {"timeout_seconds": 30, "retries": 3}

    def __init__(self, **overrides: object) -> None:
        self.__dict__.update(overrides)

    def __getattr__(self, setting_name: str) -> object:
        try:
            return self.DEFAULTS[setting_name]
        except KeyError:
            raise AttributeError(f"unknown setting {setting_name!r}") from None


app_settings = Settings(timeout_seconds=5)
print(f"{app_settings.timeout_seconds=}, {app_settings.retries=}")
print(raised_message(lambda: app_settings.colour))
print(f"{hasattr(app_settings, 'colour')=}  (hasattr relies on AttributeError)")

app_settings.timeout_seconds=5, app_settings.retries=3
AttributeError: unknown setting 'colour'
hasattr(app_settings, 'colour')=False  (hasattr relies on AttributeError)


**`__getattribute__`** intercepts every read, including `self.__dict__` and method
lookups. It must delegate to `super().__getattribute__(name)` (or
`object.__getattribute__`) to do the real work.

In [241]:
class AccessCounter:
    """Counts every attribute read."""

    def __init__(self, sku: str) -> None:
        self.sku = sku
        self.reads: dict[str, int] = {}

    def __getattribute__(self, attribute_name: str) -> Any:
        reads = super().__getattribute__("reads")
        reads[attribute_name] = reads.get(attribute_name, 0) + 1
        return super().__getattribute__(attribute_name)


counted = AccessCounter("PEN-01")
_ = counted.sku, counted.sku, counted.__class__
print(f"{object.__getattribute__(counted, 'reads')=}")

object.__getattribute__(counted, 'reads')={'sku': 2, '__class__': 1}


#### Recursion traps

Inside these hooks, `self.anything` goes through the hook again. Reading
`self.reads` inside `__getattribute__`, or assigning `self.x = ...` inside
`__setattr__`, recurses until `RecursionError`. Always go through `super()` (or
`object`) for the real access.

In [242]:
class RecursiveGetter:
    def __getattribute__(self, attribute_name: str) -> Any:
        return self.__dict__[attribute_name]  # self.__dict__ calls this method again


class RecursiveSetter:
    def __setattr__(self, attribute_name: str, value: object) -> None:
        self.attribute_name = value  # assignment calls this method again


print(raised_error_name(lambda: RecursiveGetter().anything))
print(raised_error_name(lambda: set_attribute(RecursiveSetter(), "x", 1)))

RecursionError
RecursionError


A subtler trap: `__getattr__` that reads an attribute set in `__init__`. `copy`,
`pickle` and some frameworks create instances **without calling `__init__`**, so
`self._target` does not exist yet, which calls `__getattr__("_target")`, which reads
`self._target` again... The fix is to refuse private names before delegating.

In [243]:
class NaiveProxy:
    def __init__(self, target: object) -> None:
        self._target = target

    def __getattr__(self, attribute_name: str) -> Any:
        return getattr(self._target, attribute_name)


class SafeProxy(NaiveProxy):
    def __getattr__(self, attribute_name: str) -> Any:
        if attribute_name.startswith("_"):
            raise AttributeError(attribute_name)
        return getattr(self._target, attribute_name)


print(raised_error_name(lambda: copy.copy(NaiveProxy([1, 2]))))
print(f"{copy.copy(SafeProxy([1, 2])).count(2)=}")

RecursionError
copy.copy(SafeProxy([1, 2])).count(2)=1


**`__setattr__`** sees every assignment, including those in `__init__`. This class
validates while it is being built and refuses changes afterwards ("frozen after
init"); `object.__setattr__` is the escape hatch it uses itself.

In [244]:
class ShippingLabel:
    """Validated on creation, read-only afterwards."""

    def __init__(self, tracking_number: str, weight_kg: float) -> None:
        self.tracking_number = tracking_number
        self.weight_kg = weight_kg
        object.__setattr__(self, "_frozen", True)

    def __setattr__(self, attribute_name: str, value: object) -> None:
        if getattr(self, "_frozen", False):
            raise AttributeError(f"ShippingLabel is read-only (tried to set {attribute_name})")
        if attribute_name == "weight_kg" and not 0 < value <= 30:  # type: ignore[operator]
            raise ValueError("weight_kg must be in (0, 30]")
        super().__setattr__(attribute_name, value)


label = ShippingLabel("BD123456789IN", 2.5)
print(raised_message(lambda: set_attribute(label, "weight_kg", 3)))
print(raised_message(lambda: ShippingLabel("BD1", 45)))

AttributeError: ShippingLabel is read-only (tried to set weight_kg)
ValueError: weight_kg must be in (0, 30]


**`__delattr__`** handles `del obj.name`. Here required fields cannot be deleted,
optional ones can.

In [245]:
class CustomerRecord:
    REQUIRED = frozenset({"customer_id", "email"})

    def __init__(self, customer_id: int, email: str, nickname: str) -> None:
        self.customer_id, self.email, self.nickname = customer_id, email, nickname

    def __delattr__(self, attribute_name: str) -> None:
        if attribute_name in self.REQUIRED:
            raise AttributeError(f"{attribute_name} is required and cannot be deleted")
        super().__delattr__(attribute_name)


record = CustomerRecord(7, "asha@example.com", "ash")
del record.nickname
print(f"{vars(record)=}")
print(raised_message(lambda: record.__delattr__("email")))

vars(record)={'customer_id': 7, 'email': 'asha@example.com'}
AttributeError: email is required and cannot be deleted


#### Proxies

A **proxy** stands in for another object. `__getattr__` forwards every *missing*
attribute, so the proxy only defines what it changes. This read-only proxy forwards
reads and blocks writes. Note the limit from Unit 2.4.9: implicit special-method
calls such as `len(proxy)` look on the **type**, so `__getattr__` never sees them;
each dunder you need must be written on the proxy class.

In [246]:
class ReadOnlyProxy:
    """Forwards reads to the target; refuses writes."""

    def __init__(self, target: object) -> None:
        object.__setattr__(self, "_target", target)

    def __getattr__(self, attribute_name: str) -> Any:
        if attribute_name == "_target":  # not set yet (copy, pickle): avoid recursion
            raise AttributeError(attribute_name)
        return getattr(self._target, attribute_name)

    def __setattr__(self, attribute_name: str, value: object) -> None:
        raise AttributeError(f"read-only proxy: cannot set {attribute_name}")


settings_view = ReadOnlyProxy(Settings(timeout_seconds=5))
print(f"{settings_view.timeout_seconds=}, {settings_view.retries=}")
print(raised_message(lambda: set_attribute(settings_view, "retries", 10)))
print(raised_message(lambda: len(ReadOnlyProxy([1, 2, 3]))))
print(f"{ReadOnlyProxy([1, 2, 3]).__len__()=}  (explicit call goes through __getattr__)")

settings_view.timeout_seconds=5, settings_view.retries=3
AttributeError: read-only proxy: cannot set retries
TypeError: object of type 'ReadOnlyProxy' has no len()
ReadOnlyProxy([1, 2, 3]).__len__()=3  (explicit call goes through __getattr__)


#### Lazy loading

`__getattr__` also suits **lazy loading**: fetch expensive data on first use, then
store it in the instance `__dict__` so later reads never reach `__getattr__` again.
ORMs load related rows this way. The fake database counts its queries.

In [247]:
QUERY_LOG: list[str] = []


def fetch_customer_row(customer_id: int) -> dict[str, object]:
    """Pretend database query."""
    QUERY_LOG.append(f"SELECT * FROM customers WHERE id = {customer_id}")
    return {"name": "Asha Rao", "city": "Pune", "orders": 12}


class LazyCustomer:
    """Only the id is known until a column is read."""

    def __init__(self, customer_id: int) -> None:
        self.customer_id = customer_id

    def __getattr__(self, column: str) -> object:
        if column.startswith("_"):
            raise AttributeError(column)
        row = fetch_customer_row(self.customer_id)
        self.__dict__.update(row)  # cache every column at once
        if column not in row:
            raise AttributeError(column)
        return row[column]


customer = LazyCustomer(42)
print(f"{len(QUERY_LOG)=} after creation")
print(f"{customer.name=}, {customer.city=}, {customer.orders=}")
print(f"{len(QUERY_LOG)=} after three reads")

len(QUERY_LOG)=0 after creation
customer.name='Asha Rao', customer.city='Pune', customer.orders=12
len(QUERY_LOG)=1 after three reads


### 2.5.3 The descriptor protocol

A **descriptor** is an object, stored as a **class** attribute, whose type defines
any of these methods. It takes over access to that attribute on every instance.

| Method | Called for | Arguments |
| --- | --- | --- |
| `__get__(self, instance, owner)` | `instance.attr` and `Owner.attr` | `instance` is `None` for class access |
| `__set__(self, instance, value)` | `instance.attr = value` | |
| `__delete__(self, instance)` | `del instance.attr` | |
| `__set_name__(self, owner, name)` | once, when the owning class is created | lets the descriptor learn its attribute name |

A **data descriptor** defines `__set__` or `__delete__` and beats the instance
`__dict__`; a **non-data descriptor** defines only `__get__` and loses to it
(2.5.1). The tracer below prints each call.

In [248]:
class Traced:
    """Prints every descriptor call; stores the value in the instance __dict__."""

    def __set_name__(self, owner: type, name: str) -> None:
        print(f"  __set_name__(owner={owner.__name__}, name={name!r})")
        self.name = name

    def __get__(self, instance: object, owner: type | None = None) -> object:
        print(f"  __get__(instance={type(instance).__name__}, owner={owner.__name__})")
        if instance is None:
            return self  # class access returns the descriptor itself
        return instance.__dict__[self.name]

    def __set__(self, instance: object, value: object) -> None:
        print(f"  __set__({value!r})")
        instance.__dict__[self.name] = value

    def __delete__(self, instance: object) -> None:
        print("  __delete__()")
        del instance.__dict__[self.name]


print("defining the class:")


class Parcel:
    weight_kg = Traced()

defining the class:
  __set_name__(owner=Parcel, name='weight_kg')


`__set_name__` already ran when the class statement finished. Now each kind of
access calls a different method; class access passes `instance=None`.

In [249]:
parcel = Parcel()
parcel.weight_kg = 2.5
print(f"{parcel.weight_kg=}")
del parcel.weight_kg
print(f"{type(Parcel.weight_kg).__name__=}")

  __set__(2.5)
  __get__(instance=Parcel, owner=Parcel)
parcel.weight_kg=2.5
  __delete__()
  __get__(instance=NoneType, owner=Parcel)
type(Parcel.weight_kg).__name__='Traced'


`__set_name__` is called by `type.__new__` only for attributes present when the
class is created. A descriptor attached later must be told its name by hand.

In [250]:
late_descriptor = Traced()
Parcel.length_cm = late_descriptor
print(f"{hasattr(late_descriptor, 'name')=}")
late_descriptor.__set_name__(Parcel, "length_cm")

hasattr(late_descriptor, 'name')=False
  __set_name__(owner=Parcel, name='length_cm')


**Data vs non-data.** The same `__get__` behaves differently depending on whether a
`__set__` exists. Without it, an instance attribute of the same name shadows the
descriptor after the first assignment; with it, every access keeps going through
the descriptor.

In [251]:
class Greeting:
    """Non-data descriptor: only __get__."""

    def __get__(self, instance: object, owner: type | None = None) -> str:
        return "computed hello"


class StrictGreeting(Greeting):
    """Data descriptor: adds __set__."""

    def __set__(self, instance: object, value: object) -> None:
        raise AttributeError("greeting is read-only")


class Page:
    greeting = Greeting()
    strict_greeting = StrictGreeting()


page = Page()
page.__dict__.update(greeting="from __dict__", strict_greeting="from __dict__")
print(f"{page.greeting=}  (non-data: the instance dict wins)")
print(f"{page.strict_greeting=}  (data: the descriptor wins)")
for descriptor_name in ("greeting", "strict_greeting"):
    is_data = inspect.isdatadescriptor(vars(Page)[descriptor_name])
    print(f"{descriptor_name}: data descriptor? {is_data}")

page.greeting='from __dict__'  (non-data: the instance dict wins)
page.strict_greeting='computed hello'  (data: the descriptor wins)
greeting: data descriptor? False
strict_greeting: data descriptor? True


#### A reusable `PositiveNumber` descriptor

One descriptor class, many attributes: `__set_name__` tells each instance which
name it manages, so error messages are precise. Values live in the instance's own
`__dict__` under a private name (`_price`), so instances do not share state.

In [252]:
class PositiveNumber:
    """A numeric attribute that must stay greater than zero."""

    def __set_name__(self, owner: type, name: str) -> None:
        self.public_name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance: object, owner: type | None = None) -> Any:
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance: object, value: object) -> None:
        if not isinstance(value, (int, float, Decimal)) or isinstance(value, bool):
            raise TypeError(f"{self.public_name} must be a number, got {type(value).__name__}")
        if value <= 0:
            raise ValueError(f"{self.public_name} must be > 0, got {value}")
        setattr(instance, self.storage_name, value)


class OrderLine:
    unit_price = PositiveNumber()
    quantity = PositiveNumber()

    def __init__(self, unit_price: Decimal, quantity: int) -> None:
        self.unit_price = unit_price  # goes through PositiveNumber.__set__
        self.quantity = quantity

Validation runs in `__init__` and on every later assignment, with no property
boilerplate per attribute. The values sit in `vars(line)` under the storage names.

In [253]:
line = OrderLine(Decimal("49.99"), 3)
print(f"{line.unit_price=}, {line.quantity=}, {vars(line)=}")
print(raised_message(lambda: OrderLine(Decimal("49.99"), 0)))
print(raised_message(lambda: set_attribute(line, "unit_price", "free")))
print(raised_message(lambda: set_attribute(line, "quantity", True)))

line.unit_price=Decimal('49.99'), line.quantity=3, vars(line)={'_unit_price': Decimal('49.99'), '_quantity': 3}
ValueError: quantity must be > 0, got 0
TypeError: unit_price must be a number, got str
TypeError: quantity must be a number, got bool


> **Warning:** a descriptor is shared by **all** instances of the class. Storing the
> value on `self` (the descriptor) instead of on `instance` makes every instance
> see the last value written:

In [254]:
class SharedStateBug:
    """BUG: keeps the value on the descriptor."""

    def __get__(self, instance: object, owner: type | None = None) -> object:
        return self.value

    def __set__(self, instance: object, value: object) -> None:
        self.value = value


class Account:
    balance = SharedStateBug()


first_account, second_account = Account(), Account()
first_account.balance = 100
second_account.balance = 5
print(f"{first_account.balance=}  (overwritten by second_account)")

first_account.balance=5  (overwritten by second_account)


### 2.5.4 Descriptors inside Python

**Functions are non-data descriptors.** A `def` in a class body stores a plain
function in the class `__dict__`. Every function has a `__get__`: called with an
instance it returns a **bound method**, called with `None` it returns the function
itself. That is all "method binding" is (Unit 2.1.4 previewed it).

In [255]:
class Greeter:
    def greet(self, name: str) -> str:
        return f"Hello, {name}"


raw_function = vars(Greeter)["greet"]
greeter = Greeter()
print(f"{type(raw_function).__name__=}, {hasattr(raw_function, '__get__')=}")
print(f"{type(raw_function.__get__(greeter, Greeter)).__name__=}")
print(f"{raw_function.__get__(None, Greeter) is raw_function=}  (class access)")
print(f"{raw_function.__get__(greeter, Greeter)('Asha')=}")

type(raw_function).__name__='function', hasattr(raw_function, '__get__')=True
type(raw_function.__get__(greeter, Greeter)).__name__='method'
raw_function.__get__(None, Greeter) is raw_function=True  (class access)
raw_function.__get__(greeter, Greeter)('Asha')='Hello, Asha'


Built-in methods use the C equivalent, `method_descriptor`. And because functions
are non-data descriptors, an instance attribute can shadow a method (rarely a good
idea, but it explains surprising bugs).

In [256]:
upper_descriptor = vars(str)["upper"]
print(f"{type(upper_descriptor).__name__=}")
print(f"{upper_descriptor.__get__('chai', str)()=}")
greeter.greet = lambda name: f"shadowed {name}"  # type: ignore[method-assign]
print(f"{greeter.greet('Asha')=}, {Greeter().greet('Asha')=}")

type(upper_descriptor).__name__='method_descriptor'
upper_descriptor.__get__('chai', str)()='CHAI'
greeter.greet('Asha')='shadowed Asha', Greeter().greet('Asha')='Hello, Asha'


#### Rebuilding `property` (read-only)

A minimal read-only property needs `__get__` **and** a `__set__` that refuses.
Without `__set__` it would be a non-data descriptor, and an assignment would quietly
create an instance attribute that hides it. Lab 2.5 builds the full version with
setters and deleters.

In [257]:
class ReadOnlyProperty:
    """property(fget) with no setter, rebuilt."""

    def __init__(self, fget: Callable[[Any], Any]) -> None:
        self.fget = fget
        self.__doc__ = fget.__doc__

    def __set_name__(self, owner: type, name: str) -> None:
        self.name = name

    def __get__(self, instance: object, owner: type | None = None) -> Any:
        if instance is None:
            return self
        return self.fget(instance)

    def __set__(self, instance: object, value: object) -> None:
        raise AttributeError(f"property {self.name!r} is read-only")


class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius

    @ReadOnlyProperty
    def diameter(self) -> float:
        """Twice the radius."""
        return self.radius * 2


wheel = Circle(30)
print(f"{wheel.diameter=}, {Circle.diameter.__doc__=}")
print(raised_message(lambda: set_attribute(wheel, "diameter", 1)))

wheel.diameter=60, Circle.diameter.__doc__='Twice the radius.'
AttributeError: property 'diameter' is read-only


#### Rebuilding `classmethod` and `staticmethod`

`classmethod.__get__` binds the function to the **class** (`types.MethodType(func,
owner)`); `staticmethod.__get__` returns the function unchanged. Both are non-data
descriptors of a few lines each.

In [258]:
class MyClassMethod:
    def __init__(self, func: Callable[..., Any]) -> None:
        self.func = func

    def __get__(self, instance: object, owner: type | None = None) -> Any:
        owner = owner if owner is not None else type(instance)
        return types.MethodType(self.func, owner)


class MyStaticMethod:
    def __init__(self, func: Callable[..., Any]) -> None:
        self.func = func

    def __get__(self, instance: object, owner: type | None = None) -> Any:
        return self.func

They behave like the built-ins, including on subclasses: the class method receives
the subclass it was called on.

In [259]:
class Temperature:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    @MyClassMethod
    def from_fahrenheit(cls, fahrenheit: float) -> Any:
        return cls(round((fahrenheit - 32) * 5 / 9, 1))

    @MyStaticMethod
    def is_freezing(celsius: float) -> bool:
        return celsius <= 0


class OvenTemperature(Temperature):
    pass


hot_oven = OvenTemperature.from_fahrenheit(392)
print(f"{type(hot_oven).__name__=}, {hot_oven.celsius=}")
print(f"{Temperature.is_freezing(-3)=}, {Temperature(5).is_freezing(5)=}")
print(f"{type(vars(Temperature)['from_fahrenheit']).__name__=}")

type(hot_oven).__name__='OvenTemperature', hot_oven.celsius=200.0
Temperature.is_freezing(-3)=True, Temperature(5).is_freezing(5)=False
type(vars(Temperature)['from_fahrenheit']).__name__='MyClassMethod'


### 2.5.5 `__slots__`

`__slots__ = ("x", "y")` tells Python to reserve fixed storage for those names in
each instance and to give instances **no `__dict__`**. Each slot becomes a
`member_descriptor` (a data descriptor) on the class. Assigning any other name
fails.

In [260]:
class PointWithDict:
    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y


class SlottedPoint:
    __slots__ = ("x", "y")

    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y


slotted = SlottedPoint(1, 2)
print(f"{hasattr(PointWithDict(1, 2), '__dict__')=}, {hasattr(slotted, '__dict__')=}")
slot_descriptor = vars(SlottedPoint)["x"]
print(f"{type(slot_descriptor).__name__=}, {inspect.isdatadescriptor(slot_descriptor)=}")
print(raised_message(lambda: set_attribute(slotted, "z", 3)))

hasattr(PointWithDict(1, 2), '__dict__')=True, hasattr(slotted, '__dict__')=False
type(slot_descriptor).__name__='member_descriptor', inspect.isdatadescriptor(slot_descriptor)=True
AttributeError: 'SlottedPoint' object has no attribute 'z' and no __dict__ for setting new attributes


#### Memory on 1M instances

`tracemalloc` measures one million instances of each class (the list holding them
and the small ints are included in both figures). Since Python 3.11 ordinary
instances share their key table and, since 3.13, store values inline until someone
asks for `__dict__`, so the gap is smaller than older articles claim. It is still
real: roughly 40 bytes per instance here. Takes about 2 seconds; figures vary by
version and platform.

In [261]:
INSTANCE_COUNT = 1_000_000


def traced_bytes(point_class: type) -> int:
    """Traced memory in bytes while INSTANCE_COUNT instances are alive."""
    tracemalloc.start()
    points = [point_class(number, number) for number in range(INSTANCE_COUNT)]
    current_bytes, _peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    del points
    return current_bytes


dict_bytes = traced_bytes(PointWithDict)
slots_bytes = traced_bytes(SlottedPoint)
print(f"__dict__ instances:  {dict_bytes / 1e6:6.1f} MB")
print(f"__slots__ instances: {slots_bytes / 1e6:6.1f} MB")
print(f"saved per instance:  {(dict_bytes - slots_bytes) / INSTANCE_COUNT:.0f} bytes")

__dict__ instances:   128.4 MB
__slots__ instances:   88.4 MB
saved per instance:  40 bytes


#### Speed

Attribute reads through a slot and through an inline-values `__dict__` take about
the same time on 3.15 (the specializing interpreter, Unit 1.1, handles both). Slots
mainly buy **memory** and a fixed set of attributes, not speed.

In [262]:
dict_point, slot_point = PointWithDict(1, 2), SlottedPoint(1, 2)
read_dict = min(timeit.repeat("point.x", globals={"point": dict_point}, number=1_000_000, repeat=5))
read_slot = min(timeit.repeat("point.x", globals={"point": slot_point}, number=1_000_000, repeat=5))
print(f"read via __dict__: {read_dict * 1_000:.1f} ns per read")
print(f"read via slot:     {read_slot * 1_000:.1f} ns per read")

read via __dict__: 7.5 ns per read
read via slot:     7.3 ns per read


#### Trade-offs

| Trade-off | Detail |
| --- | --- |
| No new attributes | good for value objects, bad for monkey-patching and some mocking |
| Subclasses get a `__dict__` back | unless they declare `__slots__` too (even `()`) |
| No weak references | unless `"__weakref__"` is in the slots |
| No class attribute with a slot's name | it would replace the slot descriptor (`ValueError`) |
| `functools.cached_property` needs `__dict__` | it stores the value there (`TypeError` with slots) |
| Multiple inheritance | two bases with non-empty slots cannot be combined (layout conflict) |
| Simplest route | `@dataclass(slots=True)` (Unit 2.6) writes the slots for you |

The first four rows, demonstrated:

In [263]:
class LabelledPoint(SlottedPoint):  # forgot __slots__
    pass


print(f"{hasattr(LabelledPoint(1, 2), '__dict__')=}  (the subclass brought it back)")
print(raised_message(lambda: weakref.ref(slotted)))


def class_with_conflict() -> type:
    class Conflicting:
        __slots__ = ("x",)
        x = 0  # a default value would overwrite the slot descriptor

    return Conflicting


print(raised_message(class_with_conflict))

hasattr(LabelledPoint(1, 2), '__dict__')=True  (the subclass brought it back)
TypeError: cannot create weak reference to 'SlottedPoint' object
ValueError: 'x' in __slots__ conflicts with class variable


The last two rows: `cached_property` cannot store its result, and two slotted bases
cannot share one instance layout.

In [264]:
class SlottedReport:
    __slots__ = ("rows",)

    def __init__(self, rows: list[int]) -> None:
        self.rows = rows

    @functools.cached_property
    def total(self) -> int:
        return sum(self.rows)


def combine_slotted_bases() -> type:
    class WithX:
        __slots__ = ("x",)

    class WithY:
        __slots__ = ("y",)

    class WithBoth(WithX, WithY):
        pass

    return WithBoth


print(raised_error_name(lambda: SlottedReport([1, 2]).total))
print(raised_message(combine_slotted_bases))

TypeError
TypeError: multiple bases have instance lay-out conflict


### 2.5.6 Classes as objects

A class is an ordinary object: you can pass it to functions, store it in a dict and
add attributes to it. Its type is its **metaclass**, which for normal classes is
`type` (and `type` is its own type).

In [265]:
class Invoice:
    pass


exporters = {"invoice": Invoice}
print(f"{type(Invoice)=}, {type(type)=}, {isinstance(Invoice, type)=}")
print(f"{exporters['invoice']().__class__.__name__=}")
Invoice.currency = "INR"  # classes are mutable objects
print(f"{Invoice().currency=}")

type(Invoice)=<class 'type'>, type(type)=<class 'type'>, isinstance(Invoice, type)=True
exporters['invoice']().__class__.__name__='Invoice'
Invoice().currency='INR'


**`type(name, bases, namespace)`** creates a class directly; a `class` statement is
mostly syntax for this call. The namespace dict becomes the class `__dict__`, so
functions in it become methods.

In [266]:
def describe_coupon(self: Any) -> str:
    return f"{self.code}: {self.percent}% off"


Coupon = type(
    "Coupon",
    (object,),
    {"percent": 10, "code": "WELCOME", "describe": describe_coupon, "__doc__": "A discount code."},
)
print(f"{Coupon.__name__=}, {Coupon.__doc__=}, {Coupon().describe()=}")
print(f"{mro_names(Coupon)=}")

Coupon.__name__='Coupon', Coupon.__doc__='A discount code.', Coupon().describe()='WELCOME: 10% off'
mro_names(Coupon)=['Coupon', 'object']


**Dynamic class creation.** Building classes from data is useful when the shape is
only known at run time, for example from a configuration file or an API schema. This
factory makes small record classes with a generated `__init__` and `__repr__`.
`collections.namedtuple` (Unit 1.5.6) and `dataclasses.make_dataclass` work the same
way.

In [267]:
def make_record_class(class_name: str, field_names: list[str]) -> type:
    """Build a class with an __init__ taking field_names, and a matching __repr__."""

    def __init__(self: Any, *values: object) -> None:
        if len(values) != len(field_names):
            raise TypeError(f"{class_name} takes {len(field_names)} values, got {len(values)}")
        for field_name, value in zip(field_names, values, strict=True):
            setattr(self, field_name, value)

    def __repr__(self: Any) -> str:
        pairs = ", ".join(f"{name}={getattr(self, name)!r}" for name in field_names)
        return f"{class_name}({pairs})"

    return type(
        class_name, (), {"__init__": __init__, "__repr__": __repr__, "fields": tuple(field_names)}
    )


schema = {"Shipment": ["tracking_number", "courier"], "Refund": ["order_id", "amount"]}
generated = {name: make_record_class(name, fields) for name, fields in schema.items()}
print(generated["Shipment"]("BD123", "BlueDart"))
print(generated["Refund"]("ORD-7", Decimal("250.00")))
print(raised_message(lambda: generated["Refund"]("ORD-7")))

Shipment(tracking_number='BD123', courier='BlueDart')
Refund(order_id='ORD-7', amount=Decimal('250.00'))
TypeError: Refund takes 2 values, got 1


`types.new_class` is the full equivalent of a `class` statement, including class
keywords such as `metaclass=` (which the three-argument `type()` cannot take) and a
callback that fills the namespace.

In [268]:
def fill_namespace(namespace: dict[str, Any]) -> None:
    @abstractmethod
    def export(self: Any) -> str: ...

    namespace["export"] = export


Exporter = types.new_class("Exporter", (), {"metaclass": ABCMeta}, fill_namespace)
print(f"{type(Exporter).__name__=}, {sorted(Exporter.__abstractmethods__)=}")
print(raised_error_name(Exporter))

type(Exporter).__name__='ABCMeta', sorted(Exporter.__abstractmethods__)=['export']
TypeError


### 2.5.7 Metaclasses

A **metaclass** is the class of a class. Subclass `type` and its methods run when
classes are **created** and when they are **called**:

| Metaclass method | Runs when | Use it to |
| --- | --- | --- |
| `__prepare__(name, bases, **kwargs)` | before the class body executes | supply the namespace object the body writes into |
| `__new__(mcls, name, bases, namespace)` | after the body, to build the class | inspect or change the namespace, return the class |
| `__init__(cls, name, bases, namespace)` | after `__new__` | register or validate the finished class |
| `__call__(cls, *args, **kwargs)` | when the class is called, `MyClass(...)` | control instance creation (singletons, caching) |

The trace shows the full order of a class statement, including the hooks from 2.5.3
and 2.5.8 (`__set_name__`, `__init_subclass__`) and a class decorator.

In [269]:
class TracingMeta(type):
    @classmethod
    def __prepare__(mcls, name: str, bases: tuple[type, ...], **kwargs: Any) -> dict[str, Any]:
        print(f"1. {mcls.__name__}.__prepare__({name!r})")
        return {}

    def __new__(mcls, name: str, bases: tuple[type, ...], namespace: dict[str, Any]) -> type:
        body_names = [key for key in namespace if not key.startswith("__")]
        print(f"3. {mcls.__name__}.__new__ with body names {body_names}")
        return super().__new__(mcls, name, bases, namespace)

    def __init__(cls, name: str, bases: tuple[type, ...], namespace: dict[str, Any]) -> None:
        print(f"6. {type(cls).__name__}.__init__({name!r})")
        super().__init__(name, bases, namespace)


class Announcer:
    def __set_name__(self, owner: type, name: str) -> None:
        print(f"4. __set_name__({name!r}) inside type.__new__")


class TracedBase(metaclass=TracingMeta):
    def __init_subclass__(cls, **kwargs: Any) -> None:
        print(f"5. __init_subclass__({cls.__name__!r}) inside type.__new__")
        super().__init_subclass__(**kwargs)


def class_decorator(cls: type) -> type:
    print(f"7. class decorator gets the finished {cls.__name__!r}")
    return cls

1. TracingMeta.__prepare__('TracedBase')
3. TracingMeta.__new__ with body names []
6. TracingMeta.__init__('TracedBase')


Defining `TracedBase` already ran steps 1, 3 and 6 for the base itself
(`__init_subclass__` only runs for subclasses). Now a decorated subclass with a
descriptor in its body shows every step in order:

In [270]:
@class_decorator
class Report(TracedBase):
    print("2. class body executes")
    title = Announcer()

1. TracingMeta.__prepare__('Report')
2. class body executes
3. TracingMeta.__new__ with body names ['title']
4. __set_name__('title') inside type.__new__
5. __init_subclass__('Report') inside type.__new__
6. TracingMeta.__init__('Report')
7. class decorator gets the finished 'Report'


#### `__new__`: shape the class

`__new__` receives the namespace before the class exists, so it can add, rename or
reject entries. This metaclass derives a `table_name` for every model class that
does not set one, the kind of convention Django and SQLAlchemy apply.

In [271]:
class TableNameMeta(type):
    def __new__(mcls, name: str, bases: tuple[type, ...], namespace: dict[str, Any]) -> type:
        namespace.setdefault("table_name", name.lower() + "s")
        return super().__new__(mcls, name, bases, namespace)


class Record(metaclass=TableNameMeta):
    pass


class Customer(Record):
    pass


class Person(Record):
    table_name = "people"


print(f"{Customer.table_name=}, {Person.table_name=}, {type(Customer).__name__=}")

Customer.table_name='customers', Person.table_name='people', type(Customer).__name__='TableNameMeta'


#### `__init__` and registries

The metaclass `__init__` runs for every class created with it, **including
subclasses**, which makes it a natural place for a registry. Skipping the base class
keeps it out.

In [272]:
class RegistryMeta(type):
    registry: dict[str, type] = {}

    def __init__(cls, name: str, bases: tuple[type, ...], namespace: dict[str, Any]) -> None:
        super().__init__(name, bases, namespace)
        if bases:  # not the base class itself
            RegistryMeta.registry[name.removesuffix("Notifier").lower()] = cls


class Notifier(metaclass=RegistryMeta):
    pass


class EmailNotifier(Notifier):
    pass


class SmsNotifier(Notifier):
    pass


print(f"{RegistryMeta.registry=}")
print(f"{RegistryMeta.registry['sms']().__class__.__name__=}")

RegistryMeta.registry={'email': <class '__main__.EmailNotifier'>, 'sms': <class '__main__.SmsNotifier'>}
RegistryMeta.registry['sms']().__class__.__name__='SmsNotifier'


#### Interface enforcement

An ABC complains when you **instantiate** an incomplete class (Unit 2.3.1); a
metaclass can complain when the class is **defined**, which surfaces mistakes at
import time.

In [273]:
class InterfaceMeta(type):
    required_methods = ("load", "save")

    def __new__(mcls, name: str, bases: tuple[type, ...], namespace: dict[str, Any]) -> type:
        cls = super().__new__(mcls, name, bases, namespace)
        if bases:
            missing = [
                method
                for method in mcls.required_methods
                if not callable(getattr(cls, method, None))
            ]
            if missing:
                raise TypeError(f"{name} must define {', '.join(missing)}")
        return cls


class StorageBackend(metaclass=InterfaceMeta):
    pass


def define_incomplete_backend() -> type:
    class DiskBackend(StorageBackend):
        def load(self) -> bytes:
            return b""

    return DiskBackend


print(raised_message(define_incomplete_backend))

TypeError: DiskBackend must define save


#### `__call__` and singletons

`MyClass(...)` is a call on the class, so it runs `type(MyClass).__call__`, which
normally calls `__new__` and `__init__`. Overriding it controls instance creation:
here the first instance is cached and returned every time. (2.6.5 shows why a
module-level instance is usually the more Pythonic singleton.)

In [274]:
class SingletonMeta(type):
    _instances: dict[type, object] = {}

    def __call__(cls, *args: Any, **kwargs: Any) -> Any:
        if cls not in SingletonMeta._instances:
            print(f"  creating the one {cls.__name__}")
            SingletonMeta._instances[cls] = super().__call__(*args, **kwargs)
        return SingletonMeta._instances[cls]


class AppConfig(metaclass=SingletonMeta):
    def __init__(self, environment: str = "dev") -> None:
        self.environment = environment


first_config = AppConfig("prod")
second_config = AppConfig("test")  # arguments ignored: already created
print(f"{first_config is second_config=}, {second_config.environment=}")

  creating the one AppConfig
first_config is second_config=True, second_config.environment='prod'


#### `__prepare__`: a custom namespace

`__prepare__` returns the mapping that the class body writes into. A dict subclass
that refuses duplicate names catches an easy mistake: defining the same method twice,
where the second silently replaces the first.

In [275]:
class NoDuplicatesNamespace(dict[str, Any]):
    def __setitem__(self, key: str, value: Any) -> None:
        if key in self and not key.startswith("__"):
            raise TypeError(f"{key!r} defined twice in the class body")
        super().__setitem__(key, value)


class NoDuplicatesMeta(type):
    @classmethod
    def __prepare__(
        mcls, name: str, bases: tuple[type, ...], **kwargs: Any
    ) -> NoDuplicatesNamespace:
        return NoDuplicatesNamespace()

    def __new__(mcls, name: str, bases: tuple[type, ...], namespace: dict[str, Any]) -> type:
        return super().__new__(mcls, name, bases, dict(namespace))


def define_duplicate_method() -> type:
    class PriceRules(metaclass=NoDuplicatesMeta):
        def discount(self) -> int:
            return 10

        def discount(self) -> int:  # noqa: F811 (the mistake being caught)
            return 20

    return PriceRules


print(raised_message(define_duplicate_method))

TypeError: 'discount' defined twice in the class body


#### `ABCMeta`

`abc.ABC` is a plain class whose metaclass is `ABCMeta`. Its `__new__` computes
`__abstractmethods__`; its `__instancecheck__` / `__subclasscheck__` power
`register` and `__subclasshook__` (Unit 2.3.2).

In [276]:
class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...


print(f"{type(ABC).__name__=}, {type(Shape).__name__=}")
print(f"{Shape.__abstractmethods__=}")
print(f"{[name for name in vars(ABCMeta) if name.startswith('__') and 'check' in name]=}")
Shape.register(complex)
print(f"{isinstance(1j, Shape)=}  (ABCMeta.__instancecheck__ honours register)")

type(ABC).__name__='ABCMeta', type(Shape).__name__='ABCMeta'
Shape.__abstractmethods__=frozenset({'area'})
[name for name in vars(ABCMeta) if name.startswith('__') and 'check' in name]=['__instancecheck__', '__subclasscheck__']
isinstance(1j, Shape)=True  (ABCMeta.__instancecheck__ honours register)


#### `EnumMeta`

`enum.Enum`'s metaclass is `EnumType` (`EnumMeta` is the old name, still an
alias). It uses **every** hook: `__prepare__` returns an `enum.EnumDict` (public
since 3.13) that turns assignments into members, `__new__` builds the member
objects, `__call__` makes `Colour(1)` a **lookup** instead of a construction, and
`__iter__` / `__len__` / `__contains__` on the metaclass make the class itself
iterable. That last point is Unit 2.4.9 again: `len(Colour)` looks on
`type(Colour)`.

In [277]:
class Colour(enum.Enum):
    RED = 1
    GREEN = 2


print(f"{type(Colour).__name__=}, {enum.EnumMeta is enum.EnumType=}")
print(f"{type(type(Colour).__prepare__('Size', (enum.Enum,))).__name__=}  (New in 3.13: public)")
print(f"{Colour(1)=}  (EnumType.__call__ looks the member up)")
print(f"{len(Colour)=}, {list(Colour)=}, {Colour.RED in Colour=}")
print(f"{'__len__' in vars(enum.EnumType)=}, {'__len__' in vars(Colour)=}")

type(Colour).__name__='EnumType', enum.EnumMeta is enum.EnumType=True
type(type(Colour).__prepare__('Size', (enum.Enum,))).__name__='EnumDict'  (New in 3.13: public)
Colour(1)=<Colour.RED: 1>  (EnumType.__call__ looks the member up)
len(Colour)=2, list(Colour)=[<Colour.RED: 1>, <Colour.GREEN: 2>], Colour.RED in Colour=True
'__len__' in vars(enum.EnumType)=True, '__len__' in vars(Colour)=False


> **Warning:** a class can have only one metaclass. Combining bases whose
> metaclasses are unrelated fails, which is a big reason to prefer the lighter tools
> in 2.5.8.

In [278]:
print(raised_message(lambda: type("Mixed", (Customer, Notifier), {})))

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases


### 2.5.8 Lighter alternatives

Most metaclass jobs (validate or register subclasses, add methods) are done more
simply with two tools that do not conflict with other metaclasses:

| Tool | Runs | Inherited? | Good for |
| --- | --- | --- | --- |
| `__init_subclass__` (PEP 487) | on the **parent**, each time a subclass is created | yes | validating/configuring subclasses, registries, class keywords |
| Class decorator | once, on the class it decorates | no | adding or wrapping methods on one class |
| Metaclass | at class creation and on every call | yes | changing the namespace before the body (`__prepare__`), class-level dunders, instance creation |

**`__init_subclass__`** is an implicit class method on the base. Extra keywords in
the class statement (`class Orders(ApiEndpoint, path="/orders")`) arrive as keyword
arguments. This base validates and stores a URL path for each endpoint.

In [279]:
class ApiEndpoint:
    """Every subclass must declare a path that starts with '/'."""

    path: str

    def __init_subclass__(
        cls, /, *, path: str, methods: tuple[str, ...] = ("GET",), **kwargs: Any
    ) -> None:
        super().__init_subclass__(**kwargs)
        if not path.startswith("/"):
            raise ValueError(f"{cls.__name__}: path must start with '/', got {path!r}")
        cls.path, cls.methods = path, methods


class OrdersEndpoint(ApiEndpoint, path="/orders", methods=("GET", "POST")):
    pass


print(f"{OrdersEndpoint.path=}, {OrdersEndpoint.methods=}")
print(raised_message(lambda: type("BadEndpoint", (ApiEndpoint,), {}, path="orders")))
print(raised_message(lambda: type("NoPath", (ApiEndpoint,), {})))

OrdersEndpoint.path='/orders', OrdersEndpoint.methods=('GET', 'POST')
ValueError: BadEndpoint: path must start with '/', got 'orders'
TypeError: ApiEndpoint.__init_subclass__() missing 1 required keyword-only argument: 'path'


**Class decorators** receive the finished class and return it (or a replacement).
`auto_repr` writes a `__repr__` from the `__init__` signature; `count_instances`
wraps `__init__`. Unit 1.12.5 introduced the idea; `@dataclass` (Unit 2.6) is the
best-known class decorator.

In [280]:
def auto_repr(cls: type) -> type:
    """Add a __repr__ listing the __init__ parameters."""
    parameter_names = list(inspect.signature(cls.__init__).parameters)[1:]

    def __repr__(self: Any) -> str:
        pairs = ", ".join(f"{name}={getattr(self, name)!r}" for name in parameter_names)
        return f"{type(self).__name__}({pairs})"

    cls.__repr__ = __repr__
    return cls


def count_instances(cls: type) -> type:
    """Wrap __init__ so cls.instance_count tracks how many objects were made."""
    original_init = cls.__init__

    @functools.wraps(original_init)
    def counting_init(self: Any, *args: Any, **kwargs: Any) -> None:
        original_init(self, *args, **kwargs)
        cls.instance_count += 1

    cls.instance_count = 0
    cls.__init__ = counting_init
    return cls

Stacked on one class, applied bottom-up:

In [281]:
@auto_repr
@count_instances
class Warehouse:
    def __init__(self, code: str, city: str) -> None:
        self.code, self.city = code, city


warehouses = [Warehouse("PNQ1", "Pune"), Warehouse("BLR2", "Bengaluru")]
print(warehouses)
print(f"{Warehouse.instance_count=}, {Warehouse.__init__.__name__=}")

[Warehouse(code='PNQ1', city='Pune'), Warehouse(code='BLR2', city='Bengaluru')]
Warehouse.instance_count=2, Warehouse.__init__.__name__='__init__'


### 2.5.9 How libraries do it

The famous "magic" libraries are built from the tools of this unit:

| Library | Class-creation hook | Attribute machinery | What you write |
| --- | --- | --- | --- |
| Pydantic v2 | metaclass `ModelMetaclass` reads annotations into `model_fields` | validation in `__init__` (Rust core); fields are plain instance data | `class User(BaseModel): name: str` |
| SQLAlchemy 2.0 | `DeclarativeBase.__init_subclass__` maps the class to a table | `InstrumentedAttribute` **descriptors**: class access builds SQL expressions, instance access returns values and tracks changes | `name: Mapped[str] = mapped_column()` |
| Django models | metaclass `ModelBase` collects fields and the inner `class Meta` into `_meta` | field descriptors (deferred loading, related objects) | `name = models.CharField(max_length=50)` |
| dataclasses | **class decorator**, no metaclass | generates `__init__`, `__repr__`, `__eq__` as source code and `exec`s it | `@dataclass class User: name: str` |
| attrs | class decorator (`@define`, slots by default) | generated methods, optional validators | `@define class User: name: str` |

Pydantic, SQLAlchemy and Django are Phase 6 dependencies and are not installed
yet. The next cell checks; each library then gets a tiny stand-in that uses the
same mechanism, so you can see it work.

In [282]:
for library_name in ["pydantic", "sqlalchemy", "django", "attrs"]:
    installed = importlib.util.find_spec(library_name) is not None
    status = "installed" if installed else "not installed: skipping the real library"
    print(f"{library_name:<11} {status}")
if importlib.util.find_spec("pydantic") is not None:
    import pydantic

    print(f"{type(pydantic.BaseModel).__name__=}")

pydantic    not installed: skipping the real library
sqlalchemy  not installed: skipping the real library
django      not installed: skipping the real library
attrs       installed


#### Pydantic-style: annotations + validation on `__init__`

`__init_subclass__` reads each subclass's annotations once (with
`annotationlib.get_annotations`, *New in 3.14*, which works with lazily evaluated
annotations) and `__init__` checks and converts values against them. Pydantic does
the same in Rust, with far more types and conversions.

In [283]:
class MiniModel:
    """Validates keyword arguments against the class annotations."""

    model_fields: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs: Any) -> None:
        super().__init_subclass__(**kwargs)
        cls.model_fields = dict(annotationlib.get_annotations(cls))

    def __init__(self, **values: Any) -> None:
        for field_name, field_type in self.model_fields.items():
            if field_name not in values:
                raise TypeError(f"missing field {field_name!r}")
            raw_value = values[field_name]
            try:
                setattr(self, field_name, field_type(raw_value))  # lax conversion, like Pydantic
            except TypeError, ValueError:
                raise ValueError(
                    f"{field_name}: cannot convert {raw_value!r} to {field_type.__name__}"
                ) from None


class SignupForm(MiniModel):
    username: str
    age: int


form = SignupForm(username="asha", age="29")
print(f"{SignupForm.model_fields=}, {form.age=} ({type(form.age).__name__})")
print(raised_message(lambda: SignupForm(username="ravi", age="twenty")))

SignupForm.model_fields={'username': <class 'str'>, 'age': <class 'int'>}, form.age=29 (int)
ValueError: age: cannot convert 'twenty' to int


#### SQLAlchemy-style: a descriptor that answers differently on the class

The trick behind `select(User).where(User.age > 18)`: on the **class**, `__get__`
returns an expression object whose comparison operators (Unit 2.4.4) build SQL
text instead of returning `bool`; on an **instance**, it returns the value and
records the change for the next flush.

In [284]:
class ColumnExpression:
    def __init__(self, column_name: str) -> None:
        self.column_name = column_name

    def __gt__(self, value: object) -> tuple[str, object]:
        return (f"{self.column_name} > ?", value)

    def __eq__(self, value: object) -> tuple[str, object]:  # type: ignore[override]
        return (f"{self.column_name} = ?", value)


class MappedColumn:
    def __set_name__(self, owner: type, name: str) -> None:
        self.name = name

    def __get__(self, instance: Any, owner: type | None = None) -> Any:
        if instance is None:
            return ColumnExpression(self.name)
        return instance.__dict__.get(self.name)

    def __set__(self, instance: Any, value: object) -> None:
        instance.__dict__[self.name] = value
        instance.__dict__.setdefault("_dirty", set()).add(self.name)


class UserRow:
    age = MappedColumn()
    city = MappedColumn()


print(f"{(UserRow.age > 18)=}, {(UserRow.city == 'Pune')=}")
row = UserRow()
row.age = 30
print(f"{row.age=}, {row._dirty=}  (changed columns waiting for a flush)")

(UserRow.age > 18)=('age > ?', 18), (UserRow.city == 'Pune')=('city = ?', 'Pune')
row.age=30, row._dirty={'age'}  (changed columns waiting for a flush)


#### Django-style: a metaclass that collects fields and `class Meta`

Django's `ModelBase.__new__` pulls field objects and the inner `Meta` class out of
the namespace and builds one `_meta` options object per model. A miniature:

In [285]:
class CharField:
    def __init__(self, max_length: int) -> None:
        self.max_length = max_length


class ModelOptions:
    def __init__(self, fields: dict[str, CharField], meta: type | None) -> None:
        self.fields = fields
        self.ordering = getattr(meta, "ordering", ())
        self.verbose_name = getattr(meta, "verbose_name", None)


class MiniModelBase(type):
    def __new__(mcls, name: str, bases: tuple[type, ...], namespace: dict[str, Any]) -> type:
        fields = {key: value for key, value in namespace.items() if isinstance(value, CharField)}
        meta = namespace.pop("Meta", None)
        cls = super().__new__(mcls, name, bases, namespace)
        cls._meta = ModelOptions(fields, meta)
        return cls


class Author(metaclass=MiniModelBase):
    name = CharField(max_length=80)
    city = CharField(max_length=40)

    class Meta:
        ordering = ("name",)
        verbose_name = "book author"


print(f"{list(Author._meta.fields)=}, {Author._meta.ordering=}, {Author._meta.verbose_name=}")
print(f"{hasattr(Author, 'Meta')=}  (consumed by the metaclass)")

list(Author._meta.fields)=['name', 'city'], Author._meta.ordering=('name',), Author._meta.verbose_name='book author'
hasattr(Author, 'Meta')=False  (consumed by the metaclass)


#### dataclasses: a class decorator that writes code

`@dataclass` is just a function applied to the class. It reads the annotations,
records them in `__dataclass_fields__`, **generates the source** of `__init__`,
`__repr__` and `__eq__` and runs it with `exec`, which is why the generated
`__init__` reports its file as `<string>` and `inspect.getsource` cannot find it.
No metaclass is involved: the class's type is still `type`. attrs works the same way.

In [286]:
@dataclasses.dataclass
class Book:
    title: str
    price: Decimal


print(f"{type(Book).__name__=}, {list(Book.__dataclass_fields__)=}")
print(f"{Book.__init__.__code__.co_filename=}, {inspect.signature(Book)=}")
print(raised_error_name(lambda: inspect.getsource(Book.__init__)))
print(Book("Fluent Python", Decimal("3999")))
if importlib.util.find_spec("attrs") is not None:
    import attrs

    @attrs.define
    class Magazine:
        title: str

    print(f"{type(Magazine).__name__=}, {hasattr(Magazine('Wired'), '__dict__')=}")
    print("(attrs uses slots by default)")

type(Book).__name__='type', list(Book.__dataclass_fields__)=['title', 'price']
Book.__init__.__code__.co_filename='<string>', inspect.signature(Book)=<Signature (title: str, price: decimal.Decimal) -> None>
OSError
Book(title='Fluent Python', price=Decimal('3999'))
type(Magazine).__name__='type', hasattr(Magazine('Wired'), '__dict__')=False
(attrs uses slots by default)


**Predict the output.**

```python
class Order:
    def __init__(self, items):
        self.items = items

    @property
    def total(self):
        return sum(item.price for item in self.items)  # items are plain ints!

    def __getattr__(self, name):
        return f"<missing {name}>"

print(Order([10, 20]).total)
```

<details><summary>Answer</summary>

```text
<missing total>
```

The property raises `AttributeError` (`int` has no `.price`). Python treats an
`AttributeError` escaping from normal lookup, **including from inside a property
getter**, as "attribute not found" and calls `__getattr__("total")`. The real bug is
hidden. Keep `__getattr__` narrow (raise for names you do not handle), and be
suspicious when a property seems to "disappear".

</details>

Run the next cell to check your prediction.

In [287]:
class Order:
    def __init__(self, items: list[int]) -> None:
        self.items = items

    @property
    def total(self) -> int:
        return sum(item.price for item in self.items)  # type: ignore[attr-defined]

    def __getattr__(self, name: str) -> str:
        return f"<missing {name}>"


print(Order([10, 20]).total)

<missing total>


### Lab 2.5

Five exercises. For each, complete the stub, then run the check cell. Until then the
checks print FAIL; that is expected. In exercise 4 one check ("registry is a class
attribute") already passes on the stub, because the stub declares the dict; keep it
passing. The check cells define their test classes **inside functions**, so a stub
that raises cannot stop Run All.

#### Exercise 1 — rebuild `property`

`MyProperty(fget=None, fset=None, fdel=None, doc=None)` must work like the
built-in:

- As a decorator; `.getter(f)`, `.setter(f)` and `.deleter(f)` return a **new**
  `MyProperty` with that function replaced (the original is unchanged).
- `__set_name__` remembers the attribute name; class access returns the descriptor.
- Missing functions raise `AttributeError(f"property {name!r} has no getter")`
  (or `setter` / `deleter`).
- Keep the functions in the attributes `fget`, `fset` and `fdel`.
- `__doc__` is `doc`, or else `fget.__doc__`. It must be a **data** descriptor even
  without a setter.

In [288]:
class MyProperty:
    """A pure-Python property."""

    def __init__(
        self,
        fget: Callable[[Any], Any] | None = None,
        fset: Callable[[Any, Any], None] | None = None,
        fdel: Callable[[Any], None] | None = None,
        doc: str | None = None,
    ) -> None:
        raise NotImplementedError("Lab 2.5: implement MyProperty")

Run the checks.

In [289]:
def make_thermostat_class() -> type:
    """A class using MyProperty with a validating setter, a deleter and a read-only one."""

    class Thermostat:
        def __init__(self) -> None:
            self._target = 21

        @MyProperty
        def target(self) -> int:
            """Target temperature in °C."""
            return self._target

        @target.setter
        def target(self, value: int) -> None:
            if not 5 <= value <= 30:
                raise ValueError("target must be between 5 and 30")
            self._target = value

        @target.deleter
        def target(self) -> None:
            self._target = 21

        @MyProperty
        def serial(self) -> str:
            return "TH-001"

    return Thermostat


def thermostat_session() -> list[object]:
    """Read, set, reject, delete."""
    thermostat = make_thermostat_class()()
    answers: list[object] = [thermostat.target]
    thermostat.target = 25
    answers.append(thermostat.target)
    answers.append(raised_message(lambda: set_attribute(thermostat, "target", 99)))
    del thermostat.target
    answers.append(thermostat.target)
    return answers


def setter_returns_copy() -> tuple[bool, bool]:
    """setter() builds a new object and leaves the original without a setter."""
    original = MyProperty(lambda self: 1)
    with_setter = original.setter(lambda self, value: None)
    return with_setter is not original, original.fset is None


run_checks(
    [
        (
            "session",
            thermostat_session,
            [21, 25, "ValueError: target must be between 5 and 30", 21],
        ),
        (
            "read-only",
            lambda: raised_message(lambda: set_attribute(make_thermostat_class()(), "serial", "x")),
            "AttributeError: property 'serial' has no setter",
        ),
        (
            "no deleter",
            lambda: raised_message(lambda: make_thermostat_class()().__delattr__("serial")),
            "AttributeError: property 'serial' has no deleter",
        ),
        (
            "class access",
            lambda: type(vars(make_thermostat_class())["target"]).__name__,
            "MyProperty",
        ),
        (
            "doc from getter",
            lambda: vars(make_thermostat_class())["target"].__doc__,
            "Target temperature in °C.",
        ),
        ("explicit doc", lambda: MyProperty(doc="custom").__doc__, "custom"),
        ("setter returns a copy", setter_returns_copy, (True, True)),
        ("data descriptor", lambda: inspect.isdatadescriptor(MyProperty(lambda self: 1)), True),
    ]
)

FAIL  session: raised NotImplementedError: Lab 2.5: implement MyProperty
FAIL  read-only: expected "AttributeError: property 'serial' has no setter", got 'NotImplementedError: Lab 2.5: implement MyProperty'
FAIL  no deleter: expected "AttributeError: property 'serial' has no deleter", got 'NotImplementedError: Lab 2.5: implement MyProperty'
FAIL  class access: raised NotImplementedError: Lab 2.5: implement MyProperty
FAIL  doc from getter: raised NotImplementedError: Lab 2.5: implement MyProperty
FAIL  explicit doc: raised NotImplementedError: Lab 2.5: implement MyProperty
FAIL  setter returns a copy: raised NotImplementedError: Lab 2.5: implement MyProperty
FAIL  data descriptor: raised NotImplementedError: Lab 2.5: implement MyProperty
0/8 checks passed


#### Exercise 2 — `Typed` and `Bounded` descriptors

Two reusable **data** descriptors that use `__set_name__` and store the value in the
instance `__dict__` under the **same** name (the data descriptor still wins):

- `Typed(expected_type)`: assigning a value that is not an instance raises
  `TypeError(f"{name} must be {expected_type.__name__}, got {type(value).__name__}")`.
- `Bounded(minimum=None, maximum=None)`: values below or above raise
  `ValueError(f"{name} must be >= {minimum}")` / `ValueError(f"{name} must be <= {maximum}")`;
  `None` means no bound.
- Both: reading before any assignment raises
  `AttributeError(f"{name} has not been set")`; class access returns the
  descriptor.

In [290]:
class Typed:
    def __init__(self, expected_type: type) -> None:
        raise NotImplementedError("Lab 2.5: implement Typed")


class Bounded:
    def __init__(self, minimum: float | None = None, maximum: float | None = None) -> None:
        raise NotImplementedError("Lab 2.5: implement Bounded")

Run the checks.

In [291]:
def make_booking_class() -> type:
    class Booking:
        guest_name = Typed(str)
        guests = Bounded(minimum=1, maximum=10)
        discount_percent = Bounded(maximum=50)

        def __init__(self, guest_name: str, guests: int) -> None:
            self.guest_name = guest_name
            self.guests = guests

    return Booking


run_checks(
    [
        (
            "valid",
            lambda: vars(make_booking_class()("Asha", 4)),
            {"guest_name": "Asha", "guests": 4},
        ),
        (
            "type error",
            lambda: raised_message(lambda: make_booking_class()(42, 2)),
            "TypeError: guest_name must be str, got int",
        ),
        (
            "below minimum",
            lambda: raised_message(lambda: make_booking_class()("Ravi", 0)),
            "ValueError: guests must be >= 1",
        ),
        (
            "above maximum",
            lambda: raised_message(lambda: make_booking_class()("Ravi", 11)),
            "ValueError: guests must be <= 10",
        ),
        (
            "open lower bound",
            lambda: raised_message(
                lambda: set_attribute(make_booking_class()("Ravi", 2), "discount_percent", -5)
            ),
            "no error",
        ),
        (
            "upper bound only",
            lambda: raised_message(
                lambda: set_attribute(make_booking_class()("Ravi", 2), "discount_percent", 60)
            ),
            "ValueError: discount_percent must be <= 50",
        ),
        (
            "not set yet",
            lambda: raised_message(lambda: make_booking_class()("Ravi", 2).discount_percent),
            "AttributeError: discount_percent has not been set",
        ),
        (
            "class access",
            lambda: [
                type(vars(make_booking_class())[name]).__name__ for name in ("guest_name", "guests")
            ],
            ["Typed", "Bounded"],
        ),
        (
            "data descriptors",
            lambda: inspect.isdatadescriptor(Typed(int)) and inspect.isdatadescriptor(Bounded(0)),
            True,
        ),
    ]
)

FAIL  valid: raised NotImplementedError: Lab 2.5: implement Typed
FAIL  type error: expected 'TypeError: guest_name must be str, got int', got 'NotImplementedError: Lab 2.5: implement Typed'
FAIL  below minimum: expected 'ValueError: guests must be >= 1', got 'NotImplementedError: Lab 2.5: implement Typed'
FAIL  above maximum: expected 'ValueError: guests must be <= 10', got 'NotImplementedError: Lab 2.5: implement Typed'
FAIL  open lower bound: expected 'no error', got 'NotImplementedError: Lab 2.5: implement Typed'
FAIL  upper bound only: expected 'ValueError: discount_percent must be <= 50', got 'NotImplementedError: Lab 2.5: implement Typed'
FAIL  not set yet: expected 'AttributeError: discount_percent has not been set', got 'NotImplementedError: Lab 2.5: implement Typed'
FAIL  class access: raised NotImplementedError: Lab 2.5: implement Typed
FAIL  data descriptors: raised NotImplementedError: Lab 2.5: implement Typed
0/9 checks passed


#### Exercise 3 — a caching `LazyAttribute`

`LazyAttribute(func)` is a decorator for expensive computed attributes, like
`functools.cached_property` (Unit 2.1.6):

- On first access it calls `func(instance)`, stores the result in
  `instance.__dict__` under the attribute name, and returns it. Make it a
  **non-data** descriptor (only `__get__`), so later reads find the stored value
  and never call the descriptor again.
- Class access returns the descriptor. `del instance.attribute` forgets the value,
  so the next read recomputes it. Assigning a value simply replaces it.

In [292]:
class LazyAttribute:
    """Compute once per instance, then behave like a normal attribute."""

    def __init__(self, func: Callable[[Any], Any]) -> None:
        raise NotImplementedError("Lab 2.5: implement LazyAttribute")

Run the checks.

In [293]:
def make_report_class() -> type:
    class SalesReport:
        computations = 0

        def __init__(self, amounts: list[int]) -> None:
            self.amounts = amounts

        @LazyAttribute
        def total(self) -> int:
            type(self).computations += 1
            return sum(self.amounts)

    return SalesReport


def lazy_session() -> list[object]:
    """Two reads, a second instance, delete and recompute, assign."""
    report_class = make_report_class()
    report = report_class([100, 250])
    answers: list[object] = [report.total, report.total, report_class.computations]
    answers.append("total" in vars(report))
    answers.append(report_class([1]).total)
    del report.total
    report.amounts.append(50)
    answers.extend([report.total, report_class.computations])
    report.total = 0
    answers.append(report.total)
    return answers


run_checks(
    [
        ("session", lazy_session, [350, 350, 1, True, 1, 400, 3, 0]),
        (
            "class access",
            lambda: type(vars(make_report_class())["total"]).__name__,
            "LazyAttribute",
        ),
        ("non-data descriptor", lambda: inspect.isdatadescriptor(LazyAttribute(len)), False),
        ("has __get__", lambda: hasattr(LazyAttribute(len), "__get__"), True),
    ]
)

FAIL  session: raised NotImplementedError: Lab 2.5: implement LazyAttribute
FAIL  class access: raised NotImplementedError: Lab 2.5: implement LazyAttribute
FAIL  non-data descriptor: raised NotImplementedError: Lab 2.5: implement LazyAttribute
FAIL  has __get__: raised NotImplementedError: Lab 2.5: implement LazyAttribute
0/4 checks passed


#### Exercise 4 — a `PluginBase` registry with `__init_subclass__`

Every subclass of `PluginBase` registers itself in the class attribute
`PluginBase.registry` (name → class) **without a metaclass**:

- The name comes from a class keyword, `class CsvExport(PluginBase,
  plugin_name="csv")`, defaulting to the lower-cased class name; store it on the
  subclass as `plugin_name`.
- A second plugin with a taken name raises
  `ValueError(f"duplicate plugin name: {name!r}")`.
- `PluginBase.create(name, *args, **kwargs)` instantiates a registered plugin
  (`KeyError` for unknown names); `PluginBase.available()` returns the sorted names.

In [294]:
class PluginBase:
    """Subclasses register themselves by name."""

    registry: dict[str, type[PluginBase]] = {}

    def __init_subclass__(cls, /, *, plugin_name: str | None = None, **kwargs: Any) -> None:
        raise NotImplementedError("Lab 2.5: implement PluginBase.__init_subclass__")

    @classmethod
    def create(cls, name: str, *args: Any, **kwargs: Any) -> PluginBase:
        raise NotImplementedError("Lab 2.5: implement PluginBase.create")

    @classmethod
    def available(cls) -> list[str]:
        raise NotImplementedError("Lab 2.5: implement PluginBase.available")

Run the checks. Each check starts from an empty registry.

In [295]:
def define_plugins() -> dict[str, type]:
    """Define three plugins on a fresh registry."""
    PluginBase.registry.clear()

    class CsvExport(PluginBase, plugin_name="csv"):
        def __init__(self, delimiter: str = ",") -> None:
            self.delimiter = delimiter

    class JsonExport(PluginBase):
        pass

    class PdfExport(PluginBase, plugin_name="pdf"):
        pass

    return {"CsvExport": CsvExport, "JsonExport": JsonExport, "PdfExport": PdfExport}


def duplicate_name() -> None:
    """A second plugin called 'csv'."""
    define_plugins()
    type("OtherCsv", (PluginBase,), {}, plugin_name="csv")


run_checks(
    [
        (
            "registered names",
            lambda: sorted(define_plugins()["CsvExport"].registry),
            ["csv", "jsonexport", "pdf"],
        ),
        (
            "available()",
            lambda: (define_plugins(), PluginBase.available())[1],
            ["csv", "jsonexport", "pdf"],
        ),
        (
            "plugin_name stored",
            lambda: [plugin.plugin_name for plugin in define_plugins().values()],
            ["csv", "jsonexport", "pdf"],
        ),
        (
            "create with args",
            lambda: (define_plugins(), PluginBase.create("csv", delimiter=";").delimiter)[1],
            ";",
        ),
        (
            "create type",
            lambda: (define_plugins(), type(PluginBase.create("pdf")).__name__)[1],
            "PdfExport",
        ),
        (
            "unknown name",
            lambda: (define_plugins(), raised_error_name(lambda: PluginBase.create("xml")))[1],
            "KeyError",
        ),
        (
            "duplicate",
            lambda: raised_message(duplicate_name),
            "ValueError: duplicate plugin name: 'csv'",
        ),
        ("no metaclass", lambda: type(define_plugins()["PdfExport"]).__name__, "type"),
        ("registry is a class attribute", lambda: "registry" in vars(PluginBase), True),
    ]
)

FAIL  registered names: raised NotImplementedError: Lab 2.5: implement PluginBase.__init_subclass__
FAIL  available(): raised NotImplementedError: Lab 2.5: implement PluginBase.__init_subclass__
FAIL  plugin_name stored: raised NotImplementedError: Lab 2.5: implement PluginBase.__init_subclass__
FAIL  create with args: raised NotImplementedError: Lab 2.5: implement PluginBase.__init_subclass__
FAIL  create type: raised NotImplementedError: Lab 2.5: implement PluginBase.__init_subclass__
FAIL  unknown name: raised NotImplementedError: Lab 2.5: implement PluginBase.__init_subclass__
FAIL  duplicate: expected "ValueError: duplicate plugin name: 'csv'", got 'NotImplementedError: Lab 2.5: implement PluginBase.__init_subclass__'
FAIL  no metaclass: raised NotImplementedError: Lab 2.5: implement PluginBase.__init_subclass__
PASS  registry is a class attribute
1/9 checks passed


#### Exercise 5 — a mini ORM with `Field` descriptors

Build the core of an ORM and run its SQL against an in-memory SQLite database.

- `Field(sql_type, *, python_type, primary_key=False, nullable=True)` is a data
  descriptor. `__set_name__` records `name`; instance values live in the instance
  `__dict__`; reading an unset field returns `None`. Assigning `None` to a
  non-nullable field raises `ValueError(f"{name} cannot be NULL")`; a value of the
  wrong type raises `TypeError(f"{name} must be {python_type.__name__}")`.
- `field.column_sql()` → `"id INTEGER PRIMARY KEY"`, `"name TEXT NOT NULL"`,
  `"email TEXT"` (`PRIMARY KEY` replaces `NOT NULL`).
- `Model.__init_subclass__` collects the subclass's `Field`s **in definition order**
  into `cls.fields` (a dict) and sets `cls.table_name` to the class attribute if the
  class defines one, else the lower-cased class name plus `"s"`.
- `Model(**values)` assigns every field (missing ones get `None`, so non-nullable
  fields must be given); an unknown keyword raises
  `TypeError(f"unknown field: {name}")`.
- `Model.create_table_sql()` (class method) →
  `"CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT)"`.
- `instance.insert_sql()` → `("INSERT INTO users (id, name, email) VALUES (?, ?, ?)",
  (1, 'Asha', None))`; `repr` → `User(id=1, name='Asha', email=None)`.

`IntegerField` and `TextField` are given.

In [296]:
class Field:
    """A typed column stored on model instances."""

    def __init__(
        self,
        sql_type: str,
        *,
        python_type: type,
        primary_key: bool = False,
        nullable: bool = True,
    ) -> None:
        raise NotImplementedError("Lab 2.5: implement Field")


class IntegerField(Field):
    def __init__(self, *, primary_key: bool = False, nullable: bool = True) -> None:
        super().__init__("INTEGER", python_type=int, primary_key=primary_key, nullable=nullable)


class TextField(Field):
    def __init__(self, *, primary_key: bool = False, nullable: bool = True) -> None:
        super().__init__("TEXT", python_type=str, primary_key=primary_key, nullable=nullable)


class Model:
    """Base class: collects fields and writes SQL."""

    fields: dict[str, Field] = {}
    table_name: str = ""

    def __init__(self, **values: Any) -> None:
        raise NotImplementedError("Lab 2.5: implement Model")

Run the checks. `define_models()` declares two models; the last check runs the SQL.

In [297]:
def define_models() -> tuple[type, type]:
    class User(Model):
        id = IntegerField(primary_key=True)
        name = TextField(nullable=False)
        email = TextField()

    class Person(Model):
        table_name = "people"
        person_id = IntegerField(primary_key=True)

    return User, Person


def round_trip() -> list[tuple[object, ...]]:
    """Create the table, insert two users, read them back."""
    user_class, _ = define_models()
    connection = sqlite3.connect(":memory:")
    connection.execute(user_class.create_table_sql())
    for user in (
        user_class(id=1, name="Asha"),
        user_class(id=2, name="Ravi", email="ravi@example.com"),
    ):
        connection.execute(*user.insert_sql())
    rows = connection.execute("SELECT id, name, email FROM users ORDER BY id").fetchall()
    connection.close()
    return rows


run_checks(
    [
        ("fields in order", lambda: list(define_models()[0].fields), ["id", "name", "email"]),
        (
            "table names",
            lambda: [model.table_name for model in define_models()],
            ["users", "people"],
        ),
        (
            "column sql",
            lambda: [field.column_sql() for field in define_models()[0].fields.values()],
            ["id INTEGER PRIMARY KEY", "name TEXT NOT NULL", "email TEXT"],
        ),
        (
            "create table",
            lambda: define_models()[0].create_table_sql(),
            "CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT)",
        ),
        (
            "insert sql",
            lambda: define_models()[0](id=1, name="Asha").insert_sql(),
            ("INSERT INTO users (id, name, email) VALUES (?, ?, ?)", (1, "Asha", None)),
        ),
        (
            "repr",
            lambda: repr(define_models()[0](id=1, name="Asha")),
            "User(id=1, name='Asha', email=None)",
        ),
        (
            "not null",
            lambda: raised_message(lambda: define_models()[0](id=1)),
            "ValueError: name cannot be NULL",
        ),
        (
            "type check",
            lambda: raised_message(lambda: define_models()[0](id="1", name="Asha")),
            "TypeError: id must be int",
        ),
        (
            "unknown field",
            lambda: raised_message(lambda: define_models()[0](id=1, name="A", phone="9")),
            "TypeError: unknown field: phone",
        ),
        (
            "data descriptor",
            lambda: inspect.isdatadescriptor(define_models()[0].__dict__["name"]),
            True,
        ),
        ("sqlite round trip", round_trip, [(1, "Asha", None), (2, "Ravi", "ravi@example.com")]),
    ]
)

FAIL  fields in order: raised NotImplementedError: Lab 2.5: implement Field
FAIL  table names: raised NotImplementedError: Lab 2.5: implement Field
FAIL  column sql: raised NotImplementedError: Lab 2.5: implement Field
FAIL  create table: raised NotImplementedError: Lab 2.5: implement Field
FAIL  insert sql: raised NotImplementedError: Lab 2.5: implement Field
FAIL  repr: raised NotImplementedError: Lab 2.5: implement Field
FAIL  not null: expected 'ValueError: name cannot be NULL', got 'NotImplementedError: Lab 2.5: implement Field'
FAIL  type check: expected 'TypeError: id must be int', got 'NotImplementedError: Lab 2.5: implement Field'
FAIL  unknown field: expected 'TypeError: unknown field: phone', got 'NotImplementedError: Lab 2.5: implement Field'
FAIL  data descriptor: raised NotImplementedError: Lab 2.5: implement Field
FAIL  sqlite round trip: raised NotImplementedError: Lab 2.5: implement Field
0/11 checks passed


### Recap — Unit 2.5

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Lookup order | data descriptor → instance `__dict__` → non-data descriptor / class attribute → `__getattr__` | prototype chain |
| `__getattr__` | Only for missing names: defaults, delegation, lazy loading | `Proxy` `get` trap (but that runs always) |
| `__getattribute__` / `__setattr__` | Run on every access; use `super()` inside to avoid recursion | `Proxy` traps |
| Descriptor | Class attribute with `__get__` / `__set__` / `__delete__`; `__set_name__` gives it its name | `Object.defineProperty` |
| Data vs non-data | `__set__` or `__delete__` makes it beat the instance dict | — |
| Methods | Functions are non-data descriptors; `__get__` returns the bound method | `.bind(this)`, done automatically |
| `property` / `classmethod` / `staticmethod` | Small descriptors you can rebuild in a few lines | getters / `static` |
| `__slots__` | No `__dict__`, fixed attributes, ~40 B less per instance on 3.15 | sealed objects |
| `type(name, bases, ns)` | Classes are objects created by calling their metaclass | `class` expressions |
| Metaclass | `__prepare__`, `__new__`, `__init__`, `__call__`; one per class | — |
| `__init_subclass__` | Validate, configure or register subclasses without a metaclass | — |
| Class decorator | Change one class after it is built | decorators (TC39) |
| Libraries | Pydantic/Django: metaclasses; SQLAlchemy: descriptors; dataclasses/attrs: class decorators | — |

**Further reading**

- [Descriptor HowTo Guide](https://docs.python.org/3/howto/descriptor.html) (the
  pure-Python `property`, `classmethod` and lookup code),
  [Data model — customizing attribute access](https://docs.python.org/3/reference/datamodel.html#customizing-attribute-access)
- [`__slots__`](https://docs.python.org/3/reference/datamodel.html#slots),
  [Metaclasses](https://docs.python.org/3/reference/datamodel.html#metaclasses),
  [`__init_subclass__`](https://docs.python.org/3/reference/datamodel.html#object.__init_subclass__),
  [`types.new_class`](https://docs.python.org/3/library/types.html#types.new_class)
- [`annotationlib`](https://docs.python.org/3/library/annotationlib.html),
  [`enum.EnumType`](https://docs.python.org/3/library/enum.html#enum.EnumType),
  [`inspect.getattr_static`](https://docs.python.org/3/library/inspect.html#inspect.getattr_static)
- [PEP 252 — Making types look more like classes](https://peps.python.org/pep-0252/),
  [PEP 3115 — Metaclasses in Python 3000](https://peps.python.org/pep-3115/),
  [PEP 487 — Simpler customisation of class creation](https://peps.python.org/pep-0487/),
  [PEP 749 — Implementing PEP 649 (lazy annotations)](https://peps.python.org/pep-0749/)

## Unit 2.6 — Dataclasses, Enums and Pythonic Design Patterns

**Objectives**

- Generate `__init__`, `__repr__`, `__eq__` (and more) with `@dataclass`, and control
  each field with `field()`.
- Choose between a dataclass, `NamedTuple`, `TypedDict`, a plain class, Pydantic and
  attrs.
- Describe JSON-shaped dicts with `TypedDict` (`Required`, `NotRequired`,
  `ReadOnly`).
- Model fixed sets of values with `Enum`, `IntEnum`, `StrEnum`, `Flag` and `IntFlag`.
- Recognize nine classic design patterns and write their Pythonic forms.
- Tell value objects from entities and model each correctly.

**JS / TS bridge**

| JavaScript / TypeScript | Python | The difference that bites |
| --- | --- | --- |
| `class` with a hand-written constructor | `@dataclass` | Fields are declared once, with types; methods are generated |
| `Object.freeze(obj)` | `@dataclass(frozen=True)` | Frozen dataclasses are also **hashable** (usable as dict keys) |
| `{...obj, city: "Pune"}` | `dataclasses.replace(obj, city="Pune")` | Runs `__init__` and validation again |
| TS `type User = {name: string; age?: number}` | `class User(TypedDict): name: str; age: NotRequired[int]` | Erased at run time: just a `dict` |
| TS `readonly` property | `ReadOnly[...]` in a TypedDict, or `frozen=True` | `ReadOnly` is checked only by type checkers |
| TS `enum Status { Paid = "paid" }` | `class Status(StrEnum): PAID = "paid"` | Members are singletons; `Status("paid")` looks one up |
| Bit flags with `\|` on numbers | `Flag` / `IntFlag` | Named members, `in` for membership, readable repr |
| Design patterns from Java/C# books | functions, modules, Protocols | Many patterns shrink to a few lines in Python |

The next cell is this unit's setup.

In [298]:
import copy
import dataclasses
import functools
import importlib.util
import inspect
import json
import sys
from abc import ABC, abstractmethod
from collections.abc import Callable
from dataclasses import (
    KW_ONLY,
    FrozenInstanceError,
    InitVar,
    asdict,
    astuple,
    dataclass,
    field,
    fields,
    replace,
)
from datetime import date
from decimal import Decimal
from enum import Enum, Flag, IntEnum, IntFlag, StrEnum, auto, unique
from typing import Any, NamedTuple, NotRequired, Protocol, ReadOnly, Required, TypedDict


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def set_attribute(target: object, attribute_name: str, value: object) -> None:
    """`target.attribute_name = value` as a call, for use inside raised_error_name(lambda: ...)."""
    setattr(target, attribute_name, value)


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 2.6.1 `@dataclass`

A class for "a bundle of named values" needs the same three methods every time.
Written by hand, the field names appear four times:

In [299]:
class HandWrittenProduct:
    def __init__(self, name: str, price: Decimal, in_stock: bool = True) -> None:
        self.name, self.price, self.in_stock = name, price, in_stock

    def __repr__(self) -> str:
        fields_text = f"name={self.name!r}, price={self.price!r}, in_stock={self.in_stock!r}"
        return f"HandWrittenProduct({fields_text})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, HandWrittenProduct):
            return NotImplemented
        return (self.name, self.price, self.in_stock) == (other.name, other.price, other.in_stock)


print(HandWrittenProduct("pen", Decimal("10")))

HandWrittenProduct(name='pen', price=Decimal('10'), in_stock=True)


**Generated methods.** `@dataclass` reads the class annotations (in order) and
writes `__init__`, `__repr__` and `__eq__` for you, plus `__match_args__` for
`match` (Unit 1.8.4). Because it defines `__eq__`, the class is unhashable by
default (Unit 2.4.3) unless it is frozen.

In [300]:
@dataclass
class Product:
    name: str
    price: Decimal
    in_stock: bool = True


pen = Product("pen", Decimal("10"))
print(pen)
print(f"{pen == Product('pen', Decimal('10'))=}, {inspect.signature(Product)=}")
print(f"{Product.__match_args__=}, {Product.__hash__ is None=}")
match pen:
    case Product(name, price, in_stock=True):
        print(f"matched {name} at {price}")

Product(name='pen', price=Decimal('10'), in_stock=True)
pen == Product('pen', Decimal('10'))=True, inspect.signature(Product)=<Signature (name: str, price: decimal.Decimal, in_stock: bool = True) -> None>
Product.__match_args__=('name', 'price', 'in_stock'), Product.__hash__ is None=True
matched pen at 10


#### `field()` options

Each field can be tuned with `field(...)`:

| Option | Effect |
| --- | --- |
| `default` / `default_factory` | default value / zero-argument function called for each instance |
| `repr=False` | leave the field out of `__repr__` |
| `compare=False` | leave the field out of `__eq__` and ordering (and the hash) |
| `init=False` | not an `__init__` parameter; set it in `__post_init__` or via a default |
| `kw_only=True` | keyword-only parameter in `__init__` |
| `metadata={...}` | read-only mapping for your own tools (validators, serializers) |
| `doc="..."` | docstring for the field (*New in 3.14*) |

**`default_factory`.** A mutable default would be shared by every instance (the
same trap as mutable default arguments, Unit 1.10.5), so dataclasses refuse `list`,
`dict` and `set` defaults. `default_factory=list` calls `list()` per instance.

In [301]:
def define_shared_default() -> type:
    @dataclass
    class Basket:
        items: list[str] = []  # the trap being refused

    return Basket


@dataclass
class Basket:
    owner: str
    items: list[str] = field(default_factory=list)


print(raised_message(define_shared_default))
first_basket, second_basket = Basket("asha"), Basket("ravi")
first_basket.items.append("tea")
print(f"{first_basket=}, {second_basket=}")

ValueError: mutable default <class 'list'> for field items is not allowed: use default_factory
first_basket=Basket(owner='asha', items=['tea']), second_basket=Basket(owner='ravi', items=[])


**`repr=False`** hides a field from `__repr__`: useful for secrets and for huge values
that would flood logs.

In [302]:
@dataclass
class ApiCredentials:
    client_id: str
    client_secret: str = field(repr=False)


print(ApiCredentials("shop-app", "s3cr3t-value"))

ApiCredentials(client_id='shop-app')


**`compare=False`** leaves a field out of equality (and ordering and hashing). A
cache or a "last seen" timestamp should not make two otherwise identical objects
unequal.

In [303]:
@dataclass
class SearchQuery:
    text: str
    page: int = 1
    executed_at: date | None = field(default=None, compare=False)


print(f"{SearchQuery('tea', 1, date(2026, 1, 1)) == SearchQuery('tea', 1, date(2026, 6, 1))=}")
print(f"{SearchQuery('tea', 1) == SearchQuery('tea', 2)=}")

SearchQuery('tea', 1, date(2026, 1, 1)) == SearchQuery('tea', 1, date(2026, 6, 1))=True
SearchQuery('tea', 1) == SearchQuery('tea', 2)=False


**`init=False`** removes a field from `__init__`. Give it a default, or compute it in
`__post_init__` (shown below). Here a running line number comes from a factory.

In [304]:
line_numbers = iter(range(1, 1_000))


@dataclass
class InvoiceLine:
    description: str
    amount: Decimal
    line_number: int = field(init=False, default_factory=lambda: next(line_numbers))


print(InvoiceLine("Tea 500 g", Decimal("240")), InvoiceLine("Mug", Decimal("199")))
print(f"{list(inspect.signature(InvoiceLine).parameters)=}")
print(raised_error_name(lambda: InvoiceLine("x", Decimal(1), line_number=9)))

InvoiceLine(description='Tea 500 g', amount=Decimal('240'), line_number=1) InvoiceLine(description='Mug', amount=Decimal('199'), line_number=2)
list(inspect.signature(InvoiceLine).parameters)=['description', 'amount']
TypeError


**`kw_only=True` on one field** makes that parameter keyword-only, which also lets a
field without a default follow one with a default (keyword-only parameters are
exempt from the ordering rule).

In [305]:
@dataclass
class Coupon:
    code: str
    percent: int = 10
    expires: date = field(kw_only=True)


print(Coupon("DIWALI", expires=date(2026, 11, 10)))
print(f"{inspect.signature(Coupon)=}")
print(raised_error_name(lambda: Coupon("DIWALI", 10, date(2026, 11, 10))))

Coupon(code='DIWALI', percent=10, expires=datetime.date(2026, 11, 10))
inspect.signature(Coupon)=<Signature (code: str, percent: int = 10, *, expires: datetime.date) -> None>
TypeError


**`metadata`** attaches your own information to a field; dataclasses never read it,
but your code can, through `fields()`. This is how serializers and form libraries
attach units, labels or validation rules. **`doc`** (*New in 3.14*) documents a
field.

In [306]:
@dataclass
class Parcel:
    weight: float = field(metadata={"unit": "kg", "max": 30}, doc="Gross weight.")
    length: float = field(metadata={"unit": "cm"})


for parcel_field in fields(Parcel):
    print(f"{parcel_field.name}: {dict(parcel_field.metadata)}")
if sys.version_info >= (3, 14):
    print(f"{fields(Parcel)[0].doc=}")

weight: {'unit': 'kg', 'max': 30}
length: {'unit': 'cm'}
fields(Parcel)[0].doc='Gross weight.'


#### Class options: `frozen`, `order`, `slots`, `kw_only`

**`frozen=True`** makes assignment raise `FrozenInstanceError` (a subclass of
`AttributeError`) and, because the fields can no longer change, generates
`__hash__`: frozen dataclasses work as dict keys and set members.

In [307]:
@dataclass(frozen=True)
class GeoPoint:
    latitude: float
    longitude: float


pune = GeoPoint(18.52, 73.86)
print(raised_message(lambda: set_attribute(pune, "latitude", 0.0)))
print(f"{issubclass(FrozenInstanceError, AttributeError)=}")
print(f"{ {pune: 'Pune'}[GeoPoint(18.52, 73.86)]=}")

FrozenInstanceError: cannot assign to field 'latitude'
issubclass(FrozenInstanceError, AttributeError)=True
 {pune: 'Pune'}[GeoPoint(18.52, 73.86)]='Pune'


**`order=True`** generates `<`, `<=`, `>`, `>=` that compare fields **as a tuple, in
definition order**, so put the most significant field first.

In [308]:
@dataclass(order=True)
class Release:
    major: int
    minor: int
    name: str = field(compare=False)


releases = [Release(3, 15, "latest"), Release(3, 9, "old"), Release(3, 13, "previous")]
print([release.name for release in sorted(releases)])
print(f"{Release(3, 9, 'a') < Release(3, 13, 'b')=}")
print(raised_error_name(lambda: Release(3, 9, "a") < (3, 13)))

['old', 'previous', 'latest']
Release(3, 9, 'a') < Release(3, 13, 'b')=True
TypeError


**`slots=True`** (3.10+) builds the class again with `__slots__` (Unit 2.5.5): less
memory, no stray attributes. **`kw_only=True`** on the class makes every field
keyword-only; the `KW_ONLY` marker does the same for the fields after it.

In [309]:
@dataclass(slots=True)
class Pixel:
    x: int
    y: int


@dataclass(kw_only=True)
class RetryPolicy:
    attempts: int = 3
    backoff_seconds: float = 0.5


@dataclass
class Upload:
    path: str
    _: KW_ONLY
    overwrite: bool = False
    public: bool = False


print(f"{Pixel.__slots__=}, {hasattr(Pixel(1, 2), '__dict__')=}")
print(raised_message(lambda: set_attribute(Pixel(1, 2), "colour", "red")))
print(f"{inspect.signature(RetryPolicy)=}")
print(f"{inspect.signature(Upload)=}")

Pixel.__slots__=('x', 'y'), hasattr(Pixel(1, 2), '__dict__')=False


AttributeError: 'Pixel' object has no attribute 'colour' and no __dict__ for setting new attributes
inspect.signature(RetryPolicy)=<Signature (*, attempts: int = 3, backoff_seconds: float = 0.5) -> None>
inspect.signature(Upload)=<Signature (path: str, *, overwrite: bool = False, public: bool = False) -> None>


#### `__post_init__` and `InitVar`

The generated `__init__` calls `__post_init__()` at the end: the place for
validation and normalization. In a **frozen** class, assign with
`object.__setattr__` (the generated `__init__` does the same).

In [310]:
@dataclass(frozen=True)
class EmailAddress:
    value: str

    def __post_init__(self) -> None:
        cleaned = self.value.strip().lower()
        if cleaned.count("@") != 1:
            raise ValueError(f"invalid email address: {self.value!r}")
        object.__setattr__(self, "value", cleaned)


print(EmailAddress("  Asha@Example.COM "))
print(raised_message(lambda: EmailAddress("asha.example.com")))

EmailAddress(value='asha@example.com')
ValueError: invalid email address: 'asha.example.com'


An **`InitVar`** is an `__init__` parameter that is **not** stored as a field; it is
passed to `__post_init__` instead. Typical use: a raw input you transform and then
throw away, like a plain-text password.

In [311]:
import hashlib


@dataclass
class UserAccount:
    username: str
    password: InitVar[str]
    password_digest: str = field(init=False, repr=False)

    def __post_init__(self, password: str) -> None:
        if len(password) < 8:
            raise ValueError("password must be at least 8 characters")
        self.password_digest = hashlib.sha256(password.encode()).hexdigest()


account = UserAccount("asha", "correct horse")
print(account, f"{len(account.password_digest)=}")
print(f"{[account_field.name for account_field in fields(UserAccount)]=}  (no password)")
print(raised_message(lambda: UserAccount("ravi", "short")))

UserAccount(username='asha') len(account.password_digest)=64
[account_field.name for account_field in fields(UserAccount)]=['username', 'password_digest']  (no password)
ValueError: password must be at least 8 characters


#### `asdict`, `astuple`, `replace`, `fields`

`asdict` and `astuple` convert **recursively** (nested dataclasses, lists and dicts
inside), deep-copying as they go, so the result shares nothing with the object.

In [312]:
@dataclass
class Address:
    city: str
    pin_code: str


@dataclass
class Customer:
    name: str
    addresses: list[Address] = field(default_factory=list)


asha = Customer("Asha", [Address("Pune", "411001")])
as_dict = asdict(asha)
print(f"{as_dict=}")
print(f"{astuple(asha)=}")
as_dict["addresses"][0]["city"] = "Mumbai"
print(f"{asha.addresses[0].city=}  (unchanged: asdict deep-copies)")
print(json.dumps(as_dict))

as_dict={'name': 'Asha', 'addresses': [{'city': 'Pune', 'pin_code': '411001'}]}
astuple(asha)=('Asha', [('Pune', '411001')])
asha.addresses[0].city='Pune'  (unchanged: asdict deep-copies)
{"name": "Asha", "addresses": [{"city": "Mumbai", "pin_code": "411001"}]}


`replace(obj, **changes)` returns a **new** object with some fields changed; it
calls `__init__`, so `__post_init__` validation runs again. Since 3.13 the generic
`copy.replace` does the same for dataclasses, named tuples, dates and more
(*New in 3.13*). `fields()` lists the `Field` objects, which is how generic code
(serializers, `asdict` itself) walks a dataclass.

In [313]:
work_email = EmailAddress("asha@example.com")
print(f"{replace(work_email, value='ASHA@shop.in')=}")
print(raised_message(lambda: replace(work_email, value="not-an-email")))
if sys.version_info >= (3, 13):
    print(f"{copy.replace(pune, longitude=0.0)=}")
print([(customer_field.name, customer_field.type) for customer_field in fields(Customer)])

replace(work_email, value='ASHA@shop.in')=EmailAddress(value='asha@shop.in')
ValueError: invalid email address: 'not-an-email'
copy.replace(pune, longitude=0.0)=GeoPoint(latitude=18.52, longitude=0.0)
[('name', <class 'str'>), ('addresses', list[__main__.Address])]


#### Inheritance rules

A dataclass subclass collects fields from its bases first (in MRO order), then adds
its own. Redefining a field keeps its **original position** but takes the new
default. Because `__init__` parameters follow field order, a base field **with** a
default followed by a subclass field **without** one is an error; `kw_only` (or a
default) fixes it.

In [314]:
@dataclass
class Item:
    sku: str
    quantity: int = 1


@dataclass
class GiftItem(Item):
    quantity: int = 2  # redefined: same position, new default
    message: str = ""


def define_bad_subclass() -> type:
    @dataclass
    class FragileItem(Item):
        handling_note: str  # no default after a defaulted base field

    return FragileItem


@dataclass
class FragileItem(Item):
    handling_note: str = field(kw_only=True)


print(f"{inspect.signature(GiftItem)=}")
print(raised_message(define_bad_subclass))
print(FragileItem("VASE-1", handling_note="this side up"))

inspect.signature(GiftItem)=<Signature (sku: str, quantity: int = 2, message: str = '') -> None>


TypeError: non-default argument 'handling_note' follows default argument 'quantity'
FragileItem(sku='VASE-1', quantity=1, handling_note='this side up')


#### Under the hood: what `@dataclass` stores

The decorator records its options in `__dataclass_params__` and every field in
`__dataclass_fields__`. `eq`, `frozen` and `unsafe_hash` together decide
`__hash__`:

| `eq` | `frozen` | `unsafe_hash` | `__hash__` |
| --- | --- | --- | --- |
| `True` (default) | `False` | `False` | `None` (unhashable) |
| `True` | `True` | `False` | generated from the compared fields |
| `False` | any | `False` | inherited (identity) |
| any | any | `True` | generated even though mutable (dangerous, Unit 2.4.3) |

In [315]:
print(f"{GeoPoint.__dataclass_params__}")
print(f"{list(GeoPoint.__dataclass_fields__)=}")
for point_class in (Product, GeoPoint):
    print(f"{point_class.__name__}: {point_class.__hash__ is None=}")
print(f"{dataclasses.is_dataclass(pune)=}, {dataclasses.is_dataclass(HandWrittenProduct)=}")

_DataclassParams(init=True,repr=True,eq=True,order=False,unsafe_hash=False,frozen=True,match_args=True,kw_only=False,slots=False,weakref_slot=False)
list(GeoPoint.__dataclass_fields__)=['latitude', 'longitude']
Product: point_class.__hash__ is None=True
GeoPoint: point_class.__hash__ is None=False
dataclasses.is_dataclass(pune)=True, dataclasses.is_dataclass(HandWrittenProduct)=False


### 2.6.2 Record-type comparison

Six ways to hold "a book with a title and a price":

| | dataclass | `NamedTuple` | `TypedDict` | plain class | Pydantic `BaseModel` | attrs |
| --- | --- | --- | --- | --- | --- | --- |
| Runtime type | your class | a `tuple` subclass | plain `dict` | your class | your class | your class |
| Generated `__init__` / `repr` / `eq` | yes | yes | no (dict literal) | no | yes | yes |
| Mutable | yes (or `frozen`) | no | yes | yes | yes (or frozen) | yes (or frozen) |
| Validation at run time | only what you write | no | no | what you write | **yes**, with conversion | optional validators |
| Memory | `__dict__` (or `slots=True`) | tuple: small | dict | `__dict__` | larger | slots by default |
| Unpacking / indexing | no | yes (`title, price = book`) | by key | no | no | no |
| Dependency | stdlib | stdlib | stdlib | — | third party | third party |
| Best for | internal domain objects | small immutable records, tuple APIs | JSON-shaped data you pass along | full control / custom behaviour | data from outside (APIs, config) | dataclass++ (validators, converters) |

Each one in code, same data:

In [316]:
@dataclass
class BookDataclass:
    title: str
    price: Decimal


book = BookDataclass("Clean Code", Decimal("499"))
book.price = Decimal("449")  # mutable unless frozen=True
print(f"dataclass:  {book}")

dataclass:  BookDataclass(title='Clean Code', price=Decimal('449'))


`NamedTuple`: immutable, indexable, unpackable, and equal to a plain tuple.

In [317]:
class BookTuple(NamedTuple):
    title: str
    price: Decimal


tuple_book = BookTuple("Clean Code", Decimal("499"))
title, price = tuple_book
print(
    f"NamedTuple: {tuple_book}, {tuple_book[0]=}, {tuple_book == ('Clean Code', Decimal('499'))=}"
)
print(raised_error_name(lambda: set_attribute(tuple_book, "price", Decimal(1))))

NamedTuple: BookTuple(title='Clean Code', price=Decimal('499')), tuple_book[0]='Clean Code', tuple_book == ('Clean Code', Decimal('499'))=True
AttributeError


`TypedDict`: a type annotation for a plain `dict` (details in 2.6.3).

In [318]:
class BookDict(TypedDict):
    title: str
    price: str


dict_book: BookDict = {"title": "Clean Code", "price": "499"}
print(f"TypedDict:  {dict_book}, {type(dict_book).__name__=}")

TypedDict:  {'title': 'Clean Code', 'price': '499'}, type(dict_book).__name__='dict'


A plain class: you write everything, so you control everything.

In [319]:
class BookPlain:
    def __init__(self, title: str, price: Decimal) -> None:
        self.title, self.price = title, price


plain_book = BookPlain("Clean Code", Decimal("499"))
same_data_book = BookPlain("Clean Code", Decimal("499"))
print(f"plain:      {plain_book.title}, {plain_book == same_data_book=}  (identity)")

plain:      Clean Code, plain_book == same_data_book=False  (identity)


Pydantic validates and **converts** at run time (`"499"` becomes `Decimal("499")`).
It is a Phase 6 dependency, so the cell checks first and shows the code it would
run.

In [320]:
if importlib.util.find_spec("pydantic") is None:
    print("pydantic not installed: skipping (Phase 6 installs it). The code would be:")
    print("    class BookModel(BaseModel):\n        title: str\n        price: Decimal")
    print('    BookModel(title="Clean Code", price="499")  # price converted to Decimal')
else:
    from pydantic import BaseModel

    class BookModel(BaseModel):
        title: str
        price: Decimal

    print(f"pydantic:   {BookModel(title='Clean Code', price='499')!r}")

pydantic not installed: skipping (Phase 6 installs it). The code would be:
    class BookModel(BaseModel):
        title: str
        price: Decimal
    BookModel(title="Clean Code", price="499")  # price converted to Decimal


attrs is the project dataclasses were modelled on. It happens to be installed here
(other tools depend on it), but it is not a course dependency, so the cell still
checks.

In [321]:
if importlib.util.find_spec("attrs") is None:
    print("attrs not installed: skipping")
else:
    import attrs

    @attrs.define
    class BookAttrs:
        title: str
        price: Decimal = attrs.field(validator=attrs.validators.instance_of(Decimal))

    print(f"attrs:      {BookAttrs('Clean Code', Decimal('499'))}")
    print(f"            {raised_error_name(lambda: BookAttrs('Clean Code', 499))}  (validator)")

attrs:      BookAttrs(title='Clean Code', price=Decimal('499'))
            TypeError  (validator)


### 2.6.3 `TypedDict`

A `TypedDict` describes the **keys and value types of a dict** for type checkers,
like a TypeScript object type. At run time it is a plain `dict`: no class instance,
no validation, no methods. Use it for JSON payloads you read and pass along.

In [322]:
class OrderPayload(TypedDict):
    order_id: str
    amount_paise: int
    coupon: NotRequired[str]


payload: OrderPayload = {"order_id": "ORD-1", "amount_paise": 49_900}
print(f"{type(payload) is dict=}, {payload=}")
wrong_payload: OrderPayload = {"order_id": 7}  # type: ignore[typeddict-item]
print(f"{wrong_payload=}  (no run-time check: a type checker reports this line)")
print(raised_message(lambda: isinstance(payload, OrderPayload)))

type(payload) is dict=True, payload={'order_id': 'ORD-1', 'amount_paise': 49900}
wrong_payload={'order_id': 7}  (no run-time check: a type checker reports this line)
TypeError: TypedDict does not support instance and class checks


**`total=False`** makes every key optional; **`Required[...]`** marks the exceptions.
**`NotRequired[...]`** does the opposite in a total TypedDict. The class records
the result in `__required_keys__` and `__optional_keys__`.

In [323]:
class ProductPatch(TypedDict, total=False):
    """A PATCH body: send only what changes, but the SKU is always required."""

    sku: Required[str]
    name: str
    price_paise: int


patch: ProductPatch = {"sku": "PEN-01", "price_paise": 1_200}
print(f"{sorted(ProductPatch.__required_keys__)=}")
print(f"{sorted(ProductPatch.__optional_keys__)=}")
print(f"{sorted(OrderPayload.__required_keys__)=}, {sorted(OrderPayload.__optional_keys__)=}")

sorted(ProductPatch.__required_keys__)=['sku']
sorted(ProductPatch.__optional_keys__)=['name', 'price_paise']
sorted(OrderPayload.__required_keys__)=['amount_paise', 'order_id'], sorted(OrderPayload.__optional_keys__)=['coupon']


**`ReadOnly[...]`** (*New in 3.13*, PEP 705) marks keys that must not be changed
through this type; again only type checkers enforce it, recorded in
`__readonly_keys__`.

In [324]:
class AuditedRecord(TypedDict):
    record_id: ReadOnly[str]
    note: str


audited: AuditedRecord = {"record_id": "REC-9", "note": "draft"}
audited["record_id"] = "REC-10"  # type: ignore[typeddict-readonly-mutated]
print(f"{audited=}  (a type checker reports the assignment; Python does not)")
if sys.version_info >= (3, 13):
    print(f"{AuditedRecord.__readonly_keys__=}, {AuditedRecord.__mutable_keys__=}")

audited={'record_id': 'REC-10', 'note': 'draft'}  (a type checker reports the assignment; Python does not)
AuditedRecord.__readonly_keys__=frozenset({'record_id'}), AuditedRecord.__mutable_keys__=frozenset({'note'})


> **Note:** a type checker (mypy, pyright; Phase 4) is what makes `TypedDict` useful.
> To **validate** incoming JSON at run time, use a dataclass with `__post_init__` or
> Pydantic (Phase 6).

### 2.6.4 Enums

An **enum** is a class whose members are a fixed set of named singletons. Members
have a `name` and a `value`, compare by identity, and the class is iterable (its
metaclass `EnumType` provides `__iter__` / `__len__`, Unit 2.5.7).

In [325]:
class ShippingSpeed(Enum):
    STANDARD = 1
    EXPRESS = 2
    SAME_DAY = 3


print(f"{ShippingSpeed.EXPRESS=}, {ShippingSpeed.EXPRESS.name=}, {ShippingSpeed.EXPRESS.value=}")
print(f"{list(ShippingSpeed)=}")
print(f"{len(ShippingSpeed)=}, {ShippingSpeed.EXPRESS is ShippingSpeed.EXPRESS=}")
print(f"{ShippingSpeed.EXPRESS == 2=}  (a plain Enum never equals its value)")

ShippingSpeed.EXPRESS=<ShippingSpeed.EXPRESS: 2>, ShippingSpeed.EXPRESS.name='EXPRESS', ShippingSpeed.EXPRESS.value=2
list(ShippingSpeed)=[<ShippingSpeed.STANDARD: 1>, <ShippingSpeed.EXPRESS: 2>, <ShippingSpeed.SAME_DAY: 3>]
len(ShippingSpeed)=3, ShippingSpeed.EXPRESS is ShippingSpeed.EXPRESS=True
ShippingSpeed.EXPRESS == 2=False  (a plain Enum never equals its value)


**`auto()`** picks values for you: 1, 2, 3, … for `Enum`, the lower-cased name for
`StrEnum`, powers of two for `Flag`.

In [326]:
class TicketPriority(Enum):
    LOW = auto()
    MEDIUM = auto()
    HIGH = auto()


print([(priority.name, priority.value) for priority in TicketPriority])

[('LOW', 1), ('MEDIUM', 2), ('HIGH', 3)]


**Lookup by value and by name.** `Enum(value)` finds the member with that value
(`ValueError` if none); `Enum[name]` finds it by name (`KeyError`). This is how you
turn database values or URL parameters back into members.

In [327]:
print(f"{ShippingSpeed(2)=}, {ShippingSpeed['SAME_DAY']=}")
print(raised_message(lambda: ShippingSpeed(9)))
print(raised_message(lambda: ShippingSpeed["OVERNIGHT"]))
print(f"{ShippingSpeed.__members__.keys()=}")

ShippingSpeed(2)=<ShippingSpeed.EXPRESS: 2>, ShippingSpeed['SAME_DAY']=<ShippingSpeed.SAME_DAY: 3>
ValueError: 9 is not a valid ShippingSpeed
KeyError: 'OVERNIGHT'
ShippingSpeed.__members__.keys()=dict_keys(['STANDARD', 'EXPRESS', 'SAME_DAY'])


**`IntEnum`** members **are** ints: they compare with and do arithmetic like ints,
useful when an API or a C library expects numbers. `http.HTTPStatus` is the
standard library's example.

In [328]:
from http import HTTPStatus


class RetryLimit(IntEnum):
    NONE = 0
    FEW = 3
    MANY = 10


print(f"{RetryLimit.FEW == 3=}, {RetryLimit.FEW + 1=}")
print(f"{sorted([RetryLimit.MANY, 5, RetryLimit.FEW])=}")
print(f"{HTTPStatus(404)=}, {HTTPStatus.NOT_FOUND.phrase=}, {HTTPStatus.NOT_FOUND == 404=}")

RetryLimit.FEW == 3=True, RetryLimit.FEW + 1=4
sorted([RetryLimit.MANY, 5, RetryLimit.FEW])=[<RetryLimit.FEW: 3>, 5, <RetryLimit.MANY: 10>]
HTTPStatus(404)=<HTTPStatus.NOT_FOUND: 404>, HTTPStatus.NOT_FOUND.phrase='Not Found', HTTPStatus.NOT_FOUND == 404=True


**`StrEnum`** (3.11) members are strings: they compare equal to their value, format
as the value, and serialize to JSON with no custom encoder. `auto()` gives the
lower-cased name.

In [329]:
class DeployEnvironment(StrEnum):
    DEVELOPMENT = auto()
    STAGING = auto()
    PRODUCTION = auto()


print(f"{DeployEnvironment.STAGING == 'staging'=}, {DeployEnvironment.STAGING.upper()=}")
print(f"deploying to {DeployEnvironment.PRODUCTION}")
print(json.dumps({"environment": DeployEnvironment.PRODUCTION}))

DeployEnvironment.STAGING == 'staging'=True, DeployEnvironment.STAGING.upper()='STAGING'
deploying to production
{"environment": "production"}


**`Flag`** members combine with `|`, `&`, `^` and `~`; `in` tests whether bits are
set, and iterating a combination yields its single-bit members (3.11+).

In [330]:
class TextStyle(Flag):
    BOLD = auto()
    ITALIC = auto()
    UNDERLINE = auto()


heading_style = TextStyle.BOLD | TextStyle.UNDERLINE
print(f"{heading_style=}, {TextStyle.BOLD in heading_style=}, {TextStyle.ITALIC in heading_style=}")
print(f"{list(heading_style)=}, {heading_style & ~TextStyle.BOLD=}")
print(f"{bool(TextStyle(0))=}  (no flags set is falsy)")

heading_style=<TextStyle.BOLD|UNDERLINE: 5>, TextStyle.BOLD in heading_style=True, TextStyle.ITALIC in heading_style=False
list(heading_style)=[<TextStyle.BOLD: 1>, <TextStyle.UNDERLINE: 4>], heading_style & ~TextStyle.BOLD=<TextStyle.UNDERLINE: 4>
bool(TextStyle(0))=False  (no flags set is falsy)


**`IntFlag`** is a `Flag` that is also an `int`, for bit masks that cross into
numeric APIs, such as Unix file modes.

In [331]:
class FileMode(IntFlag):
    EXECUTE = 0o1
    WRITE = 0o2
    READ = 0o4


read_write = FileMode.READ | FileMode.WRITE
print(f"{read_write=}, {int(read_write)=}, {read_write == 6=}")
print(f"{FileMode(7)=}, {oct(FileMode.READ | FileMode.EXECUTE)=}")

read_write=<FileMode.WRITE|READ: 6>, int(read_write)=6, read_write == 6=True
FileMode(7)=<FileMode.EXECUTE|WRITE|READ: 7>, oct(FileMode.READ | FileMode.EXECUTE)='0o5'


**`@unique`.** Two members with the same value are allowed: the second becomes an
**alias** of the first (handy for legacy names). `@unique` turns that into an error
when aliases would be a mistake.

In [332]:
class PaymentMethod(Enum):
    CARD = "card"
    UPI = "upi"
    CREDIT_CARD = "card"  # alias of CARD


def define_unique_enum() -> type:
    @unique
    class StrictPaymentMethod(Enum):
        CARD = "card"
        CREDIT_CARD = "card"

    return StrictPaymentMethod


print(f"{PaymentMethod.CREDIT_CARD is PaymentMethod.CARD=}, {list(PaymentMethod)=}")
print(f"{list(PaymentMethod.__members__)=}  (aliases are listed here)")
print(raised_message(define_unique_enum))

PaymentMethod.CREDIT_CARD is PaymentMethod.CARD=True, list(PaymentMethod)=[<PaymentMethod.CARD: 'card'>, <PaymentMethod.UPI: 'upi'>]
list(PaymentMethod.__members__)=['CARD', 'UPI', 'CREDIT_CARD']  (aliases are listed here)
ValueError: duplicate values found in <enum 'StrictPaymentMethod'>: CREDIT_CARD -> CARD


**Methods on enums.** Enums are classes, so members can have methods, properties
and class methods. Keeping behaviour next to the values avoids `if/elif` chains
scattered through the code.

In [333]:
class DeliverySlot(Enum):
    STANDARD = ("Standard", 5, Decimal("40"))
    EXPRESS = ("Express", 2, Decimal("99"))

    def __init__(self, label: str, days: int, fee: Decimal) -> None:
        self.label, self.days, self.fee = label, days, fee

    def arrives_by(self, ordered_on: date) -> date:
        return date.fromordinal(ordered_on.toordinal() + self.days)

    @classmethod
    def fastest(cls) -> DeliverySlot:
        return min(cls, key=lambda slot: slot.days)


print(f"{DeliverySlot.EXPRESS.label=}, {DeliverySlot.EXPRESS.fee=}")
print(f"{DeliverySlot.STANDARD.arrives_by(date(2026, 11, 7))=}")
print(f"{DeliverySlot.fastest()=}")

DeliverySlot.EXPRESS.label='Express', DeliverySlot.EXPRESS.fee=Decimal('99')
DeliverySlot.STANDARD.arrives_by(date(2026, 11, 7))=datetime.date(2026, 11, 12)
DeliverySlot.fastest()=<DeliverySlot.EXPRESS: ('Express', 2, Decimal('99'))>


**Enums in `match`.** A dotted name such as `ShippingSpeed.EXPRESS` is a **value
pattern** (compared with `==`), not a capture, so enums read naturally in `match`.

In [334]:
def shipping_message(speed: ShippingSpeed) -> str:
    match speed:
        case ShippingSpeed.SAME_DAY:
            return "arrives tonight"
        case ShippingSpeed.EXPRESS:
            return "arrives in 2 days"
        case _:
            return "arrives within a week"


print([shipping_message(speed) for speed in ShippingSpeed])

['arrives within a week', 'arrives in 2 days', 'arrives tonight']


**`_missing_`** is a class method called when `Enum(value)` finds no member. Return
a member to accept the value, or `None` to let the `ValueError` happen. A common use
is forgiving lookups for user input.

In [335]:
class Region(StrEnum):
    NORTH = auto()
    SOUTH = auto()

    @classmethod
    def _missing_(cls, value: object) -> Region | None:
        if isinstance(value, str):
            cleaned = value.strip().lower()
            for member in cls:
                if member.value == cleaned:
                    return member
        return None


print(f"{Region('  SOUTH ')=}, {Region('north')=}")
print(raised_message(lambda: Region("east")))

Region('  SOUTH ')=<Region.SOUTH: 'south'>, Region('north')=<Region.NORTH: 'north'>
ValueError: 'east' is not a valid Region


### 2.6.5 Design patterns, Pythonic style

The classic patterns (from *Design Patterns*, 1994, written for C++ and Smalltalk)
solve real problems, but many of their class hierarchies exist because those
languages lacked first-class functions, modules and duck typing. Each pattern below
is shown **non-Pythonic first** (a direct Java-style translation), then
**Pythonic**.

| Pattern | Problem | Pythonic tool |
| --- | --- | --- |
| Strategy | swap an algorithm | pass a function |
| Factory | create the right class from data | dict of classes, `classmethod` constructors |
| Singleton | one shared instance | a module-level object |
| Registry | look implementations up by name | a decorator that fills a dict |
| Observer | notify listeners of changes | a list of callables |
| Adapter | make an old interface fit a new one | a small wrapper class (or `__getattr__`) |
| Builder | build an object with many options | keyword arguments with defaults, `replace` |
| Dependency injection | give an object its collaborators | constructor parameters typed with a Protocol |
| Repository | hide storage behind a collection-like API | a Protocol + an in-memory implementation |

#### Strategy

**Non-Pythonic:** an abstract strategy class and one subclass per algorithm, each
with a single method.

In [336]:
class ShippingCostStrategy(ABC):
    @abstractmethod
    def cost(self, weight_kg: float) -> Decimal: ...


class FlatRate(ShippingCostStrategy):
    def cost(self, weight_kg: float) -> Decimal:
        return Decimal("50")


class PerKilo(ShippingCostStrategy):
    def cost(self, weight_kg: float) -> Decimal:
        return Decimal("20") * Decimal(str(weight_kg))


class CheckoutWithStrategy:
    def __init__(self, strategy: ShippingCostStrategy) -> None:
        self.strategy = strategy

    def shipping(self, weight_kg: float) -> Decimal:
        return self.strategy.cost(weight_kg)


print(CheckoutWithStrategy(PerKilo()).shipping(2.5))

50.0


**Pythonic:** a strategy with one method *is* a function. Pass the function; use a
dict when the choice comes from data. (A class is still right when the strategy has
configuration and several methods.)

In [337]:
type ShippingRule = Callable[[float], Decimal]


def flat_rate(weight_kg: float) -> Decimal:
    return Decimal("50")


def per_kilo(weight_kg: float) -> Decimal:
    return Decimal("20") * Decimal(str(weight_kg))


SHIPPING_RULES: dict[str, ShippingRule] = {"flat": flat_rate, "per_kg": per_kilo}


def shipping_cost(weight_kg: float, rule: ShippingRule = flat_rate) -> Decimal:
    return rule(weight_kg)


print(
    shipping_cost(2.5, per_kilo),
    shipping_cost(2.5, SHIPPING_RULES["flat"]),
    shipping_cost(1, lambda kg: Decimal(0)),
)

50.0 50 0


#### Factory

**Non-Pythonic:** a factory class with a growing `if/elif` chain that must be edited
for every new type.

In [338]:
class EmailMessage:
    def __init__(self, to: str) -> None:
        self.to = to


class SmsMessage:
    def __init__(self, to: str) -> None:
        self.to = to


class MessageFactory:
    def create(self, kind: str, to: str) -> object:
        if kind == "email":
            return EmailMessage(to)
        elif kind == "sms":
            return SmsMessage(to)
        else:
            raise ValueError(f"unknown kind {kind!r}")


print(type(MessageFactory().create("sms", "+91 98765 43210")).__name__)

SmsMessage


**Pythonic:** classes are objects, so a dict maps names to classes. For "build this
class from another representation", use alternative constructors
(`@classmethod`, Unit 2.1.5).

In [339]:
MESSAGE_TYPES: dict[str, type] = {"email": EmailMessage, "sms": SmsMessage}


def create_message(kind: str, to: str) -> object:
    try:
        return MESSAGE_TYPES[kind](to)
    except KeyError:
        raise ValueError(f"unknown kind {kind!r}") from None


print(type(create_message("email", "asha@example.com")).__name__)
print(raised_message(lambda: create_message("fax", "123")))
print(f"{date.fromisoformat('2026-11-07')=}  (an alternative constructor is a factory)")

EmailMessage
ValueError: unknown kind 'fax'
date.fromisoformat('2026-11-07')=datetime.date(2026, 11, 7)  (an alternative constructor is a factory)


#### Singleton

**Non-Pythonic:** override `__new__` (or use a metaclass, Unit 2.5.7) so the class
always returns the same object. Calls *look* like they create new objects, which
misleads readers, and tests cannot get a fresh instance.

In [340]:
class ConfigSingleton:
    _instance: ConfigSingleton | None = None

    def __new__(cls) -> ConfigSingleton:
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.settings = {"currency": "INR"}
        return cls._instance


print(f"{ConfigSingleton() is ConfigSingleton()=}")

ConfigSingleton() is ConfigSingleton()=True


**Pythonic:** a module is imported once and cached in `sys.modules` (Unit 3.3), so
an object created at module level **is** a singleton. Create one instance and import
it. For a lazily built one, `functools.cache` on a getter function works.

In [341]:
@dataclass
class AppSettings:
    currency: str = "INR"
    tax_rate_percent: Decimal = Decimal("18")


settings = AppSettings()  # in settings.py; everyone does `from settings import settings`


@functools.cache
def get_settings() -> AppSettings:
    """Built on first call, then cached (FastAPI apps often use this shape)."""
    return AppSettings()


print(f"{get_settings() is get_settings()=}")
print(f"{AppSettings() is settings=}  (tests can still make their own)")

get_settings() is get_settings()=True
AppSettings() is settings=False  (tests can still make their own)


#### Registry

**Non-Pythonic:** a central list maintained by hand. Adding an exporter means
editing two places, and forgetting the second is a silent bug.

In [342]:
def export_csv(rows: list[dict[str, object]]) -> str:
    return "\n".join(",".join(map(str, row.values())) for row in rows)


def export_json(rows: list[dict[str, object]]) -> str:
    return json.dumps(rows)


MANUAL_EXPORTERS = {"csv": export_csv, "json": export_json}  # remember to update!
print(MANUAL_EXPORTERS["csv"]([{"sku": "PEN-01", "qty": 3}]))

PEN-01,3


**Pythonic:** a decorator registers the function where it is defined (Unit 1.12.6);
for classes, `__init_subclass__` does the same (Lab 2.5, exercise 4).

In [343]:
REPORT_EXPORTERS: dict[str, Callable[[list[dict[str, object]]], str]] = {}


def exporter(format_name: str) -> Callable[[Callable[..., str]], Callable[..., str]]:
    def register(func: Callable[..., str]) -> Callable[..., str]:
        REPORT_EXPORTERS[format_name] = func
        return func

    return register


@exporter("tsv")
def export_tsv(rows: list[dict[str, object]]) -> str:
    return "\n".join("\t".join(map(str, row.values())) for row in rows)


print(f"{sorted(REPORT_EXPORTERS)=}, {REPORT_EXPORTERS['tsv']([{'sku': 'PEN-01', 'qty': 3}])!r}")

sorted(REPORT_EXPORTERS)=['tsv'], 'PEN-01\t3'


#### Observer

**Non-Pythonic:** an `Observer` interface with an `update()` method, and one class
per listener.

In [344]:
class PriceObserver(ABC):
    @abstractmethod
    def update(self, sku: str, new_price: Decimal) -> None: ...


class PrintingObserver(PriceObserver):
    def update(self, sku: str, new_price: Decimal) -> None:
        print(f"  observer: {sku} now {new_price}")


class ObservablePriceList:
    def __init__(self) -> None:
        self._observers: list[PriceObserver] = []

    def attach(self, observer: PriceObserver) -> None:
        self._observers.append(observer)

    def set_price(self, sku: str, new_price: Decimal) -> None:
        for observer in self._observers:
            observer.update(sku, new_price)


observable_prices = ObservablePriceList()
observable_prices.attach(PrintingObserver())
observable_prices.set_price("PEN-01", Decimal("12"))

  observer: PEN-01 now 12


**Pythonic:** listeners are callables: functions, lambdas, bound methods, objects
with `__call__`. Keep a list of them. (Lab 2.6 grows this into a typed event bus.)

In [345]:
class PriceList:
    def __init__(self) -> None:
        self.listeners: list[Callable[[str, Decimal], None]] = []
        self.prices: dict[str, Decimal] = {}

    def set_price(self, sku: str, new_price: Decimal) -> None:
        self.prices[sku] = new_price
        for listener in self.listeners:
            listener(sku, new_price)


price_changes: list[str] = []
price_list = PriceList()
price_list.listeners.append(lambda sku, price: price_changes.append(f"{sku}={price}"))
price_list.listeners.append(lambda sku, price: print(f"  listener: {sku} now {price}"))
price_list.set_price("PEN-01", Decimal("12"))
print(f"{price_changes=}")

  listener: PEN-01 now 12
price_changes=['PEN-01=12']


#### Adapter

**Non-Pythonic:** a full adapter class re-declaring every method of the target
interface, forwarding each one by hand, often through an extra abstract base.

In [346]:
class LegacySmsGateway:
    """An old client with a different method name and argument order."""

    def send_text(self, body: str, phone_number: str) -> str:
        return f"legacy sent {body!r} to {phone_number}"

    def balance(self) -> int:
        return 120


class MessageSenderBase(ABC):
    @abstractmethod
    def send(self, recipient: str, text: str) -> str: ...

    @abstractmethod
    def balance(self) -> int: ...


class VerboseSmsAdapter(MessageSenderBase):
    def __init__(self, gateway: LegacySmsGateway) -> None:
        self.gateway = gateway

    def send(self, recipient: str, text: str) -> str:
        return self.gateway.send_text(text, recipient)

    def balance(self) -> int:  # pure forwarding, repeated for every method
        return self.gateway.balance()


print(VerboseSmsAdapter(LegacySmsGateway()).send("+91 98765 43210", "Your order shipped"))

legacy sent 'Your order shipped' to +91 98765 43210


**Pythonic:** the target is a Protocol (Unit 2.3.3), so no base class is needed.
Write only the method that differs and forward the rest with `__getattr__` (Unit
2.5.2). For a single function, an adapter can be a lambda.

In [347]:
class MessageSender(Protocol):
    def send(self, recipient: str, text: str) -> str: ...


class SmsAdapter:
    def __init__(self, gateway: LegacySmsGateway) -> None:
        self._gateway = gateway

    def send(self, recipient: str, text: str) -> str:
        return self._gateway.send_text(text, recipient)

    def __getattr__(self, attribute_name: str) -> Any:
        if attribute_name == "_gateway":
            raise AttributeError(attribute_name)
        return getattr(self._gateway, attribute_name)


def notify(sender: MessageSender, recipient: str) -> str:
    return sender.send(recipient, "Your order shipped")


sms_adapter = SmsAdapter(LegacySmsGateway())
print(notify(sms_adapter, "+91 98765 43210"), f"| {sms_adapter.balance()=}")

legacy sent 'Your order shipped' to +91 98765 43210 | sms_adapter.balance()=120


#### Builder

**Non-Pythonic:** a builder with one chained setter per option, needed in Java
because it has no keyword arguments or default values.

In [348]:
class HttpRequestBuilder:
    def __init__(self, url: str) -> None:
        self._url, self._method, self._headers, self._timeout = url, "GET", {}, 10.0

    def method(self, method: str) -> HttpRequestBuilder:
        self._method = method
        return self

    def header(self, name: str, value: str) -> HttpRequestBuilder:
        self._headers[name] = value
        return self

    def timeout(self, seconds: float) -> HttpRequestBuilder:
        self._timeout = seconds
        return self

    def build(self) -> tuple[str, str, dict[str, str], float]:
        return (self._method, self._url, self._headers, self._timeout)


print(
    HttpRequestBuilder("https://api.example.com/orders")
    .method("POST")
    .header("Accept", "application/json")
    .timeout(5)
    .build()
)

('POST', 'https://api.example.com/orders', {'Accept': 'application/json'}, 5)


**Pythonic:** keyword arguments with defaults *are* the builder. A frozen dataclass
adds validation and `replace()` for variations of a base request.

In [349]:
@dataclass(frozen=True)
class HttpRequest:
    url: str
    method: str = "GET"
    headers: dict[str, str] = field(default_factory=dict)
    timeout_seconds: float = 10.0


create_order = HttpRequest(
    "https://api.example.com/orders",
    method="POST",
    headers={"Accept": "application/json"},
    timeout_seconds=5,
)
list_orders = replace(create_order, method="GET")
print(create_order)
print(list_orders.method, list_orders.headers)

HttpRequest(url='https://api.example.com/orders', method='POST', headers={'Accept': 'application/json'}, timeout_seconds=5)
GET {'Accept': 'application/json'}


#### Dependency injection

**Non-Pythonic:** the service creates its own collaborators. It is welded to a real
mail server, so testing it sends real e-mail (or needs monkey-patching).

In [350]:
class SmtpMailer:
    def send(self, to: str, subject: str) -> None:
        print(f"  (would open an SMTP connection and mail {to})")


class SignupServiceHardWired:
    def __init__(self) -> None:
        self.mailer = SmtpMailer()  # hidden, fixed dependency

    def register(self, email: str) -> str:
        self.mailer.send(email, "Welcome!")
        return f"registered {email}"


print(SignupServiceHardWired().register("asha@example.com"))

  (would open an SMTP connection and mail asha@example.com)
registered asha@example.com


**Pythonic:** take the collaborator as a constructor parameter typed with a
Protocol. Production passes the real one; tests pass a fake that records calls. No
framework required (FastAPI's `Depends`, Phase 6, automates the wiring).

In [351]:
class Mailer(Protocol):
    def send(self, to: str, subject: str) -> None: ...


class RecordingMailer:
    def __init__(self) -> None:
        self.sent: list[tuple[str, str]] = []

    def send(self, to: str, subject: str) -> None:
        self.sent.append((to, subject))


class SignupService:
    def __init__(self, mailer: Mailer) -> None:
        self.mailer = mailer

    def register(self, email: str) -> str:
        self.mailer.send(email, "Welcome!")
        return f"registered {email}"


fake_mailer = RecordingMailer()
print(SignupService(fake_mailer).register("asha@example.com"), f"| {fake_mailer.sent=}")

registered asha@example.com | fake_mailer.sent=[('asha@example.com', 'Welcome!')]


#### Repository

**Non-Pythonic:** storage details (SQL strings, connection handling) are mixed into
business logic, so the logic cannot be tested or reused without a database.

In [352]:
import sqlite3


def deactivate_stale_coupons_inline(connection: sqlite3.Connection, today: str) -> int:
    rows = connection.execute("SELECT code FROM coupons WHERE expires < ?", (today,)).fetchall()
    for (coupon_code,) in rows:
        connection.execute("UPDATE coupons SET active = 0 WHERE code = ?", (coupon_code,))
    return len(rows)


coupon_db = sqlite3.connect(":memory:")
coupon_db.execute("CREATE TABLE coupons (code TEXT, expires TEXT, active INTEGER)")
coupon_db.executemany(
    "INSERT INTO coupons VALUES (?, ?, 1)", [("OLD10", "2026-01-01"), ("NEW20", "2027-01-01")]
)
print(f"{deactivate_stale_coupons_inline(coupon_db, '2026-11-07')=}")
coupon_db.close()

deactivate_stale_coupons_inline(coupon_db, '2026-11-07')=1


**Pythonic:** a `CouponRepository` Protocol describes a collection-like API; the
business rule depends only on it. An in-memory implementation serves tests (and this
notebook); a SQLite or PostgreSQL one serves production (Phase 6). The Phase 2
Project builds an inventory repository the same way.

In [353]:
@dataclass
class StoredCoupon:
    code: str
    expires: date
    active: bool = True


class CouponRepository(Protocol):
    def list_active(self) -> list[StoredCoupon]: ...
    def save(self, coupon: StoredCoupon) -> None: ...


class InMemoryCouponRepository:
    def __init__(self, coupons: list[StoredCoupon]) -> None:
        self._coupons = {coupon.code: coupon for coupon in coupons}

    def list_active(self) -> list[StoredCoupon]:
        return [coupon for coupon in self._coupons.values() if coupon.active]

    def save(self, coupon: StoredCoupon) -> None:
        self._coupons[coupon.code] = coupon


def deactivate_stale_coupons(repository: CouponRepository, today: date) -> int:
    stale = [coupon for coupon in repository.list_active() if coupon.expires < today]
    for coupon in stale:
        repository.save(replace(coupon, active=False))
    return len(stale)


coupon_repository = InMemoryCouponRepository(
    [StoredCoupon("OLD10", date(2026, 1, 1)), StoredCoupon("NEW20", date(2027, 1, 1))]
)
print(f"{deactivate_stale_coupons(coupon_repository, date(2026, 11, 7))=}")
print(f"{[coupon.code for coupon in coupon_repository.list_active()]=}")

deactivate_stale_coupons(coupon_repository, date(2026, 11, 7))=1
[coupon.code for coupon in coupon_repository.list_active()]=['NEW20']


### 2.6.6 Value objects vs entities

Domain-driven design separates two kinds of objects:

| | Value object | Entity |
| --- | --- | --- |
| Identity | defined by its **values**: two equal values are interchangeable | defined by an **id** that stays the same while its data changes |
| Equality | compare all fields | compare the id only |
| Mutability | immutable: "changing" it means making a new one | mutable over its lifecycle |
| Hashable | yes | by id, if at all |
| Examples | `Money`, `EmailAddress`, `Address`, `DateRange`, `GeoPoint` | `Customer`, `Order`, `Warehouse`, `BankAccount` |
| Python tool | `@dataclass(frozen=True)` (or a class like Unit 2.4's `Money`) | a class (or dataclass with `eq=False` / `compare=False` fields) |

**A value object**: `DateRange` is immutable, compares by value, works as a dict key,
and its operations return new objects.

In [354]:
@dataclass(frozen=True)
class DateRange:
    start: date
    end: date

    def __post_init__(self) -> None:
        if self.end < self.start:
            raise ValueError("end must not be before start")

    def __contains__(self, day: date) -> bool:
        return self.start <= day <= self.end

    def extended_by(self, days: int) -> DateRange:
        return replace(self, end=date.fromordinal(self.end.toordinal() + days))


sale_period = DateRange(date(2026, 11, 1), date(2026, 11, 10))
longer_sale = sale_period.extended_by(5)
print(f"{sale_period == DateRange(date(2026, 11, 1), date(2026, 11, 10))=}")
print(f"{date(2026, 11, 12) in sale_period=}, {date(2026, 11, 12) in longer_sale=}")
print(f"{ {sale_period: 'Diwali sale'}[DateRange(date(2026, 11, 1), date(2026, 11, 10))]=}")

sale_period == DateRange(date(2026, 11, 1), date(2026, 11, 10))=True
date(2026, 11, 12) in sale_period=False, date(2026, 11, 12) in longer_sale=True
 {sale_period: 'Diwali sale'}[DateRange(date(2026, 11, 1), date(2026, 11, 10))]='Diwali sale'


**An entity**: a `CustomerAccount` keeps its identity while its e-mail and status
change. Equality and hashing use only `account_id`, so the same customer loaded twice
(with different data) is still "the same customer". Here `compare=False` on every
other field plus `unsafe_hash=True` makes the hash use only the id, which never
changes.

In [355]:
@dataclass(unsafe_hash=True)
class CustomerAccount:
    account_id: int
    email: EmailAddress = field(compare=False)
    active: bool = field(default=True, compare=False)

    def change_email(self, new_email: str) -> None:
        self.email = EmailAddress(new_email)  # replace the value object


loaded_yesterday = CustomerAccount(7, EmailAddress("asha@old.in"))
loaded_today = CustomerAccount(7, EmailAddress("asha@new.in"), active=False)
other_customer = CustomerAccount(8, EmailAddress("asha@old.in"))
print(f"{loaded_yesterday == loaded_today=}  (same id)")
print(f"{loaded_yesterday == other_customer=}  (same data, different id)")
loaded_yesterday.change_email("ASHA@new.in")
print(f"{loaded_yesterday.email=}, {len({loaded_yesterday, loaded_today})=}")

loaded_yesterday == loaded_today=True  (same id)
loaded_yesterday == other_customer=False  (same data, different id)
loaded_yesterday.email=EmailAddress(value='asha@new.in'), len({loaded_yesterday, loaded_today})=1


> **Note:** entities usually **contain** value objects (`CustomerAccount.email`,
> `Order.total: Money`). Change an entity by replacing its value objects, never by
> mutating them, and the bugs from shared mutable state (Unit 1.2) disappear.

**Predict the output.**

```python
from enum import Enum

class Colour(Enum):
    RED = 1
    CRIMSON = 1
    BLUE = 2

print(Colour.CRIMSON is Colour.RED)
print(Colour.CRIMSON)
print(len(Colour), [colour.name for colour in Colour])
```

<details><summary>Answer</summary>

```text
True
Colour.RED
2 ['RED', 'BLUE']
```

`CRIMSON = 1` reuses `RED`'s value, so it becomes an **alias**: the same object as
`Colour.RED`, printed with the canonical name. Iteration and `len()` skip aliases
(they appear only in `__members__`). Add `@unique` if duplicate values are a
mistake.

</details>

Run the next cell to check your prediction.

In [356]:
class Colour(Enum):
    RED = 1
    CRIMSON = 1
    BLUE = 2


print(Colour.CRIMSON is Colour.RED)
print(Colour.CRIMSON)
print(len(Colour), [colour.name for colour in Colour])

True
Colour.RED
2 ['RED', 'BLUE']


### Lab 2.6

Four exercises. For each, complete the stub, then run the check cell. Until then the
checks print FAIL; that is expected. In exercise 2 one check ("is a str") already
passes on the stub because the base class is given; keep it passing.

#### Exercise 1 — `Address` / `Customer` dataclasses with validation

Replace the stubs with dataclasses (2.6.1). Earlier labs used these class names for
other shapes; these definitions replace them.

- `Address(street, city, pin_code)` is a **frozen** dataclass. `__post_init__`
  strips `street` and `city`; if either is then empty, raise
  `ValueError("street and city are required")`. `pin_code` must be six digits not
  starting with `0`, otherwise `ValueError(f"invalid PIN code: {pin_code!r}")`.
- `Customer(customer_id, name, email, addresses=[])` is a dataclass (an entity,
  2.6.6). `name` is stripped and must not be empty
  (`ValueError("name is required")`); `email` is stripped and lower-cased and must
  contain exactly one `@` with a `.` after it, otherwise
  `ValueError(f"invalid email: {email!r}")` (showing the original input).
  `addresses` defaults to a **new** list per customer.
- Two customers are equal when their `customer_id`s are equal, whatever the other
  fields say.
- `customer.add_address(address)` appends the address and returns `True`, or returns
  `False` if an equal address is already there.

In [357]:
class Address:
    def __init__(self, street: str, city: str, pin_code: str) -> None:
        raise NotImplementedError("Lab 2.6: implement Address as a frozen dataclass")


class Customer:
    def __init__(
        self, customer_id: int, name: str, email: str, addresses: list[Address] | None = None
    ) -> None:
        raise NotImplementedError("Lab 2.6: implement Customer as a dataclass")

Run the checks.

In [358]:
def address_errors() -> list[str]:
    return [
        raised_message(lambda: Address("  ", "Pune", "411001")),
        raised_message(lambda: Address("MG Road", "Pune", "011001")),
        raised_message(lambda: Address("MG Road", "Pune", "41100")),
    ]


def address_book() -> tuple[bool, bool, int]:
    customer = Customer(1, "Asha", "asha@example.com")
    first = customer.add_address(Address("MG Road", "Pune", "411001"))
    again = customer.add_address(Address(" MG Road ", "Pune", "411001"))
    return first, again, len(customer.addresses)


run_checks(
    [
        (
            "is a dataclass",
            lambda: (dataclasses.is_dataclass(Address), dataclasses.is_dataclass(Customer)),
            (True, True),
        ),
        (
            "address stripped",
            lambda: (
                Address("  MG Road ", " Pune ", "411001").street,
                Address("  MG Road ", " Pune ", "411001").city,
            ),
            ("MG Road", "Pune"),
        ),
        (
            "address errors",
            address_errors,
            [
                "ValueError: street and city are required",
                "ValueError: invalid PIN code: '011001'",
                "ValueError: invalid PIN code: '41100'",
            ],
        ),
        (
            "address frozen",
            lambda: raised_error_name(
                lambda: set_attribute(Address("MG Road", "Pune", "411001"), "city", "Mumbai")
            ),
            "FrozenInstanceError",
        ),
        (
            "address hashable",
            lambda: len(
                {Address("MG Road", "Pune", "411001"), Address("MG Road ", "Pune", "411001")}
            ),
            1,
        ),
        (
            "replace validates",
            lambda: raised_message(
                lambda: replace(Address("MG Road", "Pune", "411001"), pin_code="0")
            ),
            "ValueError: invalid PIN code: '0'",
        ),
        (
            "email normalized",
            lambda: Customer(1, " Asha ", "  Asha@Example.COM ").email,
            "asha@example.com",
        ),
        ("name stripped", lambda: Customer(1, " Asha ", "asha@example.com").name, "Asha"),
        (
            "bad email",
            lambda: raised_message(lambda: Customer(1, "Asha", " asha@example ")),
            "ValueError: invalid email: ' asha@example '",
        ),
        (
            "empty name",
            lambda: raised_message(lambda: Customer(1, "  ", "a@b.in")),
            "ValueError: name is required",
        ),
        (
            "equal by id",
            lambda: (
                Customer(1, "Asha", "asha@example.com") == Customer(1, "A. Rao", "rao@example.com")
            ),
            True,
        ),
        (
            "different id",
            lambda: (
                Customer(1, "Asha", "asha@example.com") == Customer(2, "Asha", "asha@example.com")
            ),
            False,
        ),
        (
            "separate lists",
            lambda: (
                Customer(1, "A", "a@b.in").addresses is not Customer(2, "B", "b@b.in").addresses
            ),
            True,
        ),
        ("add_address", address_book, (True, False, 1)),
        (
            "asdict nests",
            lambda: asdict(
                Customer(1, "Asha", "asha@example.com", [Address("MG Road", "Pune", "411001")])
            )["addresses"],
            [{"street": "MG Road", "city": "Pune", "pin_code": "411001"}],
        ),
    ]
)

FAIL  is a dataclass: expected (True, True), got (False, False)
FAIL  address stripped: raised NotImplementedError: Lab 2.6: implement Address as a frozen dataclass
FAIL  address errors: expected ['ValueError: street and city are required', "ValueError: invalid PIN code: '011001'", "ValueError: invalid PIN code: '41100'"], got ['NotImplementedError: Lab 2.6: implement Address as a frozen dataclass', 'NotImplementedError: Lab 2.6: implement Address as a frozen dataclass', 'NotImplementedError: Lab 2.6: implement Address as a frozen dataclass']
FAIL  address frozen: expected 'FrozenInstanceError', got 'NotImplementedError'
FAIL  address hashable: raised NotImplementedError: Lab 2.6: implement Address as a frozen dataclass
FAIL  replace validates: expected "ValueError: invalid PIN code: '0'", got 'NotImplementedError: Lab 2.6: implement Address as a frozen dataclass'
FAIL  email normalized: raised NotImplementedError: Lab 2.6: implement Customer as a dataclass
FAIL  name stripped: raised 

#### Exercise 2 — `OrderStatus(StrEnum)` with allowed transitions

- Members `PENDING`, `PAID`, `SHIPPED`, `DELIVERED`, `CANCELLED` whose values are the
  lower-case names (use `auto()`).
- Allowed moves: pending → paid or cancelled; paid → shipped or cancelled; shipped
  → delivered. Delivered and cancelled are final.
- `status.can_transition_to(target)` → `bool`; `status.transition_to(target)`
  returns `target` or raises `ValueError(f"cannot go from {status} to {target}")`
  (e.g. `"cannot go from shipped to pending"`).
- `status.is_final` property.
- `_missing_` accepts any capitalization and surrounding spaces
  (`OrderStatus(" PAID ")`); anything else still raises `ValueError`.

In [359]:
class OrderStatus(StrEnum):
    """Add the members, the transition rules, is_final and _missing_."""

    def can_transition_to(self, target: OrderStatus) -> bool:
        raise NotImplementedError("Lab 2.6: implement OrderStatus.can_transition_to")

    def transition_to(self, target: OrderStatus) -> OrderStatus:
        raise NotImplementedError("Lab 2.6: implement OrderStatus.transition_to")

Run the checks.

In [360]:
def happy_path() -> list[str]:
    status = OrderStatus.PENDING
    visited = [str(status)]
    for target in (OrderStatus.PAID, OrderStatus.SHIPPED, OrderStatus.DELIVERED):
        status = status.transition_to(target)
        visited.append(str(status))
    return visited


run_checks(
    [
        (
            "values",
            lambda: [status.value for status in OrderStatus],
            ["pending", "paid", "shipped", "delivered", "cancelled"],
        ),
        ("is a str", lambda: issubclass(OrderStatus, str), True),
        ("equals its value", lambda: OrderStatus.PAID == "paid", True),
        ("json", lambda: json.dumps({"status": OrderStatus.SHIPPED}), '{"status": "shipped"}'),
        ("happy path", happy_path, ["pending", "paid", "shipped", "delivered"]),
        (
            "allowed",
            lambda: (
                OrderStatus.PENDING.can_transition_to(OrderStatus.CANCELLED),
                OrderStatus.PAID.can_transition_to(OrderStatus.CANCELLED),
            ),
            (True, True),
        ),
        (
            "not allowed",
            lambda: (
                OrderStatus.SHIPPED.can_transition_to(OrderStatus.CANCELLED),
                OrderStatus.PENDING.can_transition_to(OrderStatus.SHIPPED),
            ),
            (False, False),
        ),
        (
            "error",
            lambda: raised_message(lambda: OrderStatus.SHIPPED.transition_to(OrderStatus.PENDING)),
            "ValueError: cannot go from shipped to pending",
        ),
        (
            "final states",
            lambda: [status.value for status in OrderStatus if status.is_final],
            ["delivered", "cancelled"],
        ),
        ("_missing_", lambda: OrderStatus(" PAID ") is OrderStatus.PAID, True),
        ("unknown", lambda: raised_error_name(lambda: OrderStatus("lost")), "ValueError"),
    ]
)

FAIL  values: expected ['pending', 'paid', 'shipped', 'delivered', 'cancelled'], got []
PASS  is a str
FAIL  equals its value: raised AttributeError: type object 'OrderStatus' has no attribute 'PAID'
FAIL  json: raised AttributeError: type object 'OrderStatus' has no attribute 'SHIPPED'
FAIL  happy path: raised AttributeError: type object 'OrderStatus' has no attribute 'PENDING'
FAIL  allowed: raised AttributeError: type object 'OrderStatus' has no attribute 'PENDING'
FAIL  not allowed: raised AttributeError: type object 'OrderStatus' has no attribute 'SHIPPED'
FAIL  error: expected 'ValueError: cannot go from shipped to pending', got "AttributeError: type object 'OrderStatus' has no attribute 'SHIPPED'"
FAIL  final states: expected ['delivered', 'cancelled'], got []
FAIL  _missing_: raised TypeError: <enum 'OrderStatus'> has no members; specify `names=()` if you meant to create a new, empty, enum
FAIL  unknown: expected 'ValueError', got 'TypeError'
1/11 checks passed


#### Exercise 3 — `Permission(Flag)`

- Single-bit members `READ`, `WRITE`, `DELETE`, `SHARE` (`auto()`), plus combined
  aliases `VIEWER = READ`, `EDITOR = READ | WRITE`,
  `OWNER = READ | WRITE | DELETE | SHARE`.
- `has_permission(granted, required)` → `True` only if **every** bit of `required`
  is in `granted`.
- `parse_permissions(text)` turns `"read, Write"` into `Permission.READ |
  Permission.WRITE` (comma-separated, any case and spacing; `""` gives
  `Permission(0)`); an unknown name raises
  `ValueError(f"unknown permission: {name!r}")` with the cleaned name.
- `permission_names(permission)` → the single-bit member names in definition order,
  e.g. `["READ", "WRITE"]` for `EDITOR`.

In [361]:
class Permission(Flag):
    """Add the members and aliases."""


def has_permission(granted: Permission, required: Permission) -> bool:
    raise NotImplementedError("Lab 2.6: implement has_permission")


def parse_permissions(text: str) -> Permission:
    raise NotImplementedError("Lab 2.6: implement parse_permissions")


def permission_names(permission: Permission) -> list[str]:
    raise NotImplementedError("Lab 2.6: implement permission_names")

Run the checks.

In [362]:
run_checks(
    [
        ("single bits", lambda: [member.value for member in Permission], [1, 2, 4, 8]),
        (
            "aliases",
            lambda: (
                Permission.EDITOR == Permission.READ | Permission.WRITE,
                Permission.VIEWER is Permission.READ,
            ),
            (True, True),
        ),
        (
            "in",
            lambda: (Permission.WRITE in Permission.EDITOR, Permission.DELETE in Permission.EDITOR),
            (True, False),
        ),
        (
            "has_permission",
            lambda: (
                has_permission(Permission.OWNER, Permission.EDITOR),
                has_permission(Permission.EDITOR, Permission.READ | Permission.DELETE),
            ),
            (True, False),
        ),
        ("parse", lambda: parse_permissions("read, Write") == Permission.EDITOR, True),
        ("parse spacing", lambda: parse_permissions("  SHARE,delete ").value, 12),
        ("parse empty", lambda: parse_permissions("").value, 0),
        (
            "parse unknown",
            lambda: raised_message(lambda: parse_permissions("read, Fly")),
            "ValueError: unknown permission: 'fly'",
        ),
        ("names", lambda: permission_names(Permission.EDITOR), ["READ", "WRITE"]),
        (
            "names after removing",
            lambda: permission_names(Permission.OWNER & ~Permission.READ),
            ["WRITE", "DELETE", "SHARE"],
        ),
        ("names of nothing", lambda: permission_names(Permission(0)), []),
    ]
)

FAIL  single bits: expected [1, 2, 4, 8], got []
FAIL  aliases: raised AttributeError: type object 'Permission' has no attribute 'EDITOR'
FAIL  in: raised AttributeError: type object 'Permission' has no attribute 'WRITE'
FAIL  has_permission: raised AttributeError: type object 'Permission' has no attribute 'OWNER'
FAIL  parse: raised NotImplementedError: Lab 2.6: implement parse_permissions
FAIL  parse spacing: raised NotImplementedError: Lab 2.6: implement parse_permissions
FAIL  parse empty: raised NotImplementedError: Lab 2.6: implement parse_permissions
FAIL  parse unknown: expected "ValueError: unknown permission: 'fly'", got 'NotImplementedError: Lab 2.6: implement parse_permissions'
FAIL  names: raised AttributeError: type object 'Permission' has no attribute 'EDITOR'
FAIL  names after removing: raised AttributeError: type object 'Permission' has no attribute 'OWNER'
FAIL  names of nothing: raised TypeError: <flag 'Permission'> has no members; specify `names=()` if you meant to 

#### Exercise 4 — an `EventBus`

A publish/subscribe hub that dispatches on the **event class**:

- `subscribe(event_type, handler)` registers a handler and returns a function that
  unsubscribes it. `on(event_type)` is the decorator form (returns the handler).
- `publish(event)` calls the handlers for `type(event)` **and its base classes**
  (most specific class first, following the MRO; within a class, in subscription
  order) and returns how many handlers ran. A handler that raises must not stop the
  others: collect the exceptions and, after all handlers ran, raise
  `ExceptionGroup("event handlers failed", errors)` (Unit 1.14.6).
- `subscriber_count(event_type)` → number of handlers registered for exactly that
  class.

In [363]:
class EventBus:
    """Dispatch events to handlers subscribed to their class or a base class."""

    def __init__(self) -> None:
        raise NotImplementedError("Lab 2.6: implement EventBus")

Run the checks. The events are frozen dataclasses; `OrderPlaced` and `OrderShipped` extend `Event`.

In [364]:
@dataclass(frozen=True)
class Event:
    order_id: str


@dataclass(frozen=True)
class OrderPlaced(Event):
    amount_paise: int


@dataclass(frozen=True)
class OrderShipped(Event):
    courier: str


def dispatch_log() -> tuple[int, int, list[str]]:
    bus, log = EventBus(), []
    bus.subscribe(Event, lambda event: log.append(f"audit {event.order_id}"))
    bus.subscribe(OrderPlaced, lambda event: log.append(f"invoice {event.amount_paise}"))
    bus.subscribe(OrderPlaced, lambda event: log.append("thank-you mail"))
    placed_count = bus.publish(OrderPlaced("ORD-1", 49_900))
    shipped_count = bus.publish(OrderShipped("ORD-1", "BlueDart"))
    return placed_count, shipped_count, log


def unsubscribe_flow() -> tuple[int, int, int]:
    bus, seen = EventBus(), []
    stop = bus.subscribe(OrderShipped, seen.append)
    first = bus.publish(OrderShipped("ORD-2", "DTDC"))
    stop()
    return first, bus.publish(OrderShipped("ORD-3", "DTDC")), len(seen)


def decorator_form() -> tuple[int, str]:
    bus = EventBus()

    @bus.on(OrderPlaced)
    def record(event: Event) -> None:
        pass

    return bus.subscriber_count(OrderPlaced), record.__name__


def failing_handlers() -> tuple[str, int, list[str], list[str]]:
    bus, log = EventBus(), []
    bus.subscribe(Event, lambda event: 1 / 0)
    bus.subscribe(Event, lambda event: log.append("still runs"))
    bus.subscribe(Event, lambda event: {}["missing"])
    try:
        bus.publish(Event("ORD-4"))
    except ExceptionGroup as group:
        return (
            group.message,
            len(group.exceptions),
            sorted(type(error).__name__ for error in group.exceptions),
            log,
        )
    return "no group", 0, [], log


run_checks(
    [
        (
            "dispatch by MRO",
            dispatch_log,
            (3, 1, ["invoice 49900", "thank-you mail", "audit ORD-1", "audit ORD-1"]),
        ),
        ("unsubscribe", unsubscribe_flow, (1, 0, 1)),
        ("no subscribers", lambda: EventBus().publish(Event("ORD-5")), 0),
        ("decorator form", decorator_form, (1, "record")),
        ("subscriber_count is exact", lambda: EventBus().subscriber_count(Event), 0),
        (
            "errors grouped",
            failing_handlers,
            ("event handlers failed", 2, ["KeyError", "ZeroDivisionError"], ["still runs"]),
        ),
    ]
)

FAIL  dispatch by MRO: raised NotImplementedError: Lab 2.6: implement EventBus
FAIL  unsubscribe: raised NotImplementedError: Lab 2.6: implement EventBus
FAIL  no subscribers: raised NotImplementedError: Lab 2.6: implement EventBus
FAIL  decorator form: raised NotImplementedError: Lab 2.6: implement EventBus
FAIL  subscriber_count is exact: raised NotImplementedError: Lab 2.6: implement EventBus
FAIL  errors grouped: raised NotImplementedError: Lab 2.6: implement EventBus
0/6 checks passed


### Recap — Unit 2.6

| Concept | One-line takeaway | JS / TS equivalent |
| --- | --- | --- |
| `@dataclass` | Declare typed fields; get `__init__`, `__repr__`, `__eq__`, `__match_args__` | class fields + constructor |
| `field()` | `default_factory` for mutable defaults; `repr`, `compare`, `init`, `kw_only`, `metadata`, `doc` | — |
| `frozen` / `order` / `slots` / `kw_only` | Immutable + hashable / tuple ordering / no `__dict__` / keyword-only | `Object.freeze` |
| `__post_init__` / `InitVar` | Validate and normalize; init-only inputs | constructor body |
| `asdict` / `replace` / `fields` | Recursive dict copy / copy with changes (re-validated) / introspection | `{...obj, x: 1}` |
| Record types | dataclass inside, NamedTuple for tuples, TypedDict for JSON, Pydantic at the edges | TS types vs classes |
| `TypedDict` | Type-checker-only description of a dict; `Required`, `NotRequired`, `ReadOnly` | TS object types |
| `Enum` | Fixed set of singletons; lookup by value `E(v)` and name `E[n]` | TS `enum` |
| `IntEnum` / `StrEnum` | Members are ints / strs (JSON-friendly) | string enums |
| `Flag` / `IntFlag` | Combinable bits with names, `in`, iteration | bit masks |
| `@unique`, `_missing_` | Forbid aliases; customize failed lookups | — |
| Patterns | Functions, dicts, modules, decorators and Protocols replace most hierarchies | same |
| Value object vs entity | Equal by value and immutable vs equal by id with a lifecycle | — |

**Further reading**

- [`dataclasses`](https://docs.python.org/3/library/dataclasses.html),
  [`copy.replace`](https://docs.python.org/3/library/copy.html#copy.replace)
- [`enum`](https://docs.python.org/3/library/enum.html) and the
  [Enum HOWTO](https://docs.python.org/3/howto/enum.html)
- [`typing.TypedDict`](https://docs.python.org/3/library/typing.html#typing.TypedDict),
  [`typing.NamedTuple`](https://docs.python.org/3/library/typing.html#typing.NamedTuple)
- [PEP 557 — Data Classes](https://peps.python.org/pep-0557/),
  [PEP 435 — Enums](https://peps.python.org/pep-0435/),
  [PEP 589 — TypedDict](https://peps.python.org/pep-0589/),
  [PEP 655 — Required / NotRequired](https://peps.python.org/pep-0655/),
  [PEP 705 — ReadOnly](https://peps.python.org/pep-0705/)

## Phase 2 Project — Inventory Domain Model

Model the core of an inventory system for a small chain of stores: products,
warehouses, stock movements and the rules between them. No database and no web
framework yet; the goal is a domain model that is correct, readable and easy to test.
Phase 6 puts a FastAPI app and a real database in front of the same design.

**Requirements and where each is met**

| Requirement | Step |
| --- | --- |
| `Money` from Unit 2.4 | Step 1 |
| A frozen, slotted `Product` dataclass | Step 2 |
| `StockMovement` with a `MovementType` enum | Step 3 |
| Custom exceptions | Step 4 |
| A `Warehouse` container class with dunders | Step 5 |
| An `InventoryRepository` Protocol with an in-memory implementation | Step 6 |
| An event bus emitting low-stock alerts | Step 7 |
| An `InventoryService` receiving the repository by injection | Step 8 |
| A full demo script | Step 9 |
| Self-check | Step 10 |

**Design**

```mermaid
flowchart LR
    Demo["demo script"] --> Service["InventoryService"]
    Service -->|"depends on (Protocol)"| Repo["InventoryRepository"]
    Repo -.implemented by.-> Memory["InMemoryInventoryRepository"]
    Service --> Bus["EventBus"]
    Bus --> Alerts["low-stock handlers"]
    Memory --> Warehouse["Warehouse (entity)"]
    Warehouse --> Product["Product (value object)"]
    Product --> Money["Money (value object)"]
    Memory --> Movement["StockMovement + MovementType"]
```

<details><summary>ASCII fallback</summary>

```text
demo script
    |
    v
InventoryService ---(Protocol)---> InventoryRepository
    |                                    ^ implemented by
    |                              InMemoryInventoryRepository
    v                                    |
EventBus --> low-stock handlers          +--> Warehouse (entity) --> Product --> Money
                                         +--> StockMovement + MovementType
```

</details>

Value objects (`Money`, `Product`, `StockMovement`) are frozen; the `Warehouse`
entity changes over time and is equal by its code (Unit 2.6.6). The service is the
only place that coordinates the others.

> **Note:** every class this project uses is defined again here, so the project runs
> on its own after the shared setup cell.

In [365]:
import functools
import re
import sys
from collections import defaultdict
from collections.abc import Callable, Iterator
from dataclasses import dataclass, replace
from datetime import datetime, timedelta
from decimal import ROUND_HALF_UP, Decimal
from enum import StrEnum, auto
from typing import Any, Protocol, runtime_checkable

PAISE = Decimal("0.01")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def set_attribute(target: object, attribute_name: str, value: object) -> None:
    """`target.attribute_name = value` as a call, for use inside raised_error_name(lambda: ...)."""
    setattr(target, attribute_name, value)


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### Step 1 — `Money`

The immutable `Money` from Lab 2.4 (exercise 1), copied unchanged: amounts rounded
to the paisa, read-only fields and `__slots__`, value equality and hashing,
ordering within one currency, `+`, `-`, `*`, `sum()` support and custom format
specs. If you have not solved the lab yet, this cell is a spoiler for it.

In [366]:
class CurrencyMismatchError(ValueError):
    """Two Money values in different currencies were combined or compared."""


CURRENCY_SYMBOLS = {"INR": "₹", "USD": "$", "EUR": "€"}


@functools.total_ordering
class Money:
    """An immutable amount of money in one currency (Lab 2.4, exercise 1)."""

    __slots__ = ("_amount", "_currency")

    def __init__(self, amount: Decimal | int | str, currency: str = "INR") -> None:
        if isinstance(amount, float):
            raise TypeError("amount must be Decimal, int or str, not float")
        if not (len(currency) == 3 and currency.isalpha() and currency.isupper()):
            raise ValueError(f"invalid currency code: {currency!r}")
        self._amount = Decimal(amount).quantize(PAISE, rounding=ROUND_HALF_UP)
        self._currency = currency

    @property
    def amount(self) -> Decimal:
        return self._amount

    @property
    def currency(self) -> str:
        return self._currency

    def __repr__(self) -> str:
        return f"Money({str(self._amount)!r}, {self._currency!r})"

    def __str__(self) -> str:
        return f"{self._currency} {self._amount:,.2f}"

    def __format__(self, spec: str) -> str:
        if spec == "":
            return str(self)
        if spec == "symbol":
            symbol = CURRENCY_SYMBOLS.get(self._currency, self._currency + " ")
            return f"{symbol}{self._amount:,.2f}"
        return f"{self._currency} {format(self._amount, spec)}"

    def _same_currency(self, other: Money) -> None:
        if other._currency != self._currency:
            raise CurrencyMismatchError(f"currency mismatch: {self._currency} vs {other._currency}")

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Money):
            return NotImplemented
        return (self._amount, self._currency) == (other._amount, other._currency)

    def __hash__(self) -> int:
        return hash((self._amount, self._currency))

    def __lt__(self, other: object) -> bool:
        if not isinstance(other, Money):
            return NotImplemented
        self._same_currency(other)
        return self._amount < other._amount

    def __add__(self, other: object) -> Money:
        if not isinstance(other, Money):
            return NotImplemented
        self._same_currency(other)
        return Money(self._amount + other._amount, self._currency)

    def __radd__(self, other: object) -> Money:
        if isinstance(other, int) and not isinstance(other, bool) and other == 0:
            return self  # sum() starts from 0
        return NotImplemented

    def __sub__(self, other: object) -> Money:
        if not isinstance(other, Money):
            return NotImplemented
        self._same_currency(other)
        return Money(self._amount - other._amount, self._currency)

    def __mul__(self, factor: object) -> Money:
        if isinstance(factor, bool) or not isinstance(factor, (int, Decimal)):
            return NotImplemented
        return Money(self._amount * factor, self._currency)

    __rmul__ = __mul__

    def __neg__(self) -> Money:
        return Money(-self._amount, self._currency)

    def __abs__(self) -> Money:
        return Money(abs(self._amount), self._currency)

    def __bool__(self) -> bool:
        return bool(self._amount)


print(f"{sum([Money('49.99'), Money(100)])=}, {Money('19.99') * 3:symbol}")

sum([Money('49.99'), Money(100)])=Money('149.99', 'INR'), ₹59.97


### Step 2 — a frozen, slotted `Product`

A product is a **value object**: SKU, name, unit price and the reorder level below
which stock counts as low. `frozen=True` makes it immutable and hashable,
`slots=True` drops the per-instance `__dict__` (catalogues can be large), and
`__post_init__` rejects bad data at the boundary. A price change produces a new
`Product` with `replace()`.

In [367]:
SKU_PATTERN = re.compile(r"[A-Z]{2,5}-\d{2,4}")


@dataclass(frozen=True, slots=True)
class Product:
    """A catalogue item. Immutable: change it with dataclasses.replace()."""

    sku: str
    name: str
    unit_price: Money
    reorder_level: int = 5

    def __post_init__(self) -> None:
        if not SKU_PATTERN.fullmatch(self.sku):
            raise ValueError(f"invalid SKU {self.sku!r}: expected e.g. 'TEA-500'")
        if not self.name.strip():
            raise ValueError("product name is required")
        if not isinstance(self.unit_price, Money) or self.unit_price.amount <= 0:
            raise ValueError("unit_price must be a positive Money")
        if self.reorder_level < 0:
            raise ValueError("reorder_level must be >= 0")

What the options buy: no stray attributes, no mutation, hashing (a `set` of
products), and validation that also runs on `replace()`.

In [368]:
masala_tea = Product("TEA-500", "Masala tea 500 g", Money("240"), reorder_level=10)
print(masala_tea)
print(f"{hasattr(masala_tea, '__dict__')=}, {Product.__slots__=}")
print(raised_message(lambda: set_attribute(masala_tea, "unit_price", Money(1))))
print(f"{len({masala_tea, Product('TEA-500', 'Masala tea 500 g', Money('240.00'), 10)})=}")
print(f"{replace(masala_tea, unit_price=Money('255')).unit_price=}")
print(raised_message(lambda: Product("tea", "Tea", Money(1))))
print(raised_message(lambda: replace(masala_tea, unit_price=Money(0))))

Product(sku='TEA-500', name='Masala tea 500 g', unit_price=Money('240.00', 'INR'), reorder_level=10)
hasattr(masala_tea, '__dict__')=False, Product.__slots__=('sku', 'name', 'unit_price', 'reorder_level')
FrozenInstanceError: cannot assign to field 'unit_price'
len({masala_tea, Product('TEA-500', 'Masala tea 500 g', Money('240.00'), 10)})=1
replace(masala_tea, unit_price=Money('255')).unit_price=Money('255.00', 'INR')
ValueError: invalid SKU 'tea': expected e.g. 'TEA-500'
ValueError: unit_price must be a positive Money


### Step 3 — `StockMovement` with a `MovementType` enum

Every change to stock is recorded as a movement, never as a bare number, so the
history explains the current quantity (an audit trail). `MovementType` is a
`StrEnum`, so it prints and serializes as plain text; each member knows its
direction. A movement is a frozen, slotted dataclass whose `delta` property gives the
signed change. Adjustments (stock counts, damage) carry their own sign.

In [369]:
class MovementType(StrEnum):
    RECEIVED = auto()
    SHIPPED = auto()
    RETURNED = auto()
    ADJUSTED = auto()

    @property
    def direction(self) -> int:
        """+1 adds stock, -1 removes it, 0 means the quantity carries its own sign."""
        match self:
            case MovementType.RECEIVED | MovementType.RETURNED:
                return 1
            case MovementType.SHIPPED:
                return -1
            case _:
                return 0


@dataclass(frozen=True, slots=True)
class StockMovement:
    warehouse_code: str
    sku: str
    movement_type: MovementType
    quantity: int
    occurred_at: datetime
    note: str = ""

    def __post_init__(self) -> None:
        if self.movement_type is MovementType.ADJUSTED:
            if self.quantity == 0:
                raise ValueError("an adjustment must change the quantity")
        elif self.quantity <= 0:
            raise ValueError(f"{self.movement_type} quantity must be positive")

    @property
    def delta(self) -> int:
        """Signed change in stock."""
        if self.movement_type is MovementType.ADJUSTED:
            return self.quantity
        return self.quantity * self.movement_type.direction

Movements from each type, and the validation. `occurred_at` is passed explicitly;
Step 8 injects a clock so the whole project is deterministic.

In [370]:
opening_time = datetime(2026, 11, 7, 9, 0)
sample_movements = [
    StockMovement("PNQ1", "TEA-500", MovementType.RECEIVED, 40, opening_time),
    StockMovement("PNQ1", "TEA-500", MovementType.SHIPPED, 3, opening_time),
    StockMovement("PNQ1", "TEA-500", MovementType.ADJUSTED, -2, opening_time, "damaged"),
]
print([(str(movement.movement_type), movement.delta) for movement in sample_movements])
print(f"{sum(movement.delta for movement in sample_movements)=}")
print(
    raised_message(lambda: StockMovement("PNQ1", "TEA-500", MovementType.SHIPPED, 0, opening_time))
)
print(f"{MovementType('shipped') is MovementType.SHIPPED=}, {list(MovementType)=}")

[('received', 40), ('shipped', -3), ('adjusted', -2)]
sum(movement.delta for movement in sample_movements)=35
ValueError: shipped quantity must be positive
MovementType('shipped') is MovementType.SHIPPED=True, list(MovementType)=[<MovementType.RECEIVED: 'received'>, <MovementType.SHIPPED: 'shipped'>, <MovementType.RETURNED: 'returned'>, <MovementType.ADJUSTED: 'adjusted'>]


### Step 4 — custom exceptions

One base class, `InventoryError`, lets callers catch every domain failure at once;
each subclass also derives from the matching built-in (`LookupError`, `ValueError`)
so generic handlers still work (Unit 1.14.5). Each carries an `error_code` and a
`to_dict()` for the API layer of Phase 6, plus structured attributes instead of
information buried in the message.

In [371]:
class InventoryError(Exception):
    """Base class for every inventory rule violation."""

    error_code = "inventory_error"

    def to_dict(self) -> dict[str, str]:
        return {"error": self.error_code, "message": str(self)}


class UnknownWarehouseError(InventoryError, LookupError):
    error_code = "unknown_warehouse"

    def __init__(self, warehouse_code: str) -> None:
        super().__init__(f"no warehouse with code {warehouse_code!r}")
        self.warehouse_code = warehouse_code


class UnknownProductError(InventoryError, LookupError):
    error_code = "unknown_product"

    def __init__(self, sku: str, warehouse_code: str) -> None:
        super().__init__(f"{sku} is not stocked in {warehouse_code}")
        self.sku, self.warehouse_code = sku, warehouse_code


class DuplicateProductError(InventoryError, ValueError):
    error_code = "duplicate_product"

    def __init__(self, sku: str, warehouse_code: str) -> None:
        super().__init__(f"{sku} is already stocked in {warehouse_code}")
        self.sku, self.warehouse_code = sku, warehouse_code


class InsufficientStockError(InventoryError):
    error_code = "insufficient_stock"

    def __init__(self, sku: str, available: int, requested: int) -> None:
        super().__init__(f"not enough {sku}: {available} available, {requested} requested")
        self.sku, self.available, self.requested = sku, available, requested

The hierarchy and one error as the API would report it:

In [372]:
for error_class in (
    UnknownWarehouseError,
    UnknownProductError,
    DuplicateProductError,
    InsufficientStockError,
):
    print(f"{error_class.__name__:<24} {[klass.__name__ for klass in error_class.__mro__[1:-2]]}")
shortage = InsufficientStockError("TEA-500", 2, 5)
print(f"{shortage.to_dict()=}, {shortage.available=}")

UnknownWarehouseError    ['InventoryError', 'LookupError', 'Exception']
UnknownProductError      ['InventoryError', 'LookupError', 'Exception']
DuplicateProductError    ['InventoryError', 'ValueError', 'Exception']
InsufficientStockError   ['InventoryError', 'Exception']
shortage.to_dict()={'error': 'insufficient_stock', 'message': 'not enough TEA-500: 2 available, 5 requested'}, shortage.available=2


### Step 5 — a `Warehouse` container with dunders

A warehouse is an **entity**: it is identified by its code while its stock changes,
so `__eq__` and `__hash__` use the code only. It behaves like a container of
products (Unit 2.4):

| Syntax | Method | Meaning |
| --- | --- | --- |
| `len(warehouse)` | `__len__` | number of products carried |
| `sku in warehouse` / `product in warehouse` | `__contains__` | is this product carried? |
| `warehouse["TEA-500"]` | `__getitem__` | quantity on hand (`UnknownProductError` if not carried) |
| `for product in warehouse` | `__iter__` | products in SKU order |
| `bool(warehouse)` | `__bool__` | is **anything** in stock? (not the same as `len > 0`) |
| `repr(warehouse)` | `__repr__` | code, city, product count |
| `==`, `hash()` | `__eq__`, `__hash__` | identity by code |

Stock changes only through `apply(movement)`, which enforces the rules: the product
must be carried and stock may never go negative.

In [373]:
class Warehouse:
    """A stock location: the products it carries and their quantities."""

    def __init__(self, code: str, city: str) -> None:
        self.code, self.city = code, city
        self._products: dict[str, Product] = {}
        self._quantities: dict[str, int] = {}

    def __repr__(self) -> str:
        return f"Warehouse({self.code!r}, {self.city!r}, products={len(self)})"

    def __eq__(self, other: object) -> bool:
        return self.code == other.code if isinstance(other, Warehouse) else NotImplemented

    def __hash__(self) -> int:
        return hash(self.code)

    def __len__(self) -> int:
        return len(self._products)

    def __contains__(self, item: object) -> bool:
        sku = item.sku if isinstance(item, Product) else item
        return sku in self._products

    def __getitem__(self, sku: str) -> int:
        if sku not in self._products:
            raise UnknownProductError(sku, self.code)
        return self._quantities[sku]

    def __iter__(self) -> Iterator[Product]:
        return iter(sorted(self._products.values(), key=lambda product: product.sku))

    def __bool__(self) -> bool:
        return any(self._quantities.values())

    def product(self, sku: str) -> Product:
        """The catalogue entry for a carried SKU."""
        if sku not in self._products:
            raise UnknownProductError(sku, self.code)
        return self._products[sku]

    def add_product(self, product: Product) -> None:
        """Start carrying a product, with zero stock."""
        if product.sku in self._products:
            raise DuplicateProductError(product.sku, self.code)
        self._products[product.sku] = product
        self._quantities[product.sku] = 0

    def apply(self, movement: StockMovement) -> int:
        """Apply a movement and return the new quantity."""
        available = self[movement.sku]
        if available + movement.delta < 0:
            raise InsufficientStockError(movement.sku, available, -movement.delta)
        self._quantities[movement.sku] = available + movement.delta
        return self._quantities[movement.sku]

    def stock_value(self) -> Money:
        """Value of everything on hand; start=Money(0) covers an empty warehouse."""
        return sum((product.unit_price * self[product.sku] for product in self), Money(0))

    def low_stock(self) -> list[Product]:
        """Products at or below their reorder level."""
        return [product for product in self if self[product.sku] <= product.reorder_level]

Exercising every dunder on a small warehouse. Note that a warehouse that carries
products but has none in stock is falsy, which reads well in `if not warehouse:`.

In [374]:
demo_warehouse = Warehouse("TEST", "Pune")
demo_warehouse.add_product(masala_tea)
print(f"{demo_warehouse!r}, {len(demo_warehouse)=}, {bool(demo_warehouse)=}")
demo_warehouse.apply(StockMovement("TEST", "TEA-500", MovementType.RECEIVED, 12, opening_time))
print(f"{demo_warehouse['TEA-500']=}, {bool(demo_warehouse)=}")
print(f"{'TEA-500' in demo_warehouse=}, {masala_tea in demo_warehouse=}")
print(f"{[product.sku for product in demo_warehouse]=}, {demo_warehouse.stock_value()=}")
print(f"{demo_warehouse == Warehouse('TEST', 'Mumbai')=}  (entity: same code)")
print(raised_message(lambda: demo_warehouse["MUG-02"]))
print(
    raised_message(
        lambda: demo_warehouse.apply(
            StockMovement("TEST", "TEA-500", MovementType.SHIPPED, 20, opening_time)
        )
    )
)
print(raised_message(lambda: demo_warehouse.add_product(masala_tea)))

Warehouse('TEST', 'Pune', products=1), len(demo_warehouse)=1, bool(demo_warehouse)=False
demo_warehouse['TEA-500']=12, bool(demo_warehouse)=True
'TEA-500' in demo_warehouse=True, masala_tea in demo_warehouse=True
[product.sku for product in demo_warehouse]=['TEA-500'], demo_warehouse.stock_value()=Money('2880.00', 'INR')
demo_warehouse == Warehouse('TEST', 'Mumbai')=True  (entity: same code)
UnknownProductError: MUG-02 is not stocked in TEST
InsufficientStockError: not enough TEA-500: 12 available, 20 requested
DuplicateProductError: TEA-500 is already stocked in TEST


### Step 6 — an `InventoryRepository` Protocol and an in-memory implementation

The service needs somewhere to load and save warehouses and to record movements.
It depends on a **Protocol** (Unit 2.3.3), not on a concrete class, so the storage
can change (SQLite in tests, PostgreSQL in Phase 6) without touching the business
rules (the repository pattern, Unit 2.6.5). `@runtime_checkable` lets the demo
assert that the in-memory class satisfies it, although it never inherits from it.

In [375]:
@runtime_checkable
class InventoryRepository(Protocol):
    def get_warehouse(self, code: str) -> Warehouse: ...
    def save_warehouse(self, warehouse: Warehouse) -> None: ...
    def list_warehouses(self) -> list[Warehouse]: ...
    def record_movement(self, movement: StockMovement) -> None: ...
    def movements(self, *, sku: str | None = None) -> list[StockMovement]: ...


class InMemoryInventoryRepository:
    """Dict-backed storage for tests and demos."""

    def __init__(self) -> None:
        self._warehouses: dict[str, Warehouse] = {}
        self._movements: list[StockMovement] = []

    def get_warehouse(self, code: str) -> Warehouse:
        try:
            return self._warehouses[code]
        except KeyError:
            raise UnknownWarehouseError(code) from None

    def save_warehouse(self, warehouse: Warehouse) -> None:
        self._warehouses[warehouse.code] = warehouse

    def list_warehouses(self) -> list[Warehouse]:
        return sorted(self._warehouses.values(), key=lambda warehouse: warehouse.code)

    def record_movement(self, movement: StockMovement) -> None:
        self._movements.append(movement)

    def movements(self, *, sku: str | None = None) -> list[StockMovement]:
        return [movement for movement in self._movements if sku is None or movement.sku == sku]


print(f"{isinstance(InMemoryInventoryRepository(), InventoryRepository)=}")
print(f"{InventoryRepository in InMemoryInventoryRepository.__mro__=}  (structural, not inherited)")
print(raised_message(lambda: InMemoryInventoryRepository().get_warehouse("NOPE")))

isinstance(InMemoryInventoryRepository(), InventoryRepository)=True
InventoryRepository in InMemoryInventoryRepository.__mro__=False  (structural, not inherited)
UnknownWarehouseError: no warehouse with code 'NOPE'


### Step 7 — an event bus for low-stock alerts

The service should not know who cares about stock changes (a purchasing e-mail, a
dashboard, a log). It **publishes events**; handlers subscribe by event class (the
observer pattern, Unit 2.6.5). Events are frozen dataclasses. This bus is kept
deliberately small: exact-class dispatch, handlers called in order. Lab 2.6's
`EventBus` (base-class dispatch, error groups) could replace it unchanged.

In [376]:
@dataclass(frozen=True)
class StockChanged:
    warehouse_code: str
    sku: str
    old_quantity: int
    new_quantity: int
    movement_type: MovementType


@dataclass(frozen=True)
class LowStockAlert:
    warehouse_code: str
    sku: str
    quantity: int
    reorder_level: int


class EventBus:
    """Calls the handlers subscribed to the exact class of each published event."""

    def __init__(self) -> None:
        self._handlers: defaultdict[type, list[Callable[[Any], None]]] = defaultdict(list)

    def subscribe(self, event_type: type, handler: Callable[[Any], None]) -> None:
        self._handlers[event_type].append(handler)

    def publish(self, event: object) -> int:
        handlers = self._handlers.get(type(event), [])
        for handler in handlers:
            handler(event)
        return len(handlers)


trial_bus = EventBus()
trial_bus.subscribe(
    LowStockAlert, lambda alert: print(f"  ALERT {alert.sku}: {alert.quantity} left")
)
print(f"{trial_bus.publish(LowStockAlert('PNQ1', 'TEA-500', 3, 10))=}")
unwatched_event = StockChanged("PNQ1", "TEA-500", 5, 3, MovementType.SHIPPED)
print(f"{trial_bus.publish(unwatched_event)=}  (no subscribers)")

  ALERT TEA-500: 3 left
trial_bus.publish(LowStockAlert('PNQ1', 'TEA-500', 3, 10))=1
trial_bus.publish(unwatched_event)=0  (no subscribers)


### Step 8 — an `InventoryService` with injected dependencies

The service holds the use cases: open a warehouse, register a product, receive,
ship, accept returns, adjust after a stock count, and transfer between warehouses.
Its collaborators arrive through the constructor (**dependency injection**, Unit
2.6.5): the repository (any object matching the Protocol), the event bus, and a
`clock`, so tests control time. `_record` is the single place where stock changes:
it builds the movement, applies it, saves, and publishes `StockChanged`, plus a
`LowStockAlert` when the quantity **crosses** the reorder level (not on every later
sale, which would flood the purchasing team).

In [377]:
class InventoryService:
    def __init__(
        self,
        repository: InventoryRepository,
        events: EventBus,
        clock: Callable[[], datetime] = datetime.now,
    ) -> None:
        self.repository, self.events, self.clock = repository, events, clock

    def open_warehouse(self, code: str, city: str) -> Warehouse:
        warehouse = Warehouse(code, city)
        self.repository.save_warehouse(warehouse)
        return warehouse

    def register_product(self, warehouse_code: str, product: Product) -> None:
        warehouse = self.repository.get_warehouse(warehouse_code)
        warehouse.add_product(product)
        self.repository.save_warehouse(warehouse)

    def receive(self, warehouse_code: str, sku: str, quantity: int, note: str = "") -> int:
        return self._record(warehouse_code, sku, MovementType.RECEIVED, quantity, note)

    def ship(self, warehouse_code: str, sku: str, quantity: int, note: str = "") -> int:
        return self._record(warehouse_code, sku, MovementType.SHIPPED, quantity, note)

    def accept_return(self, warehouse_code: str, sku: str, quantity: int, note: str = "") -> int:
        return self._record(warehouse_code, sku, MovementType.RETURNED, quantity, note)

    def adjust(self, warehouse_code: str, sku: str, change: int, note: str) -> int:
        return self._record(warehouse_code, sku, MovementType.ADJUSTED, change, note)

The rest of the class: transfers, reporting, and the shared `_record` step. A
transfer checks **both** sides before moving anything, so a failure leaves stock
untouched; the destination starts carrying the product if it did not already.

In [378]:
class InventoryService(InventoryService):  # the second half of the class
    def transfer(self, from_code: str, to_code: str, sku: str, quantity: int) -> None:
        source = self.repository.get_warehouse(from_code)
        destination = self.repository.get_warehouse(to_code)
        if source[sku] < quantity:
            raise InsufficientStockError(sku, source[sku], quantity)
        if sku not in destination:
            self.register_product(to_code, source.product(sku))
        self.ship(from_code, sku, quantity, note=f"transfer to {to_code}")
        self.receive(to_code, sku, quantity, note=f"transfer from {from_code}")

    def total_value(self) -> Money:
        return sum(
            (warehouse.stock_value() for warehouse in self.repository.list_warehouses()), Money(0)
        )

    def stock_report(self, warehouse_code: str) -> list[str]:
        warehouse = self.repository.get_warehouse(warehouse_code)
        lines = [
            f"{warehouse.code} ({warehouse.city})",
            f"{'SKU':<9}{'Product':<20}{'Qty':>5}{'Value (INR)':>14}",
        ]
        for product in warehouse:
            flag = "  LOW" if product in warehouse.low_stock() else ""
            value = product.unit_price * warehouse[product.sku]
            quantity = warehouse[product.sku]
            lines.append(
                f"{product.sku:<9}{product.name[:19]:<20}{quantity:>5}{value.amount:>14,.2f}{flag}"
            )
        lines.append(f"{'Total':<34}{warehouse.stock_value().amount:>14,.2f}")
        return lines

    def _record(
        self, warehouse_code: str, sku: str, movement_type: MovementType, quantity: int, note: str
    ) -> int:
        warehouse = self.repository.get_warehouse(warehouse_code)
        old_quantity = warehouse[sku]
        movement = StockMovement(warehouse_code, sku, movement_type, quantity, self.clock(), note)
        new_quantity = warehouse.apply(movement)
        self.repository.record_movement(movement)
        self.repository.save_warehouse(warehouse)
        self.events.publish(
            StockChanged(warehouse_code, sku, old_quantity, new_quantity, movement_type)
        )
        reorder_level = warehouse.product(sku).reorder_level
        if old_quantity > reorder_level >= new_quantity:
            self.events.publish(LowStockAlert(warehouse_code, sku, new_quantity, reorder_level))
        return new_quantity

### Step 9 — the full demo script

Wire everything together: a fake clock that ticks one minute per movement, the
in-memory repository, the bus with two subscribers (an alert collector standing in
for "e-mail purchasing", and a counter of stock changes), and the service.

In [379]:
def make_clock(start: datetime, step: timedelta = timedelta(minutes=1)) -> Callable[[], datetime]:
    """A deterministic clock: start, start + step, start + 2 * step, ..."""
    readings = iter(start + step * tick for tick in range(1_000_000))
    return lambda: next(readings)


repository = InMemoryInventoryRepository()
event_bus = EventBus()
alerts: list[LowStockAlert] = []
change_counts: defaultdict[MovementType, int] = defaultdict(int)
event_bus.subscribe(LowStockAlert, alerts.append)
event_bus.subscribe(
    StockChanged,
    lambda event: change_counts.__setitem__(
        event.movement_type, change_counts[event.movement_type] + 1
    ),
)
service = InventoryService(repository, event_bus, clock=make_clock(datetime(2026, 11, 7, 9, 0)))

catalogue = [
    Product("TEA-500", "Masala tea 500 g", Money("240"), reorder_level=10),
    Product("MUG-02", "Steel mug", Money("199"), reorder_level=4),
    Product("JAR-10", "Glass jar 1 L", Money("149.50"), reorder_level=6),
]
for warehouse_code, city in [("PNQ1", "Pune"), ("BLR2", "Bengaluru")]:
    service.open_warehouse(warehouse_code, city)
for product in catalogue:
    service.register_product("PNQ1", product)
print(repository.list_warehouses())

[Warehouse('BLR2', 'Bengaluru', products=0), Warehouse('PNQ1', 'Pune', products=3)]


A day of business at the Pune warehouse: a delivery arrives, orders ship, one
customer returns a mug, a stock count finds two broken jars. Shipping the mugs below
their reorder level of 4 raises exactly one alert.

In [380]:
service.receive("PNQ1", "TEA-500", 40, note="PO-1001")
service.receive("PNQ1", "MUG-02", 12, note="PO-1001")
service.receive("PNQ1", "JAR-10", 20, note="PO-1002")
for sku, quantity in [("TEA-500", 5), ("MUG-02", 6), ("MUG-02", 3), ("TEA-500", 8), ("JAR-10", 2)]:
    service.ship("PNQ1", sku, quantity, note="customer order")
service.accept_return("PNQ1", "MUG-02", 1, note="wrong colour")
service.adjust("PNQ1", "JAR-10", -2, note="stock count: 2 broken")
print(f"{alerts=}")
print(f"{dict(change_counts)=}")

alerts=[LowStockAlert(warehouse_code='PNQ1', sku='MUG-02', quantity=3, reorder_level=4)]
dict(change_counts)={<MovementType.RECEIVED: 'received'>: 3, <MovementType.SHIPPED: 'shipped'>: 5, <MovementType.RETURNED: 'returned'>: 1, <MovementType.ADJUSTED: 'adjusted'>: 1}


Failures are domain exceptions with structured data. One `except InventoryError`
catches them all, and `to_dict()` is what an API would return. None of the failed
calls changed any stock.

In [381]:
failing_operations = {
    "ship too many mugs": lambda: service.ship("PNQ1", "MUG-02", 50),
    "unknown product": lambda: service.receive("PNQ1", "PEN-01", 5),
    "unknown warehouse": lambda: service.receive("DEL9", "TEA-500", 5),
    "register twice": lambda: service.register_product("PNQ1", catalogue[0]),
}
for description, operation in failing_operations.items():
    try:
        operation()
    except InventoryError as error:
        print(f"{description:<18} -> {error.to_dict()}")
print(f"{repository.get_warehouse('PNQ1')['MUG-02']=}  (unchanged by the failures)")

ship too many mugs -> {'error': 'insufficient_stock', 'message': 'not enough MUG-02: 4 available, 50 requested'}
unknown product    -> {'error': 'unknown_product', 'message': 'PEN-01 is not stocked in PNQ1'}
unknown warehouse  -> {'error': 'unknown_warehouse', 'message': "no warehouse with code 'DEL9'"}
register twice     -> {'error': 'duplicate_product', 'message': 'TEA-500 is already stocked in PNQ1'}
repository.get_warehouse('PNQ1')['MUG-02']=4  (unchanged by the failures)


A transfer to Bengaluru: the destination starts carrying the product, and the
Pune tea stock crosses its reorder level of 10, so a second alert fires. A
transfer that is too large fails before anything moves.

In [382]:
service.transfer("PNQ1", "BLR2", "TEA-500", 20)
pune, bengaluru = repository.get_warehouse("PNQ1"), repository.get_warehouse("BLR2")
print(f"{pune['TEA-500']=}, {bengaluru['TEA-500']=}")
print(f"{alerts[-1]=}")
print(raised_message(lambda: service.transfer("PNQ1", "BLR2", "JAR-10", 100)))
print(f"{'JAR-10' in repository.get_warehouse('BLR2')=}  (nothing registered or moved)")

pune['TEA-500']=7, bengaluru['TEA-500']=20
alerts[-1]=LowStockAlert(warehouse_code='PNQ1', sku='TEA-500', quantity=7, reorder_level=10)
InsufficientStockError: not enough JAR-10: 16 available, 100 requested
'JAR-10' in repository.get_warehouse('BLR2')=False  (nothing registered or moved)


The reports and the audit trail. Every line of the report comes from the dunders
(`for product in warehouse`, `warehouse[sku]`), `Money` arithmetic and
`Money.__format__`; the history replays every movement of one SKU in order with
its timestamp.

In [383]:
for warehouse in repository.list_warehouses():
    print("\n".join(service.stock_report(warehouse.code)))
    print()
print(f"Total stock value: {service.total_value():symbol}")
low_stock_skus = [product.sku for product in repository.get_warehouse("PNQ1").low_stock()]
print(f"Low stock in PNQ1: {low_stock_skus}")

BLR2 (Bengaluru)
SKU      Product               Qty   Value (INR)
TEA-500  Masala tea 500 g       20      4,800.00
Total                                   4,800.00

PNQ1 (Pune)
SKU      Product               Qty   Value (INR)
JAR-10   Glass jar 1 L          16      2,392.00
MUG-02   Steel mug               4        796.00  LOW
TEA-500  Masala tea 500 g        7      1,680.00  LOW
Total                                   4,868.00

Total stock value: ₹9,668.00
Low stock in PNQ1: ['MUG-02', 'TEA-500']


The movement history for the mugs, and a check that it explains the current quantity:

In [384]:
mug_history = repository.movements(sku="MUG-02")
for movement in mug_history:
    when, kind = f"{movement.occurred_at:%H:%M}", f"{movement.movement_type:<9}"
    print(f"{when}  {kind} {movement.delta:+4d}  {movement.note}")
mug_total = sum(movement.delta for movement in mug_history)
print(f"{mug_total=}, {repository.get_warehouse('PNQ1')['MUG-02']=}")

09:01  received   +12  PO-1001
09:04  shipped     -6  customer order
09:05  shipped     -3  customer order
09:08  returned    +1  wrong colour
mug_total=4, repository.get_warehouse('PNQ1')['MUG-02']=4


### Step 10 — self-check

The project's own checks: each builds a **fresh** service, so they do not depend on
the demo above. All should pass.

In [385]:
def fresh_service() -> tuple[InventoryService, InMemoryInventoryRepository, list[object]]:
    """A service with one warehouse, one product and a recording bus."""
    fresh_repository, fresh_bus, published = InMemoryInventoryRepository(), EventBus(), []
    fresh_bus.subscribe(LowStockAlert, published.append)
    fresh_bus.subscribe(StockChanged, published.append)
    fresh = InventoryService(fresh_repository, fresh_bus, clock=make_clock(datetime(2026, 1, 1)))
    fresh.open_warehouse("W1", "Pune")
    fresh.register_product("W1", Product("TEA-500", "Tea", Money("100"), reorder_level=3))
    return fresh, fresh_repository, published


def alert_only_on_crossing() -> int:
    fresh, _, published = fresh_service()
    fresh.receive("W1", "TEA-500", 10)
    for _ in range(5):
        fresh.ship("W1", "TEA-500", 2)  # 8, 6, 4, 2 (crosses 3), 0
    return sum(isinstance(event, LowStockAlert) for event in published)


def failed_transfer_moves_nothing() -> tuple[int, bool, int]:
    fresh, fresh_repository, _ = fresh_service()
    fresh.open_warehouse("W2", "Mumbai")
    fresh.receive("W1", "TEA-500", 5)
    try:
        fresh.transfer("W1", "W2", "TEA-500", 9)
    except InsufficientStockError:
        pass
    return (
        fresh_repository.get_warehouse("W1")["TEA-500"],
        "TEA-500" in fresh_repository.get_warehouse("W2"),
        len(fresh_repository.movements()),
    )


def history_matches_stock() -> bool:
    fresh, fresh_repository, _ = fresh_service()
    fresh.receive("W1", "TEA-500", 7)
    fresh.ship("W1", "TEA-500", 4)
    fresh.accept_return("W1", "TEA-500", 1)
    fresh.adjust("W1", "TEA-500", -1, "count")
    return (
        sum(movement.delta for movement in fresh_repository.movements())
        == fresh_repository.get_warehouse("W1")["TEA-500"]
    )

Run the self-check:

In [386]:
run_checks(
    [
        ("Money sum", lambda: sum([Money("0.10")] * 3), Money("0.30")),
        (
            "Product frozen and slotted",
            lambda: (
                raised_error_name(lambda: set_attribute(catalogue[0], "name", "x")),
                hasattr(catalogue[0], "__dict__"),
            ),
            ("FrozenInstanceError", False),
        ),
        (
            "repository satisfies the Protocol",
            lambda: isinstance(InMemoryInventoryRepository(), InventoryRepository),
            True,
        ),
        (
            "receive then ship",
            lambda: (
                lambda fresh: (fresh.receive("W1", "TEA-500", 10), fresh.ship("W1", "TEA-500", 4))
            )(fresh_service()[0]),
            (10, 6),
        ),
        (
            "cannot oversell",
            lambda: raised_error_name(lambda: fresh_service()[0].ship("W1", "TEA-500", 1)),
            "InsufficientStockError",
        ),
        (
            "errors share a base",
            lambda: all(
                issubclass(error, InventoryError)
                for error in (
                    UnknownProductError,
                    UnknownWarehouseError,
                    DuplicateProductError,
                    InsufficientStockError,
                )
            ),
            True,
        ),
        (
            "lookup errors are LookupErrors",
            lambda: issubclass(UnknownProductError, LookupError),
            True,
        ),
        ("alert only on crossing", alert_only_on_crossing, 1),
        ("failed transfer moves nothing", failed_transfer_moves_nothing, (5, False, 1)),
        ("history explains stock", history_matches_stock, True),
        (
            "warehouse equality by code",
            lambda: Warehouse("W1", "Pune") == Warehouse("W1", "Goa"),
            True,
        ),
        (
            "demo totals",
            lambda: (
                repository.get_warehouse("PNQ1")["TEA-500"],
                repository.get_warehouse("PNQ1")["MUG-02"],
                len(alerts),
            ),
            (7, 4, 2),
        ),
    ]
)

PASS  Money sum
PASS  Product frozen and slotted
PASS  repository satisfies the Protocol
PASS  receive then ship
PASS  cannot oversell
PASS  errors share a base
PASS  lookup errors are LookupErrors
PASS  alert only on crossing
PASS  failed transfer moves nothing
PASS  history explains stock
PASS  warehouse equality by code
PASS  demo totals
12/12 checks passed


**Ideas to extend it** (good practice before Phase 6): replace `EventBus` with the
Lab 2.6 version and subscribe to the `object` class to log everything; add a
`Reservation` value object so orders hold stock before shipping; store
`StockMovement`s in SQLite behind a second repository class that satisfies the same
Protocol (the service code does not change); add `Warehouse.__format__` with a
`"report"` spec.

### Phase 2 review — 20 predict-the-output questions

Twenty short snippets, at least three from every unit of the phase. For each one,
decide what it prints **before** opening the answer, then run the proof cell. If a
question surprises you, go back to the unit it names. Snippets leave out type hints
to stay short; the proof cells add them.

| Unit | Questions |
| --- | --- |
| 2.1 | 3 |
| 2.2 | 3 |
| 2.3 | 3 |
| 2.4 | 4 |
| 2.5 | 4 |
| 2.6 | 3 |

#### Question 1 — Unit 2.1

**Predict the output.**

```python
class Counter:
    count = 0

    def __init__(self):
        self.count += 1

first, second = Counter(), Counter()
print(first.count, second.count, Counter.count)
```

<details><summary>Answer</summary>

`1 1 0`. `self.count += 1` reads `self.count` (found on the **class**: 0), adds 1,
then **assigns** to `self.count`, which creates an instance attribute. Each instance
gets its own `1`; the class attribute never changes.

</details>

Run the next cell to check your prediction.

In [387]:
class Counter:
    count = 0

    def __init__(self) -> None:
        self.count += 1


first, second = Counter(), Counter()
print(first.count, second.count, Counter.count)

1 1 0


#### Question 2 — Unit 2.1

**Predict the output.**

```python
class Cart:
    items = []

    def add(self, item):
        self.items.append(item)

first_cart, second_cart = Cart(), Cart()
first_cart.add("tea")
print(second_cart.items, first_cart.items is second_cart.items)
```

<details><summary>Answer</summary>

`['tea'] True`. `self.items.append` **mutates** the one list stored on the class
(no assignment, so no instance attribute is created). Every cart shares it. Create
the list in `__init__` instead (Unit 2.1.3).

</details>

Run the next cell to check your prediction.

In [388]:
class Cart:
    items: list[str] = []  # the shared mutable class attribute trap

    def add(self, item: str) -> None:
        self.items.append(item)


first_cart, second_cart = Cart(), Cart()
first_cart.add("tea")
print(second_cart.items, first_cart.items is second_cart.items)

['tea'] True


#### Question 3 — Unit 2.1

**Predict the output.**

```python
class Vault:
    def __init__(self):
        self.__pin = 1234

vault = Vault()
print(hasattr(vault, "__pin"), vault._Vault__pin)
```

<details><summary>Answer</summary>

`False 1234`. Inside a class body, `__pin` is **name-mangled** to `_Vault__pin`.
It is a guard against accidental clashes in subclasses, not privacy: the mangled name
is still reachable (Unit 2.1.7).

</details>

Run the next cell to check your prediction.

In [389]:
class Vault:
    def __init__(self) -> None:
        self.__pin = 1234


vault = Vault()
print(hasattr(vault, "__pin"), vault._Vault__pin)

False 1234


#### Question 4 — Unit 2.2

**Predict the output.**

```python
class A:
    def hello(self): return "A"
class B(A):
    def hello(self): return "B" + super().hello()
class C(A):
    def hello(self): return "C" + super().hello()
class D(B, C):
    def hello(self): return "D" + super().hello()

print(D().hello())
```

<details><summary>Answer</summary>

`DBCA`. `super()` means "the next class in the **instance's** MRO", which is
`D, B, C, A, object`. So `B`'s `super()` call reaches `C`, not `A` (Unit 2.2.3).

</details>

Run the next cell to check your prediction.

In [390]:
class A:
    def hello(self) -> str:
        return "A"


class B(A):
    def hello(self) -> str:
        return "B" + super().hello()


class C(A):
    def hello(self) -> str:
        return "C" + super().hello()


class D(B, C):
    def hello(self) -> str:
        return "D" + super().hello()


print(D().hello())

DBCA

#### Question 5 — Unit 2.2

**Predict the output.**

```python
class Shipment: pass
class Express(Shipment): pass

parcel = Express()
print(isinstance(parcel, Shipment), type(parcel) == Shipment, issubclass(Express, object))
```

<details><summary>Answer</summary>

`True False True`. `isinstance` respects inheritance; `type(x) ==` compares the exact
class and breaks substitutability (Liskov, Unit 2.2.5). Every class ultimately
inherits from `object`.

</details>

Run the next cell to check your prediction.

In [391]:
class Shipment:
    pass


class Express(Shipment):
    pass


parcel = Express()
exact_type_match = type(parcel) == Shipment  # noqa: E721 (the point of the question)
print(isinstance(parcel, Shipment), exact_type_match, issubclass(Express, object))

True False True


#### Question 6 — Unit 2.2

**Predict the output.**

```python
class Base:
    def __init__(self):
        self.created = True

class Child(Base):
    def __init__(self):
        self.name = "child"

print(hasattr(Child(), "created"))
```

<details><summary>Answer</summary>

`False`. Defining `__init__` in `Child` **replaces** `Base.__init__`; Python never
calls the parent initializer automatically. Call `super().__init__()` (Unit 2.2.1).

</details>

Run the next cell to check your prediction.

In [392]:
class Base:
    def __init__(self) -> None:
        self.created = True


class Child(Base):
    def __init__(self) -> None:
        self.name = "child"


print(hasattr(Child(), "created"))

False


#### Question 7 — Unit 2.3

**Predict the output.**

```python
from abc import ABC, abstractmethod

class Repository(ABC):
    @abstractmethod
    def get(self, key): ...

class Partial(Repository):
    pass

try:
    Partial()
except TypeError:
    print("TypeError")
print(Repository.__abstractmethods__)
```

<details><summary>Answer</summary>

`TypeError`, then `frozenset({'get'})`. Defining `Partial` is fine; **instantiating**
a class that still has abstract methods fails. `ABCMeta` records them in
`__abstractmethods__` (Unit 2.3.1).

</details>

Run the next cell to check your prediction.

In [393]:
from abc import ABC, abstractmethod


class Repository(ABC):
    @abstractmethod
    def get(self, key: str) -> object: ...


class Partial(Repository):
    pass


try:
    Partial()
except TypeError:
    print("TypeError")
print(Repository.__abstractmethods__)

TypeError
frozenset({'get'})


#### Question 8 — Unit 2.3

**Predict the output.**

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class Closer(Protocol):
    def close(self) -> None: ...

class Door:
    close = "not a method"

print(isinstance(Door(), Closer))
```

<details><summary>Answer</summary>

`True`. A runtime-checkable Protocol only checks that the **names** exist, not their
types or signatures (Unit 2.3.3). A type checker would reject `Door` as a `Closer`.

</details>

Run the next cell to check your prediction.

In [394]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Closer(Protocol):
    def close(self) -> None: ...


class Door:
    close = "not a method"


print(isinstance(Door(), Closer))

True


#### Question 9 — Unit 2.3

**Predict the output.**

```python
from collections.abc import Sized

class Box:
    def __len__(self):
        return 0

print(isinstance(Box(), Sized), bool(Box()))
```

<details><summary>Answer</summary>

`True False`. `Sized.__subclasshook__` accepts any class with `__len__`, no
inheritance needed (Unit 2.3.2), and `bool()` falls back to `__len__`, so an empty box
is falsy (Unit 2.4.5).

</details>

Run the next cell to check your prediction.

In [395]:
from collections.abc import Sized


class Box:
    def __len__(self) -> int:
        return 0


print(isinstance(Box(), Sized), bool(Box()))

True False


#### Question 10 — Unit 2.4

**Predict the output.**

```python
class Points:
    def __init__(self, value):
        self.value = value

    def __add__(self, other):
        return Points(self.value + other.value)

print(sum([Points(1), Points(2)]).value)
```

<details><summary>Answer</summary>

`TypeError: unsupported operand type(s) for +: 'int' and 'Points'`. `sum()` starts
from `0`, so the first step is `0 + Points(1)`: `int.__add__` declines and `Points`
has no `__radd__` (Unit 2.4.8).

</details>

Run the next cell to check your prediction.

In [396]:
class Points:
    def __init__(self, value: int) -> None:
        self.value = value

    def __add__(self, other: Points) -> Points:
        return Points(self.value + other.value)


try:
    print(sum([Points(1), Points(2)]).value)  # type: ignore[arg-type]
except TypeError as error:
    print(f"TypeError: {error}")

TypeError: unsupported operand type(s) for +: 'int' and 'Points'


#### Question 11 — Unit 2.4

**Predict the output.**

```python
class Tag:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Tag):
            return NotImplemented
        return self.name == other.name

print(Tag("a") == Tag("a"), Tag("a") == "a", Tag("a") != "a")
```

<details><summary>Answer</summary>

`True False True`. Against a `str`, both sides return `NotImplemented`, so `==` falls
back to identity (`False`) and `!=` to non-identity (`True`); no exception (Unit
2.4.3).

</details>

Run the next cell to check your prediction.

In [397]:
class Tag:
    def __init__(self, name: str) -> None:
        self.name = name

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Tag):
            return NotImplemented
        return self.name == other.name


print(Tag("a") == Tag("a"), Tag("a") == "a", Tag("a") != "a")

True False True


#### Question 12 — Unit 2.4

**Predict the output.**

```python
class Queue:
    def __len__(self):
        return 3

    def __bool__(self):
        return False

queue = Queue()
print(len(queue), bool(queue), "yes" if queue else "no")
```

<details><summary>Answer</summary>

`3 False no`. `__bool__` takes precedence over `__len__` for truth tests (Unit 2.4.5).

</details>

Run the next cell to check your prediction.

In [398]:
class Queue:
    def __len__(self) -> int:
        return 3

    def __bool__(self) -> bool:
        return False


queue = Queue()
print(len(queue), bool(queue), "yes" if queue else "no")

3 False no


#### Question 13 — Unit 2.4

**Predict the output.**

```python
class Echo:
    def __getitem__(self, index):
        return index

echo = Echo()
print(echo[2], echo[1:3], echo[::2])
```

<details><summary>Answer</summary>

`2 slice(1, 3, None) slice(None, None, 2)`. Slicing syntax builds a `slice` object
and passes it to `__getitem__`; your class decides what to do with it (Unit 2.4.6).

</details>

Run the next cell to check your prediction.

In [399]:
class Echo:
    def __getitem__(self, index: int | slice) -> int | slice:
        return index


echo = Echo()
print(echo[2], echo[1:3], echo[::2])

2 slice(1, 3, None) slice(None, None, 2)


#### Question 14 — Unit 2.5

**Predict the output.**

```python
class Item:
    @property
    def price(self):
        return 100

item = Item()
item.__dict__["price"] = 5
print(item.price)
```

<details><summary>Answer</summary>

`100`. A property is a **data** descriptor, which wins over the instance `__dict__`
in attribute lookup (Unit 2.5.1).

</details>

Run the next cell to check your prediction.

In [400]:
class Item:
    @property
    def price(self) -> int:
        return 100


item = Item()
item.__dict__["price"] = 5
print(item.price)

100


#### Question 15 — Unit 2.5

**Predict the output.**

```python
class Item:
    def price(self):
        return 100

item = Item()
item.price = 5
print(item.price, Item().price())
```

<details><summary>Answer</summary>

`5 100`. A function is a **non-data** descriptor, so the instance attribute shadows
the method on that one instance; other instances still see the method (Unit 2.5.4).

</details>

Run the next cell to check your prediction.

In [401]:
class Item:
    def price(self) -> int:
        return 100


item = Item()
item.price = 5  # type: ignore[method-assign]
print(item.price, Item().price())

5 100


#### Question 16 — Unit 2.5

**Predict the output.**

```python
class Defaults:
    colour = "red"

    def __getattr__(self, name):
        return f"default-{name}"

defaults = Defaults()
print(defaults.colour, defaults.size)
```

<details><summary>Answer</summary>

`red default-size`. `__getattr__` runs only when normal lookup **fails**; `colour` is
found on the class first (Unit 2.5.2).

</details>

Run the next cell to check your prediction.

In [402]:
class Defaults:
    colour = "red"

    def __getattr__(self, name: str) -> str:
        return f"default-{name}"


defaults = Defaults()
print(defaults.colour, defaults.size)

red default-size


#### Question 17 — Unit 2.5

**Predict the output.**

```python
class Base:
    registry = []

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Base.registry.append(cls.__name__)

class A(Base): pass
class B(A): pass

print(Base.registry)
```

<details><summary>Answer</summary>

`['A', 'B']`. `__init_subclass__` runs for **every** descendant, including
grandchildren, but not for `Base` itself (Unit 2.5.8).

</details>

Run the next cell to check your prediction.

In [403]:
class Base:
    registry: list[str] = []

    def __init_subclass__(cls, **kwargs: object) -> None:
        super().__init_subclass__(**kwargs)
        Base.registry.append(cls.__name__)


class A(Base):
    pass


class B(A):
    pass


print(Base.registry)

['A', 'B']


#### Question 18 — Unit 2.6

**Predict the output.**

```python
from dataclasses import dataclass, field

@dataclass(order=True)
class Task:
    priority: int
    name: str = field(compare=False)

tasks = [Task(2, "b"), Task(1, "z"), Task(2, "a")]
print([task.name for task in sorted(tasks)], Task(1, "x") == Task(1, "y"))
```

<details><summary>Answer</summary>

`['z', 'b', 'a'] True`. Only `priority` is compared. The two priority-2 tasks are
equal for sorting, and `sorted` is **stable**, so `b` stays before `a` (Unit 2.6.1).

</details>

Run the next cell to check your prediction.

In [404]:
from dataclasses import dataclass, field


@dataclass(order=True)
class Task:
    priority: int
    name: str = field(compare=False)


tasks = [Task(2, "b"), Task(1, "z"), Task(2, "a")]
print([task.name for task in sorted(tasks)], Task(1, "x") == Task(1, "y"))

['z', 'b', 'a'] True


#### Question 19 — Unit 2.6

**Predict the output.**

```python
from dataclasses import dataclass

@dataclass
class Point:
    x: int

@dataclass(frozen=True)
class FrozenPoint:
    x: int

print(Point.__hash__ is None, hash(FrozenPoint(1)) == hash(FrozenPoint(1)))
```

<details><summary>Answer</summary>

`True True`. A plain dataclass generates `__eq__`, which switches hashing off (Unit
2.4.3); `frozen=True` makes the fields immutable, so a `__hash__` is generated.

</details>

Run the next cell to check your prediction.

In [405]:
from dataclasses import dataclass


@dataclass
class Point:
    x: int


@dataclass(frozen=True)
class FrozenPoint:
    x: int


print(Point.__hash__ is None, hash(FrozenPoint(1)) == hash(FrozenPoint(1)))

True True


#### Question 20 — Unit 2.6

**Predict the output.**

```python
from enum import StrEnum, auto

class Env(StrEnum):
    DEV = auto()
    PROD = auto()

print(Env.PROD == "prod", f"{Env.DEV}", Env("prod") is Env.PROD)
```

<details><summary>Answer</summary>

`True dev True`. `auto()` in a `StrEnum` gives the lower-cased name; members are
real strings, format as their value, and `Env("prod")` looks up the existing
singleton (Unit 2.6.4).

</details>

Run the next cell to check your prediction.

In [406]:
from enum import StrEnum, auto


class Env(StrEnum):
    DEV = auto()
    PROD = auto()


print(Env.PROD == "prod", f"{Env.DEV}", Env("prod") is Env.PROD)

True dev True


### Phase 2 complete

You can now design classes that behave like built-ins (dunders), choose between
inheritance, composition, ABCs and Protocols, explain how attribute lookup,
descriptors and metaclasses work (and so how Pydantic, SQLAlchemy and dataclasses
work), and model a domain with dataclasses, enums, value objects, entities, a
repository, dependency injection and events.

**Next: Phase 3 — Python Internals: Memory, Bytecode and Imports.** Reference
counting and the garbage collector behind the objects you have been creating, the
bytecode that `BINARY_OP` and `LOAD_ATTR` belong to, and the import system that
makes a module-level object a singleton.